Copyright 2026 Google DeepMind Technologies Limited

All software is licensed under the Apache License, Version 2.0 (Apache 2.0);
you may not use this file except in compliance with the Apache 2.0 license.
You may obtain a copy of the Apache 2.0 license at:
https://www.apache.org/licenses/LICENSE-2.0

All other materials are licensed under the Creative Commons Attribution 4.0
International License (CC-BY). You may obtain a copy of the CC-BY license at:
https://creativecommons.org/licenses/by/4.0/legalcode

Unless required by applicable law or agreed to in writing, all software and
materials distributed here under the Apache 2.0 or CC-BY licenses are
distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND,
either express or implied. See the licenses for the specific language governing
permissions and limitations under those licenses.

This is not an official Google product.

In [ ]:
#@title Imports (run this first)

import numpy as np

# Tensor rank decompositions (algorithms for matrix multiplication)

This section contains several low-rank tensor decompositions, discovered by AlphaEvolve, and the code to verify their correctness. A tensor with parameters $\langle n,m,p\rangle$ corresponds to multiplying an $n \times m$ and an $m \times p$ matrix, and the rank of the decomposition corresponds to the number of scalar multiplications needed to perform the matrix multiplication.

In [ ]:
#@title Verification function

def verify_tensor_decomposition(decomposition: tuple[np.ndarray, np.ndarray, np.ndarray], n: int, m: int, p: int, rank: int):
  """Verifies the correctness of the tensor decomposition.

  Args:
    decomposition: a tuple of 3 factor matrices with the same number of columns.
      (The number of columns specifies the rank of the decomposition.) To
      construct a tensor, we take the outer product of the i-th column of the
      three factor matrices, for 1 <= i <= rank, and add up all these outer
      products.
    n: the first parameter of the matrix multiplication tensor.
    m: the second parameter of the matrix multiplication tensor.
    p: the third parameter of the matrix multiplication tensor.
    rank: the expected rank of the decomposition.

  Raises:
    AssertionError: If the decomposition does not have the correct rank, or if
    the decomposition does not construct the 3D tensor which corresponds to
    multiplying an n x m matrix by an m x p matrix.
  """
  # Check that each factor matrix has the correct shape.
  factor_matrix_1, factor_matrix_2, factor_matrix_3 = decomposition
  assert factor_matrix_1.shape == (n * m, rank), f'Expected shape of factor matrix 1 is {(n * m, rank)}. Actual shape is {factor_matrix_1.shape}.'
  assert factor_matrix_2.shape == (m * p, rank), f'Expected shape of factor matrix 1 is {(m * p, rank)}. Actual shape is {factor_matrix_2.shape}.'
  assert factor_matrix_3.shape == (p * n, rank), f'Expected shape of factor matrix 1 is {(p * n, rank)}. Actual shape is {factor_matrix_3.shape}.'

  # Form the matrix multiplication tensor <n, m, p>.
  matmul_tensor = np.zeros((n * m, m * p, p * n), dtype=np.int32)
  for i in range(n):
    for j in range(m):
      for k in range(p):
        matmul_tensor[i * m + j][j * p + k][k * n + i] = 1

  # Check that the tensor is correctly constructed.
  constructed_tensor = np.einsum('il,jl,kl -> ijk', *decomposition)
  assert np.array_equal(constructed_tensor, matmul_tensor), f'Tensor constructed by decomposition does not match the matrix multiplication tensor <{(n,m,p)}>: {constructed_tensor}.'
  print(f'Verified a decomposition of rank {rank} for matrix multiplication tensor <{n},{m},{p}>.')

  # Print the set of values used in the decomposition.
  np.set_printoptions(linewidth=100)
  print('This decomposition uses these factor entries:\n', np.array2string(np.unique(np.vstack((factor_matrix_1, factor_matrix_2, factor_matrix_3))), separator=', '))

## Rank-32 decomposition of <2,4,5> over 0.5*Z

In [ ]:
#@title Data

decomposition_245 = (np.array([[-0. ,  0.5, -0. ,  0.5,  0. ,  0. ,  0.5,  0. , -0.5, -0.5, -0. ,
        -0.5, -0. , -0. ,  0.5,  0. ,  0.5, -1. , -0.5, -0. , -0.5, -0. ,
         0.5, -0. , -0.5,  0.5, -1. ,  0. ,  0.5,  0. ,  0.5, -0. ],
       [ 0.5, -0.5,  0. ,  0.5, -0. ,  0.5, -0.5, -0. ,  0. , -0.5,  0. ,
         0. , -0.5,  0.5,  0. ,  0.5,  0. , -0. , -0. , -0.5, -0. ,  0. ,
         0. ,  0.5, -0.5,  0. , -0. ,  0. ,  0. ,  0. , -0. ,  0.5],
       [ 0. , -0. , -0.5, -0. ,  0.5, -0.5, -0. ,  0. , -0. ,  0. ,  0. ,
         0.5, -0.5, -0. ,  0.5,  0. ,  0.5, -0.5,  0.5,  0. ,  0. , -0.5,
        -0.5,  0. , -0. ,  0.5,  0.5, -0. ,  0. , -0. ,  0. ,  0. ],
       [-0.5,  0. ,  0.5, -1. ,  0.5,  0. ,  0. , -0. , -0.5,  0. , -0. ,
         0. , -0. , -0.5, -0. ,  0.5, -0. , -0.5, -0. , -0.5, -0.5,  0.5,
        -0. ,  0.5, -1. ,  0. ,  0.5, -0. , -0.5, -0. , -0.5, -0.5],
       [ 0. ,  0.5,  0. ,  0.5,  0. ,  0. , -0.5, -0. , -0.5, -0. , -0. ,
         0.5, -0. , -0. ,  0. , -0. , -0. ,  0. ,  0. , -0. , -0. , -0. ,
        -0. , -0. , -0.5, -0.5,  0. ,  0.5,  0.5, -0. ,  0. , -0. ],
       [-0.5, -0.5, -0. , -0.5, -0. , -0.5,  0.5, -0.5, -0. ,  0. ,  0.5,
         0. ,  0.5, -0.5, -0. , -0. ,  0.5, -0. , -0.5,  0.5, -0. , -0. ,
        -0. , -0.5,  0.5, -0. ,  0. ,  0.5,  0. ,  0. ,  0. , -0. ],
       [-0. ,  0. ,  0.5,  0. , -0. ,  0.5, -0. ,  0.5,  0. ,  0. ,  0.5,
        -0.5,  0.5, -0.5, -0. ,  0. , -0.5,  0.5, -0.5,  0. ,  0. , -0.5,
        -0. ,  0.5, -0. , -0.5, -0.5, -0. ,  0. , -0.5, -0. , -0. ],
       [ 0.5, -0. , -0.5, -0. , -0. , -0. ,  0. ,  0. , -0.5,  0. ,  0. ,
        -0. , -0. ,  0. , -0. , -0. , -0. , -0.5, -0. ,  0.5, -0. ,  0.5,
        -0. ,  0. , -0. , -0. ,  0.5,  0. , -0.5, -0.5,  0. , -0. ]],
      dtype=np.float32), np.array([[ 0.5, -0. , -0. ,  0. , -0. , -0.5,  0. ,  0. ,  1. , -0. , -0. ,
        -0. ,  0.5, -0.5,  0.5,  0. , -0. , -0.5, -0. ,  0.5, -0.5,  0. ,
        -0.5, -0.5,  0. , -0. ,  0.5,  0. , -1. ,  0. ,  0.5,  0. ],
       [ 0.5,  0. ,  0. ,  1. ,  0. ,  0.5,  2. ,  0. , -0. , -0. ,  0. ,
         0. , -0.5,  0.5, -0.5,  0. ,  0. ,  0.5, -0. ,  0.5,  0.5,  0. ,
         0.5,  0.5, -1. ,  0. , -0.5, -0. ,  0. , -0. , -0.5,  0. ],
       [ 0.5,  0. , -0. , -0. ,  0. , -0.5, -0. , -1. , -0. ,  0. ,  1. ,
         1. ,  0.5, -0.5, -0.5,  0. , -1. ,  0.5,  1. ,  0.5, -0.5, -0. ,
         0.5, -0.5, -0. ,  1. , -0.5, -2. , -0. , -0. ,  0.5, -0. ],
       [-0.5, -0. , -0. ,  0. ,  0. ,  0.5,  0. ,  0. , -0. ,  0. , -0. ,
        -1. , -0.5,  0.5, -0.5,  0. , -0. , -0.5,  0. , -0.5,  0.5,  0. ,
         0.5,  0.5, -0. , -1. ,  0.5, -0. ,  0. , -0. , -0.5,  0. ],
       [ 0. ,  0. ,  0. ,  1. , -0. , -1. ,  0. , -0. , -0. , -0. , -0. ,
        -1. , -1. , -0. , -0. , -1. , -1. , -0. , -1. , -0. , -1. , -0. ,
        -0. , -0. ,  1. ,  1. , -0. , -0. , -0. , -0. , -1. , -1. ],
       [ 0.5, -2. , -0. ,  1. ,  0. , -0.5,  0. ,  0. ,  1. ,  0. , -0. ,
         0. ,  0.5, -0.5,  0.5,  1. , -0. , -0.5, -0. ,  0.5,  0.5, -0. ,
        -0.5, -0.5, -1. , -0. ,  0.5, -0. , -1. , -0. , -0.5, -1. ],
       [ 0.5, -0. ,  0. ,  0. ,  0. ,  0.5,  0. , -0. , -0. , -0. ,  0. ,
        -0. , -0.5,  0.5, -0.5, -1. ,  0. ,  0.5, -0. ,  0.5, -0.5,  0. ,
         0.5,  0.5, -0. ,  0. , -0.5, -0. , -0. ,  0. ,  0.5,  1. ],
       [-0.5, -0. , -0. , -0. , -0. , -0.5,  0. ,  1. ,  0. ,  0. , -1. ,
         0. ,  0.5,  0.5,  0.5, -0. ,  0. , -0.5, -0. , -0.5,  0.5, -0. ,
        -0.5,  0.5, -0. ,  0. ,  0.5, -0. , -0. , -0. , -0.5,  0. ],
       [ 0.5,  0. , -0. , -0. , -0. ,  0.5,  0. ,  0. ,  0. ,  2. ,  0. ,
         0. , -0.5, -0.5,  0.5, -0. ,  1. ,  0.5, -1. ,  0.5, -0.5, -0. ,
        -0.5, -0.5, -0. ,  0. , -0.5, -0. , -0. ,  0. ,  0.5, -0. ],
       [ 0. , -0. , -0. , -1. ,  0. , -1. ,  0. ,  0. , -0. ,  0. , -0. ,
        -1. , -1. ,  0. , -0. ,  1. , -1. , -0. , -1. , -0. ,  1. , -0. ,
        -0. , -0. , -1. ,  1. , -0. , -0. , -0. , -0. ,  1. ,  1. ],
       [ 0. ,  0. , -0. , -0.5,  0. ,  0.5,  0. , -0. ,  1. ,  0. ,  0. ,
         0.5,  0.5,  0. ,  1. ,  0.5,  0.5, -1. ,  0.5, -0. ,  0.5,  2. ,
         1. , -0. , -0.5, -0.5, -1. ,  0. ,  1. , -0. ,  0.5,  0.5],
       [ 0. ,  0. , -0. ,  0.5,  2. , -0.5,  0. ,  0. , -0. ,  0. , -0. ,
         0.5, -0.5,  1. ,  0. ,  0.5,  0.5, -0. ,  0.5,  0. , -0.5, -0. ,
         0. , -1. ,  0.5, -0.5, -0. , -0. , -0. ,  0. , -0.5,  0.5],
       [-0. , -0. ,  0. , -0.5,  0. ,  0.5,  0. , -1. , -0. , -0. , -1. ,
        -0.5,  0.5, -0. ,  0. ,  0.5, -0.5,  0. , -0.5, -0. ,  0.5,  0. ,
        -0. ,  0. , -0.5,  0.5, -0. , -0. , -0. ,  0. ,  0.5,  0.5],
       [-0. ,  0. ,  0. ,  0.5, -0. , -0.5, -0. , -0. ,  0. ,  0. , -0. ,
         0.5, -0.5,  0. , -1. , -0.5, -0.5,  0. , -0.5, -0. , -0.5,  0. ,
        -1. , -0. ,  0.5, -0.5,  0. ,  0. ,  0. , -0. , -0.5, -0.5],
       [-1. ,  0. , -0. , -0. , -0. ,  1. , -0. , -0. ,  0. , -0. ,  0. ,
         0. , -1. ,  1. ,  1. ,  0. , -0. , -1. , -0. , -1. ,  1. ,  0. ,
        -1. ,  1. ,  0. , -0. ,  1. , -0. , -0. ,  0. , -1. , -0. ],
       [-0. , -0. ,  0. , -0.5, -0. ,  0.5, -0. ,  0. ,  1. ,  0. ,  0. ,
         0.5,  0.5,  0. , -0. ,  0.5,  0.5, -0. ,  0.5,  0. , -0.5, -0. ,
        -0. , -0. , -0.5, -0.5, -0. ,  0. ,  1. ,  0. , -0.5,  0.5],
       [-1. , -0. , -0. , -0.5,  0. , -0.5, -0. ,  0. ,  0. ,  0. , -0. ,
        -0.5, -0.5,  0. ,  0. , -0.5, -0.5, -0. , -0.5,  1. ,  0.5,  0. ,
         0. , -0. , -0.5,  0.5,  0. , -0. ,  0. ,  0. ,  0.5, -0.5],
       [ 1. , -0. , -0. ,  0.5,  0. ,  0.5,  0. ,  1. , -0. , -0. ,  1. ,
         0.5,  0.5, -1. ,  0. , -0.5,  0.5,  0. ,  0.5, -1. , -0.5,  0. ,
        -0. ,  1. ,  0.5, -0.5, -0. ,  0. ,  0. ,  2. , -0.5, -0.5],
       [ 0. ,  0. ,  2. ,  0.5,  0. , -0.5, -0. ,  0. , -0. , -0. , -0. ,
         0.5, -0.5,  0. ,  0. , -0.5, -0.5, -1. , -0.5, -0. ,  0.5, -0. ,
        -0. ,  0. ,  0.5, -0.5, -1. , -0. , -0. ,  0. ,  0.5, -0.5],
       [-1. , -0. , -0. ,  0. ,  0. ,  1. , -0. ,  0. , -0. ,  0. ,  0. ,
        -0. , -1. ,  1. , -1. , -0. ,  0. ,  1. ,  0. , -1. , -1. ,  0. ,
         1. ,  1. , -0. ,  0. , -1. , -0. ,  0. , -0. ,  1. , -0. ]],
      dtype=np.float32), np.array([[ 0. ,  0.5,  0.5,  0.5, -0. , -0. ,  0.5,  0. , -0. , -0. , -0. ,
        -0. , -0. ,  0. ,  0. ,  0. , -0. ,  0.5,  0. , -0. ,  1. , -0.5,
        -0. , -0. ,  0.5,  0. , -0.5, -0. , -0. ,  0. ,  1. ,  0. ],
       [-0. ,  0.5, -0.5, -0.5, -0. , -0. , -0.5,  0. , -1. , -0. , -0. ,
        -0. ,  0. ,  0. ,  0. ,  0. , -0. , -0.5,  0. , -0. , -1. , -0.5,
         0. , -0. , -0.5, -0. ,  0.5,  0. , -1. ,  0. , -1. ,  0. ],
       [ 0. ,  0.5, -0. ,  0.5,  1. , -0. ,  0.5, -0. ,  0. ,  0. , -0. ,
         0. ,  0. , -0. ,  0. , -1. , -0. ,  0. ,  0. , -0. ,  0. ,  0. ,
         0. ,  0. ,  0.5,  0. ,  0. , -0. ,  0. ,  0. , -0. ,  1. ],
       [-1. ,  0.5,  0. ,  0.5, -0. ,  0. , -0.5,  0. ,  0. , -0. ,  0. ,
        -0. , -0. , -1. ,  0. , -1. , -0. ,  0. ,  0. ,  1. ,  0. ,  0. ,
         0. , -1. ,  0.5, -0. ,  0. , -0. , -0. , -1. ,  0. ,  1. ],
       [-0. , -0. ,  0. , -0. ,  1. , -1. ,  0. , -1. , -0. , -1. , -1. ,
        -0. , -1. ,  1. ,  0. ,  0. , -1. , -0. , -1. , -0. ,  0. ,  0. ,
         0. ,  1. , -0. , -0. ,  0. , -0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , -0. , -0. ,  0. ,  0. , -1. , -0. ,  0. , -1. ,
         0. ,  0. , -0. , -0. ,  0. ,  0. ,  0. ,  0. , -0. , -0. ,  0. ,
         0. , -0. ,  0. , -0. ,  0. , -1. ,  0. , -1. ,  0. ,  0. ],
       [-0. , -0. ,  0.5, -0. ,  0. ,  0. ,  0. ,  0. , -0. , -1. , -0. ,
         0. , -0. ,  0. , -1. ,  0. , -0. ,  0.5, -0. , -0. , -0. , -0.5,
         1. , -0. , -0. ,  0. , -0.5,  0. , -0. , -0. ,  0. ,  0. ],
       [ 0. , -0. , -0.5,  0. ,  0. ,  0. , -0. , -0. , -0. ,  0. ,  0. ,
        -1. ,  0. , -0. , -1. ,  0. ,  1. ,  0.5,  1. ,  0. , -0. , -0.5,
         1. ,  0. ,  0. ,  1. , -0.5, -1. , -0. ,  0. ,  0. , -0. ],
       [ 0. ,  0. , -0. , -0. , -0. , -0.5, -0. , -0.5, -0. ,  0. ,  0.5,
        -0. ,  0.5, -0.5,  0.5,  0.5, -0.5,  0. ,  0.5, -0. ,  0.5,  0. ,
         0.5,  0.5, -0. ,  0. ,  0. , -0. ,  0. , -0. , -0.5,  0.5],
       [-0.5, -0. ,  0. ,  0.5,  0. ,  0. ,  0. , -0.5, -0.5, -0. ,  0.5,
        -0.5, -0. , -0.5,  0.5,  0.5, -0.5, -0.5,  0.5, -0.5, -0.5, -0. ,
         0.5,  0.5, -0.5, -0.5, -0.5,  0. ,  0.5,  0. ,  0.5,  0.5]],
      dtype=np.float32))

In [ ]:
#@title Verification
n = 2
m = 4
p = 5
rank = 32

verify_tensor_decomposition(decomposition_245, n, m, p, rank)

## Rank-45 decomposition of <2,4,7> over Z

In [ ]:
#@title Data

decomposition_247 = (np.array([[ 0., -1., -1.,  0.,  1.,  1.,  1., -1., -1.,  0., -0., -0.,  0.,
         0., -0.,  1., -0., -1.,  0.,  0., -0., -0., -0., -1., -0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  1., -1., -0., -0., -0., -0.,  1., -0.,
        -0.,  0., -1., -1., -0.,  0.],
       [ 0.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,  1., -0.,  0., -1.,
        -0., -0., -0., -0., -0.,  1., -0.,  1., -0.,  1.,  1., -1., -1.,
         0., -1., -0.,  0., -0., -1., -0., -0.,  1., -1., -0., -0.,  0.,
         0., -0.,  1.,  0., -0.,  1.],
       [ 0.,  0., -0.,  0.,  1.,  1., -0., -1.,  0., -1.,  0., -0., -0.,
         1., -0.,  1.,  1.,  0., -0.,  0.,  0., -1., -0., -1.,  0.,  0.,
        -0.,  0., -0.,  0., -1.,  0., -1., -0.,  0.,  0.,  1.,  1., -0.,
        -0.,  0., -1., -1., -1., -0.],
       [-0.,  1., -0.,  0., -1., -0., -1.,  0.,  1.,  0.,  0.,  0.,  0.,
        -0., -1., -1.,  0.,  1., -0.,  1.,  0.,  0.,  1.,  1.,  0.,  1.,
        -0., -0., -0.,  0.,  1., -0., -0.,  0., -0., -0., -0., -1.,  0.,
         0.,  0., -0.,  0.,  0., -0.],
       [-1., -1., -1., -0.,  1.,  1.,  0., -1., -0., -0., -0., -0.,  0.,
        -0., -0.,  1.,  1., -1., -0.,  0.,  0., -0., -0., -0., -0., -0.,
         0.,  0.,  1., -1., -0., -0.,  0., -0.,  1., -0., -0., -0.,  0.,
         1.,  0., -1., -0., -0.,  0.],
       [ 1.,  0., -0.,  0., -1., -0., -0.,  0., -1.,  1.,  0.,  1.,  0.,
        -0., -0., -0., -1., -0., -0., -0.,  1., -0.,  1., -0.,  0., -0.,
        -1., -1., -1., -0.,  0., -0.,  0., -0.,  0., -1., -0., -0., -0.,
        -1., -1.,  1., -1.,  0.,  1.],
       [-1.,  0.,  0., -1.,  1., -0.,  0., -1.,  1., -1., -0., -0., -0.,
         0.,  0.,  1.,  1.,  0., -0., -0., -1., -1., -0.,  0.,  0., -0.,
         0., -0.,  1., -0.,  0.,  0.,  0.,  1., -0.,  0., -0., -0., -0.,
         0.,  1., -1.,  0., -0., -0.],
       [ 0., -0., -0.,  0.,  0., -0., -0., -0.,  0., -0.,  1., -1.,  0.,
         1., -0., -1., -0.,  1.,  1.,  1., -0., -0.,  1.,  0.,  0.,  0.,
        -1., -0., -1.,  0.,  0.,  0., -0., -1., -0., -1., -0.,  0., -1.,
         0.,  0.,  0., -1., -0., -0.]], dtype=np.float32), np.array([[-0.,  0.,  0.,  1.,  1.,  1., -0., -1.,  0., -0., -0., -1., -0.,
        -1.,  0.,  0.,  1.,  0.,  1., -0., -0.,  1.,  0., -0.,  0., -1.,
         0., -0.,  0., -0.,  0., -1., -0., -1.,  1., -0.,  0.,  1., -0.,
         1., -0., -0., -0., -1.,  0.],
       [-0.,  1.,  1.,  1., -0.,  1., -0., -0., -1., -0.,  0.,  0.,  0.,
         0., -1.,  0.,  0., -0.,  0.,  0., -1., -0.,  0.,  1.,  1., -0.,
         0., -1., -1., -0., -0., -0.,  0.,  0.,  1.,  1., -1.,  0., -1.,
         1.,  0., -0., -1.,  0., -1.],
       [ 0.,  0.,  1., -0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  1.,
        -0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,  0.,
        -0.,  0., -0., -0.,  0.,  1.,  0., -0., -1., -0., -0.,  0., -0.,
        -0.,  0., -0., -0.,  0., -0.],
       [-0.,  1.,  1., -0., -1., -0.,  0.,  0., -0.,  0.,  0.,  1., -0.,
         1.,  0., -1., -1., -1., -1., -0.,  0., -1.,  0., -0.,  0.,  1.,
         0.,  0., -0., -1., -1.,  1.,  0.,  0., -1.,  0., -0.,  0., -1.,
        -1., -0., -0.,  0.,  1., -0.],
       [-0., -1., -0., -0.,  0.,  0.,  1.,  0.,  0.,  0., -0.,  0., -0.,
         0.,  1., -0., -0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,  0.,
         0.,  0.,  0.,  1., -0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,
        -0.,  0., -0., -0., -0.,  0.],
       [ 0., -0.,  0., -0.,  0.,  1.,  0.,  0.,  0., -0., -0., -0., -0.,
         0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,
        -0., -0., -0., -1., -0., -0.,  1., -0., -0., -0., -1., -0., -0.,
         0.,  0., -0.,  0., -0.,  0.],
       [ 0., -0., -1., -0., -0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,
         0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,
         0., -0., -0., -1., -0., -0., -0., -0., -0., -0.,  0.,  0., -0.,
        -0.,  0., -0.,  0., -0., -0.],
       [ 0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0., -1., -1.,  0.,
         0.,  0.,  0.,  0.,  0.,  1.,  0., -0., -0.,  0., -0., -0., -0.,
         0.,  1., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0.,
        -0., -0.,  0.,  0.,  0.,  0.],
       [ 0.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0.,
         0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0.,  1., -0.,
        -0., -0., -0.,  0.,  0., -0.,  0., -0., -0., -0.,  0.,  0., -0.,
        -0.,  0.,  0., -0.,  0.,  1.],
       [-0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0., -0., -0.,  1.,
         0., -0., -0., -0., -0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,
         0., -1., -0., -0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
        -0., -0.,  0.,  0., -0., -0.],
       [-0.,  1.,  1.,  0., -1., -0., -0.,  0.,  0.,  0., -0.,  1., -0.,
         1., -0., -1.,  0., -1., -1.,  0., -0., -0.,  0.,  0., -0.,  1.,
         0., -0.,  0., -1., -1., -0.,  0., -0., -1., -0.,  0., -0., -1.,
        -1., -0., -1.,  0.,  1., -0.],
       [ 0., -0.,  0., -0., -0., -0., -0., -0., -0., -0., -0.,  0.,  0.,
         0., -0., -0., -0., -0.,  0., -0., -0.,  0.,  0., -0., -1., -0.,
        -1., -0.,  0., -0.,  0.,  0., -0., -0.,  0., -1.,  0., -0.,  1.,
         0., -0., -0., -0., -0., -0.],
       [ 0.,  0.,  0.,  1., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,
         0., -0., -0., -0., -0.,  0., -0., -1., -0.,  0., -0.,  1.,  0.,
         0.,  0., -0., -0., -0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,
         0.,  1., -0., -0.,  0., -0.],
       [-0.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,
        -0.,  0.,  0., -0., -0., -0.,  0., -0.,  0.,  0.,  0., -0., -0.,
        -0.,  1., -0., -1., -0.,  0.,  0.,  0., -1., -0., -0.,  0., -0.,
        -1.,  0.,  0.,  0.,  0., -0.],
       [ 0.,  0.,  0., -0., -0., -0., -0., -0.,  0.,  0., -1., -0., -0.,
         1.,  0., -0., -0.,  0.,  0., -0., -0., -1.,  0., -0., -0., -0.,
         0.,  0.,  0., -0.,  0., -0., -0.,  1., -0., -0., -0., -0., -0.,
        -0., -0., -0., -0., -0.,  0.],
       [-0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,  1., -0., -0., -0.,
        -0., -0., -0.,  0., -0., -0.,  0.,  1.,  0., -0., -0., -0.,  0.,
        -0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,  0.,  1., -0.,  0.,
        -0.,  0.,  0., -0.,  0.,  1.],
       [-1., -0., -1., -1., -0., -1., -0.,  1., -0.,  0.,  0., -0.,  0.,
         0.,  0.,  0., -1.,  0., -0.,  0., -0., -0., -0., -0., -0.,  0.,
        -0.,  0., -0., -0.,  0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,
        -1., -0.,  0.,  0.,  1., -0.],
       [ 0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,
        -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  1., -0., -0., -0., -0.,
         0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0., -0.,  0., -0.,
        -0., -0.,  0., -0.,  1.,  0.],
       [ 0., -0., -1., -1.,  0., -1., -1.,  1.,  1., -1.,  0., -0., -0.,
         0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -1., -1., -0.,
        -1.,  1., -0., -1., -0.,  0.,  0., -0., -1., -1.,  0.,  0.,  1.,
        -1.,  0.,  0., -0., -0.,  0.],
       [-0.,  0., -0.,  1.,  0.,  0., -0., -0., -0.,  0., -0.,  0., -0.,
         0.,  0., -0., -0., -0., -0.,  0., -0., -0., -0., -0.,  0., -0.,
        -0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  1.,  0.,  0.,
        -0., -0., -0.,  0., -0.,  0.],
       [ 0.,  0.,  1.,  1.,  0.,  1., -0., -1., -0.,  0.,  0., -0.,  0.,
        -0.,  0., -0., -0., -0., -0., -0., -0.,  0.,  0., -0., -0.,  0.,
        -0.,  0., -0.,  0.,  0., -0.,  0., -0., -0.,  0., -0., -0.,  0.,
        -0., -0., -0.,  0., -0.,  0.],
       [ 0.,  0., -0., -0., -0.,  0.,  0., -0., -0.,  0., -1., -0., -0.,
        -0., -0.,  0.,  0.,  0.,  0., -1., -0.,  0., -0., -0., -0.,  0.,
        -0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,
        -0., -0., -0.,  0., -0.,  0.],
       [-0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,
        -0., -1.,  0., -0., -0., -0., -0., -0.,  0., -1., -0.,  0.,  0.,
         0., -0., -0., -0.,  0., -0.,  0.,  0.,  0.,  1., -0., -0.,  0.,
         0., -0., -0.,  0., -0., -1.],
       [ 0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,
         0.,  0.,  0., -0.,  0.,  1.,  1., -0., -0., -0., -0., -0., -1.,
         0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0.,
        -0., -0.,  0., -0.,  0.,  0.],
       [ 0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0.,
         1.,  0.,  0., -0.,  0., -0.,  1.,  0., -0., -0.,  0., -0.,  0.,
         0., -0.,  0.,  0., -1.,  0., -0.,  0., -0., -0., -0.,  0., -0.,
        -0.,  0., -0.,  0.,  1.,  0.],
       [-0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0.,
         0.,  1.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,
        -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0., -1.,
        -0.,  0.,  0.,  0.,  0., -0.],
       [-0., -1., -1., -1.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,
        -0., -0., -0.,  0.,  1.,  0.,  0.,  1., -0., -1., -1., -1.,  0.,
         0.,  1.,  0., -1.,  0., -0.,  1., -0., -1., -0.,  0., -0.,  1.,
        -1., -1.,  0.,  1., -0., -0.],
       [ 0., -1., -1., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,
         0.,  0., -0.,  0.,  1., -0.,  0.,  0.,  0., -0.,  0.,  0., -0.,
         0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0., -0., -0., -0.,  1.,
        -0.,  0., -0.,  0., -0., -0.]], dtype=np.float32), np.array([[ 0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  1., -0.,  0.,
         1., -0., -0.,  0.,  0.,  1., -1.,  0.,  0., -0., -0., -0., -1.,
        -0.,  0.,  0., -0., -1., -0.,  0., -0., -0.,  0., -0.,  1.,  0.,
        -0., -0., -0.,  0.,  0.,  0.],
       [-0.,  0., -0., -0.,  1.,  0., -0.,  0., -0.,  0., -1., -1.,  0.,
         0.,  0., -1.,  0., -0., -0., -0., -0.,  0.,  0.,  0., -0.,  0.,
        -0.,  0.,  0.,  0., -0.,  0., -0.,  1.,  0., -0.,  0., -1.,  0.,
        -0., -0.,  0., -0.,  0., -0.],
       [ 0., -0., -0., -0., -0., -0.,  0.,  0.,  1., -1.,  0., -0., -0.,
        -0., -0., -0.,  0.,  0., -0., -0.,  1.,  0., -1.,  1., -1.,  0.,
        -1.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  1.,  0., -0.,  0.,
         0., -1.,  0.,  1., -0.,  0.],
       [ 0., -0.,  0., -0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,  0.,
         0., -0.,  0., -0., -0.,  0.,  0., -1.,  0.,  0.,  0.,  1.,  0.,
         1.,  0., -1.,  0.,  0.,  0.,  0.,  0.,  0., -1., -0., -0., -0.,
        -0.,  1.,  0., -0., -0.,  1.],
       [-1.,  0., -0.,  0.,  1.,  0.,  0.,  0.,  0.,  0., -0., -0., -1.,
         0., -0.,  0., -1., -0., -0.,  0., -0.,  0.,  0.,  0.,  0., -1.,
        -0., -0.,  0.,  0.,  0.,  1., -0.,  0.,  0.,  0., -0.,  0.,  0.,
         0., -0.,  1.,  0.,  0.,  0.],
       [ 1.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0., -1.,  1.,
        -0., -0., -0.,  0., -0.,  1.,  0.,  0.,  0.,  0., -0., -0.,  0.,
         0.,  1., -0.,  0., -0., -0.,  0.,  0., -1., -0.,  0., -0.,  0.,
         1., -0., -0.,  0., -0., -0.],
       [ 1., -0., -0., -0., -1.,  0., -0., -0.,  0., -0.,  0.,  0., -0.,
         0.,  0., -0.,  1.,  0.,  0., -0., -0., -0., -0., -0., -0., -0.,
        -0.,  0.,  0., -0., -1., -0., -0., -0.,  0., -0.,  0.,  1.,  0.,
        -0.,  0., -1.,  0., -1., -0.],
       [-1.,  0., -0., -0.,  1.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0.,
         1., -0., -1., -1., -0., -0., -0.,  0., -1.,  0.,  0.,  0.,  0.,
        -0.,  0., -0.,  0.,  0., -0., -0.,  1., -0.,  0., -0., -1.,  0.,
        -0., -0.,  0.,  0.,  1., -0.],
       [-0.,  0.,  0., -0.,  0.,  0.,  1.,  0., -0.,  0., -0., -0.,  0.,
         0., -1.,  0., -0., -0.,  0.,  0.,  0.,  0., -1.,  1.,  0.,  0.,
        -1.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,  1., -0., -0.,  0.,
        -0., -0.,  0.,  1.,  0., -0.],
       [ 0.,  1., -0.,  0., -0.,  0., -1., -0.,  1., -0.,  0.,  0., -0.,
        -0., -0., -0.,  0.,  1.,  0., -0.,  0., -0., -0., -0.,  0., -0.,
         1.,  0., -1.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,  1.,
        -0.,  0., -0.,  0.,  0., -0.],
       [ 0., -0., -0.,  0.,  0.,  0.,  0.,  0., -1.,  1.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0., -0., -0., -1.,  0., -0., -1., -0., -0.,
         0.,  0., -0., -0., -0.,  0., -1.,  0.,  0., -0.,  1.,  0.,  0.,
         0.,  1., -0.,  0.,  0.,  0.],
       [ 0.,  0.,  0., -1.,  0.,  1., -0., -1.,  0., -0.,  0.,  0.,  0.,
         0., -0.,  0., -0., -0., -0., -0., -0., -0.,  0.,  0., -0., -0.,
        -0.,  0.,  1., -0.,  0.,  0.,  1., -0.,  0., -0.,  0.,  0., -0.,
         0., -1., -0., -1., -0.,  0.],
       [-1., -1.,  1.,  0.,  1.,  1.,  1.,  0.,  0., -0.,  0., -0., -0.,
         0.,  0.,  0., -1., -0., -0.,  0.,  0.,  0., -0.,  1., -0., -0.,
        -0., -0.,  0., -1.,  0.,  1.,  1.,  0., -1., -0.,  0., -1., -0.,
         0.,  0.,  1., -0.,  0.,  0.],
       [ 1.,  1., -0., -0., -1., -1., -1.,  1.,  1., -0., -0.,  0.,  0.,
        -0., -0.,  1.,  0.,  1., -0.,  0., -0., -0., -0., -0.,  0., -0.,
         0., -0., -1.,  1.,  0.,  0., -1., -0., -0., -0.,  0.,  1., -0.,
         1., -0., -0.,  1., -0., -0.]], dtype=np.float32))

In [ ]:
#@title Verification
n = 2
m = 4
p = 7
rank = 45

verify_tensor_decomposition(decomposition_247, n, m, p, rank)

## Rank-51 decomposition of <2,4,8> over Z


In [ ]:
#@title Data

decomposition_248 = (np.array([[-1., -0., -0.,  1.,  0.,  0., -0.,  1.,  0., -0.,  0.,  0., -0.,
        -1.,  0.,  0.,  1.,  0.,  0.,  1.,  1., -0., -0., -1., -0., -0.,
         0.,  0.,  0.,  1., -0.,  0.,  1., -0.,  0., -0.,  0., -0., -0.,
        -1.,  1., -0., -0., -0., -0.,  0.,  0.,  1.,  0.,  1., -0.],
       [-0.,  0.,  0., -0.,  0., -0.,  0.,  0., -1., -0., -0.,  0.,  0.,
         1.,  0., -0.,  0., -0.,  0., -1.,  0.,  1., -1.,  1.,  0., -0.,
        -0.,  0.,  0.,  0., -1.,  1.,  1., -0.,  0., -0., -0.,  1.,  1.,
         1.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0., -0., -1.],
       [ 0.,  0., -0.,  0., -0.,  0., -0., -1., -1.,  0., -0.,  1., -1.,
         0.,  0., -0., -1.,  0., -0., -0., -1., -0., -1.,  0.,  0., -0.,
        -0., -0., -0., -1.,  1.,  0., -0., -1.,  1., -1., -0., -0., -1.,
         0., -1., -0., -1., -0., -0.,  0.,  0.,  0., -0.,  0.,  0.],
       [-0., -0., -1.,  0.,  0., -0.,  0., -0.,  1., -1.,  0.,  0.,  1.,
        -0., -0.,  0.,  0.,  0.,  1.,  0.,  0., -0., -0.,  0.,  1., -0.,
        -0.,  0.,  1.,  0.,  0., -1.,  0., -0.,  0., -0.,  1.,  0., -1.,
        -0., -0., -1., -0., -1.,  0.,  0.,  1., -1., -0., -1., -0.],
       [-1.,  0., -0.,  1., -0., -1.,  0., -0., -0., -0., -1., -0., -0.,
         0., -1.,  0.,  1., -1., -0., -0., -0., -0., -0.,  0.,  1.,  1.,
         0., -0.,  1., -0., -0.,  0., -0., -0.,  0., -1.,  0.,  0., -0.,
        -1., -0.,  0., -0., -0., -0., -1., -0., -0.,  0., -0.,  1.],
       [-0., -1., -1., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,
        -0., -0.,  1.,  0.,  0., -0., -1., -1.,  0., -1.,  1., -0.,  1.,
         1., -1.,  0.,  0., -1.,  1., -0.,  0.,  0., -0.,  1., -0., -0.,
         1.,  0.,  0., -1., -0.,  1., -0., -0.,  0., -1.,  0., -0.],
       [-0., -1.,  1.,  0., -0.,  0.,  1.,  0.,  0.,  0.,  1., -0., -1.,
         0., -0.,  0., -1., -0., -0.,  1., -1., -1., -1.,  0.,  0., -0.,
         1.,  1.,  0.,  0.,  1.,  0.,  0., -1.,  1.,  0.,  1.,  0.,  0.,
         0., -1.,  0., -0., -0., -0., -0.,  0., -0.,  1.,  0.,  0.],
       [-0.,  1.,  0.,  0., -1.,  1., -1., -0., -0., -1.,  0.,  0.,  1.,
         0.,  1., -1.,  0., -0., -0.,  0., -0.,  1., -0., -1., -0.,  0.,
         0.,  0., -0.,  0.,  0., -1., -0.,  1.,  0., -0., -0., -0.,  0.,
        -0.,  1.,  0.,  1., -1.,  0., -0., -0.,  0.,  1.,  0.,  0.]],
      dtype=np.float32), np.array([[ 0.,  0.,  0.,  1.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  0.,
        -0.,  0., -0.,  0., -1., -0.,  0., -0., -0., -0., -0.,  0., -0.,
         0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0.,
        -0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0.],
       [-1., -0.,  0.,  0., -0., -0., -0., -1.,  0., -0., -0., -1.,  0.,
        -0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,
         0., -0., -0.,  0., -0., -0.,  0., -0., -0.,  1.,  0., -0., -0.,
        -0., -0., -0., -0., -0., -0., -0., -0., -0.,  0., -0.,  0.],
       [-0., -0., -1.,  0., -0., -0., -0., -0.,  0., -0., -0., -0., -1.,
        -0., -1.,  0.,  0.,  0.,  1.,  0.,  0., -0.,  0.,  1.,  0.,  0.,
         0.,  1.,  0., -0., -0., -1.,  0.,  1.,  1.,  0.,  0., -0.,  0.,
         1., -0., -0.,  0., -0., -0.,  0.,  1.,  0., -1.,  0.,  0.],
       [-0., -0.,  0.,  0.,  1.,  1.,  0., -0.,  0., -0., -0., -0.,  0.,
        -0.,  0., -0., -1., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0.,
         0., -0.,  0.,  0., -0.,  0.,  0.,  1.,  1.,  0.,  0., -0.,  0.,
         0.,  1.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0.],
       [-0., -0., -0., -1., -0., -0., -0., -0.,  0., -0., -0., -0.,  0.,
        -0., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,
         0.,  0., -0.,  0., -0., -0., -1., -0., -0.,  0.,  0.,  1.,  0.,
         0., -0., -0.,  0.,  0., -0.,  0.,  0., -0., -0., -0., -1.],
       [ 0., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0., -0., -1., -0., -0.,  0.,  0.,  0., -0., -1.,  0.,
         0., -0.,  0.,  0.,  0.,  0., -0., -0., -0., -0.,  0., -0.,  0.,
        -0., -0.,  1., -0., -0.,  0., -0.,  0.,  0.,  0., -1.,  0.],
       [-1.,  0., -0., -0., -0., -0.,  0.,  0., -0., -0.,  0.,  0., -0.,
        -0., -0.,  0.,  0.,  0.,  1., -0., -0.,  0., -0.,  0.,  0., -0.,
        -0.,  0.,  1., -0.,  0., -0.,  0.,  0.,  0., -0., -0., -0., -0.,
         0., -0., -0., -0., -0.,  0., -0., -0.,  1., -0.,  0.,  0.],
       [-1.,  0., -0.,  0., -0., -0., -0.,  0., -0., -0.,  0.,  0., -0.,
         0., -0., -0.,  0.,  0.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,
         0.,  0., -0., -0., -0., -0.,  0.,  0.,  0., -0.,  0., -0., -0.,
         0., -0., -0., -0., -0., -0.,  1.,  0., -0., -0.,  0.,  0.],
       [ 0., -0.,  0.,  0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,
         0., -0.,  0.,  0.,  1., -0., -0.,  0.,  0., -0.,  0., -0., -1.,
        -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,
         0.,  0., -0., -0.,  0.,  1., -0., -0., -0.,  0., -0.,  1.],
       [ 0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -1.,  1.,
         0., -0.,  0.,  0., -0., -0., -0., -0.,  0., -0.,  0., -0.,  0.,
        -0.,  0., -0., -0., -0.,  1.,  0., -1., -1.,  0., -0.,  0.,  1.,
         0., -0., -0.,  1.,  0.,  0., -0., -1., -0., -0., -0., -0.],
       [ 0., -0., -1.,  0., -0., -0.,  0.,  0.,  0., -0., -0., -0., -1.,
        -0.,  0., -1.,  0.,  0.,  1.,  0.,  0.,  0.,  0., -0.,  0.,  0.,
         0.,  1.,  0., -0., -0., -1., -0.,  1.,  1., -0.,  0.,  0.,  0.,
        -0., -0., -0.,  0., -0.,  0.,  0.,  1., -0., -1.,  0., -0.],
       [ 0.,  1., -0., -0., -0., -0., -1.,  0.,  0.,  0.,  0., -0.,  0.,
        -0., -0., -0., -0.,  0.,  0., -0.,  0., -1.,  0.,  0.,  0., -0.,
         0.,  0., -0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,
         0., -0., -0.,  0.,  0., -1.,  0., -0., -0., -0.,  0., -0.],
       [ 0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0., -0.,  0.,
        -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,
        -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0., -0., -0., -1.,  0.,
        -0., -0.,  0.,  0.,  0., -1.,  0., -0., -0., -0.,  0.,  0.],
       [ 0., -1.,  0.,  0., -1., -1.,  1.,  0., -0., -0.,  0.,  0., -0.,
         0., -0., -0., -0.,  1.,  0.,  0.,  1.,  1.,  1., -0.,  1.,  0.,
        -0., -0.,  0., -0.,  0., -0., -1., -1., -1., -0., -0.,  0., -0.,
         0., -1., -1., -0., -0.,  1., -0.,  0.,  0.,  0.,  1., -0.],
       [-0.,  0., -0.,  0.,  0.,  0.,  0.,  1., -0.,  0., -1.,  0.,  1.,
         0.,  1., -1., -0., -0.,  1.,  1., -0.,  0., -0., -1., -0., -0.,
        -0., -1.,  1.,  0.,  0.,  1.,  0., -1., -1., -1., -0., -0.,  1.,
         0.,  0.,  0.,  1., -0.,  0., -0., -1.,  1., -0., -0.,  0.],
       [-0.,  0., -0.,  0.,  0.,  0.,  0.,  1., -0.,  0., -1.,  0.,  1.,
        -1., -0.,  0., -0., -0.,  0.,  1., -0., -0., -0.,  0., -0., -0.,
        -0., -1., -0.,  0.,  0.,  1.,  0., -1., -1., -1., -0.,  0.,  1.,
         0.,  0.,  0.,  1.,  0.,  0.,  1., -1., -0., -0., -0.,  0.],
       [ 0., -1.,  0., -0., -1.,  0., -0., -0.,  1., -0.,  0., -0., -0.,
        -0.,  0.,  0., -0., -1.,  0.,  0.,  1.,  1., -0., -0.,  0.,  1.,
        -0., -0.,  0., -1., -0., -0., -1., -1., -1., -0.,  1.,  0.,  0.,
         0.,  0., -1., -0.,  0., -0., -0., -0.,  0.,  0.,  0., -1.],
       [ 0., -0., -0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0., -1., -0.,
        -0., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0., -0., -0.,
         0.,  0., -0.,  0.,  0., -0.,  0.,  0., -1.,  0.,  0., -0., -0.,
        -0.,  0., -0., -0.,  0., -0., -0.,  0., -0., -0., -0.,  0.],
       [ 0., -0.,  0., -0., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,
         0.,  0., -1., -0., -0., -0., -0., -0.,  0., -0., -0., -0., -0.,
         0., -0., -0., -0.,  0.,  0.,  0., -0.,  1.,  0.,  0., -0., -0.,
        -0.,  0., -0.,  1., -0., -0.,  0., -0., -0., -1., -0.,  0.],
       [ 0., -0.,  0., -0.,  1., -0.,  1.,  0.,  0., -0., -0., -0.,  0.,
        -0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,
         0., -0.,  0., -0., -0.,  0., -0.,  1.,  1., -0.,  0., -0.,  0.,
        -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0., -0.],
       [-0.,  1., -0.,  0.,  1., -0.,  0., -0., -1.,  0., -0., -0.,  0.,
        -0., -0., -0.,  0., -0., -0., -0.,  0., -1.,  0.,  0.,  0.,  0.,
        -1.,  0., -0., -0.,  0.,  0., -0.,  1.,  1.,  0., -1.,  1.,  0.,
        -0., -0.,  1.,  0.,  0.,  0.,  0.,  0., -0., -0., -0., -0.],
       [-0., -0.,  0.,  0., -0.,  1., -1., -0., -0.,  0., -0.,  0.,  0.,
         0.,  0.,  0.,  0., -1.,  0.,  0.,  0.,  0.,  0., -0., -1., -0.,
         0.,  0.,  0., -1., -0.,  0., -0., -0.,  0., -0., -0., -0.,  0.,
         0.,  1.,  1., -0.,  0.,  0., -0.,  0.,  0.,  0., -1.,  0.],
       [-0.,  0.,  1.,  0.,  0., -0.,  0.,  1., -0.,  0., -1.,  0.,  1.,
         0.,  1., -1., -0.,  0., -0.,  1., -0., -0., -0., -1., -0., -0.,
        -0., -1.,  1.,  0., -1.,  1.,  0., -1., -1., -1., -0., -0., -0.,
         0.,  0.,  0.,  1., -0.,  0.,  0., -1.,  1., -0.,  0.,  0.],
       [ 0.,  0., -0.,  0., -0., -0.,  0., -0., -0., -0., -1.,  0., -0.,
         0., -0., -0., -0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0., -0.,
        -0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -1., -1., -0., -0., -0.,
        -0., -0., -0., -0.,  0., -0.,  1.,  0., -0., -0.,  0.,  0.],
       [-0.,  0.,  0., -0.,  0.,  1.,  0.,  0., -0., -0., -0., -0., -0.,
        -0.,  0., -0.,  0., -1., -0.,  0., -0., -0.,  0., -0., -1., -0.,
        -0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0.,
         0., -0.,  0.,  0., -1.,  0., -0., -0.,  0.,  0.,  0.,  0.],
       [ 0., -0., -0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,  1.,
        -0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,
         0.,  0.,  0., -0.,  0., -0., -0., -1., -1., -0.,  0., -0., -0.,
        -0.,  0., -0., -0.,  0.,  0., -0., -1.,  0., -0., -0.,  0.],
       [ 0., -0., -1.,  0., -0., -0., -0.,  0.,  0., -0., -0., -0., -1.,
        -0.,  0.,  0., -0.,  0.,  1.,  0.,  0., -0.,  0., -0.,  0.,  0.,
         0.,  1.,  0., -0., -0.,  0., -0.,  1.,  1., -0.,  0., -0.,  0.,
        -0., -0., -0.,  0.,  0., -0.,  0.,  1.,  0., -1.,  0., -0.],
       [-0., -0., -0., -0.,  1., -0.,  0.,  0.,  0.,  0., -0.,  0., -1.,
         0., -0., -0.,  0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,
         0.,  0., -0., -0., -0.,  0., -0.,  1.,  1., -0.,  0., -0.,  0.,
         0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0., -0.],
       [-0.,  1., -0.,  0.,  1., -0.,  0.,  0.,  0., -0., -0.,  0., -1.,
        -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0.,
        -1.,  0., -0., -0., -0.,  0.,  0.,  1.,  1., -0., -1., -0., -0.,
        -0., -0.,  1.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,  0.],
       [-0., -0.,  0.,  0.,  0.,  1.,  0., -0.,  0.,  0., -0., -0., -0.,
        -0.,  0., -0.,  0., -1., -0.,  0.,  0.,  0.,  0., -0., -1.,  0.,
         0., -0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0., -0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.],
       [ 0., -0., -0.,  0., -0., -0.,  0., -0., -0.,  1.,  0., -0., -0.,
        -0.,  0.,  0., -0.,  0.,  1., -0.,  0.,  0., -0.,  0.,  0.,  0.,
         0.,  0., -0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,
        -0.,  0., -0.,  0.,  0.,  0., -0., -0., -0., -0., -0., -0.],
       [ 0., -0., -0., -0., -0.,  0., -0.,  0., -0.,  1.,  0.,  0., -0.,
         0., -1., -0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,  0., -0.,
         0.,  0., -1., -0., -0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,
        -0., -0., -0.,  0., -0., -0.,  1.,  0.,  0., -0., -0., -0.]],
      dtype=np.float32), np.array([[-0., -0.,  0.,  1., -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,
         0., -0.,  0.,  0., -1.,  0.,  0., -1., -0.,  1., -0., -1., -0.,
         1.,  0.,  0.,  0., -0., -0., -1.,  0., -0.,  0., -1.,  0.,  0.,
         0.,  0.,  1., -0.,  0.,  0., -0.,  0.,  0., -0., -0., -1.],
       [ 0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,  0.,
        -0.,  0., -0., -0.,  1., -0.,  0., -0., -0.,  0., -0.,  1., -1.,
        -1., -0.,  0., -0., -0., -0., -0., -0., -0., -0.,  1., -0., -0.,
         0., -0., -1., -0.,  1.,  0.,  0., -0.,  0., -0.,  0.,  0.],
       [ 0.,  0.,  1., -0., -0., -0.,  0., -1.,  0., -0.,  0., -1.,  0.,
         0.,  0., -0.,  0.,  0., -0.,  1., -0.,  0.,  0.,  0.,  0., -0.,
         0.,  1., -0., -0.,  1.,  0.,  0.,  0., -0.,  0., -0.,  0.,  1.,
        -0., -0., -0.,  0.,  0., -0.,  0., -1.,  0., -0., -0.,  0.],
       [ 0.,  0.,  0.,  0., -1.,  0., -0., -0.,  0., -0.,  1.,  1.,  0.,
        -0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0., -0.,  0.,
        -0., -1.,  0.,  0.,  0.,  0., -0., -1., -1., -1., -0., -0.,  0.,
         0.,  0.,  0., -1., -0.,  0., -0., -0., -0., -1., -0.,  0.],
       [-0., -0.,  1.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,
        -1., -0.,  1., -0., -0., -0.,  1.,  0.,  0., -0., -1., -0.,  0.,
        -0.,  1.,  0., -0.,  1., -1.,  0.,  0.,  0., -0.,  0.,  0.,  1.,
        -0., -0.,  0., -1., -0.,  0., -0.,  0.,  0.,  0., -0., -0.],
       [ 0., -0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,
         1.,  0., -1.,  0.,  0.,  0., -1.,  0., -0.,  0.,  1.,  0.,  0.,
         0., -1., -0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,
        -1.,  0., -0., -0.,  0.,  0.,  0.,  0., -0., -1.,  0.,  0.],
       [ 0., -0.,  0.,  0., -0.,  0., -1., -0., -1.,  0., -0., -0., -1.,
        -0., -0.,  0., -0., -0., -0.,  0.,  1., -1., -1., -0., -0.,  0.,
        -1.,  0., -0.,  1.,  0., -0., -0., -1.,  0.,  0.,  1., -0.,  0.,
        -0.,  1.,  0., -0., -0., -0.,  0., -1.,  0., -0.,  0., -0.],
       [ 0., -1.,  0., -0., -1., -0.,  1., -0., -0., -0.,  0.,  0., -0.,
         0.,  0., -0., -1.,  0., -0., -0., -1., -0., -0.,  0.,  0., -0.,
         1., -0.,  0., -1., -0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,
        -0., -1., -0., -0.,  0., -0., -0., -0., -0., -0., -0., -0.],
       [ 0.,  0.,  0., -0., -0., -0., -0., -0.,  1.,  0.,  0., -0., -0.,
        -0., -0.,  0.,  0.,  0.,  0.,  0., -1., -0.,  1., -0.,  0., -0.,
         1.,  0., -0.,  0.,  0., -0., -1., -0.,  0.,  0., -1., -1.,  0.,
        -0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0.],
       [ 0.,  1.,  0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,
         0., -0., -0., -0., -0., -0., -0., -0.,  1.,  0.,  0., -0., -1.,
        -1., -0.,  0., -0.,  0.,  0., -0., -0., -0., -0., -0.,  1., -0.,
        -0., -0.,  0., -0., -0., -1., -0., -0.,  0., -0., -0., -1.],
       [ 0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0.,
        -0.,  0., -0., -0., -0.,  0., -0.,  1.,  0., -1.,  0., -0.,  0.,
        -1., -0., -0.,  1., -0.,  0., -0.,  0., -0.,  0.,  1.,  0., -0.,
         0., -0., -1.,  0., -0.,  0.,  0.,  0.,  0., -0., -1., -0.],
       [ 0.,  0.,  0., -0.,  0.,  1., -0., -0., -0., -0., -0., -0., -0.,
        -0.,  0.,  0., -1.,  0., -0.,  0., -1., -0., -0., -0., -1., -0.,
         1.,  0.,  0., -1., -0.,  0.,  0., -0.,  0.,  0., -1., -0., -0.,
         0., -1.,  1., -0., -1.,  0., -0.,  0., -0.,  0., -0.,  0.],
       [ 0., -0., -1., -0., -0.,  0.,  0.,  0.,  0., -0., -0., -0., -0.,
         1.,  0.,  0.,  0., -0.,  1., -1., -0., -0.,  0., -0., -0.,  0.,
         0., -1., -0., -0., -1., -0., -0., -0., -0.,  0., -0., -0.,  0.,
         0., -0.,  0., -0., -0.,  0.,  0., -0.,  1., -0.,  0.,  0.],
       [ 0., -0.,  1., -0., -0., -0., -0.,  0.,  0., -1., -0.,  0., -0.,
        -1.,  1., -0.,  0., -0., -1.,  1.,  0.,  0., -0., -1.,  0., -0.,
        -0.,  1.,  1.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,  0., -0.,
         1.,  0., -0., -0.,  0., -0., -0.,  0.,  0., -0., -0., -0.],
       [ 1.,  0.,  0.,  0., -0., -0.,  0.,  1.,  0., -0., -0., -0.,  0.,
        -1., -0., -0., -0., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,
         0.,  0., -1., -0.,  0.,  0., -0.,  0., -0.,  1., -0.,  0.,  0.,
        -0., -0.,  0.,  0.,  0., -0.,  1., -0., -1., -0., -0., -0.],
       [ 0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0., -0., -1.,  0.,  0.,
         1., -1.,  0.,  0.,  0., -0., -1.,  0., -0.,  0.,  1.,  0., -0.,
        -0.,  0., -0.,  0.,  0.,  0., -0., -0., -0., -0.,  0., -0., -0.,
        -1.,  0., -0., -0.,  0.,  0., -1., -0., -0., -0.,  0.,  0.]],
      dtype=np.float32))

In [ ]:
#@title Verification
n = 2
m = 4
p = 8
rank = 51

verify_tensor_decomposition(decomposition_248, n, m, p, rank)

## Rank-47 decomposition of <2,5,6> over Z

In [ ]:
#@title Data

decomposition_256 = (np.array([[ 0.,  0.,  1., -0.,  0.,  0., -1., -1.,  0., -0., -1.,  0., -0.,
         0.,  1., -0., -0., -0.,  0.,  0., -1.,  1., -0.,  0.,  1.,  0.,
         0., -0., -0., -1., -0., -1.,  0.,  0.,  0., -0.,  0.,  1., -0.,
        -0., -0., -0.,  0., -0.,  0.,  0., -0.],
       [ 0., -0., -0.,  1., -0., -0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,
         0.,  1., -0., -1.,  0., -0., -1.,  0.,  0.,  0., -0., -1., -1.,
         0., -0.,  0.,  0.,  0.,  0.,  1.,  0., -0., -1.,  0.,  1., -0.,
         0.,  0., -0., -0., -0.,  0., -0.,  0.],
       [-0., -0., -0., -1., -0.,  0., -0., -0.,  0., -1.,  0.,  1.,  0.,
         0.,  0., -1.,  1.,  0., -0., -0., -1.,  0., -1., -1.,  1.,  0.,
        -0.,  0.,  0., -0., -0.,  0., -1., -0., -0.,  1.,  0., -0.,  0.,
         0.,  0., -1., -0., -1.,  1.,  0.,  0.],
       [ 0., -0., -1.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  1.,  0.,
        -0.,  0., -1.,  0.,  0.,  1.,  0.,  0.,  0., -1.,  0., -1., -1.,
        -0., -1., -1., -0., -0., -0., -0., -0.,  1.,  0.,  0.,  0., -0.,
        -0.,  0.,  0.,  1., -1.,  1., -0.,  0.],
       [-1., -1., -0., -0., -1.,  0., -1., -1.,  0., -0., -1.,  0.,  0.,
         0.,  1.,  1.,  0.,  1., -0., -0.,  0.,  1., -0., -0., -0.,  0.,
        -0.,  0., -0., -1., -0.,  0., -0., -0., -0.,  0., -0., -0., -0.,
        -0.,  1.,  1., -0.,  0., -0., -0.,  0.],
       [ 0., -0., -1., -1., -1.,  0., -0.,  1.,  1., -1., -0., -0., -0.,
        -0.,  0., -0., -0., -0.,  1., -0., -0.,  0., -0., -0.,  0., -0.,
         1.,  0.,  0., -0., -1., -0., -0.,  1.,  0., -0.,  0., -1., -1.,
         0.,  0.,  0.,  0.,  0.,  1.,  1.,  0.],
       [-0., -0., -0.,  0., -1., -1., -0., -0.,  1., -1., -0.,  0.,  0.,
         1.,  0., -0.,  1.,  0., -0.,  1.,  0.,  1., -0., -0., -0., -0.,
         1., -1., -0.,  0.,  0.,  0., -1.,  0., -0., -0.,  0., -1.,  0.,
         0., -0.,  0.,  0., -0.,  0., -1., -0.],
       [ 0., -1.,  0.,  0.,  0.,  0., -1.,  0.,  0., -0.,  0., -1.,  1.,
         0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,  0.,  0.,
        -0., -0., -0.,  0., -1.,  0.,  1., -0., -0.,  0.,  1.,  0.,  0.,
         1., -0.,  1.,  1.,  0., -0.,  1.,  0.],
       [-0.,  0.,  1.,  1.,  0., -1.,  0.,  0.,  0.,  0., -0., -1.,  0.,
        -0.,  0., -0.,  0., -0., -1.,  0., -0., -0., -0., -0., -0., -0.,
        -0., -0.,  1., -0., -0., -0.,  0., -1.,  0., -0.,  1.,  0., -0.,
        -0., -0., -0.,  0., -0., -1., -1., -0.],
       [ 1.,  1.,  0.,  0.,  0., -0.,  1.,  1.,  1.,  0.,  1., -0.,  0.,
         0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0., -0.,
        -0., -0., -0., -0.,  0.,  0.,  0., -0., -0.,  0., -1.,  0.,  0.,
        -1., -1., -1., -1.,  1., -0., -0.,  1.]], dtype=np.float32), np.array([[ 0., -1., -0., -0.,  0.,  0.,  1., -1.,  0., -0.,  0.,  0., -1.,
        -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,
         0.,  0., -0.,  0., -1., -0., -0., -0., -0.,  0.,  0.,  0., -0.,
        -0.,  1., -0., -0.,  0., -0., -0., -0.],
       [-0.,  0.,  1., -0., -0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,
        -0.,  0.,  0., -0.,  0., -1., -0., -0.,  0., -0.,  0.,  0.,  0.,
        -0., -0.,  1.,  0., -0.,  0., -0., -0., -0., -0.,  0., -0., -1.,
        -0., -0., -0.,  0.,  0.,  0.,  0.,  0.],
       [ 0., -0., -0.,  0.,  0., -0., -0.,  0., -0.,  1.,  0.,  0., -0.,
        -0., -0., -0.,  1.,  0.,  0., -1., -1.,  0., -0., -1., -0., -0.,
        -0., -0.,  0., -0., -0., -0.,  0., -0.,  0., -0., -0.,  1., -0.,
         0.,  0.,  0., -0.,  0., -0.,  0., -0.],
       [ 0.,  0.,  1., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,
         0.,  0.,  0., -0., -0., -1., -0., -0.,  0., -0., -0.,  0.,  0.,
        -0., -0.,  1., -0., -0.,  1.,  0., -0.,  0., -0.,  0.,  0.,  0.,
        -0., -0., -0., -0.,  0., -0., -0., -0.],
       [-1., -0., -0.,  0., -1.,  0.,  0., -1., -0., -0., -1.,  0., -0.,
         1.,  0., -0., -0., -1., -0.,  0., -0., -1.,  0.,  0.,  0.,  0.,
         1.,  0., -0.,  1., -0.,  1.,  0.,  0.,  0.,  0.,  0., -0., -0.,
         0.,  0., -0.,  0., -0.,  0., -0., -0.],
       [-1., -0.,  0.,  0.,  0.,  0.,  0., -1.,  0., -0., -1., -0., -0.,
        -0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,
        -0., -0., -0., -0., -0.,  1., -0.,  0., -0.,  0.,  0.,  0., -0.,
         0.,  0., -0.,  0., -0.,  0.,  0., -0.],
       [-1.,  1., -0., -0.,  0., -0., -1., -0., -0., -1., -1.,  0.,  1.,
         0., -1.,  0., -0.,  0.,  0.,  1.,  1., -1.,  0.,  0., -0., -0.,
         1.,  0., -0.,  1.,  1.,  1.,  0.,  0., -0.,  1.,  0.,  0., -0.,
         0., -1., -0.,  0., -0.,  0., -0.,  0.],
       [-0.,  0., -0., -1., -0., -0.,  0.,  0.,  0., -0.,  0.,  1., -0.,
         0., -0.,  0., -0., -0.,  1.,  0.,  0., -0., -0., -0.,  0., -0.,
        -0.,  0., -1.,  0., -1., -0., -1., -1., -0., -0.,  0.,  0.,  1.,
         0., -0.,  0., -0.,  0.,  1.,  1.,  0.],
       [-0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,
         0., -0., -0., -1.,  0.,  0.,  1., -0., -0., -0.,  1., -0.,  0.,
         0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0., -1., -0.,  0., -0.,
         0., -0., -0.,  0.,  0.,  0., -0., -0.],
       [ 0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,
         0.,  0., -0.,  0., -0., -0.,  1., -0.,  0., -0.,  0.,  0.,  1.,
        -0., -1.,  0., -0.,  0.,  0.,  0.,  0.,  1., -0., -0., -0., -0.,
        -0.,  0.,  0.,  0., -0., -0., -0., -0.],
       [-0., -0., -0., -0., -0., -0.,  0., -0., -0.,  0., -0., -0., -0.,
        -1.,  0.,  0.,  0., -0., -0.,  1., -0., -0., -0., -0., -0., -0.,
         0.,  0., -0.,  0., -0.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,
         0.,  0., -0., -0., -0.,  0., -0., -0.],
       [ 0., -0., -0., -0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0., -0.,
         0., -1., -0., -0.,  0.,  0.,  1., -0., -1., -0., -0.,  0., -0.,
        -0.,  0., -0.,  1., -0., -0., -0., -0., -0., -0., -0.,  0.,  0.,
         0.,  0.,  0., -0.,  0.,  0.,  0., -0.],
       [-0.,  1., -0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,  1.,
        -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,
        -0.,  0.,  0., -0., -0.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,
         0., -1., -1.,  0., -0.,  0.,  0.,  0.],
       [ 0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  1., -0.,
        -0., -0., -0., -0., -0.,  1.,  0., -0., -0.,  0.,  0.,  0.,  0.,
         0., -0., -1.,  0., -1., -0.,  0.,  0., -0.,  0.,  0., -0.,  1.,
         0.,  0., -0.,  0., -0.,  1., -0.,  0.],
       [ 0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,
        -0., -0.,  0., -1.,  0., -0.,  1., -0.,  0., -0.,  1., -0.,  0.,
        -0., -0.,  0., -0., -0.,  0.,  1., -0.,  0.,  0.,  0.,  0.,  0.,
        -0., -0., -0.,  0., -0.,  0., -0.,  0.],
       [ 0.,  1., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0.,  1.,
        -0.,  0., -0., -0.,  0., -0., -0., -0.,  0., -1., -0., -0., -0.,
        -0.,  0.,  0., -0., -0., -0., -0., -0.,  1., -0.,  0.,  0., -0.,
        -1., -1., -1., -1.,  1., -0.,  0., -1.],
       [-0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0., -0., -0.,  1.,
         0.,  0., -0., -1., -0., -0.,  1.,  0., -0.,  0., -0., -0., -0.,
         0.,  0., -0.,  0.,  0.,  0.,  1., -0.,  0., -0.,  0., -0., -0.,
        -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.],
       [ 1.,  0.,  0., -0.,  0., -0.,  1.,  0., -0.,  0.,  1.,  0., -0.,
        -0.,  0.,  0.,  0.,  0.,  0., -0., -1.,  0., -0.,  0., -0., -0.,
        -0., -0.,  0., -0.,  0., -1., -0., -0., -0., -0., -0.,  0.,  0.,
        -0., -0., -1., -0., -0., -0.,  0.,  0.],
       [ 0.,  0., -0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0.,
        -0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -0., -0.,
         0.,  0.,  1.,  0., -0., -0., -0.,  0.,  0.,  0., -1.,  0.,  0.,
        -1., -0., -0., -1., -0.,  0., -0.,  0.],
       [-0.,  0., -0., -0., -0.,  0., -0., -0., -0.,  0., -0.,  0.,  0.,
        -0.,  0.,  0.,  0.,  0., -1., -0.,  0.,  0., -0., -0.,  0., -0.,
        -0.,  0.,  1., -0., -0.,  0., -0.,  1.,  0., -0., -0., -0., -1.,
        -0., -0., -0., -0., -0.,  0.,  0.,  0.],
       [ 0.,  0., -1., -1.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0., -0.,
        -0., -0.,  0.,  1.,  0., -0., -1., -1.,  0.,  0., -1., -1.,  0.,
        -0.,  0., -0.,  0., -0., -0., -0.,  0., -0.,  1., -0.,  1.,  0.,
        -0., -0., -0., -0.,  0.,  0., -0.,  0.],
       [ 0.,  0., -0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0.,
        -0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0.,
        -0., -0.,  1., -0., -0., -0.,  0., -0., -1., -0.,  0., -0.,  0.,
        -0., -0., -0., -0.,  0.,  0., -0., -0.],
       [ 0., -0.,  0.,  0.,  0., -1.,  0.,  0., -0., -0.,  0.,  0.,  0.,
         1., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        -0., -1.,  1., -0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0., -0., -0.],
       [-1.,  1.,  0., -0.,  0.,  0., -1., -0., -0.,  0., -1.,  0.,  1.,
        -0.,  0., -0.,  0.,  0., -0., -0.,  1.,  0., -1., -0., -0., -0.,
        -0.,  0.,  1., -0.,  1.,  1., -0., -1.,  0., -0., -1.,  0., -0.,
        -1., -1., -0., -1., -0., -1.,  0.,  0.],
       [-0.,  1.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  1.,
        -0.,  0., -0.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,
        -0., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,  0., -0.,
        -1., -1.,  0.,  0., -0.,  0.,  0.,  0.],
       [ 0.,  1., -0., -0.,  0., -0.,  0., -0., -0.,  0., -1., -0.,  1.,
         0.,  0.,  1., -0.,  0., -0., -0., -0.,  0., -1., -0.,  0., -0.,
        -0., -0.,  0.,  1.,  0.,  1., -0.,  0.,  0., -0.,  0., -0., -0.,
        -1., -1., -0.,  0.,  1., -0.,  0.,  0.],
       [ 1., -1.,  0.,  0.,  1., -0., -0.,  0., -1.,  0.,  1., -0., -1.,
        -0., -0.,  0.,  0., -0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,
        -1., -0.,  0., -1.,  0., -1., -0., -0., -0.,  0., -0.,  0., -0.,
         1., -0.,  0., -0.,  0., -0., -0., -0.],
       [ 0.,  1., -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  1.,
         0.,  0.,  1., -0., -0.,  0., -0., -0.,  0., -1.,  0.,  0.,  0.,
        -0., -0., -0., -0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,
        -1., -1.,  0.,  0.,  1.,  0., -0., -1.],
       [ 1.,  0., -0., -0.,  1.,  0., -0.,  0., -1.,  0.,  1.,  0.,  0.,
         0., -0.,  0.,  0.,  1.,  0., -0.,  0., -0., -0.,  0., -0., -0.,
        -1.,  0.,  0., -1.,  0., -1., -0., -0., -0., -0., -0., -0.,  0.,
        -0., -0.,  0., -0., -0., -0., -0.,  0.],
       [ 1.,  0., -0., -0., -0.,  0., -0.,  0.,  0.,  0.,  1., -0.,  0.,
        -0., -0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0.,
         0., -0.,  0., -1.,  0., -1., -0., -0.,  0., -0., -0., -0., -0.,
        -0., -0.,  0., -0.,  0., -0., -0.,  0.]], dtype=np.float32), np.array([[-0., -1., -0., -0., -1.,  0., -1.,  0., -0., -1.,  0., -0., -0.,
        -0., -0., -1., -0., -1.,  0., -0., -1., -0., -1.,  0., -0., -0.,
         1.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,  0.,  1.,  0.,
         1., -0.,  1., -1.,  0.,  0.,  0.,  0.],
       [-0., -1., -0.,  0., -1., -0.,  0., -0.,  0.,  0.,  0.,  1.,  0.,
        -0.,  0.,  0.,  0., -1., -0.,  0.,  0., -0., -1.,  0.,  0., -0.,
         1.,  0.,  0.,  0.,  1.,  0., -0., -0., -0., -0., -1., -0., -0.,
         1.,  1., -0.,  0., -1., -1., -0., -0.],
       [-1., -0.,  1., -1., -0.,  0., -0., -0., -0., -0.,  1.,  0., -0.,
         0.,  0.,  0., -0.,  0., -1., -0., -0.,  0.,  1., -0., -1., -0.,
         0., -0., -0., -0., -0.,  1., -0., -1., -0., -0., -0.,  0., -0.,
         0., -0., -0.,  0.,  1.,  1.,  0.,  1.],
       [ 0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0., -1., -0.,
        -0., -0., -0., -0.,  0., -0., -0., -0.,  0.,  1., -0.,  0.,  0.,
         0.,  0.,  0., -0.,  0., -0., -0., -1., -0., -0.,  0., -0.,  1.,
         0., -0., -0., -0.,  1.,  1., -1.,  1.],
       [ 0.,  0.,  0.,  0., -1., -0.,  0., -0.,  0., -1., -0.,  0.,  0.,
        -0.,  0.,  0.,  0., -1.,  0., -0.,  0., -0.,  0., -1.,  1.,  0.,
         1.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  1.,  0.,  1.,  0.,
        -0., -0., -0.,  0.,  0.,  0., -0.,  0.],
       [-0., -1.,  0., -1., -1.,  0., -0.,  0., -0., -1., -0., -0., -1.,
        -0., -0., -0., -1., -1.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,
         1., -0.,  0., -0., -0., -0.,  1., -0., -0.,  1.,  0., -0.,  0.,
        -0.,  1., -0.,  0.,  0.,  0.,  1., -0.],
       [ 1., -0.,  0., -0., -0.,  0., -0.,  0., -0.,  0., -1., -0., -0.,
         0., -0.,  1.,  0., -0., -0.,  0., -0.,  0.,  0.,  0., -0., -1.,
         0., -0., -0.,  0.,  0., -1., -0., -0., -1.,  0., -0.,  0.,  0.,
         0.,  0.,  0.,  0., -1.,  0.,  0., -1.],
       [-0., -0., -0.,  0., -0., -1., -0., -0.,  0., -0.,  0., -0.,  0.,
        -0., -0., -0.,  0., -0., -1., -0.,  0., -0.,  0.,  0.,  0.,  0.,
        -0.,  1.,  1.,  0., -0.,  0.,  0., -0., -1.,  0.,  1., -0., -1.,
         0.,  0., -0., -1.,  0.,  0., -0., -1.],
       [-0., -0., -0.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,
        -1.,  1., -0., -1.,  1.,  0., -1.,  0., -1.,  0.,  1.,  0.,  1.,
         0.,  1.,  0.,  0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,
         0.,  0.,  0., -0.,  0., -0., -0.,  0.],
       [ 0.,  1., -0., -0.,  1.,  1., -0., -0., -1., -0.,  0., -0.,  1.,
        -1., -0.,  0., -0.,  1., -0.,  0., -0., -0.,  0., -0., -0., -0.,
         0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0., -0., -0.,
         0., -1., -0., -0.,  0.,  0.,  0.,  0.],
       [-1., -0., -0.,  0.,  1.,  0., -0.,  0., -0.,  1.,  1.,  0.,  0.,
        -0., -1., -0., -0., -0.,  0., -0.,  1.,  0.,  1.,  0.,  0., -0.,
        -1.,  0., -0.,  1.,  0., -0., -0.,  0., -0.,  0.,  0., -1.,  0.,
         0., -0.,  0., -0.,  1., -0.,  0.,  1.],
       [ 0., -0.,  0., -0., -0., -0., -1., -1.,  1., -0.,  1., -1.,  0.,
        -0.,  0.,  0., -0., -0., -0.,  0., -0., -1.,  1., -0., -0., -0.,
        -1.,  0.,  0.,  1., -1.,  0., -0.,  0., -0., -0., -0.,  0., -0.,
         0.,  0., -0., -0.,  1.,  1.,  0.,  1.]], dtype=np.float32))

In [ ]:
#@title Verification
n = 2
m = 5
p = 6
rank = 47

verify_tensor_decomposition(decomposition_256, n, m, p, rank)

## Rank-23 decomposition of <3,3,3> over Z

In [ ]:
#@title Data
decomposition_333 = (np.array([[ 1.,  0., -0., -0., -1.,  0.,  0., -0., -0.,  1., -0.,  0.,  0.,
        -0., -0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0.],
       [-1., -0.,  0.,  0., -0.,  0.,  0.,  0., -1.,  0., -0.,  1., -0.,
         0.,  0.,  0., -0., -0., -1., -0.,  0.,  0.,  1.],
       [ 0., -0., -0., -0.,  0.,  0., -1., -0.,  0., -0.,  1.,  1.,  0.,
         0., -0., -0.,  1.,  0.,  0.,  1.,  0., -0., -0.],
       [ 0., -0.,  1.,  1.,  0., -0., -0., -0.,  0., -0.,  1., -0., -1.,
         0.,  0., -0.,  0.,  0., -0., -0., -1., -0., -0.],
       [-0.,  1.,  0., -1.,  0., -1., -0., -1.,  0., -0., -0.,  1., -0.,
        -0., -0., -0.,  0.,  1., -0.,  1., -0., -0.,  0.],
       [ 0.,  1., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  1.,  1., -0.,
        -0.,  0.,  0., -0., -0., -0.,  1.,  0.,  1., -0.],
       [ 1., -0., -1.,  0.,  0.,  0., -0.,  1.,  0., -0.,  0., -0.,  0.,
        -0.,  1., -0.,  0.,  0., -0.,  0., -0.,  0., -1.],
       [-1.,  0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,
        -1., -1., -0.,  0.,  0., -1.,  0., -0.,  0.,  1.],
       [ 0.,  0.,  0.,  0.,  0., -1.,  0.,  0., -0.,  0., -0., -0., -1.,
        -1.,  0.,  1., -0., -0., -1., -0.,  0., -1., -0.]], dtype=np.float32), np.array([[ 0., -0., -1., -1.,  1., -0., -0., -1., -0.,  0., -0., -0.,  0.,
         0.,  0.,  0., -0.,  1., -0., -0.,  1., -0.,  0.],
       [-0.,  0., -1.,  0., -0., -0., -0.,  0., -0.,  1., -0., -0.,  0.,
         0.,  0., -0., -0., -0.,  0., -0.,  1., -0., -0.],
       [ 1.,  0., -1.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,  1.,
         0.,  0.,  1., -0., -0., -1.,  0., -0., -0.,  1.],
       [ 0., -0.,  0.,  0., -0.,  1.,  0., -0., -0.,  0.,  1.,  1., -1.,
        -1.,  0.,  0.,  0.,  1.,  0.,  1.,  1.,  0., -0.],
       [ 0.,  0.,  0., -0.,  0.,  1., -0.,  1.,  1., -0.,  1.,  1., -1.,
        -1., -1., -0., -0., -0.,  0.,  1.,  1., -0.,  1.],
       [-0.,  0.,  0., -0., -0.,  1.,  0.,  1., -0.,  0., -0., -0.,  0.,
        -1., -1.,  1., -0., -0., -1., -0., -0., -0.,  1.],
       [ 0., -1., -0., -0.,  0., -1.,  0., -0., -0.,  0., -1.,  0.,  1.,
        -0.,  0., -0.,  0., -1., -0., -1., -1.,  0., -0.],
       [ 0.,  0., -0.,  0.,  0.,  0., -1.,  0.,  0.,  0., -1.,  0.,  1.,
        -0., -0.,  0., -0.,  0.,  0.,  0., -1., -1.,  0.],
       [ 0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,
        -0., -0., -1., -1.,  0., -0.,  0.,  0., -1.,  0.]], dtype=np.float32), np.array([[-0.,  1.,  0., -0., -1.,  0.,  0., -0.,  1.,  0., -0.,  1., -0.,
         0.,  0., -0., -0.,  0., -0., -1., -0., -0., -0.],
       [-0., -1., -0., -1., -0., -0., -0.,  0.,  0.,  0., -0.,  0., -0.,
        -0.,  0., -0., -0.,  1.,  0.,  0., -0., -0., -0.],
       [-0.,  0.,  0.,  0., -0.,  1.,  0., -1.,  0., -0.,  0.,  0., -0.,
         1., -1., -0., -0.,  1., -0., -0., -0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0., -0.,  1.,  0., -1.,  1., -0.,  0., -0.,
         0., -0., -0., -0., -0., -0., -0., -0., -0.,  0.],
       [ 0., -0., -0.,  1., -0., -0., -1., -0., -0.,  0., -1.,  0., -0.,
        -0., -0., -0., -0., -1.,  0.,  1., -1., -0., -0.],
       [-0.,  0.,  1., -0.,  0., -1.,  0.,  1.,  1.,  0.,  0.,  0., -1.,
         0.,  0., -0., -0., -1., -1., -0., -1.,  0.,  1.],
       [ 1.,  0.,  0., -0.,  0., -0.,  0.,  0.,  1.,  0.,  0.,  0., -0.,
         0., -1., -0., -1., -0.,  0., -0., -0., -0.,  1.],
       [ 0., -0., -0.,  0.,  0., -1.,  1., -0.,  0.,  0.,  1.,  0., -1.,
        -0., -0., -1., -0., -0., -0., -1., -0., -1.,  0.],
       [ 0.,  0.,  0.,  0.,  0., -0., -0.,  0., -1.,  0., -0., -0., -0.,
         0.,  1., -1., -0., -0.,  1.,  0., -0.,  0., -1.]], dtype=np.float32))

In [ ]:
#@title Verification
n = 3
m = 3
p = 3
rank = 23

verify_tensor_decomposition(decomposition_333, n, m, p, rank)

## Rank-54 decomposition of <3,4,6> over 0.5*Z


In [ ]:
#@title Data

decomposition_346 = (np.array([[ 0.5,  0. ,  0. ,  0. , -1. ,  1. ,  0. ,  1. , -1. ,  0. ,  0. ,
         0. ,  0.5,  1. , -0.5,  0. ,  0.5,  0. ,  0. ,  0. ,  0.5,  0. ,
         0. ,  0. ,  0. ,  1. , -1. ,  1. ,  0.5, -0.5,  0. ,  0. ,  0.5,
         0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         1. ,  0. ,  0.5, -1. ,  1. , -1. ,  0. ,  1. ,  0. , -1. ],
       [ 0.5,  0. ,  0. ,  0. ,  1. ,  1. ,  0. ,  1. ,  1. ,  0. ,  0. ,
         0. ,  0.5, -1. ,  0.5,  0. ,  0.5,  0. ,  0. ,  0. ,  0.5,  0. ,
         0. ,  0. ,  0. ,  1. ,  1. , -1. ,  0.5, -0.5,  0. ,  0. , -0.5,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
        -1. ,  0. ,  0.5,  0. ,  0. , -1. ,  0. ,  1. ,  0. ,  0. ],
       [-0.5,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. , -0.5,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. , -0.5,  0. , -1. ,  0. , -1. ,  0. ,  0. ,  1. ,  0. ,  0. ,
         0. ,  0. ,  0. , -0.5,  0. ,  0.5,  0. ,  0. ,  0. ,  0. ,  0.5,
        -1. ,  0. , -0.5,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  1. ,  0. ,  0. , -1. ,  1. ,  0. , -1. ,  1. ,  0. ,  0. ,
         0. ,  0.5, -1. , -0.5,  0. ,  0. ,  0. ,  0. ,  0. , -0.5,  0. ,
         0. ,  0.5,  0. ,  0. ,  1. ,  0. , -0.5, -0.5, -1. ,  0. , -0.5,
         0. ,  0. ,  0. ,  0.5,  0. , -0.5,  0. ,  0. ,  0. ,  0. ,  0.5,
         0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  1. ,  0. ,  0. ],
       [-0.5,  0. , -1. , -0.5,  0. ,  0. , -1. ,  0. ,  0. , -1. , -1. ,
        -1. ,  0. ,  0. ,  0.5, -1. , -0.5,  0. ,  0. , -0.5,  0.5,  0. ,
         0. , -0.5,  1. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,
        -0.5,  0. ,  1. ,  0. , -0.5,  0.5,  0. , -1. , -0.5,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0.5,  0. , -1. ,  0. ],
       [-0.5,  0. ,  0. , -0.5,  0. ,  0. ,  1. ,  0. ,  0. ,  1. , -1. ,
         0. ,  0. ,  0. , -0.5,  1. ,  0.5,  0. ,  0. , -0.5, -0.5,  0. ,
         0. ,  0.5,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0.5,  0. ,  1. ,  0. , -0.5, -0.5,  0. ,  1. ,  0.5,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0.5,  0. ,  0. ,  0. ],
       [ 0.5,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  1. ,  0.5,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0.5,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  1. ,  0. ,
         0. ,  0. , -1. ,  0.5,  0. ,  0. ,  1. ,  0. ,  0. ,  0. , -0.5,
         0. ,  0. ,  0.5,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , -0.5,  0. ,  0. ,  1. ,  1. ,  0. ,  1. , -1. ,
         0. ,  0. ,  0. ,  0.5,  0. ,  0. ,  0. ,  0. , -0.5,  0.5,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,
         0.5,  0. ,  0. , -0.5,  0.5,  0.5,  1. , -1. ,  0.5,  0. , -0.5,
         0. ,  0. ,  0.5,  0. ,  0. , -1. ,  0.5,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  1. ,  0.5, -1. ,  1. ,  0. ,  0. , -1. ,  0. ,  0. ,
         0.5,  0.5,  0. ,  0. ,  0. ,  0. , -0.5, -1. ,  0.5,  0. ,  1. ,
         0. ,  0. , -0.5,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0.5,  0. , -1. ,  0. ,  0.5,  0. ,  0. ,  0. ,  0.5,  1. ,  0. ,
         1. ,  0. ,  0. , -1. ,  0.5,  0. ,  0. ,  0. ,  0. , -0.5],
       [ 0. ,  0. ,  0. ,  0.5,  1. , -1. ,  0. ,  0. ,  1. ,  0. ,  0. ,
         0.5,  0.5,  0. ,  0. ,  0. ,  0. , -0.5, -1. ,  0.5,  0. , -1. ,
         0. ,  0. , -0.5,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
        -0.5,  0. , -1. ,  0. ,  0.5,  0. ,  0. ,  0. ,  0.5, -1. ,  0. ,
        -1. ,  0. ,  0. ,  0. , -0.5,  0. ,  0. ,  0. ,  0. ,  0.5],
       [ 0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. , -1. ,
        -0.5,  0. , -1. ,  0. ,  0. ,  0. ,  0.5,  0. , -0.5,  0. , -1. ,
         0.5,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,
         0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
        -1. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0.5],
       [ 0. ,  1. ,  0. ,  0.5, -1. ,  1. ,  0. ,  0. ,  1. , -1. ,  1. ,
         0. ,  0.5, -1. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,
        -0.5,  0. ,  0.5,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,
        -0.5,  0. ,  0. ,  0. , -0.5,  0. ,  0. ,  0. , -0.5,  1. ,  0. ,
         0. ,  1. ,  0. ,  0. ,  0.5,  0. ,  0. ,  1. ,  0. ,  0. ]],
      dtype=np.float32), np.array([[ 0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0.5, -1. ,
         0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  1. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         1. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. , -1. ],
       [ 0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,
        -1. ,  1. , -0.5,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0.5,  0. ,  0. ],
       [ 0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. , -1. ,  0. ],
       [-1. ,  0. , -1. ,  0. ,  0. , -1. ,  0.5,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  1. , -1. ,  0. ,  0. ,  1. ,  0. , -1. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. , -1. ,  0.5, -1. ,
         0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. , -0.5,  1. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
        -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ],
       [ 0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,
         0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0.5,  0. ,
         0. ,  0. ,  0. ,  1. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [-1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. , -0.5,  0. ,
         0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  1. ,  1. , -1. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0.5,  0. ,
        -1. ,  1. , -0.5,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. , -1. ,
         0. ,  0. ,  1. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  1. ],
       [ 0. , -0.5,  1. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  1. ,  0.5,  0. ,  0. ,  0. ,  1. ,  0. ,  1. ,  0. ,  0. ,
         0. ,  0. ,  1. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0.5,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  1. , -1. ,  0. , -0.5,  0. ,  0. ,  0. ,
        -0.5,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0.5,  0. ,  0. ],
       [ 0. ,  0. , -1. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
        -1. ,  0. , -0.5,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  1. ,  0. ,
         0. , -1. , -1. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. , -1. ,
         0. ,  0. ,  0. ,  1. , -1. ,  0. ,  0. ,  0.5,  0. ,  0. ,  0. ,
         0. , -1. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. , -1. ,  0. ],
       [-1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,
        -1. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. , -1. , -1. ,  0.5,  1. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  1. ,
         1. ,  0. ,  0. ,  0. ,  1. ,  0. , -1. ,  0. ,  0. , -0.5,  0. ,
         1. ,  0. ,  0.5,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  1. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. , -1. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
        -1. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 1. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. , -1. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0.5,  0. ,
         0. ,  0. ,  0.5,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. , -1. ,
         0. ,  0. , -1. ,  0. ,  0. ,  1. ,  1. ,  0. ,  0. ,  0. ],
       [ 0. ,  0.5,  0. ,  1. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,
        -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,
         0. ,  0. ,  1. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0.5,  0. ,  0. ,
         0. ,  0. , -1. ,  0. ,  1. , -1. ,  0. , -0.5,  0. ,  0. ,  0. ,
        -0.5,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         1. ,  0. , -0.5,  1. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,
         0. , -1. , -1. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. , -1. ,
         0. ,  0. ,  1. , -1. , -1. ,  2. ,  0. ,  0.5,  0. ,  0. ,  0. ,
         0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. , -1. ,  0. ,  0. , -1. ,  0.5,  0. ,  0. , -1. ,  0. ,
         0. ,  0. ,  0. ,  0. , -1. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  1. , -1. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  1. ,  0. ,
        -1. ,  0. ,  0. , -1. , -1. ,  0. ,  0. ,  0. ,  0. ,  0.5, -1. ,
         0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  1. ,  1. ,  0. ,  0. ,  1. ,  1. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,
        -1. ,  0. ,  0. ,  0. , -1. ,  0. ,  1. ,  0. ,  0. , -0.5,  0. ,
         0. ,  0. , -0.5,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,
         1. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  1. ],
       [ 0. ,  1. , -1. ,  0. , -1. , -1. ,  0. ,  0. ,  0. , -1. ,  0. ,
         0. , -1. ,  0. ,  0. , -1. ,  2. ,  0. , -1. ,  0. ,  0. ,  0. ,
         0. ,  0. , -1. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,
        -1. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  1. ,  0.5,  0. ,
         0. ,  0. ,  0. ,  1. , -1. ,  0. ,  0. , -1. ,  0. ,  0. ],
       [-1. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  1. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -0.5,  0. ,
         0. ,  0. ,  0.5,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. , -1. ,
         0. ,  0. ,  1. ,  0. ,  0. , -1. , -1. ,  0. ,  0. ,  0. ],
       [ 0. ,  0.5,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,
         1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,
         0. ,  0. , -1. ,  0. , -1. ,  0. ,  0. ,  0. , -0.5,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. , -1. ,  1. ,  0. ,  0.5,  0. ,  0. ,  0. ,
         0.5,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
        -1. ,  0. , -0.5,  1. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,
         0. , -1. ,  1. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. , -1. ,
         0. ,  0. ,  0. ,  1. ,  1. ,  0. ,  0. , -0.5,  0. ,  0. ,  0. ,
         0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. , -1. ,  0. ,  0. , -1. , -0.5,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. , -1. , -1. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,
        -1. ,  0. ,  0. ,  1. , -1. ,  0. ,  0. ,  0. ,  0. ,  0.5, -1. ,
         0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  1. ,  1. ,  0. ,  0. ,  1. , -1. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,
         1. ,  0. ,  0. ,  0. ,  1. ,  0. ,  1. ,  0. ,  0. ,  0.5,  0. ,
         0. ,  0. , -0.5,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  1. ],
       [ 0. ,  0. , -1. ,  0. ,  1. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,
         0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
        -1. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. , -1. ,  0.5,  0. ,
         0. ,  0. ,  0. , -1. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ]],
      dtype=np.float32), np.array([[ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0.5,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. , -1. ,  0. ,  0. ,  0. ,
         0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0.5,  0. ,  0. ,  0. , -1. ,
         0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0.5,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0.5,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0.5, -1. ,  0. ,  1. ,  0. ],
       [ 0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0.5,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -0.5,  1. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0.5,  0. , -1. ,  1. ,  0. , -1. ,  0. ,  0. ,  1. ],
       [ 0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  1. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. , -0.5,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,
         1. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. , -1. ,  0. ,  0. ,
         0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,
         0. ,  0. ,  1. , -1. ,  0. ,  1. ,  0.5,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0.5,  0. ,  0. ,
         0. ,  0. ,  1. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,
        -1. ,  0. ,  0. ,  0. ,  0.5,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0.5,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  1. ,
         0. ,  1. ,  0. ,  0. ,  0. ,  1. ,  0. ,  1. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,
         0. ,  0. ,  0. , -1. ,  0. ,  1. ,  0. ,  1. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ],
       [ 0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0.5,  0. ,  0. ,
        -1. ,  0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
        -1. ,  0. ,  1. ,  0. ,  0.5,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. , -1. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. , -1. ,  0. , -1. ,  0. ,  0.5,  0. ,  1. ,  0. ,  0. , -1. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [ 1. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. , -1. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  1. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ,  0. , -0.5, -1. ,  0. ,  0. , -1. ,  0. ,
         0. , -1. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. , -0.5,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,
         1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0.5,
         0. ,  0. ,  0. ,  0. ,  0.5,  0. , -1. ,  0. ,  1. ,  0. ,  0. ,
         0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
        -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ],
       [ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0.5,  0. ,  1. , -0.5,
         0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,
        -1. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. , -1. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0.5,  0. ,  0. ,  1. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -0.5,  0. ,  0. , -0.5,
        -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. , -1. ,  1. ,  0. ,  0. ,  0. ,  0. ,  1. ],
       [ 0. ,  0. ,  0. ,  0. , -0.5, -0.5,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  1. , -1. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,
         0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ,  0. ,  0. ],
       [-1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0.5,  0. ,  0. ,  0. ,
         0. ,  0. , -1. ,  1. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  1. , -1. ,  0. ,  0. ,
         0. ,  0. , -1. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ,  0. ,  0.5,  0. ,  0. ,  0. ,  0. ,  0. ,
         0. ,  1. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0.5,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ,  0. ,
         0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  0. , -1. ,  0. ,  0. ]],
      dtype=np.float32))

In [ ]:
#@title Verification
n = 3
m = 4
p = 6
rank = 54

verify_tensor_decomposition(decomposition_346, n, m, p, rank)

## Rank-63 decomposition of <3,4,7> over 0.5*Z[i]

In [ ]:
#@title Data

decomposition_347 = (np.array([[ 0. -0.j , -1. +0.j ,  0. +0.j ,  0. +1.j ,  0. +0.j , -0. +0.j ,
        -0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. -0.j , -0. +0.j ,  1. +0.j ,  0. +0.j ,  0. -1.j ,  0. +0.j ,
         1. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  1. +0.j , -0. +1.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j , -1. -0.j ,  0. +0.j ,  0. -0.j ,  0. -1.j , -0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. -1.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  1. +0.j ,
         0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,
        -0. +1.j ,  0. -0.j , -0. +0.j , -0. +0.j ,  1. +0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.j ,  0. +0.j ],
       [ 0. +0.j , -0.5-0.j , -0. +0.j ,  0. +0.j ,  0. +0.5j,  0. +0.j ,
         0. -0.j ,  0. -0.j ,  0. +0.j ,  0.5+0.j , -0. +0.j ,  0. -0.j ,
        -0. +0.j ,  0.5+0.j , -0. +0.j ,  0. +0.j ,  0. +0.5j, -0. +0.5j,
        -0.5-0.j , -0. +0.j ,  0. +0.j ,  0.5+0.j , -0.5+0.j ,  0. -0.5j,
         0. +0.j , -0. +0.5j,  0. +0.j , -0. +0.j ,  0. -0.j ,  0.5+0.j ,
        -0.5+0.j ,  0.5+0.j , -0. +0.j , -0. +0.j ,  0. +0.5j, -0.5+0.j ,
         0. -0.5j, -0.5+0.j ,  0. +0.j , -1. +0.j , -0. +0.5j,  0. -0.5j,
        -0.5-0.j ,  0.5+0.j ,  0. -0.5j,  0. +0.j ,  0. +0.j , -0.5+0.j ,
         0.5+0.j , -0.5-0.j ,  0. +0.j , -0. +0.j ,  0. -0.5j, -0. +0.5j,
         0. -0.5j, -0.5+0.j ,  0. +0.j ,  0. +0.j ,  0.5+0.j ,  0.5+0.j ,
         0. +0.j ,  0.5+0.j ,  0. +0.j ],
       [ 0. -0.j , -0. +0.j ,  0. -0.j , -1. -0.j ,  0. -0.j , -0. +0.j ,
        -0. +0.j , -0. +0.j ,  0. -0.j ,  0. -1.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. -1.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  1. +0.j ,
         0. +0.j ,  0. -0.j ,  0. -1.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  1. +0.j ,  0. -0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,
         0. -1.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +1.j ,
         1. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,
        -0. +1.j , -0. +0.j , -1. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -1. +0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -1.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ],
       [-0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,  1. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +1.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j ,  0. -1.j ,  0. -1.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,  0. -1.j ,
         0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,
        -0. +0.j , -0. +1.j , -0. +0.j ,  0. -0.j , -0. +0.j ,  1. +0.j ,
         0. -0.j ,  0. -1.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,
         0. -1.j , -0. +1.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,  1. +0.j ,
         0. -0.j , -0. +1.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +1.j ,  0. +0.j ],
       [-1. -0.j ,  0. +1.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. -1.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. -1.j , -0. +0.j , -1. -0.j ,  0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. -1.j ,  0. +0.j ,  0. -0.j ,  0. -1.j ,  0. -0.j ,
        -1. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  0. -1.j ,
        -0. +0.j , -0. +1.j ,  0. +0.j , -0.5-0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
        -0. +0.j ,  0. -0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -1.j , -0. +0.j ,  0. +0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. +0.5j,  0. -0.5j, -0. +0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.5j, -0.5+0.j ,  0. +0.j ,  0. +0.j ,  0. -0.5j,
        -0. +0.j ,  0. -0.5j, -0. +0.j ,  0.5+0.j ,  0. +0.j , -0. +0.j ,
        -0. +0.5j,  0. +0.5j,  0. +0.j ,  0. +0.j , -0. +0.5j,  0. +0.j ,
         0.5+0.j ,  0.5+0.j , -0. +0.j ,  0.5+0.j , -0. +0.j ,  0. -0.5j,
         0. +0.j , -0. +0.5j,  0.5+0.j , -0. +0.j , -0. +0.j ,  0. -0.5j,
        -0.5+0.j ,  0. -0.5j, -0.5+0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0.5+0.j , -0.5+0.j ,  0. +0.j ,
         0. -0.5j,  0. +0.j , -0. +0.5j,  0. -0.5j, -0.5+0.j ,  0. -0.j ,
        -0.5-0.j ,  0. +0.5j,  0. +0.j ,  0. +0.5j, -0. +0.j , -0. +0.j ,
         0. -0.j ,  0. -0.5j,  1. +0.j ],
       [ 0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j , -1. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
        -1. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -1.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j , -0. +1.j ,  0. -0.5j,  0. +0.j ,  1. +0.j ,
         0. -1.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +1.j ,  0. +0.j ,
         1. +0.j ,  0. +0.j ,  1. +0.j ,  0. +0.j ,  0. +1.j ,  0. -0.j ,
        -0. +0.j ,  0. +0.j , -0.5+0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.j ,  0. +0.j ],
       [ 0. -1.j ,  0. +0.j , -1. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j , -0. +1.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j , -0. +1.j , -0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,
        -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j ,  1. +0.j , -0. +1.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j ,  0. -1.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j , -0. +0.j ,  1. +0.j , -0. +0.j ,  0. +0.j ,
         0. -1.j ,  1. +0.j , -0.5-0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  1. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  1. +0.j ,
         1. +0.j ,  1. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. -0.j ,  0. -1.j ,  0. -1.j ,  0. +0.j ,
         0. -0.j ,  1. +0.j ,  0. -0.j , -0. +0.j , -0. +0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j , -0. +1.j ,  0. +0.j , -0. +0.j ,
         0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. +1.j ,  0. -0.j ,  1. +0.j ,
         0. -0.j , -1. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  1. +0.j ,  1. +0.j ,
         1. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j , -0. +0.j ,  0.5+0.j ,  0. -0.j ,  0. +0.5j,  0. -0.j ,
         0.5+0.j ,  0.5+0.j ,  0. +0.5j,  0.5+0.j ,  0.5+0.j ,  0.5+0.j ,
         0. -0.5j, -0. +0.j , -0. +0.j ,  0. -0.5j,  0. -0.5j,  0. +0.5j,
        -0. +0.j , -0.5+0.j ,  0. +0.j ,  0.5+0.j ,  0. +0.j ,  0. -0.5j,
         0. +0.5j,  0. -0.j , -0. +1.j , -0. +0.5j,  0. +0.j ,  0. +0.j ,
        -0.5-0.j , -0. +0.j ,  0. +0.5j,  0. -0.j , -0. +0.5j,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. -0.5j,  0. +0.j ,  0. -0.j ,  0. -0.5j,
         0. +0.j ,  0. -0.j , -0. +0.5j,  0. +0.5j,  0. +0.5j, -0.5-0.j ,
         0. +0.j , -0.5-0.j , -0.5+0.j ,  0.5+0.j ,  0. +0.j ,  0. -0.5j,
         0. +0.j ,  0. +0.j ,  0. +0.j , -0.5-0.j ,  0.5+0.j ,  0.5+0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +1.j ,  0. -1.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,  1. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +1.j , -0. +0.j , -0. +0.j ,
        -1. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  1. +0.j ,  0. +0.j ,
         0. -1.j ,  0. +0.j , -1. -0.j , -0. +0.j , -1. +0.j , -0. +0.j ,
         0. -0.j ,  0. -0.j ,  1. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j , -1. -0.j ,  0. +0.j , -1. -0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,
        -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. +1.j ,  0. -0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. +0.j ,  0. -1.j ,  0. -0.j ,  1. +0.j , -0. +1.j ,
         0. +0.j ,  0. +0.j , -1. -0.j ,  0. +1.j ,  0. -0.j ,  0. +0.j ,
         1. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  1. +0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  1. +0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  1. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. -1.j , -0. +1.j , -0. +0.j ,  1. +0.j ,
        -0. +0.j ,  0. +0.j , -0. +0.j ,  0. -1.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ]]), np.array([[ 0. +0.j , -0. +0.j ,  0. -0.5j,  0. +0.j ,  0.5+0.j , -0. +1.j ,
         0. +0.j ,  0. +0.j , -0. +0.5j,  0. +0.j , -0. +0.j ,  0. -0.j ,
         0. +1.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. -0.j , -1. -0.j , -0. +0.j ,  0. -0.j , -0. +0.j , -1. -0.j ,
        -0. +0.j ,  0. +0.j ,  0. -0.5j,  0. +0.j , -0. +0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.5j,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -1. +0.j ,
         0. +0.j ,  0. -0.j ,  0. -0.j , -0.5-0.j ,  0. -0.j ,  0. +0.5j,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. -1.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ],
       [-0. +0.j ,  0.5+0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. -0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.5j,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. -0.j ,  0. -0.j , -0. +0.5j,  0. +0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j , -0. +0.j ,
         0. +0.j ,  0. -0.5j,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j ,  0.5+0.j ,  1. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  0.5+0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j , -0.5-0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. -0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  1. +0.j , -0.5+0.j ,
         0. -0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j , -0.5-0.j ,
         0. -0.j ,  0. -1.j , -0. +0.j ,  0. -0.j , -0. +0.j , -0. +0.j ,
         0. -1.j ,  0. +0.j ,  0.5+0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,
         0. -0.5j, -0. +0.j ,  0.5+0.j ,  0. -0.j , -1. -0.j , -0. +0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.5j,  0. +0.j ,  0. -0.5j,  0. -1.j ,
         0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,  0. -0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,  0. -0.j ,
         0. -1.j , -0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  1. +0.j ,  0. +0.j , -0. +0.j ,  0.5+0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j , -0. +0.j ,  0. -0.j , -0. +0.j , -0. +0.j ,
         0. -0.j , -0. +0.j , -0. +1.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  1. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,
        -0. +0.j , -0.5+0.j ,  0. +0.j ,  0.5+0.j ,  0. -0.j ,  0. -0.5j,
         0. -0.j ,  0. -1.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  1. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.5j,
         0. +0.j ,  0. -0.5j,  0. +0.j ,  0. +0.j , -1. +0.j , -0. +0.j ,
         0. +0.j , -0.5-0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. -1.j , -0. +0.j , -1. +0.j ,  0. -0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
        -0. +0.j ,  0. -1.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0.5+0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j , -0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,
        -0. +0.5j,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.5j,
         0. +0.5j, -0. +0.j ,  0. +0.j ,  0. -0.5j, -0. +0.j , -0. +0.j ,
         0. -1.j ,  0. +0.j ,  0. -0.5j,  0. +0.j , -0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j , -0. +0.j , -0.5-0.j ,  0. -0.j ,
         0. -0.j , -0. +0.j ,  0. -0.j ,  0. +0.j , -0. +1.j ,  0. -1.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j , -0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,
         1. +0.j ,  0. +0.5j,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j ,  0. -0.5j,  0. -0.5j, -0. +0.j ,
        -0. +0.j ,  0.5+0.j ,  0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.5j,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j , -0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0.5+0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j ,  0. -0.j , -0. +0.j , -0.5+0.j ,  0. +0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j ],
       [ 0. +1.j ,  0. -1.j ,  0. +0.j , -0.5-0.j ,  0. +0.j ,  0.5+0.j ,
         0. +0.j ,  1. +0.j ,  0. -0.j ,  0. -0.5j,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.5j,  0.5+0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,
        -0.5-0.j , -0. +1.j , -0.5+0.j ,  0. +0.5j, -1. -0.j ,  0. +0.5j,
        -0. +0.5j, -0. +0.5j,  0. +0.j ,  0.5+0.j , -0.5-0.j ,  0. -0.5j,
         0. +0.j ,  0. +0.j , -0. +0.j ,  0. -1.j ,  0.5+0.j ,  0. -0.j ,
         0. +0.j ,  0. -0.j , -0.5-0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.5j,  0. +0.j ,  0. +1.j ,
         0. +0.5j,  0. -0.5j,  0. +0.5j, -0. +0.j , -0. +0.j , -0. +0.j ,
        -0. +0.5j, -0. +0.j ,  0. +1.j , -0.5-0.j , -0. +1.j ,  0. -0.5j,
        -0. +0.5j, -0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. +0.j ,  0. -1.j ,  0. +0.j ,  1. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j , -0. +1.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. -0.j ,  0. -1.j , -0. +0.j ,  0. -0.j ,  0. -0.j ,
         0. -0.j ,  0. -0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,  0. -1.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j , -1. +0.j ,  0. -0.j , -0. +1.j ,
         0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  1. +0.j ,  0. -0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +1.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. -1.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. -1.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j ,  1. +0.j , -0. +0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j , -1. -0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j , -1. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j , -1. -0.j ,
         0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j , -0. +0.j , -1. -0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  1. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -1.j ,  0. +0.j ,  1. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,
         0. -0.j ,  0. -0.j ,  0. +1.j ,  0. -0.j ,  0. -1.j ,  0. +0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ],
       [-0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -1. +0.j , -0. +0.j ,
         0. -0.j ,  0. -0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  1. +0.j , -0. +0.j , -1. -0.j ,  0. +0.j ,  0. +1.j ,
         0. +0.j ,  0. -0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,  0. -1.j ,
         0. -0.j ,  0. +1.j ,  0. +0.j ,  0. -0.j , -0. +0.j , -0. +0.j ,
         0. +0.j ,  1. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,
        -0. +0.j , -0. +0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j , -0. +0.j , -0. +0.j , -0. +0.j , -1. -0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -1.j ,  0. -0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j , -0. +1.j ,
         0. -1.j , -0. +0.j ,  0. +0.j ,  0. +1.j ,  0. -0.j , -0. +0.j ,
         0. -0.j , -0. +0.j ,  0. +1.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  1. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j , -0. +1.j ,  0. -0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j , -0. +1.j ,  0. +1.j ,  0. +0.j ,
         0. +0.j , -1. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -1.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -1. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j , -1. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,
        -0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,  0. -0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j , -1. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j ,  0. +1.j ,  0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,
        -0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  1. +0.j ],
       [ 0. +0.j ,  0. +0.j , -0.5+0.j ,  0. -0.j ,  0. -0.5j,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0.5+0.j ,  0. +1.j ,  0. +0.j ,  1. +0.j ,
         1. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  1. +0.j ,
         0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
        -0. +0.j , -0. +0.j , -0.5+0.j ,  0. +0.j ,  1. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j , -0.5-0.j ,
         0. +0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. -1.j , -0. +0.5j,  0. +0.j ,  0.5+0.j ,
        -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ],
       [ 0. -0.j ,  0. +0.5j,  0. +0.j ,  1. +0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j , -0.5+0.j ,  0. +0.j ,
         1. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0.5+0.j ,  0. +0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,
         0. -1.j ,  0.5+0.j ,  0. -0.j ,  0. -0.j , -1. -0.j ,  0. -0.j ,
         0. -1.j ,  0. +0.j ,  0. -0.j , -0. +0.5j, -0. +1.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.5j,
        -0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.5j,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ],
       [-0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. +1.j ,  0. -0.5j,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.5j,
        -0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.5j,  0. -0.j ,  0. +0.j ,  0. +0.j ,
        -0.5-0.j , -0. +0.j ,  0. -0.5j, -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j , -0.5-0.j ,  0. -0.j ,  0.5+0.j ,  0. -0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.5j,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
        -0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j , -0. +0.j ,  1. +0.j ,  0. -1.j , -0. +0.j , -0. +0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j , -0. +0.j ,
         0. +1.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,
        -0. +1.j ,  0. +0.5j, -0. +0.j ,  0. -0.5j,  0. -0.j , -0.5+0.j ,
        -0. +0.j , -1. -0.j ,  0. +0.j , -0. +0.j , -0. +0.j , -0. +0.j ,
         0. -1.j , -0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,  0.5+0.j ,
         0. +0.j , -0.5-0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
         0. -0.j , -0. +0.5j,  0. +0.j ],
       [ 0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.5j,
         0. -0.j ,  0. -0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
        -0.5+0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,  0.5+0.j ,
        -0.5+0.j ,  0. +0.j , -0. +0.j ,  0.5+0.j ,  0. -0.j ,  0. +0.j ,
         1. +0.j ,  0. +0.j ,  0.5+0.j , -0. +0.j , -0. +0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.5j,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ],
       [-0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. +1.j ,  0.5+0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  1. +0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j ,  0.5+0.j ,  0.5+0.j ,  1. +0.j ,
         0. +0.j , -0. +0.5j, -0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j , -0. +0.j , -0.5-0.j ,  1. +0.j ,  0. -0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
        -0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.5j,
         0. -0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.5j,  0. -1.j ,
        -0. +1.j , -0. +0.j ,  0. +0.j ],
       [-1. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.5j, -0. +0.j ,  0. +0.5j,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0.5+0.j ,  0. -0.j ,  0. -1.j ,
         0. +0.j , -0.5+0.j ,  0. +0.5j,  0. +0.j , -0. +0.j ,  0. -1.j ,
        -0. +0.5j,  0. +0.j ,  0. -0.5j, -0.5-0.j ,  0. +0.j , -0.5-0.j ,
         0.5+0.j , -0.5-0.j ,  0. -0.j ,  0. -0.5j,  0. -0.5j,  0.5+0.j ,
         1. +0.j ,  0. +0.j ,  0. +1.j , -1. +0.j ,  0. -0.5j,  0. +0.j ,
         1. +0.j , -0. +0.j ,  0. -0.5j,  0. +0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j , -0.5-0.j ,  0. -0.j ,  0. -0.j ,
        -0.5-0.j ,  0.5+0.j , -0.5+0.j ,  0. +0.j ,  0. +1.j ,  0. +0.j ,
        -0.5-0.j ,  0. +0.j , -1. -0.j ,  0. -0.5j,  0. -0.j , -0.5+0.j ,
         0.5+0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. +0.j ,  0.5+0.j , -0. +0.j ,  0. +0.5j,  0. +0.j ,
         0. +0.j , -0. +0.j , -0.5+0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,
        -1. -0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. -0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0.5+0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j , -0.5+0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.5j,  0. +0.j , -0.5+0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. +0.5j,  0. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,  0. -0.j ,
         0. +0.j , -0. +0.j , -1. -0.j , -0. +0.j , -0.5-0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0.5+0.j ,  0. -1.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j , -0. +0.j ,  0. -0.j ,
         0. -0.j ,  0.5+0.j ,  0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.5j,  0. +1.j ,  1. +0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.5j,
        -0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -1.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.5j,  0. +0.j ,
         0. +0.j ,  0. -1.j ,  0. +0.j ],
       [-0. +0.j ,  0. +0.j ,  0. -1.j ,  0. +0.j ,  1. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -1.j , -0. +0.5j,
         0. -0.j ,  0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.5j,
         0. -0.j ,  0. +0.j ,  0. -0.j ,  1. +0.j ,  0. -0.j , -0. +0.j ,
         0. +0.j , -0. +0.j ,  0. -0.5j,  0. +0.j , -0. +1.j ,  0. +0.j ,
        -0.5+0.j , -0. +0.j ,  0. -0.5j,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +1.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0.5+0.j ,  0. +0.j , -0.5-0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
        -0. +0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j , -0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.5j,  0. -0.j ,
         0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. -0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.5j,  0. -0.j , -0. +0.5j,  0. +0.j ,  0.5+0.j ,
        -0. +0.j ,  1. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0.5-0.j ,
        -0. +0.j , -0.5-0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. -0.5j,  0. +0.j ],
       [ 0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j ,  1. +0.j , -0. +0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.5j,
        -0. +0.j ,  0. +0.j ,  0. -1.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j , -1. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j , -0. +0.j ,
         0.5+0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -0.5-0.j ,
         0.5+0.j ,  0. -0.j ,  0. +0.j , -0.5-0.j ,  0. +0.j , -0. +1.j ,
        -1. -0.j ,  0. +0.j , -0.5+0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. +0.5j, -0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  1. +0.j ,  0. +0.j ],
       [ 0. +0.j , -0. +0.j , -1. +0.j ,  0. +0.j ,  0. -1.j ,  1. +0.j ,
        -0. +1.j ,  0.5+0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
        -0. +0.j ,  0. -0.j ,  0. -0.j ,  0.5+0.j ,  0.5+0.j ,  0. +0.j ,
        -0. +0.j , -0. +0.5j,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. -0.j , -0.5-0.j ,  0. -0.j , -0. +0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +1.j ,  0. -0.j , -0. +0.5j,
         0. +0.j ,  0. -1.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.5j,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ],
       [ 1. +0.j , -0. +0.j ,  0. +1.j ,  0. -0.5j, -1. -0.j ,  0. -0.5j,
        -0. +0.j ,  0. +0.j ,  0. +0.j ,  0.5+0.j ,  0. +0.j ,  0. -0.j ,
        -0. +0.j ,  0.5+0.j ,  0. -0.5j,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. -0.5j,  0. +0.j ,  0. -0.5j, -0.5-0.j , -0. +0.j ,  0.5+0.j ,
        -0.5-0.j , -0.5-0.j ,  0. +0.j , -0. +0.5j,  0. -0.5j, -0.5+0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j ,  1. +0.j , -0. +0.5j,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. -0.5j,  0. +0.j ,  0. +0.j , -0. +1.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0.5+0.j ,  0. +0.j , -0. +0.j ,
        -0.5+0.j , -0.5-0.j , -0.5-0.j ,  1. +0.j , -0. +0.j , -0. +0.j ,
         0.5+0.j ,  0. +1.j , -1. -0.j ,  0. +0.5j, -0. +0.j ,  0.5+0.j ,
        -0.5+0.j ,  1. +0.j ,  0. +0.j ]]), np.array([[ 0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. -0.j ,  1. +0.j ,  0. -0.j ,  0. +0.j ,
         1. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j , -0. +1.j ,
         0. -0.j ,  0. -1.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,  0. -0.j , -0. +0.j ,
         0. -0.j , -1. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j , -1. +0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
        -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j , -0. +0.j , -1. +0.j ,
         0. -1.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  1. +0.j ,  0. +0.j ],
       [-0.5+0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,
        -0. +0.j , -0. +0.j , -0. +1.j ,  0. +0.j , -0. +0.j , -0. +0.j ,
        -0. +1.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. -0.j , -0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +1.j ,  0. -1.j ,  0. +0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j , -0. +1.j , -1. -0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ],
       [-0. +0.j , -0. +0.j , -0. +1.j ,  0. +0.j ,  0. +0.j ,  0. -1.j ,
         0. +0.j , -0. +0.j , -1. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
        -1. +0.j ,  0. -0.j , -0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  1. +0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. -0.j , -0. +0.j , -1. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j ,  0. -1.j ,  0. -0.j , -0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j , -1. +0.j , -0. +0.j , -1. +0.j ,  0. -0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j , -1. -0.j , -0. +1.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j , -0. +0.j , -1. +0.j ,
         0. -0.j ,  0. +1.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +1.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0.5+0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
        -1. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,  1. +0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
        -0. +0.j ,  1. +0.j ,  0. -0.j , -1. +0.j ,  0. -0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  1. +0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,  0. -0.j ,
        -1. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. -0.j , -0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.j , -0. +0.j , -1. -0.j ,  1. +0.j , -0. +0.j ,
         0. +0.j , -1. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j , -0. +1.j ,
        -0. +1.j , -0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  1. +0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. -1.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  1. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. -1.j , -0. +0.j , -0. +0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j ],
       [ 0. -0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  1. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
        -1. -0.j ,  0. -0.j ,  0. -0.j , -0. +1.j , -0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,
         0. -1.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -1.j ,  0. -1.j ,
         1. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. -0.j ,  1. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,
         0. -1.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
        -0. +0.j ,  0. -0.j ,  0. +0.j ],
       [ 0. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +1.j ,  0. -0.j ,
         0. +0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
         0. -1.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j , -0. +0.j ,  1. +0.j ,  0. -1.j , -0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j , -1. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. -1.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  1. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. -0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j , -1. +0.j , -1. -0.j ,
         0. +0.j ,  0. -0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,
         0. +0.j , -0. +0.j ,  0. +0.j ,  0. -1.j ,  0. -1.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  1. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. -1.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,
         0. -0.j ,  0. -0.j ,  0. -0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,
         0. +1.j , -0. +0.j ,  0. +0.j ],
       [-0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,
        -0. +0.j , -0. +0.j ,  0. -1.j ,  0. -0.j , -0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j , -0. +1.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. -0.j ,  0. -1.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,
         0. +0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,
         0. -0.j , -1. -0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. +1.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j , -0. +0.j ,  0. -0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,
         0. -1.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j , -0. +0.j ,
         0. +0.j ,  1. +0.j ,  0. +0.j ],
       [-0.5-0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
        -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. +1.j ,
         0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j , -0. +1.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
         0. -0.j ,  1. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. +1.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j , -1. +0.j , -0. +1.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. +0.j ,  0. +1.j ,  0. -0.j , -0. +1.j ,  0. +0.j ,
         0. -0.j , -0. +0.j , -1. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j , -0. +0.j ,  0. -0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  1. +0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. -0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j , -0. +0.j , -1. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. -1.j ,  0. +0.j ,  0. -1.j ,  0. -0.j , -0. +0.j ,
         0. +0.j , -1. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -1. -0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. +0.j , -0. +0.j , -0. +1.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         1. +0.j ,  0. +0.j , -1. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. -0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,  0. -0.j ,
         0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +1.j ,
        -1. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -1.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,
         0. +1.j , -0. +0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. -0.j ,  0. +0.j ],
       [-0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j , -0. +0.j , -0. +0.j ,
        -0. +0.j , -1. -0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +1.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j , -0. +1.j ,  0. +0.j ,  1. +0.j ,
         0. -0.j , -0. +0.j ,  0. -0.j , -0. +0.j ,  0. -0.j ,  0. -0.j ,
        -1. -0.j ,  0. -0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -1. +0.j ,  0. -0.j ,
         0. -0.j , -0. +0.j , -1. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. -0.j ,  0. -1.j , -0. +0.j ,  1. +0.j ,
         0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +1.j ,
         0. +0.j ,  0. -0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j , -0. +1.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
        -0. +0.j ,  0. -0.j ,  0. -0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,
        -0. +1.j ,  0. +0.j , -1. -0.j ,  0. +0.j , -1. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +1.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j , -0. +1.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. -0.j ,  1. +0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,
        -1. -0.j ,  0. +0.j ,  0. -0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,
         0. -0.j ,  1. +0.j , -0. +0.j ,  0. +0.j , -1. -0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  1. +0.j ,
        -0. +0.j ,  0. -1.j , -0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
        -0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  1. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,
        -0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j , -1. +0.j , -1. +0.j ,
        -0. +0.j ,  0. -0.j ,  0. +0.j ],
       [-0.5+0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -1.j ,  1. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -1.j ,  0. +0.j ,  0. -0.j ,
         0. -0.j , -0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +1.j ,  0. -0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j , -0. +0.j ,  1. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j ,  1. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,
        -0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. -1.j ,
         1. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j , -0. +0.j ,
        -0. +0.j ,  0. +0.j , -0. +0.j ,  1. +0.j ,  0. -0.j ,  0. +0.j ,
        -0. +0.j ,  1. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,
         0. -1.j ,  0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,
        -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +1.j ,  0. -0.j , -0. +0.j ,
        -1. +0.j ,  0. +0.j ,  0. +0.j ],
       [ 0. +0.j ,  0. -1.j ,  0. +0.j ,  0. -1.j ,  0. -0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,
        -0. +0.j ,  0. -1.j , -1. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
        -1. -0.j ,  0. -0.j ,  1. +0.j ,  0. -0.j ,  0. -0.j ,  0. -0.j ,
         0. -0.j , -1. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. -1.j ,
        -0. +0.j , -1. -0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,  0. -1.j ,
         1. +0.j ,  0. +1.j ,  0. +0.j , -0. +1.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. -0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. -1.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,
        -1. -0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -1.j ,  0. +0.j ],
       [ 0. +0.5j,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,
        -0. +0.j , -0. +0.j , -0. +0.j ,  0. -1.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,
         0. -0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
        -0. +0.j ,  0. +0.j ,  1. +0.j ,  0. -0.j , -0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j ,  1. +0.j ],
       [ 0. +0.j ,  0. +0.j ,  1. +0.j ,  0. -0.j ,  0. -0.j , -1. -0.j ,
         0. +0.j ,  0. +0.j , -0. +1.j ,  0. -0.j ,  0. +0.j , -1. -0.j ,
         0. -0.j ,  0. +0.j ,  0. -0.j ,  0. -1.j ,  0. +0.j , -0. +0.j ,
        -0. +0.j ,  0. -1.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,
        -1. -0.j , -0. +0.j ,  0. +1.j ,  0. -1.j ,  0. -1.j ,  0. +0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j ,  0. -0.j ,  0. -0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j , -0. +1.j ,  0. +0.j , -0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.j , -1. +0.j ,  1. +0.j ,  0. +0.j ,
         0. +0.j ,  0. -0.j ,  0. -1.j ,  0. -0.j ,  0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j ,  0. -0.j ,  1. +0.j ,  0. +0.j , -0. +0.j ,
         0. +1.j ,  0. +0.j ,  0. +0.j ]]))

In [ ]:
#@title Verification
n = 3
m = 4
p = 7
rank = 63

verify_tensor_decomposition(decomposition_347, n, m, p, rank)

## Rank-74 decomposition of <3,4,8> over Z


In [ ]:
#@title Data

decomposition_348 = (np.array([[ 1., -0.,  0., -0.,  0.,  1.,  0., -1.,  0., -1., -0., -0.,  1.,
         1.,  0., -1.,  0., -1., -0., -0.,  0.,  1.,  0.,  1.,  0.,  1.,
        -1., -1., -1.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,
         1.,  1., -1.,  1.,  1.,  0., -0., -0.,  0., -0., -0., -0.,  0.,
        -0., -1., -0., -1.,  1., -1.,  0.,  0., -1., -1., -0.,  0., -0.,
         1., -1., -1.,  1., -0.,  0., -0.,  0., -1.],
       [-0., -0., -0.,  1., -0.,  1., -0.,  0., -1.,  0.,  0., -0.,  1.,
         1., -0., -1., -0.,  0., -0.,  0., -0., -0., -0.,  1., -0., -0.,
         0.,  0.,  0., -1., -0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,
        -0., -0.,  0.,  1., -0., -0.,  0.,  0., -0., -0., -0.,  0., -0.,
         0., -1., -0.,  0., -0., -1.,  0., -0.,  0.,  0., -0.,  0., -0.,
         1.,  0.,  0., -0.,  0., -0., -0., -0., -1.],
       [-0.,  0.,  0., -0., -0.,  1.,  0., -1.,  0.,  0., -0.,  1., -0.,
        -0., -0.,  0.,  0., -1.,  0., -0.,  0., -0., -0.,  1.,  0., -0.,
         0.,  0., -1.,  0.,  0., -1.,  0., -0.,  0.,  0.,  0., -0.,  1.,
         1., -0., -1.,  1.,  1., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,
         0.,  0.,  0., -1.,  1., -1., -0.,  0.,  0.,  0.,  0., -0.,  1.,
        -0.,  0., -1.,  1., -0.,  0.,  1.,  0., -1.],
       [-1.,  0.,  0.,  1., -0.,  0., -0.,  1., -1., -0., -0.,  0.,  0.,
         0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,  1., -1.,
        -0.,  1.,  1.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0., -1., -0.,  0., -1.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,
        -0., -0.,  0.,  1.,  0., -0.,  0., -0., -0.,  1., -0., -0.,  0.,
         0., -0., -0., -1., -0.,  0.,  0.,  0., -0.],
       [ 0.,  0.,  0.,  0.,  1.,  0.,  0.,  1.,  0., -0.,  0., -0.,  0.,
         0., -0., -0.,  1., -0., -1., -0.,  0.,  0., -0., -1.,  0.,  0.,
        -0., -0., -0.,  0., -1.,  1., -0.,  0.,  1., -0.,  0.,  0.,  0.,
        -1.,  0.,  1., -1.,  0., -0., -0.,  0.,  1., -1.,  0., -0.,  0.,
        -1., -0., -1.,  1.,  0., -0.,  0.,  0., -0., -0., -1.,  0.,  0.,
         0., -0.,  1., -1., -0., -0., -0., -0.,  1.],
       [-0., -0., -1.,  0., -0.,  0., -1., -0., -0.,  0., -0., -0.,  0.,
         0.,  0., -0., -0., -0., -1.,  1., -1., -0.,  0.,  0., -0., -0.,
         0.,  0., -0., -0., -1., -0.,  1., -0.,  1., -0., -0., -1.,  0.,
         0., -0., -0.,  0., -0.,  0., -0., -0., -0., -1., -1.,  1., -1.,
         0., -0., -1., -0., -0., -0., -0., -1.,  0.,  0., -1.,  0.,  0.,
         0.,  0., -0.,  0.,  1.,  0., -0.,  0.,  1.],
       [-0., -0.,  0., -0., -0.,  0.,  0.,  1.,  0.,  0.,  0.,  0., -0.,
        -0.,  0.,  0.,  1.,  1.,  0., -0.,  0., -0.,  0., -1., -0., -0.,
         0.,  0.,  1.,  0.,  0.,  1., -0., -0.,  1., -0., -0.,  0., -1.,
        -1., -0.,  1., -1.,  0.,  0.,  0.,  0.,  1.,  0.,  0., -0.,  0.,
         0.,  0.,  0.,  1.,  0.,  1.,  0.,  0.,  0.,  0.,  0., -1., -1.,
        -0.,  0.,  1., -1., -0., -0.,  0., -0.,  1.],
       [-0., -0.,  0.,  0., -1.,  0., -1., -1., -0.,  0.,  0.,  0.,  0.,
        -0., -0.,  0.,  0.,  0.,  0.,  1., -1.,  0.,  0., -0.,  0., -0.,
        -0.,  0.,  0.,  0.,  0.,  0.,  1.,  1.,  0., -0., -0.,  0., -0.,
        -0., -0., -1.,  1., -0., -0., -0., -0., -1.,  0., -1., -0.,  0.,
        -0., -0., -0., -1.,  0.,  0., -0., -1., -0.,  0.,  0., -0., -0.,
         0., -0.,  0., -0., -0., -0.,  0., -0.,  0.],
       [ 0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0., -1.,  0., -0.,  1.,
         0., -1., -1.,  0.,  0.,  1.,  0., -0.,  1., -1., -0., -0.,  1.,
        -0., -1.,  0., -0., -0.,  0., -1.,  0., -1., -1.,  0.,  1.,  0.,
        -0., -0.,  0., -0., -0., -1.,  0.,  1.,  0.,  1., -0.,  0., -0.,
        -0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -1., -0.,  0., -0.,
         1., -1.,  0., -0.,  0., -1., -0., -0.,  0.],
       [ 0.,  0., -0., -0.,  0., -0.,  1., -0.,  0.,  0., -1.,  0., -0.,
        -0.,  0.,  0., -0., -0.,  1., -1., -0., -0.,  0., -0.,  0., -0.,
        -0.,  0., -0.,  0.,  1., -0., -1., -0., -1., -1.,  0.,  1.,  0.,
         0., -0., -0., -0.,  0.,  0., -1., -0., -0.,  1.,  1., -1.,  1.,
         0.,  0.,  1., -0.,  0.,  0., -1.,  1., -0.,  0., -0.,  0.,  0.,
         1.,  0., -0.,  0., -1.,  0., -0.,  0.,  0.],
       [-1., -1.,  0., -0., -0., -0., -0.,  0., -0.,  0.,  0., -0., -0.,
        -1.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0., -0., -0.,
         1.,  0.,  0., -0., -0.,  0., -1.,  0., -1., -1., -1.,  1., -0.,
        -0.,  0.,  0., -0., -0.,  0.,  0.,  1.,  0.,  1., -0.,  0., -0.,
         0., -0., -0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,
        -0.,  0.,  0., -0.,  0., -1.,  1., -0.,  0.],
       [-0., -0.,  0., -0., -0., -0.,  1., -0.,  0.,  1.,  0., -0.,  0.,
        -0., -0.,  1.,  0.,  0., -0.,  0., -0.,  0.,  1., -0., -0., -1.,
         0.,  1., -0., -0.,  1.,  0.,  0.,  0., -0., -0., -0.,  0., -0.,
        -0.,  0., -0.,  0.,  0., -0., -1., -1., -0.,  0.,  1., -1., -0.,
        -0., -0., -0., -0., -0.,  0., -1.,  1.,  0., -0., -0., -0., -0.,
         0., -0.,  0.,  0.,  0., -0.,  0., -1.,  0.]], dtype=np.float32), np.array([[ 0., -0., -0., -0., -0., -0., -0., -0.,  0., -0., -0.,  0.,  0.,
         0., -0., -0., -0., -1., -0., -0.,  0., -0., -0., -0., -0., -0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  1.,
        -1.,  0.,  0., -0., -0., -0.,  0.,  0.,  0., -0.,  0., -0., -0.,
         0.,  0., -0., -0., -0., -0.,  0., -0., -1., -0., -0., -0., -0.,
        -0.,  1., -0., -0., -0.,  0.,  0.,  0.,  0.],
       [-0., -0., -0.,  0., -0.,  0.,  0., -0., -0.,  0., -0.,  0., -0.,
        -0., -0., -0., -0.,  0.,  1., -0., -0., -1., -0.,  0., -0., -0.,
        -0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0., -0.,  0., -0.,  0.,
         0.,  0.,  0.,  0., -0., -1.,  0., -0.,  0.,  0.,  0.,  0.,  1.,
        -0., -0., -1.,  0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0.,
         0., -0., -0., -0., -0., -0., -0.,  0., -0.],
       [ 0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0., -1.,  0., -0.,  0.,
        -0., -0.,  0.,  0., -1.,  1.,  1.,  0.,  0.,  1.,  0., -0.,  0.,
        -0.,  0.,  0., -0., -1.,  0., -0.,  1., -0., -0., -0.,  0.,  1.,
        -1.,  0.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0.,  1.,
         0., -0.,  0.,  0.,  0., -0.,  0., -1., -1., -0.,  0., -0.,  0.,
         0., -0.,  0., -0., -0., -0., -0.,  0.,  0.],
       [-0.,  1.,  0., -0.,  0., -0., -0.,  0.,  0., -0., -0.,  0.,  0.,
        -0., -0.,  0., -0., -0., -1.,  0.,  0., -0.,  0., -0.,  0.,  0.,
         1., -0., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,  1., -0.,
         0., -0., -0., -0.,  0., -0., -0., -0., -0.,  1., -0., -0., -0.,
        -0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,
        -0.,  1., -0.,  0.,  1., -1.,  0.,  0.,  0.],
       [-0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,
        -0., -0.,  0.,  0., -1.,  1., -0., -0.,  0., -0.,  0.,  0.,  0.,
         0., -0.,  0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,  1.,
        -1., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,  1.,
        -1.,  0., -1., -0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0.,
        -0.,  0., -0.,  0., -0., -0., -0., -0.,  0.],
       [ 0.,  0.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0.,
        -0.,  1.,  0., -0.,  0.,  1.,  0., -0., -0., -0., -0.,  0.,  0.,
         0.,  0., -0.,  0.,  0., -1.,  0., -0., -0., -0., -0., -0., -0.,
        -0., -0., -0.,  0.,  0.,  0., -0., -0., -0., -0., -0., -0.,  1.,
         0.,  0., -1., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0., -1.,
        -0.,  0.,  1.,  0., -0., -0., -0., -0.,  0.],
       [-0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0., -0.,  0., -0.,  1.,
         0.,  0., -0., -0., -0.,  0.,  0.,  0., -1.,  0., -1., -0.,  0.,
         0., -0.,  0.,  1., -0., -0., -0.,  0., -0., -0., -0.,  0., -1.,
        -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,
        -0.,  1., -0., -0., -0., -1., -0., -0., -0., -0., -0.,  0.,  0.,
         0.,  0., -1., -0.,  0., -0., -0., -0., -0.],
       [ 0.,  0., -0., -0.,  0.,  0., -0., -0., -0., -0., -0., -0., -0.,
        -0.,  0.,  0., -0., -0.,  0.,  0.,  0., -1.,  0.,  0.,  1., -0.,
         0.,  0., -1., -0., -0., -0., -0.,  0., -0., -0., -0., -0., -1.,
        -0., -1.,  0., -0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,  0.,
        -0., -0., -0.,  0., -0.,  0.,  0., -0., -0.,  1., -0.,  0., -0.,
        -0.,  0., -1.,  1.,  0., -0., -0., -0., -0.],
       [ 0., -0., -0., -0., -0., -0., -0.,  0., -0., -1., -0., -0., -0.,
         0., -0., -1.,  0., -0., -0.,  0.,  1.,  0., -0.,  0., -0.,  0.,
         0.,  0., -0., -0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0.,
         1., -0.,  1., -1.,  0., -0.,  1.,  0., -0., -0.,  0., -0.,  0.,
        -0., -0., -0.,  0., -0.,  0.,  0., -1., -0., -0., -0., -0., -0.,
        -1., -1.,  0.,  0.,  0., -0.,  0.,  0.,  1.],
       [ 0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -1.,  0., -1.,
        -0., -0., -0., -0., -0.,  0.,  0., -0.,  1., -0.,  0.,  0.,  0.,
        -0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0., -0., -0.,
        -0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0., -1.,
        -0., -0., -0., -0.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,  0.,
         1., -0.,  0.,  0., -0.,  0., -0., -0., -0.],
       [ 0.,  0., -0., -0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,
        -0.,  0.,  0., -0., -0.,  0., -1.,  1., -0.,  0., -0., -0., -0.,
        -0.,  0.,  0.,  0., -0.,  0., -0., -1., -0., -0., -0., -0.,  0.,
         1.,  0.,  1., -1.,  0., -0., -0.,  0., -0.,  0.,  0., -0., -1.,
        -0.,  0., -0., -0.,  0., -0.,  0., -0., -0., -0.,  0., -0.,  0.,
         0.,  0., -0., -0.,  0.,  0., -0.,  0.,  1.],
       [ 0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,
        -1.,  0.,  0., -0.,  0., -0., -0., -0.,  0.,  0., -0., -0., -0.,
        -1.,  0., -0., -0.,  0.,  0., -0., -0.,  0., -1.,  0., -1.,  0.,
         0., -0., -0.,  0.,  0., -0., -0.,  0., -0.,  0., -0., -0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0., -0., -0.,
        -1., -1.,  0., -0., -1.,  1.,  0., -0.,  0.],
       [ 0., -0., -0., -0., -0., -0., -0., -0., -0.,  0.,  0.,  0.,  0.,
        -0.,  0.,  0., -0.,  1.,  0.,  0., -0., -0., -0.,  0., -0., -0.,
        -0.,  0., -0., -0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,  0.,
         1., -0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0., -1.,
         1.,  0.,  1.,  0., -0.,  1.,  0.,  0.,  0., -0., -1.,  0.,  0.,
        -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.],
       [ 0.,  0.,  1., -0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0.,
        -0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,  0., -1., -0., -0.,
        -0.,  0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0.,  0.,
         0., -0., -0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0.,
        -0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0., -0.,
        -0., -0., -1., -0.,  1.,  0., -0.,  0.,  1.],
       [ 0.,  0.,  0.,  0., -0., -0., -0., -0., -0., -0.,  0., -0., -1.,
         0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  1., -0.,  1.,  0., -0.,
        -0., -0., -0., -1.,  0., -0., -0., -0.,  0., -0., -0., -0., -0.,
         0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,
         0.,  0.,  0.,  0., -0., -0., -0., -0., -0.,  0., -0.,  0., -0.,
         1., -0.,  1., -0.,  0.,  0., -0.,  0., -1.],
       [ 0.,  0., -1., -1., -0., -0., -0.,  0., -0., -0., -0., -0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  1., -1., -0.,
        -0.,  1., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0., -0., -0.,
        -0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,  0., -1., -0.,
         0., -0.,  0., -0.,  0.,  0.,  1., -0., -0., -1., -0.,  0.,  0.,
        -0.,  0.,  1.,  0.,  0.,  0., -0., -0., -1.],
       [-0.,  0., -0., -0., -0., -0., -0., -0.,  0., -0.,  0.,  0.,  0.,
         0., -0., -0., -0.,  1.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,
         1.,  0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0., -0., -1.,
        -0., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,
        -0., -0.,  0., -0., -1.,  0.,  0., -0.,  1., -0., -0., -0.,  0.,
        -0.,  0.,  0., -0., -0.,  0.,  1.,  0.,  0.],
       [-0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0., -0.,  1., -0.,
        -0., -1.,  0., -1., -0., -1., -0., -0., -0.,  0.,  0.,  0., -0.,
        -0.,  0., -0.,  0., -0.,  1.,  0.,  0., -1.,  0.,  1., -1., -0.,
        -0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,  0., -1.,
        -0.,  0.,  1.,  0.,  0., -0.,  0., -0., -0., -0.,  0.,  0., -0.,
         0.,  0., -0., -0., -1., -0., -0.,  0., -0.],
       [ 0., -0., -0.,  0.,  0., -0.,  0., -0., -0.,  1.,  0., -0.,  0.,
        -0.,  0.,  0.,  0.,  1., -1.,  0.,  0., -0., -1.,  0.,  0.,  0.,
        -0., -0., -0.,  0.,  1., -0., -1.,  0., -1.,  0., -0.,  0., -1.,
        -0., -0., -1., -0.,  0.,  0.,  0., -0., -1., -0., -0., -0.,  0.,
         0.,  0., -0., -0.,  0., -0.,  0.,  1.,  1.,  0.,  0., -0., -0.,
         0., -0., -0.,  0.,  0., -0., -0., -0.,  0.],
       [-0., -1.,  0., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0., -0.,
        -0.,  0., -0., -0., -0.,  1., -0.,  0., -0., -0., -0.,  0.,  0.,
         0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  1., -0.,  0.,  0.,  0.,
        -0.,  0., -0., -0.,  0., -0.,  0., -0., -0., -1., -0., -0., -0.,
        -0., -0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,
        -0.,  0., -0.,  0.,  0., -0.,  1., -0., -0.],
       [ 0., -0., -0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0.,
         0.,  0.,  0.,  0., -0., -1., -0., -0., -0.,  0.,  0., -0.,  0.,
        -0.,  0.,  0.,  0., -0., -0.,  0., -0., -1., -0., -0.,  0., -1.,
        -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  1., -0.,  0.,  0.,
        -0., -0., -0., -0.,  0.,  0.,  0., -0., -0., -0., -0.,  1.,  0.,
        -0., -0., -0., -0.,  0.,  0., -0.,  0.,  0.],
       [ 0., -0.,  0., -0., -0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0.,
         0., -1.,  0., -1., -0., -1.,  0., -0., -0.,  0., -0.,  0.,  0.,
        -0.,  0.,  0.,  0., -0.,  1.,  0.,  0., -1.,  0., -0., -1.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0., -0., -1.,
        -0., -0.,  1., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  1.,
        -0.,  0.,  0.,  0., -1., -0., -0., -0.,  0.],
       [-0., -0., -0.,  0.,  0., -1.,  0.,  0.,  0.,  0.,  0., -0.,  0.,
         1.,  0., -0., -0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,
         0., -0., -0.,  0.,  0., -0., -0.,  0., -0., -0., -0., -0.,  1.,
        -0., -0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0., -0.,  0.,
        -0., -1., -0., -0., -0.,  1., -0.,  0.,  0.,  0.,  0.,  0., -0.,
         0.,  0., -0., -0., -0., -0., -1., -0., -0.],
       [ 1., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0.,
        -0., -0.,  0., -0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0., -0.,
         0.,  0.,  1.,  0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,  1.,
         0.,  1., -0., -0.,  1.,  0.,  0., -0., -0.,  0., -0., -0., -0.,
        -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,
        -0.,  0., -0.,  0., -0.,  0., -1., -0., -0.],
       [-0., -0., -0., -0.,  0.,  0.,  0.,  1., -0.,  1., -0.,  0.,  0.,
         0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,  1.,
        -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,
        -1.,  0., -1., -0., -0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,
         0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,
        -0.,  1., -0., -0., -0.,  0., -0., -0.,  0.],
       [ 0., -0., -0., -0.,  0., -0., -1.,  0.,  0.,  0., -0., -0., -0.,
         0.,  0., -0.,  0.,  0., -0.,  1., -0., -1.,  0.,  0., -0.,  0.,
         0., -1.,  0., -0., -0., -0., -0., -0.,  0., -0., -0.,  0.,  0.,
         0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0., -0., -0.,  1.,
         0.,  0.,  0., -0., -0., -0., -0., -0.,  0.,  1.,  0., -0., -0.,
         0.,  0., -0.,  0.,  0., -0.,  0., -1., -0.],
       [ 0., -0.,  0., -0.,  0.,  0., -1.,  1., -0., -0.,  0., -0.,  0.,
        -0., -0.,  0.,  0.,  0., -0.,  1.,  0.,  0., -0.,  0.,  0., -0.,
         0.,  0.,  0., -0.,  0., -0.,  0.,  1.,  0., -0.,  0.,  0., -0.,
        -1.,  0., -1., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  1.,
         0., -0.,  0., -0., -0., -0.,  0., -0.,  0., -0.,  0., -0., -0.,
         0., -0.,  0.,  0., -0., -0.,  0.,  0.,  0.],
       [ 1., -0.,  0.,  0., -0., -0.,  1., -0., -0., -0., -0., -0.,  0.,
        -0., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0.,
         1.,  1., -0.,  0.,  0.,  0., -1.,  0., -0., -0., -0.,  1.,  0.,
         0.,  0.,  0.,  0., -0.,  0.,  0.,  1.,  0., -0.,  0.,  0., -0.,
        -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,
        -0.,  1., -0., -0.,  1., -1., -0., -0.,  0.],
       [ 0., -0., -0., -0., -1.,  0., -1., -0., -0., -0.,  0., -0.,  0.,
        -0., -0.,  0.,  0., -1., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,
         0.,  0., -1.,  0.,  1., -0., -0., -0.,  0., -0.,  0.,  0., -0.,
        -1.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0., -0.,  1.,
        -1., -0., -1.,  1., -0.,  0., -0.,  1., -0., -0., -0., -0.,  0.,
        -0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0.],
       [ 0., -0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,
        -0., -0.,  0.,  0.,  0., -0., -0., -0., -0., -0., -0.,  0., -0.,
         0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,  0., -0.,
        -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  1.,  1., -0.,
         0.,  0.,  0.,  1., -0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
        -0.,  0.,  1., -1., -1., -0., -0.,  0.,  0.],
       [ 0., -0.,  0.,  0., -0.,  0., -0., -1., -1.,  0.,  0.,  0.,  1.,
        -0., -0.,  1., -0., -0., -0., -0.,  0., -1.,  0., -1., -0., -1.,
         0.,  0., -0.,  1.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0.,
        -0., -0., -0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        -0., -0., -0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,
         0.,  0., -1., -0., -0., -0.,  0.,  0., -0.],
       [-0., -0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,
        -0., -0., -0., -0., -0., -0., -0.,  0., -1., -0.,  0.,  1.,  0.,
         0., -1.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0.,
        -0.,  0., -0., -0., -0., -0.,  0.,  0.,  0., -0.,  0., -0., -0.,
        -0.,  0., -0., -1.,  0.,  0.,  0.,  0.,  0.,  1., -0., -0.,  0.,
        -0.,  0., -1.,  1., -0., -0.,  0.,  0., -0.]], dtype=np.float32), np.array([[-0., -0., -0., -0., -0.,  0.,  0.,  0.,  1., -1.,  0.,  0.,  0.,
        -0.,  0.,  1., -0., -0.,  0.,  0.,  0.,  0., -1., -0.,  0., -1.,
         0., -0., -0., -0.,  0., -0.,  0.,  0., -0., -0., -0., -0., -0.,
        -0., -0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,
         0.,  0.,  0., -0., -1., -0., -0., -0.,  1., -0.,  0., -0., -0.,
        -0., -0., -0.,  0., -0., -0.,  0., -0., -0.],
       [-0., -0.,  0.,  0.,  1.,  0., -0., -0., -0.,  0., -0.,  0.,  0.,
        -0.,  0., -0., -0.,  1.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,
        -0., -0., -0., -0., -1.,  0., -0., -1.,  0.,  0.,  0.,  0.,  0.,
         1.,  0.,  1., -0., -0.,  0., -1.,  0., -0., -0., -0.,  0., -0.,
        -1.,  0., -0., -0., -1., -0., -0.,  1., -0.,  0., -0., -0.,  0.,
         0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.],
       [ 0.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,  1.,  0., -0., -0.,
         0.,  0., -0.,  0., -0.,  0.,  0., -0., -0.,  1.,  0.,  0., -0.,
         1.,  0.,  0., -0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,
        -0., -0., -0., -0.,  0., -0., -1., -0.,  0.,  0., -0., -0.,  0.,
         0., -0.,  0.,  0.,  0.,  0., -0., -0., -1.,  0.,  0.,  0., -0.,
        -0., -1.,  0., -0., -0., -1.,  0., -0., -0.],
       [ 0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0., -0.,  1., -1.,
         0., -0., -0., -0.,  0.,  0., -0.,  0., -1.,  0., -0., -1.,  0.,
         0., -0., -0., -1., -0., -0., -0.,  0.,  0.,  0., -0.,  0., -0.,
         0., -0.,  0., -0., -0., -1., -0., -0., -0., -0.,  0.,  0., -0.,
         0., -0., -0., -0., -0., -0.,  0.,  0., -0.,  1., -0., -0., -0.,
         0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0.],
       [-0., -0., -0.,  0.,  0., -0.,  0., -0., -0.,  0.,  1.,  1.,  0.,
        -0.,  0.,  0., -0., -0., -0.,  1., -0.,  0., -0.,  0.,  0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  1.,  0., -1., -0., -0.,  0., -0.,  0.,
        -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  1.,
        -1.,  0.,  1.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,  1.,
         0.,  0.,  0., -0., -0., -0.,  0., -0., -0.],
       [-0.,  0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,  1., -0., -0.,
        -0., -0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,  0.,
        -0., -0., -0., -0.,  0., -0.,  0.,  0.,  0., -0., -1., -0., -0.,
         0.,  0.,  0., -0., -0.,  1.,  0., -0., -0.,  0.,  0., -0., -0.,
         0.,  0., -0., -0., -0., -0., -0., -0.,  0.,  0.,  0., -0., -0.,
         0.,  0., -0.,  0., -0.,  0., -0.,  1.,  0.],
       [ 0.,  0., -0.,  0.,  0.,  0., -0.,  1.,  0.,  1., -0.,  0., -0.,
         0.,  0., -1., -0.,  0.,  0.,  0.,  0., -0.,  1., -0., -0.,  1.,
         0.,  0., -0., -0., -0., -0.,  0., -0., -0., -0., -0., -0., -0.,
        -0., -0.,  1., -1., -0., -0., -0., -0., -1.,  0., -0.,  0., -0.,
         0., -0., -0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,
         0., -0., -0.,  0.,  0., -0., -0., -0., -0.],
       [-0., -0.,  0., -0., -1.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,
        -0., -0.,  0.,  0.,  0., -0., -0., -1.,  0., -0., -0.,  0., -0.,
        -0., -0., -0.,  0.,  1.,  0., -0.,  1., -0., -0.,  0., -0.,  0.,
         0.,  0.,  0.,  0., -0.,  0.,  1.,  0., -1.,  0.,  0., -0.,  0.,
        -0.,  0.,  0., -0., -0., -0.,  0., -1.,  0., -0.,  0., -0.,  0.,
         0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.],
       [ 0.,  0., -0.,  0.,  0.,  0., -1.,  0.,  0.,  0.,  0., -0., -0.,
         0.,  0., -0., -0., -0.,  0.,  1., -1., -0., -1., -0.,  0.,  0.,
         0.,  0.,  0., -0., -0., -0.,  1., -0.,  0., -0., -0., -0., -0.,
        -0., -0.,  0., -0.,  0., -0.,  1., -1.,  0.,  0., -0.,  0., -0.,
         0., -0.,  0., -0.,  0., -0.,  0., -1., -0.,  0.,  0., -0., -0.,
        -0., -0.,  0.,  0., -0., -0., -0., -1., -0.],
       [-1., -1.,  0.,  0., -0., -1.,  0., -0.,  0., -0.,  0.,  0.,  0.,
        -1.,  0.,  0.,  0., -0., -0., -0.,  0.,  0., -0.,  0., -0., -0.,
        -1.,  0.,  0., -0.,  0.,  0.,  0., -0., -0., -0., -0., -0.,  0.,
         0.,  0., -0.,  0.,  1.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,
        -0.,  0.,  0., -0.,  1.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
        -0., -0.,  0., -0.,  0.,  0.,  1.,  0.,  0.],
       [ 0.,  0.,  0., -0.,  0.,  0., -0., -0., -0., -0., -0., -0.,  0.,
         0.,  0., -0., -1.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,
         0., -0., -0.,  0., -0., -0., -1., -0.,  1.,  1.,  0.,  1., -0.,
         0., -0.,  0., -0., -0., -0., -0.,  0., -1., -1.,  0.,  0., -0.,
        -0., -0., -0., -0., -0., -0.,  0.,  0.,  0., -0., -0., -1., -0.,
        -0.,  0., -0.,  0., -0., -0., -0., -0.,  0.],
       [-0.,  1.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,
        -0.,  0., -0.,  0., -0., -0., -0.,  0., -0., -0.,  0., -0.,  0.,
        -0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,  1.,  0., -0.,  0.,
        -0., -0., -0.,  0.,  0.,  0., -0., -1.,  0.,  0., -0.,  0., -0.,
         0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,
        -0., -0.,  0., -0.,  0.,  1.,  0.,  0., -0.],
       [ 0.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,
         0.,  0., -0.,  0.,  1.,  0.,  0., -0., -0.,  0.,  0., -0.,  0.,
         0., -0., -1., -0.,  0.,  0.,  0.,  0., -0., -0., -0., -0., -1.,
        -0., -1.,  0., -0., -0., -0., -0., -0., -0.,  0., -0., -0., -0.,
        -0., -1.,  0.,  0., -0., -1., -0.,  0., -1.,  0., -0., -1., -0.,
         0.,  0.,  0., -0., -0.,  0., -0., -0., -0.],
       [-0., -0., -0., -0.,  1.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,
        -0., -0.,  0.,  0., -0., -0.,  0., -0.,  0., -0., -0.,  0., -0.,
        -0., -0.,  0.,  0., -0., -0.,  0., -0., -0., -0.,  0., -0.,  0.,
         0.,  0.,  0., -0., -0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,
         1.,  0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  1., -1., -0.,
         0., -0., -0.,  0., -0.,  0.,  0., -0.,  0.],
       [ 0.,  1., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,
         0.,  1., -0., -0.,  0.,  1.,  0., -0., -0.,  1., -0., -0., -0.,
         0.,  0., -0., -0.,  1.,  0.,  0.,  0.,  0., -0., -0., -0., -0.,
         0., -0., -0., -0.,  0., -1.,  0., -0., -0.,  1., -0., -0., -0.,
        -0., -0.,  1.,  0., -0., -0.,  0., -0., -0.,  0.,  1.,  0.,  0.,
        -0., -0., -0., -0., -0., -0.,  0.,  0.,  0.],
       [ 0.,  0.,  0., -1.,  0.,  0.,  0., -0., -0.,  0., -0., -1., -0.,
         0., -0., -0., -1.,  0.,  0., -0., -0., -0.,  0., -1.,  1.,  0.,
         0.,  0., -0.,  1., -0., -1., -0.,  0., -0., -0.,  0., -0., -0.,
        -0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,
         0., -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,
         0., -0., -1.,  1., -0.,  0., -0.,  0., -0.],
       [-0., -0., -1.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -1.,  0.,
        -0.,  0.,  0., -1.,  0., -0.,  0.,  0.,  0., -0.,  0., -0., -0.,
        -0., -0., -0., -0.,  0., -1.,  0., -0., -0., -0., -0.,  0., -0.,
         0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,  0., -1., -1.,  0.,
        -0., -0.,  0., -0., -0., -0., -1., -0.,  0., -0., -0.,  0., -1.,
         0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.],
       [ 0.,  0., -1., -0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,
         0., -1., -0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  1., -1.,  0.,
        -0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0., -1.,  0.,
         0., -0.,  0.,  0., -0.,  0., -1.,  0., -0.,  0.,  0., -0.,  0.,
        -0., -0., -0.,  0., -1.,  1.,  0.,  0., -0.],
       [ 0.,  0.,  0.,  0.,  0., -1., -0., -0.,  1.,  0., -0.,  0.,  0.,
        -0., -0.,  0.,  0., -0., -0., -0.,  0., -0.,  0., -0., -0.,  0.,
         0.,  0.,  0.,  1., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,
         0., -0., -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,
         0., -1., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0., -0.,
         0., -0.,  0., -0., -0., -0., -0.,  0.,  0.],
       [ 0., -0., -1., -0., -0., -1.,  0.,  0., -0.,  0.,  0.,  0., -0.,
         0., -0., -0.,  0., -0., -0.,  0., -1.,  0.,  0.,  1.,  0.,  0.,
         0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,
        -0., -0., -0.,  1.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  1., -0., -0., -0., -0.,  1.,  0.,  0.,
        -0., -0., -0., -0.,  0., -0., -0.,  0., -1.],
       [-0., -0.,  0.,  0., -0.,  0., -0.,  0.,  0., -0., -1.,  0.,  1.,
        -1.,  0.,  1.,  0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,
        -0., -0., -0.,  0., -0., -0.,  0.,  0., -0.,  1., -0.,  0., -0.,
        -0.,  0.,  0., -0., -0.,  0., -1., -0., -0., -0.,  0.,  0.,  0.,
        -0.,  1., -0., -0., -0., -0., -0., -0.,  0., -0., -0., -0., -0.,
         1.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0.],
       [-0.,  0.,  0., -1., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,
         0.,  0., -0.,  0., -0., -0., -0.,  0., -0.,  0.,  0.,  1.,  0.,
         0., -0., -0., -0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,
        -0., -1., -0.,  0.,  1., -0., -0., -0.,  0., -0.,  0.,  0., -0.,
         0., -0., -0., -0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,
         0., -0.,  0., -0.,  0., -0., -0., -0.,  0.],
       [ 0.,  0.,  0.,  0.,  1., -0., -0., -0., -0., -0.,  0.,  0., -0.,
         0., -0., -0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,
         0.,  0.,  1.,  0.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,
         0.,  0.,  0.,  0.,  1., -0., -0.,  0.,  0.,  0., -1., -1.,  0.,
        -0., -0.,  0.,  1., -0.,  0., -1., -0., -0.,  0.,  0.,  0.,  0.,
        -0., -0., -0., -1., -0.,  0., -0.,  0., -0.],
       [-1., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0.,
        -0.,  0.,  0., -0.,  0., -0., -0.,  0., -0., -0.,  0.,  0., -0.,
         0., -1., -0., -0., -0., -0., -0.,  0., -0.,  0., -0.,  0., -0.,
         0.,  1., -0., -0., -0.,  0., -0., -1., -0., -0.,  0.,  0., -0.,
         0.,  0., -0., -0., -0., -0., -1.,  0., -0., -1., -0., -0., -0.,
         0., -0.,  0.,  0.,  0., -0., -0., -1.,  0.]], dtype=np.float32))

In [ ]:
#@title Verification
n = 3
m = 4
p = 8
rank = 74

verify_tensor_decomposition(decomposition_348, n, m, p, rank)

## Rank-68 decomposition of <3,5,6> over Z


In [ ]:
#@title Data

decomposition_356 = (np.array([[-0.,  0., -0., -0.,  1.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,
        -0.,  0.,  0., -0.,  0.,  1.,  1.,  0.,  0.,  0., -1.,  0.,  0.,
        -0.,  0., -0.,  0.,  0.,  0., -1., -0., -0., -1., -0., -0., -0.,
        -0.,  0.,  1.,  0., -1., -1., -0., -1.,  0.,  1.,  1., -0.,  0.,
        -0., -0.,  0., -0., -0.,  0., -1., -1.,  0.,  0., -0., -0.,  0.,
         0.,  0.,  0.],
       [-0.,  1.,  0.,  0.,  1.,  1., -0.,  0., -0.,  0.,  0.,  0., -0.,
        -0.,  0., -1., -0.,  0.,  1., -0., -0.,  0., -0., -0., -0.,  0.,
        -0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0., -0., -0., -0., -0.,
         0., -0.,  1.,  0., -0., -1.,  1.,  0.,  0.,  1.,  1.,  1.,  0.,
         0., -0., -0.,  1., -0.,  0., -1., -0., -0., -0., -0.,  1.,  1.,
        -0., -0., -0.],
       [-0.,  0.,  0.,  0., -0., -0., -0., -0., -0., -0.,  0.,  0.,  0.,
         0., -1., -0.,  0., -0.,  0., -1., -0.,  0.,  0.,  1.,  0.,  1.,
        -0., -1., -0.,  0.,  1.,  0.,  1.,  0.,  0.,  1., -0., -0., -0.,
         0., -0.,  0.,  0., -0., -0.,  0.,  1.,  0., -0.,  0., -0.,  0.,
        -0., -0.,  0.,  0.,  1., -0.,  0.,  1., -1.,  0.,  0., -0.,  1.,
         0.,  1., -0.],
       [-0.,  0., -0., -0., -0.,  0., -0.,  0.,  0.,  1., -0.,  1.,  1.,
         0., -0., -0.,  0., -0.,  0., -0.,  0., -1., -1.,  0.,  0., -0.,
         0., -0.,  0., -0., -0.,  0., -0.,  0.,  0., -0., -0., -0., -1.,
        -0., -0., -0.,  0.,  1., -0., -0.,  0.,  1.,  0., -0., -0.,  0.,
        -1.,  1., -1., -0., -0., -0., -0., -0., -0.,  0.,  0.,  0.,  1.,
        -0.,  0., -0.],
       [ 0., -0.,  0.,  0., -1.,  0., -0.,  0.,  0.,  1.,  0.,  1.,  0.,
        -0., -0.,  1.,  0.,  0., -1.,  0., -0., -0.,  0.,  1.,  0.,  1.,
         0., -1., -0., -0.,  1.,  0.,  0.,  0., -0., -0., -0., -0.,  0.,
        -0.,  0., -1.,  0., -0.,  1.,  0.,  0., -0., -0., -1.,  0.,  0.,
        -0., -0.,  0., -0.,  1.,  0.,  1.,  0., -1., -0., -0.,  0., -0.,
        -0.,  1., -0.],
       [-1.,  0.,  0.,  0., -0., -0., -1.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0., -0.,  0., -0., -0., -0., -0.,  0., -0., -0., -0., -1.,
        -0.,  0.,  0., -0.,  0.,  0.,  1., -0.,  1., -0.,  1.,  0., -0.,
         0., -0., -1.,  0.,  1.,  0.,  0.,  0., -0.,  0., -1.,  1.,  1.,
        -0., -0., -0.,  0., -0., -1., -0.,  1., -0., -0.,  0., -0., -0.,
         0., -0.,  0.],
       [-0., -1., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -1., -0.,  0.,
         0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,
        -0., -0.,  0., -0., -1., -0.,  0., -0.,  0., -1., -0.,  0., -0.,
        -1.,  0., -1.,  1., -0.,  1., -1., -0., -0., -0.,  0., -0.,  0.,
         0., -0.,  0.,  0., -0., -1., -0.,  0., -0., -0.,  0.,  0., -1.,
         0., -0.,  1.],
       [ 0., -1., -0.,  0., -0., -0.,  1., -0.,  0.,  0.,  0.,  0.,  0.,
        -0.,  0.,  0., -0., -0., -0., -0.,  0., -0., -0.,  0.,  0., -0.,
         0.,  1.,  0.,  0., -1., -0., -1., -0., -0., -1., -0.,  0., -0.,
        -1.,  0.,  0.,  1., -0.,  1., -0., -0.,  0.,  0., -0.,  0.,  0.,
        -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,  1., -0.,  0., -1.,
         0.,  0.,  1.],
       [ 0.,  0., -0.,  1., -0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,
        -1.,  0.,  0.,  1., -0., -0., -0., -1., -0., -0., -0.,  1.,  0.,
        -0., -1.,  0., -1., -0.,  0.,  1., -1., -0.,  0., -0.,  0., -0.,
        -0.,  0., -1.,  0.,  0., -0., -1., -0., -0.,  0., -0.,  0., -0.,
        -0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,  0., -1.,  0.,  0.,
         0., -0., -0.],
       [ 0., -0., -1.,  0., -0., -0.,  1.,  0., -0., -1.,  1., -1., -0.,
        -0.,  0.,  0., -0.,  0.,  1., -1., -0., -0., -0., -0.,  0.,  0.,
         0.,  1., -1.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
         0.,  0.,  1.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0., -0.,  0., -1.,  0.,
        -0., -1., -0.],
       [-0.,  0., -0., -0.,  0., -0., -1., -0., -1., -0.,  0.,  0., -1.,
         1.,  0., -0., -0.,  1.,  0., -0., -0., -0.,  1.,  0.,  0., -1.,
        -0.,  0., -0.,  0., -0., -0., -0., -1.,  0.,  0.,  1., -0., -0.,
         0.,  0., -1.,  0., -0.,  0.,  0., -0.,  0., -0., -1., -0.,  1.,
         0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0., -1.,
        -1., -0., -1.],
       [ 1., -0.,  0.,  0.,  0.,  0., -0., -0., -1.,  0., -1.,  0.,  0.,
         0.,  0.,  0.,  0.,  1.,  0., -0., -0.,  0.,  1.,  0., -1.,  0.,
         0.,  0.,  0.,  0., -1., -0., -0., -1.,  0.,  0., -0.,  0.,  0.,
         0., -1., -1.,  1.,  0.,  1., -0.,  0., -1., -0.,  0., -0.,  0.,
         0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0., -0., -0.,  0., -1.,
        -0., -0.,  0.],
       [ 0., -0.,  0.,  0., -0.,  0.,  1., -1., -0., -0., -0.,  0.,  1.,
        -1.,  0.,  0.,  0., -0., -0., -0.,  0., -1.,  0., -0.,  0.,  0.,
        -1.,  1.,  0.,  1., -1.,  0., -0., -0., -0.,  0., -1.,  0., -0.,
         0.,  0., -0.,  0.,  1.,  1.,  0., -0.,  0., -0.,  0.,  0., -0.,
        -0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0.,
         1.,  0.,  1.],
       [-0., -0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0., -1.,
         0.,  0., -0., -0.,  1.,  0.,  0., -1.,  1.,  1.,  0., -0., -0.,
         1., -1.,  0., -1.,  0.,  0.,  0., -1., -0.,  0., -0., -0.,  0.,
        -0.,  0., -1., -0., -1.,  0.,  0., -0., -1., -0.,  0.,  0., -0.,
        -0.,  0., -0., -0.,  0., -0., -0.,  0.,  0., -0., -1.,  0., -1.,
         0.,  0., -0.],
       [-0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,
         0., -0., -1.,  0.,  0.,  0.,  0., -1., -0., -0., -1., -0., -1.,
        -0.,  0., -0., -0., -1., -1.,  0., -0., -0.,  0., -0., -1., -1.,
        -0.,  0.,  0.,  0.,  0., -1.,  0.,  0., -0.,  0.,  1.,  0.,  0.,
        -0., -0., -1., -0., -1., -0., -1.,  0.,  0.,  0., -1.,  0., -0.,
         0., -0., -0.]], dtype=np.float32), np.array([[-0.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0., -0., -1.,
        -0.,  1.,  0., -0., -0.,  0.,  0.,  0.,  1.,  0.,  0., -0.,  0.,
         0.,  0., -0., -0.,  0.,  0., -0., -0., -0., -0.,  0., -0., -0.,
        -0., -0.,  0., -0., -1.,  0., -0., -1.,  0.,  0.,  0.,  0.,  0.,
         0., -0.,  0., -0.,  0.,  0., -0.,  1.,  0., -0., -0., -0., -0.,
        -0., -0., -0.],
       [ 1., -0., -0., -0.,  0., -0., -0.,  0., -1.,  0.,  0.,  0., -0.,
        -0.,  0., -0.,  0.,  0., -0.,  0., -0., -0., -0.,  0.,  0., -0.,
        -0., -0.,  0.,  0., -0., -0., -0., -0.,  1., -0., -0.,  0., -0.,
        -0.,  1., -0.,  0.,  0., -0., -0.,  0., -0., -1., -0.,  1., -0.,
        -0., -0., -0.,  1.,  0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0.,
        -0., -0.,  0.],
       [-0., -0., -1.,  1., -0.,  0., -0.,  0., -0.,  0.,  1.,  1.,  0.,
         0.,  0., -0., -0., -0., -1.,  0., -0.,  0., -0., -0., -1.,  0.,
        -0., -0., -0.,  0., -0., -1.,  0., -1.,  0.,  0.,  0.,  0.,  0.,
        -0., -0., -1.,  0.,  0., -0., -1., -0., -0., -0., -0., -0., -0.,
         0.,  1., -1.,  0.,  0., -1., -0., -0.,  0.,  0.,  1.,  1.,  0.,
        -0., -0., -0.],
       [ 0.,  0., -0.,  0.,  0.,  0., -0., -1., -0.,  0., -0., -0.,  0.,
         0., -1.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,
        -0., -0., -0.,  0., -0.,  0.,  0.,  0., -1.,  0.,  1.,  0., -0.,
        -0.,  0.,  0., -0., -0.,  0.,  0.,  1.,  0.,  0.,  0.,  0., -0.,
        -0., -0., -0., -0.,  0., -0.,  0., -1.,  0.,  0.,  0., -0., -0.,
         1.,  0.,  0.],
       [-0., -0.,  0., -0.,  0., -0., -0., -1., -0., -0.,  0.,  0., -0.,
        -0.,  0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,  1., -0., -1.,
         0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,  0.,  1.,  0., -0.,
         0., -0.,  0., -0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  1.,
         0.,  0.,  0.,  0.,  1.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,
         1.,  0., -0.],
       [ 1., -0.,  0.,  0.,  0., -0., -0., -0., -1., -0.,  0.,  0., -0.,
        -0., -0.,  1., -0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,
         0.,  0., -0., -0., -0., -0., -0.,  0., -0., -0.,  0.,  0., -0.,
         0.,  1.,  0.,  0.,  0., -0., -0., -0.,  0.,  0., -1.,  0., -1.,
         0., -0., -0., -0.,  0.,  0., -1.,  0., -0., -0.,  0., -0., -0.,
        -0., -0., -0.],
       [-0.,  0., -0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  2.,
         1., -1., -0.,  1.,  0.,  0., -0., -0., -1.,  1., -0., -0., -0.,
         0., -0.,  0., -0., -0.,  0.,  1., -0., -0., -1.,  0.,  0., -0.,
         0.,  0.,  0.,  0.,  1., -0.,  0.,  1.,  0.,  1., -0.,  0., -0.,
        -0., -0., -0.,  0., -0.,  0., -0., -1., -0.,  0., -0., -0.,  1.,
        -1.,  0., -0.],
       [-1., -0., -0., -0., -0., -0., -0., -0., -0.,  0., -0., -0., -0.,
         0., -0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0., -0., -0.,
        -0., -0., -0.,  0., -0., -0.,  0., -0., -1., -0.,  0., -0., -0.,
        -0., -1.,  0., -0.,  0., -0., -0.,  0.,  0.,  0., -0., -1.,  0.,
        -0., -0.,  0., -1., -0., -1., -0.,  0., -0.,  0., -0., -0., -0.,
        -0.,  0., -0.],
       [-0.,  0.,  1., -1.,  0., -0., -0.,  0., -0.,  0., -1.,  0.,  0.,
         0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  1., -0.,
        -0.,  0., -0., -0.,  0., -0., -0., -0., -0., -0., -0., -0., -0.,
        -0., -0., -0.,  0., -0., -0.,  1.,  0., -0.,  0., -0.,  0., -0.,
        -0.,  0.,  0., -0., -0., -0., -0., -0.,  0.,  0., -0., -1., -0.,
        -0.,  0., -0.],
       [-0., -1., -0., -0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,
         0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0., -0., -0.,  0., -0.,
         0.,  0., -0., -0., -0.,  0.,  0., -0., -0.,  0., -0., -0., -0.,
         1., -1., -0., -1., -0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,
        -0.,  0.,  0., -1., -0.,  0., -0.,  0., -0.,  1.,  0., -0.,  0.,
        -0.,  0., -0.],
       [ 0., -0., -0.,  0., -0.,  0.,  1.,  1., -1.,  0., -0.,  0., -0.,
        -0., -0., -0., -0., -0.,  0., -1., -0.,  0.,  0., -2.,  0.,  1.,
         0.,  0.,  1., -0.,  0.,  0., -0.,  0., -0.,  0., -1.,  0., -0.,
        -0., -0., -0.,  0., -0., -1., -0.,  1., -0.,  0., -0., -0., -1.,
         0., -0., -0.,  0., -1., -0., -1., -0., -0., -0., -0.,  0.,  0.,
        -1.,  0.,  1.],
       [-0., -0.,  1.,  0.,  0.,  1.,  0.,  0., -0.,  0., -1., -0.,  0.,
        -0., -0.,  0.,  0., -0., -0., -0.,  0., -0., -0., -0.,  0., -0.,
        -0., -0., -0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0., -0.,  0.,
        -0., -1.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,
         0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0., -1., -0.,
        -0.,  0., -0.],
       [ 0.,  0., -0., -0., -0.,  0., -0.,  0.,  0., -0., -0.,  0., -1.,
        -1.,  1.,  0., -1., -0.,  0., -0., -0.,  1., -0.,  0.,  0.,  0.,
        -0.,  0.,  0., -0., -0.,  0., -1., -0.,  0.,  0.,  0.,  0.,  0.,
        -0., -0., -0.,  0., -1.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,
        -0.,  0.,  0., -0., -0.,  0., -0.,  1.,  0.,  0., -0., -0.,  0.,
         1., -0., -0.],
       [ 1., -0.,  0.,  0.,  0., -0., -0.,  0., -1., -0., -0., -0.,  0.,
        -0., -0., -0.,  0., -0.,  0., -0., -0., -0.,  0., -0., -0.,  0.,
         0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  1.,  1.,  0.,  0., -0.,
        -1.,  1., -0., -0., -0., -0.,  0., -1.,  0., -1., -0.,  1.,  0.,
        -0., -0., -0.,  1., -0.,  1.,  0., -0.,  0.,  0.,  0.,  0., -0.,
        -1.,  0.,  1.],
       [ 0.,  0., -0., -0., -0., -0.,  0.,  0.,  0., -0.,  0., -1.,  0.,
        -0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0.,
         0.,  1.,  0.,  1.,  0.,  1., -0., -0.,  0., -0.,  0., -0.,  0.,
         0., -0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0., -0.,  0., -0.,
        -0., -1.,  1., -0., -0.,  0.,  0., -0., -0., -1., -1., -0., -0.,
        -0.,  1., -0.],
       [-0.,  0., -0., -0., -0., -0., -0., -1.,  0.,  0., -0., -0., -0.,
        -0., -1.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0., -0.,
         0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0.,
         0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0.,
        -0., -0., -0., -0.,  0.,  0., -0.,  0.,  0., -1.,  0.,  0., -0.,
        -0., -0.,  0.],
       [-0.,  0.,  0.,  0., -0., -0., -1., -1., -0., -0.,  0., -0.,  0.,
         0., -0., -0., -0.,  0., -0.,  1.,  0., -0., -0.,  1., -0., -1.,
        -0.,  0., -1.,  0., -0.,  0., -0.,  0.,  0., -0.,  1., -0., -0.,
         0.,  0., -0.,  0., -0.,  0.,  0., -1.,  0., -0., -0., -0.,  1.,
         0.,  0.,  0.,  0.,  1.,  0., -0., -0., -0., -0.,  0.,  0., -0.,
         0., -0.,  0.],
       [ 0.,  0., -1., -0.,  0., -1., -0., -1.,  0., -0.,  1.,  0.,  0.,
        -0., -0., -1., -0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,
         0.,  0., -0., -0., -1., -0., -0., -0.,  0., -0.,  0., -0.,  0.,
         0., -0.,  0., -1., -0., -0., -0., -0.,  0.,  0.,  0., -0., -0.,
         0.,  0.,  0., -0.,  1., -0., -0., -0., -0.,  0., -0.,  1., -0.,
        -0., -0., -0.],
       [ 0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -1.,
        -1.,  0.,  0., -1.,  0.,  0.,  0., -0., -0., -0.,  0.,  0., -0.,
        -0., -0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,
        -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,
         1., -0., -0., -0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0., -0.,
         1., -0.,  0.],
       [ 0.,  0., -0.,  1.,  0., -0.,  0.,  0.,  1.,  0., -0., -0., -0.,
        -0., -0.,  0., -0., -1., -0., -0.,  0.,  0., -1.,  0.,  0.,  0.,
         0.,  0., -0., -0.,  0.,  0.,  0., -1.,  0.,  0.,  0., -0.,  0.,
         0.,  0.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,
        -1., -0., -0., -0., -0., -0.,  0., -0., -0.,  0., -0.,  0.,  0.,
        -0., -0.,  0.],
       [-0.,  0., -0.,  1., -0.,  0., -0., -0.,  0., -0., -0., -0.,  0.,
         0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0., -0., -0., -0.,
         0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,  0.,
         0., -1.,  1.,  0., -0., -0., -0., -0.,  0.,  0., -1., -0., -0.,
         0.,  0.,  0.],
       [ 0.,  0., -0.,  1., -0.,  0., -0., -1., -0.,  0., -0.,  0.,  0.,
        -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  1., -0.,  0.,  0., -0.,
        -1.,  0., -0., -1.,  0., -0., -0.,  0., -0.,  0., -0., -0., -0.,
         0., -0.,  0., -0., -0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0.,
        -1.,  0.,  0., -0.,  0.,  0., -0.,  0., -0., -0., -0., -0., -0.,
         0., -0., -0.],
       [-0., -0., -0., -0.,  0.,  0.,  0., -0.,  0., -0., -0., -0.,  1.,
         1.,  0., -0., -0.,  0., -0.,  0.,  1.,  0.,  0., -0., -0., -0.,
        -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0., -0.,  0.,  1.,  1.,
         0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,  0.,
         0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0.,
        -1.,  0., -0.],
       [-0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0., -0., -0., -0.,  0.,
         0., -0.,  0., -0.,  0., -0.,  0., -0., -0.,  0., -0.,  1., -0.,
        -0.,  0.,  0.,  0., -0.,  1.,  0., -0., -0., -0., -0., -0., -0.,
         0.,  1., -0., -0.,  0., -0., -0.,  0., -1., -0.,  0.,  0.,  0.,
         0.,  0.,  1., -0.,  0.,  0., -0.,  0.,  0., -0., -1.,  0.,  0.,
         0., -0.,  0.],
       [ 0., -0., -0.,  0., -0., -0.,  0., -0., -0.,  1.,  0., -0., -0.,
        -0., -0., -0., -0.,  0.,  0.,  1.,  0., -0., -0.,  1.,  0.,  0.,
        -0.,  0., -0.,  0.,  0., -0., -0., -0., -0., -0.,  0., -0., -1.,
        -0.,  0., -0., -0., -0.,  0.,  0., -1., -0., -0.,  0.,  0., -0.,
        -1.,  0., -0., -0.,  0.,  0., -0., -0., -0., -0., -0., -0.,  0.,
         0., -0.,  0.],
       [ 0.,  0.,  1.,  0., -1., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,
        -0., -0.,  0.,  0., -0.,  1., -0.,  0., -0.,  0.,  0., -0.,  0.,
         0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0., -0.,  0., -1.,  0.,
         0., -0.,  0., -0., -0.,  0., -0., -0., -0., -1.,  0.,  0., -0.,
         0., -0., -0., -0.,  0., -0.,  1., -0.,  0.,  0.,  0.,  0., -0.,
        -0.,  0.,  0.],
       [-0.,  0.,  1., -0., -0., -0.,  0., -0.,  0.,  0., -0.,  1., -0.,
        -0.,  0., -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0., -0., -0.,
         0., -0.,  0.,  0.,  0., -1., -0., -0.,  0.,  0.,  0.,  0.,  0.,
         0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,
         0.,  1., -1.,  0., -0., -0., -0., -0., -0.,  0., -0., -0.,  0.,
        -0.,  0.,  0.],
       [-0.,  0.,  1., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0.,
        -0.,  1.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0.,
        -0.,  0., -0., -0.,  0., -1.,  0.,  0., -0.,  0., -0., -0.,  0.,
         0., -0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,
        -0.,  0.,  0., -0.,  1.,  0., -0.,  0., -1., -0.,  0.,  0., -0.,
        -0.,  1.,  0.],
       [ 0.,  0., -0., -0.,  0., -0., -0., -0.,  0., -0., -0., -0.,  0.,
         0.,  0., -0., -0.,  0., -0., -1., -0.,  0., -0., -1.,  0., -0.,
         0.,  0.,  1., -0.,  0.,  0., -0., -0.,  0., -0., -0., -1.,  0.,
         0., -0.,  0., -0., -0.,  0.,  0.,  1., -0., -0., -0., -0., -0.,
         0.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,
         0., -0.,  0.],
       [-0., -0.,  1.,  0.,  0.,  1.,  0.,  0., -0., -0., -0.,  0., -0.,
         0., -0.,  1., -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0., -0.,
         0., -0., -0., -0., -0., -1.,  0., -0.,  0.,  0., -0.,  0.,  0.,
         0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,
        -0., -0.,  0.,  0., -0.,  0., -0., -0.,  0., -0.,  0., -1., -0.,
        -0., -0., -0.]], dtype=np.float32), np.array([[ 0.,  1., -0., -0.,  0.,  0., -0.,  0., -0.,  1.,  0.,  0.,  0.,
        -0., -0.,  0.,  0., -1., -0., -1., -0., -1.,  1., -0.,  0.,  0.,
        -1.,  0., -1.,  0., -0.,  0., -0., -0., -1.,  1., -0., -0.,  0.,
         1., -0., -0., -0.,  1.,  0.,  0.,  1.,  0.,  0.,  0.,  1.,  0.,
        -1., -0.,  0., -1., -0.,  0., -0.,  1., -0.,  0.,  0., -0.,  1.,
         0., -0.,  0.],
       [ 0.,  1.,  0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,  0.,  0.,
        -0., -0., -0., -1., -0.,  0., -1.,  0., -0., -0.,  0., -0., -0.,
        -0., -0., -1., -0.,  0., -0.,  1.,  0., -1.,  1.,  0.,  0.,  0.,
         1., -0.,  0.,  0.,  0., -0.,  0.,  1.,  0., -0., -0.,  1.,  0.,
         0., -0., -0., -1.,  0.,  0., -0.,  1., -0., -0., -0.,  0., -0.,
         0., -0., -0.],
       [ 0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  1.,
         1.,  0., -0.,  1., -1.,  0., -0., -1., -1.,  1.,  0., -0., -0.,
        -1., -0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  1.,
         0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,
        -1.,  0., -0., -0.,  0., -0., -0., -0., -0., -0.,  0.,  0., -0.,
        -0.,  0., -0.],
       [-0.,  1.,  0., -0.,  1.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,
        -0., -0., -0., -0., -1., -0., -0.,  0.,  0.,  1.,  0.,  0., -0.,
         0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,  1.,  0., -0.,  0.,
         1.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0., -1., -0.,  0., -0.,
         0.,  0., -0., -1., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  1.,
        -0.,  0., -0.],
       [ 0.,  1., -0.,  0.,  1., -0., -0.,  0., -0.,  0.,  0.,  0., -0.,
         0., -0., -0.,  0.,  1.,  1.,  0., -0.,  0.,  0., -0.,  0., -0.,
         0., -0.,  0., -0., -0., -0.,  0.,  1., -0., -0., -0., -0.,  0.,
         1.,  0., -1.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,  1., -0.,
         0., -0., -0., -1.,  0.,  1.,  0.,  0., -0., -0.,  0., -0.,  0.,
         0., -0.,  0.],
       [-1., -1., -0., -0.,  1.,  0., -0., -0.,  1.,  0., -0.,  0., -0.,
         0., -0., -0.,  0., -1.,  0., -0., -0., -0.,  0., -0., -0., -0.,
        -0., -0., -0., -0., -0., -0., -0., -0., -0.,  0.,  0., -0.,  0.,
        -1.,  0., -0., -0., -0.,  1., -0.,  0.,  0.,  0., -1., -1., -0.,
        -0., -0., -0.,  1., -0.,  0., -1.,  0.,  0.,  0.,  0., -0., -0.,
         0.,  0.,  1.],
       [ 0., -0., -1., -0., -1., -1.,  0., -0.,  0., -0.,  0.,  1.,  0.,
        -0., -0.,  0.,  0., -0., -1.,  0.,  0.,  0., -0., -0., -0.,  0.,
        -0., -0., -0., -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,
         0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,
         0., -1.,  0.,  0., -0.,  0., -0.,  0., -1.,  0., -0., -1., -0.,
         0.,  1.,  0.],
       [ 0., -0., -1.,  1., -1., -1.,  0., -0.,  0., -0.,  0.,  0.,  0.,
        -0., -0., -0., -0., -1., -1.,  0.,  0., -0., -0., -0., -0.,  0.,
        -1.,  1.,  0., -1., -0.,  0.,  0., -1.,  0., -0.,  0., -0., -0.,
        -0., -0.,  1., -0., -0., -0., -1.,  0., -0.,  0., -0., -0.,  0.,
         0., -0.,  0., -0., -0., -0., -0., -0., -1.,  0., -0., -1.,  0.,
        -0.,  1.,  0.],
       [ 0., -0., -0., -1.,  0.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,
         0., -0.,  0., -0.,  1., -0.,  0., -0., -0.,  0.,  0., -1., -0.,
         1.,  0.,  0.,  1., -0.,  0.,  0.,  1., -0.,  0.,  0.,  0.,  0.,
         0., -0., -0.,  0., -0.,  0., -0., -0., -1.,  0.,  0., -0.,  0.,
        -0., -1.,  1., -0.,  0.,  0., -0., -0.,  0., -0.,  1., -0., -0.,
        -0.,  0.,  0.],
       [ 0.,  0.,  0., -0., -0.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,
        -0.,  1., -0., -0., -0., -0.,  0., -0., -1.,  0., -0., -0.,  0.,
        -1.,  0., -0.,  0.,  0., -0.,  0., -0., -1.,  0., -0.,  0.,  0.,
         0.,  0., -0., -0.,  1.,  0., -0., -0., -0.,  0., -0.,  1.,  0.,
        -0.,  0., -0., -1., -0.,  0.,  0.,  1.,  1.,  0.,  0.,  0., -0.,
         0.,  0.,  0.],
       [ 0.,  1., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,
         0., -0., -0., -0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,
         1., -1.,  0.,  1.,  0., -0., -0., -0., -1., -0., -0., -0.,  0.,
         0.,  0., -0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,  1., -0.,
        -0., -0., -0., -1.,  0., -0.,  0.,  0.,  1., -1., -0., -0.,  0.,
        -0., -1.,  0.],
       [-0., -1.,  0.,  0., -0., -0., -0.,  1., -0., -0., -0., -0., -0.,
         0.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  0.,  0., -0., -1.,
        -1., -0., -0., -0., -1., -0.,  0., -0.,  1., -0.,  1.,  0.,  0.,
        -0., -0.,  0., -1.,  0.,  0.,  0.,  0., -0., -0.,  0., -1.,  0.,
         0.,  0., -0.,  1., -1.,  0., -0.,  0.,  1.,  0., -0.,  0., -0.,
         0., -0., -0.],
       [-0.,  0.,  0.,  0., -1.,  0., -0.,  0., -0.,  1.,  0., -0.,  0.,
        -0., -0.,  0., -0.,  0.,  0., -1., -0.,  0., -0., -1., -0., -1.,
         0.,  0., -1.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,  1., -1.,
         0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,  1., -0.,  1.,
        -0., -0.,  0., -0.,  0., -0.,  1., -0.,  0.,  0., -0., -0.,  0.,
         0.,  0.,  0.],
       [ 0., -1., -0.,  0., -0.,  0., -1., -0.,  0., -0., -0., -0., -0.,
        -1.,  0.,  0., -1.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,
         0., -0., -1., -0.,  0., -0., -0.,  0.,  1.,  0.,  1.,  0., -0.,
        -1., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -1.,  0.,
        -0., -0., -0.,  1., -0., -0.,  0., -0.,  0.,  0.,  0., -0., -0.,
         1.,  0.,  1.],
       [ 0.,  1.,  0.,  0., -1.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,
         1., -0., -0.,  1.,  0.,  0., -0., -1.,  0., -0.,  0., -0., -0.,
         0., -0., -0., -0.,  0.,  0.,  0., -0., -1., -0., -1.,  1., -0.,
         1., -0., -0., -0.,  0., -1.,  0., -0., -0.,  0.,  1.,  1.,  1.,
        -0., -0., -0., -1.,  0.,  0.,  1., -0., -0., -0., -0., -0., -0.,
        -1., -0., -1.],
       [ 0., -0.,  0., -0.,  0.,  1.,  0.,  0.,  0.,  0.,  0., -0., -0.,
         0., -0.,  1.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,  0.,  1.,
        -0.,  0., -0., -0., -0.,  1., -0.,  0., -0.,  0.,  0., -0., -0.,
        -0., -0., -0.,  0., -0., -0.,  0., -0., -0.,  0., -1.,  0., -1.,
        -0.,  1., -1.,  0.,  1.,  0.,  0.,  0., -1.,  0., -0.,  0.,  0.,
        -0.,  0.,  0.],
       [-1., -1.,  0., -0., -0.,  1., -0.,  0.,  0., -0.,  1.,  0.,  0.,
        -0., -0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,  0.,  1., -0.,
        -0., -0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,
         0., -1., -0., -1.,  0., -0.,  1.,  0.,  0.,  0.,  0., -1., -0.,
        -0., -0., -0.,  1.,  0., -0., -0., -0., -0., -0., -0.,  1.,  0.,
         0.,  0., -0.],
       [ 1.,  1.,  0., -0.,  0., -0.,  0., -0.,  0.,  0., -0., -0.,  0.,
        -0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0., -0.,  1.,
        -0.,  0., -0., -0.,  1.,  1., -0.,  0.,  0., -0.,  0.,  0., -0.,
         0.,  1.,  0.,  1.,  0., -0.,  0.,  0.,  1.,  0.,  0.,  1., -1.,
         0.,  1., -1., -1.,  1., -0.,  0.,  0., -1.,  0.,  0., -0., -0.,
        -0.,  0.,  0.]], dtype=np.float32))

In [ ]:
#@title Verification
n = 3
m = 5
p = 6
rank = 68

verify_tensor_decomposition(decomposition_356, n, m, p, rank)

## Rank-80 decomposition of <3,5,7> over Z

In [ ]:
#@title Data

decomposition_357 = (np.array([[ 0., -0., -0., -0., -0., -0., -0.,  0., -0.,  0., -0.,  0., -1.,
         0., -1., -0.,  0., -0., -0.,  1.,  0., -0., -0., -0., -0., -0.,
         0., -0.,  0.,  0.,  0., -1., -0.,  0.,  0.,  0.,  0.,  1., -0.,
        -0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  1., -0.,
        -0.,  0., -1.,  0.,  0., -0.,  0.,  0., -0.,  1., -0.,  0.,  1.,
         0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0., -1.,  1., -0.,
         0.,  1.],
       [-1., -1., -0.,  0., -0.,  0., -0.,  0., -1., -0., -0., -1.,  1.,
         1., -0., -0.,  0., -1., -0., -0., -0., -0.,  0.,  1.,  0.,  0.,
         0., -1.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,
         0., -0., -1.,  0., -0.,  0., -1.,  0., -0., -0.,  0.,  0.,  0.,
        -0., -0., -0.,  0., -0., -0., -1., -0., -0.,  0., -0.,  0., -0.,
         0.,  0., -1.,  0., -0.,  1.,  1., -0., -0., -1., -0., -0., -0.,
        -0., -1.],
       [ 0.,  0.,  1.,  1.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,  1.,
         0.,  0.,  1., -0.,  0.,  0.,  1., -0., -0., -1., -0.,  0.,  0.,
         0., -0.,  0., -0.,  1., -0.,  0.,  0.,  1., -0., -0.,  0.,  1.,
         0., -0.,  0., -0., -1.,  0.,  0., -0.,  1., -0.,  0., -0., -1.,
        -0., -0., -0., -0., -0.,  0.,  0., -0., -1., -0., -0.,  0.,  0.,
        -0., -0., -0., -0., -0., -0.,  0., -0., -0.,  0., -0., -0.,  0.,
        -1., -0.],
       [-1., -0., -0., -0.,  0.,  0., -0., -0., -1.,  0., -0., -0.,  0.,
         1., -1., -0.,  0., -0.,  0., -0., -0.,  0.,  0.,  1.,  0., -0.,
         0.,  0.,  0.,  0.,  0., -1.,  0.,  0.,  0., -0., -0.,  1., -0.,
         0., -0., -1.,  1.,  1.,  0.,  0.,  0.,  0., -0., -0.,  1., -0.,
         0., -0., -1., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  1.,
        -0.,  1., -0., -0., -0.,  0.,  0.,  0.,  0., -1., -1.,  1., -0.,
         0., -0.],
       [ 1., -0., -1., -0., -0., -0.,  0., -0.,  1.,  0.,  0., -0.,  0.,
        -1.,  1., -1.,  0., -1., -0., -0., -0., -0., -0., -0.,  0.,  0.,
         0., -1., -0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0., -1.,  0.,
        -1.,  0.,  1., -0., -0., -0., -1.,  1.,  0., -0.,  0.,  0.,  1.,
         0., -0.,  0.,  0., -0., -0.,  0.,  0.,  1., -0.,  0., -0., -1.,
         0., -1.,  0.,  0., -0.,  1., -0.,  0., -0.,  1., -0., -1.,  0.,
        -0.,  0.],
       [-0.,  0., -0., -0.,  0.,  1., -0., -0.,  0.,  0., -0., -0., -0.,
         0.,  1.,  0., -0., -0.,  1., -0.,  0.,  0.,  0.,  0., -1., -0.,
         0., -0., -1., -0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,
         0.,  0., -0.,  0.,  0.,  1.,  0., -0.,  0.,  0., -1.,  0.,  0.,
         0., -0., -0., -1.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,
        -0., -0., -0.,  0., -0., -0., -0.,  0., -0.,  0., -0., -0.,  0.,
        -0., -0.],
       [ 0.,  0., -0.,  0., -0., -1., -0., -1., -0., -0., -0.,  0., -0.,
         0.,  0., -0.,  0.,  0.,  0.,  0., -0., -1., -0., -0.,  0., -0.,
        -1.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  1., -1.,  0.,  0.,
         0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        -0., -0.,  0.,  1.,  0., -0., -0., -0., -0., -0., -0.,  0.,  0.,
         1., -0.,  0., -1.,  1., -1.,  0., -0., -0., -0.,  0., -0., -1.,
         0.,  0.],
       [-0., -0., -0., -0., -1.,  0., -1., -1.,  0.,  0., -0.,  0., -0.,
         0., -0., -1., -0., -0.,  1.,  0., -0., -1.,  0.,  0., -0., -1.,
         0.,  0.,  0., -1.,  0.,  0.,  1.,  0.,  0.,  0., -1., -0., -0.,
         0.,  0., -0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,
        -0., -0.,  0.,  1.,  0., -0., -0.,  0.,  0., -0.,  1.,  1.,  0.,
         1.,  0.,  0., -1., -0.,  0.,  0., -0., -1.,  0.,  0., -0., -1.,
         0., -0.],
       [ 0., -0., -0., -0.,  0., -0., -0.,  0., -0.,  0., -1.,  0., -0.,
        -1.,  1., -0., -0., -0., -0.,  0.,  0., -0.,  0.,  0., -1., -0.,
        -1.,  0., -0.,  0., -0., -0.,  0.,  0.,  0.,  1.,  0., -1., -0.,
         0., -0.,  0., -0., -0.,  1., -1.,  0., -0.,  0., -0.,  0.,  1.,
        -0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,  0., -0., -1., -0.,
        -0., -1., -0., -0.,  1.,  0.,  0., -0., -0., -0., -0., -0.,  0.,
         0.,  0.],
       [-1., -0.,  1.,  0., -0.,  0.,  0., -0.,  0., -0.,  1.,  0.,  0.,
         1., -1.,  1.,  0.,  0.,  0.,  0.,  1.,  0., -0.,  0.,  0., -0.,
         0.,  1.,  0., -0.,  0.,  0., -1.,  0.,  0.,  0., -0.,  1., -0.,
        -0., -0., -0.,  0.,  0., -1.,  1., -1.,  0., -0.,  0., -0., -1.,
         0., -0.,  0., -0.,  0., -0., -0.,  0.,  0., -0., -0., -0., -0.,
        -0.,  1.,  0., -0., -0., -1., -0., -0., -0., -0., -0.,  1.,  0.,
        -0.,  0.],
       [ 0., -0., -0., -1., -0.,  0.,  1.,  0., -0.,  0.,  0.,  0., -0.,
         0.,  0.,  0., -1., -0.,  1., -1., -0., -0., -0., -0., -0., -1.,
        -0.,  0., -0., -0.,  0., -0., -0., -0., -0., -0., -0., -0., -1.,
         0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,  0., -0., -1.,  0.,
         0., -1.,  0., -1., -1., -0., -0.,  1.,  0., -0.,  0.,  0.,  0.,
         0.,  0., -1., -1.,  0., -0., -0.,  1.,  0., -0., -0., -0.,  1.,
        -0., -1.],
       [-0.,  0.,  0.,  1., -0.,  0., -1., -0., -0.,  0., -0., -0., -0.,
         0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,  0.,  0., -0., -0.,
         0., -0., -0.,  0., -0., -0.,  0.,  1.,  0., -0., -1.,  0.,  0.,
        -0., -0., -0., -0., -0., -0., -0., -0.,  0.,  1.,  0.,  0.,  0.,
         0.,  1., -0.,  1.,  1., -0.,  0., -0.,  0.,  0.,  0., -0., -0.,
        -0.,  0.,  1., -0.,  0.,  0., -1.,  0., -0.,  1.,  0.,  0., -1.,
        -0.,  1.],
       [ 0., -0.,  0.,  0.,  0.,  0., -1., -0.,  0.,  1.,  0., -0.,  0.,
        -0.,  0.,  0.,  0., -0.,  1., -1.,  0., -1.,  0., -0.,  0., -1.,
        -0., -0., -0., -1.,  0.,  0.,  1., -0., -1., -0., -1.,  0., -1.,
         0.,  1., -0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0.,
         0.,  1.,  0.,  1.,  0.,  0., -0.,  1., -0., -0.,  1.,  1.,  0.,
         1., -0., -0., -1., -0., -0., -0., -0., -1.,  0., -0., -0., -1.,
        -0., -0.],
       [ 0., -1.,  0.,  0., -0., -0.,  0., -0.,  0., -1.,  0., -0., -0.,
        -0., -0.,  0., -1.,  0., -0., -0., -0.,  1.,  0.,  0.,  0.,  0.,
        -0.,  0., -0.,  1., -1.,  0., -0.,  1.,  0.,  0.,  0.,  0., -0.,
        -0.,  0., -0., -1.,  0.,  0.,  0., -0.,  0.,  0.,  0., -1., -0.,
        -0., -0., -0., -0.,  0., -1.,  0.,  0., -0.,  0., -0., -1., -0.,
         0.,  0., -0., -0., -0., -0.,  0., -0.,  0.,  1.,  1.,  0.,  0.,
         0.,  0.],
       [-0.,  0., -0.,  0.,  0.,  0., -0.,  0., -1.,  0.,  0.,  0., -0.,
         0.,  0., -0.,  1., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0.,
         0.,  0., -0., -0.,  0., -0., -1., -0.,  0., -0.,  0.,  0., -0.,
         1., -1.,  0., -0., -0.,  0., -0.,  0., -1.,  1., -0.,  0., -0.,
        -1., -0.,  1., -0.,  0., -0., -1., -0.,  0., -0., -1., -0., -0.,
        -1.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0., -1.,  0., -0.,  0.,
         0., -0.]], dtype=np.float32), np.array([[-0.,  0., -0., -0.,  0.,  0., -0.,  0., -1., -0.,  0., -0.,  0.,
         0., -0., -0., -1., -0.,  0.,  0.,  0.,  1., -0., -0., -0.,  0.,
        -1.,  0.,  0., -0., -0., -0.,  1.,  1., -0., -0.,  0., -0.,  0.,
        -1., -0., -0., -0.,  0.,  1., -0.,  0., -0.,  1.,  0., -1., -0.,
         0., -0., -1.,  0.,  0., -0., -1., -0., -0.,  0.,  0.,  1.,  0.,
        -1.,  0., -0., -0.,  0.,  0.,  1.,  0., -0.,  1.,  0., -0., -0.,
        -0., -0.],
       [-0., -0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,
        -0.,  1., -0., -0., -0., -0.,  0.,  0., -0., -0., -0.,  0., -0.,
        -0.,  0., -1.,  0., -0.,  0., -0.,  0.,  0., -0.,  1., -1., -0.,
        -0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0., -0., -1., -0., -0.,
         0., -0.,  0.,  1.,  0., -0.,  0.,  0.,  0.,  0., -0., -0., -0.,
        -0.,  1.,  0., -0.,  0., -0., -0., -0., -0.,  0.,  0., -0., -1.,
        -0.,  0.],
       [-1.,  0.,  1.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,
         1., -1.,  0.,  0., -0.,  1., -0.,  0.,  0.,  0., -0., -1., -1.,
         0., -1.,  1.,  0., -0.,  0., -0., -0., -0.,  0., -0., -0.,  0.,
         0.,  0., -0.,  0.,  0., -0.,  1., -0.,  0.,  0., -0., -0., -1.,
        -0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -0., -0.,
         0., -1., -0.,  0.,  0., -0., -0.,  0., -1., -0., -0., -1.,  0.,
         1., -0.],
       [-0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,
        -0.,  1., -0.,  0., -0.,  1., -0.,  0.,  0.,  0., -0.,  0., -1.,
        -0., -0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -1., -0.,
        -0., -0., -0.,  1.,  0., -0., -0., -0.,  0., -0., -0., -1.,  0.,
        -0., -0.,  0., -0.,  0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,
        -0.,  1.,  0., -0., -0.,  0., -0., -1., -1., -0.,  1., -0.,  0.,
        -0., -0.],
       [-0., -0.,  0., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0., -0.,
         0., -1.,  0., -0.,  0.,  0., -1., -0.,  0.,  0.,  0., -0., -0.,
        -0., -0.,  1.,  0., -0., -0., -0., -0., -1., -0., -0.,  1.,  1.,
        -0.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,
         0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  1.,  0.,  0.,  0.,
         0., -1., -0., -0.,  0., -0., -0.,  0., -0.,  0., -0., -0., -0.,
        -0., -0.],
       [ 0., -0., -0., -0., -0., -0., -0.,  0.,  0., -0., -0.,  0.,  0.,
         0., -0.,  0., -0.,  0., -1., -0., -0.,  0., -0., -0.,  0.,  1.,
         0.,  0.,  0.,  0., -0., -0., -0., -0., -0., -0., -0., -0.,  0.,
         0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  1.,  0.,  0.,  0.,
         0.,  0.,  0., -0., -0., -0., -0.,  1.,  1., -0.,  0., -0.,  0.,
        -0., -0.],
       [ 1., -0., -1.,  0.,  0.,  0., -0., -0., -0., -0.,  0.,  0., -0.,
        -1.,  1., -0., -0., -0.,  0., -0.,  0.,  0., -0.,  0.,  1.,  0.,
        -0.,  1., -1., -0.,  0., -0., -0., -0., -0.,  0., -0.,  0.,  0.,
        -0.,  0.,  0.,  0., -0.,  0., -1.,  0., -0., -0.,  1., -0.,  1.,
         0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0., -1.,  0.,  0., -0.,
         0.,  1., -1.,  0., -0.,  0.,  1.,  0., -0., -0., -0.,  1., -0.,
        -1., -1.],
       [-0.,  0., -0., -0., -0.,  0.,  0.,  0., -0., -0., -0., -0.,  0.,
        -0.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,
        -0.,  0., -0.,  0., -0., -0., -0.,  0.,  0., -0.,  1.,  0.,  0.,
        -0., -0.,  0.,  0., -0., -0., -0., -0., -0.,  1., -0., -0., -0.,
         1., -0., -0., -0., -0., -0., -1., -0.,  0.,  0., -1.,  0.,  0.,
        -1.,  0., -0., -0., -0.,  0.,  1., -0.,  1.,  0., -0.,  0.,  0.,
         0.,  0.],
       [-0.,  0.,  1., -0., -0.,  1.,  0.,  0., -0., -0.,  0., -0.,  0.,
         0., -0.,  0.,  0.,  0., -1.,  0.,  0.,  0.,  0., -0., -0.,  0.,
        -0., -0.,  0.,  0., -0., -0.,  0., -0., -0., -0.,  2., -0., -0.,
         0.,  0.,  0.,  0.,  0., -0.,  1.,  1., -0., -0., -1.,  0., -1.,
         0.,  0., -0.,  1., -0.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,
        -0.,  0.,  0.,  1., -1., -1., -0.,  0.,  0., -0., -0., -0., -1.,
         1., -0.],
       [ 0.,  0., -0., -0., -0., -1., -1., -0.,  0., -0.,  0., -0., -0.,
         0., -0., -0., -0., -0.,  1.,  0., -0., -0., -0., -0., -0., -1.,
         0., -1.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,
        -0.,  0.,  0.,  0.,  0., -0., -0., -1.,  0., -0.,  1., -0., -0.,
        -0., -0., -0., -1.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,
        -0.,  0., -0.,  0.,  0.,  1., -0.,  0., -2.,  0.,  0.,  0., -0.,
         0.,  0.],
       [ 0.,  1.,  1., -0., -0.,  0., -0.,  0., -0., -0.,  0.,  0., -0.,
         0.,  0., -0., -0., -0., -0., -0., -0.,  1., -0., -0.,  0., -0.,
        -0.,  0.,  0.,  1.,  0.,  0.,  0.,  1., -0.,  0., -1., -0.,  0.,
        -0., -0., -0.,  0., -0., -0.,  1.,  1., -0., -0., -0., -0., -1.,
        -0.,  0.,  0.,  0.,  0., -1., -0., -0.,  0., -0.,  0.,  0., -0.,
         0.,  0.,  0.,  0., -1., -1.,  1., -0., -1.,  0., -0.,  0.,  0.,
         1., -0.],
       [ 0., -0., -1., -1.,  0., -0., -0., -0., -0.,  0., -0.,  0.,  1.,
        -0.,  0., -0.,  0.,  0., -0., -1.,  0., -0., -1., -0.,  0.,  0.,
        -0.,  0., -0., -0., -0., -0.,  0.,  0., -1., -0., -0.,  0.,  1.,
         0., -0.,  0., -0., -0.,  0., -1., -1., -0., -0.,  0.,  0.,  1.,
         0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,  1.,  0.,  0.,  0.,
         0., -0., -0.,  0.,  1.,  1.,  0., -0.,  0., -0.,  0.,  0.,  0.,
        -0., -1.],
       [-0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -1.,  0.,
         0., -0.,  0.,  0., -0., -0., -0., -0., -0., -0.,  0., -0.,  1.,
        -0., -0., -0., -0.,  0., -0., -0.,  0., -0., -0., -1., -0., -0.,
         0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,
        -0., -0.,  0.,  0., -1., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,
        -0., -0., -1., -1., -0., -0., -0.,  1.,  1., -0.,  0., -0.,  0.,
        -0.,  0.],
       [-0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  1., -0.,
         0.,  0., -0., -0., -0., -0.,  0., -0.,  0., -0.,  0.,  0., -0.,
        -0.,  1., -0., -0.,  0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,
        -0., -0., -0.,  0., -0.,  0.,  0.,  1.,  0., -0., -0.,  0.,  0.,
         0.,  0., -0.,  0.,  0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,
        -0., -0., -0.,  0., -0., -1.,  1., -0.,  0., -0., -0., -0.,  0.,
         0.,  0.],
       [ 0.,  0., -0., -0., -0., -0.,  0., -0., -0., -0.,  0.,  0., -0.,
         0.,  0., -0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,
         0., -0.,  0., -0.,  0., -0., -0., -0., -1., -0., -0., -0.,  0.,
         0.,  1., -0.,  0., -0., -0.,  0.,  0.,  1., -0.,  0.,  0., -0.,
        -1.,  0.,  0., -0., -0.,  0., -0., -0., -0., -0.,  1.,  0.,  0.,
        -0., -0.,  0., -0.,  0., -0., -0., -0., -1.,  0., -0., -0.,  0.,
        -0., -0.],
       [ 0., -0., -1., -0., -0., -1.,  0.,  1.,  0., -0.,  0., -0.,  0.,
         0., -0., -1., -0.,  0.,  1., -0., -0.,  0.,  0., -0., -0.,  0.,
        -0.,  0., -0., -0., -0., -0.,  0., -0.,  0., -0., -1.,  0.,  0.,
         0., -0., -0.,  0., -0., -0., -0., -1., -0., -0.,  1., -0., -0.,
        -0., -0., -0., -1.,  0., -0., -0., -0.,  0., -0., -0., -0.,  0.,
        -0., -0., -0., -1., -0., -0., -0., -0., -0., -0.,  0.,  0.,  1.,
        -1., -0.],
       [ 0.,  0.,  1.,  0., -1.,  0.,  0., -0., -0., -0.,  0., -0., -0.,
        -0.,  0.,  1., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0.,
         0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,
         0., -0., -0., -0.,  0., -0., -0.,  1.,  0., -0.,  0., -0., -0.,
        -0., -0.,  0.,  0., -0., -0., -0., -0., -0.,  0., -0.,  0.,  0.,
        -0., -0., -0.,  0.,  0., -0., -0.,  0.,  1.,  0., -0.,  0., -0.,
         0.,  0.],
       [ 0., -0., -1., -0.,  0.,  0., -0., -0., -0., -1.,  0., -0.,  0.,
         0., -0., -1.,  0., -0.,  0., -0., -0.,  0., -0.,  0., -0., -0.,
         0., -0.,  0., -1.,  1.,  0.,  0., -0., -1., -0.,  0., -0.,  0.,
        -0.,  0., -0., -0.,  0., -0., -0., -1., -0.,  0., -0.,  0.,  0.,
        -0.,  0., -0.,  0., -0.,  1.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
        -0., -0.,  0.,  0., -0., -0.,  0., -0.,  1.,  0.,  0.,  0.,  0.,
        -1., -0.],
       [-0., -0.,  1., -0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,
        -0.,  0.,  1.,  0.,  0.,  0.,  0., -0.,  0.,  1., -0.,  0., -0.,
         0.,  0.,  0., -0.,  0., -0., -0.,  0.,  1.,  0.,  0., -0., -0.,
        -0.,  0., -0.,  0.,  0.,  0., -0.,  1.,  0., -0.,  0., -0.,  0.,
         0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,
         0., -0., -0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0., -0.,
        -0., -0.],
       [ 0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,
        -0.,  0.,  0., -0., -0., -0., -0.,  0., -0.,  1., -0.,  0., -1.,
        -0.,  0.,  0., -0., -0., -0., -0., -0.,  0.,  0., -0.,  0.,  1.,
        -0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,
        -0., -0.,  0.,  0., -0., -0.,  0., -1., -0.,  0.,  0., -0.,  0.,
        -0.,  0.,  0., -0., -0., -0.,  0., -1., -1., -0.,  0.,  0., -0.,
        -1.,  0.],
       [ 0., -0., -1.,  1.,  1.,  1.,  1., -1., -0., -0.,  0., -0., -0.,
        -0., -0., -1., -0.,  0., -1.,  0.,  0., -0., -1.,  0., -0.,  1.,
        -0.,  0., -0., -0., -0., -0.,  0.,  0., -1.,  0.,  1., -0., -1.,
        -0., -0.,  0.,  0., -0.,  0.,  0., -1.,  0.,  0., -1., -0.,  0.,
         0., -1.,  0.,  1.,  1.,  0., -0.,  1., -0.,  0.,  0.,  0.,  0.,
         0., -0., -0.,  1., -0., -0.,  0.,  1.,  1., -0.,  0.,  0., -1.,
         1., -0.],
       [ 0., -0.,  0.,  0., -0., -0., -0., -0.,  1.,  0., -0., -0., -0.,
        -0., -0.,  0.,  0., -0., -0., -0.,  0., -1., -0.,  0., -0.,  0.,
         1.,  0.,  0., -0., -0.,  0., -0., -1., -0.,  0.,  0.,  0., -0.,
         1.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -1., -0.,  0., -0.,
        -1.,  0.,  0., -0., -0., -0.,  1.,  0.,  0., -0.,  1., -1., -0.,
         1., -0., -0.,  0.,  0., -0., -1.,  0., -1., -1.,  0.,  0., -0.,
        -0., -0.],
       [-0.,  0., -1.,  0., -0., -1.,  0., -0., -0., -0., -0.,  0., -0.,
         0., -0., -0.,  0., -0.,  1., -0., -0., -1., -0.,  0., -0., -0.,
         1.,  0., -0., -0., -0.,  0.,  0.,  0.,  0., -1., -1., -0., -0.,
        -0., -0.,  0.,  0.,  0., -0.,  0., -1., -0., -0.,  1.,  0.,  1.,
        -0.,  0.,  0., -1.,  0., -0.,  0.,  0., -0.,  0.,  0., -1., -0.,
        -0., -1.,  0., -1.,  0.,  0.,  0., -0., -0.,  0.,  0., -0.,  1.,
        -1., -0.],
       [ 1.,  0., -1.,  0.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,
        -1., -0., -0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,
        -0.,  1.,  0., -1., -0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,
         0.,  0.,  0., -0.,  0.,  0., -1.,  1., -0., -0.,  0.,  0.,  1.,
        -0., -0., -0., -0.,  0., -0.,  0., -0., -0.,  0.,  0.,  1.,  0.,
        -0.,  1., -0., -0.,  0.,  0., -0., -0.,  1., -0., -0., -0.,  0.,
        -1., -0.],
       [-0., -0., -1.,  0.,  0., -0., -0.,  0., -0., -0., -0., -0., -0.,
         0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0., -0.,
         0.,  0.,  0., -1.,  0., -0., -0.,  0.,  0., -0., -0., -0.,  0.,
        -0., -0.,  0., -1., -0.,  0.,  0., -1.,  0.,  0., -0., -0.,  1.,
        -0., -0., -0., -0.,  0.,  1., -0., -0., -0., -0., -0.,  1., -0.,
         0., -1., -0.,  0., -0.,  0.,  0.,  0.,  1.,  0., -0.,  0.,  0.,
        -1., -0.],
       [-0., -0.,  1.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,
        -0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  1.,  0., -0., -0.,
         0.,  0.,  0.,  0.,  1., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,
         0., -0., -0.,  1., -1., -0.,  0.,  1.,  0., -0.,  0.,  0., -1.,
        -0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,
         0.,  1., -0.,  0., -0., -0., -0.,  0.,  0., -0., -0., -0., -0.,
         0., -0.],
       [-0.,  0.,  1.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,
        -0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,
        -0., -0.,  0.,  1.,  0.,  1., -0.,  0., -0.,  0.,  0.,  1., -0.,
         0.,  0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -1.,
        -0., -0., -0., -0., -0., -1., -0.,  0., -0., -1.,  0., -1.,  0.,
        -0., -0.,  0., -0., -0.,  0.,  0., -0., -1., -0., -1.,  1.,  0.,
         1., -0.],
       [ 0.,  1., -0.,  0.,  0., -0.,  0., -0., -0., -0.,  0., -1., -0.,
         0.,  0.,  0., -0.,  0.,  0., -0., -0., -0., -0., -1., -0., -0.,
        -0., -1.,  0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,
        -0., -0.,  0., -1., -0., -0.,  1., -1., -0., -0., -0.,  0., -0.,
         0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0.,
         0., -1., -0., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,
         0.,  0.],
       [-0.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0., -0.,  0.,
         0.,  0., -0., -0., -0., -0., -0.,  0.,  0., -0., -0.,  0., -0.,
        -0.,  0., -0.,  0., -0.,  0.,  1., -0., -0., -0., -0., -0., -0.,
         1., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0., -0., -0., -0.,
        -1.,  0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0.,  1.,  0., -0.,
        -0.,  0.,  0., -0.,  0.,  0., -0., -0., -1.,  0.,  0., -0., -0.,
        -0.,  0.],
       [-0., -0., -1.,  0.,  0., -1., -0., -0.,  0.,  0.,  1., -0.,  0.,
         0.,  0., -0.,  0., -0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        -0.,  0., -0., -0., -0., -0.,  1., -0., -0., -1., -1.,  0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0., -0., -1.,  0.,  0.,  1., -0.,  1.,
         0., -0.,  0., -1.,  0., -0.,  0., -0., -0., -0., -0.,  0.,  0.,
        -1., -0., -0., -1., -0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  1.,
        -1.,  0.],
       [-0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0., -0.,  0.,
         0.,  0.,  0.,  0., -0., -0.,  0.,  1.,  0.,  0.,  0., -0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0., -1.,  0.,  0., -0., -0., -0., -0.,
        -0., -0.,  0.,  0.,  0., -0., -0.,  1., -0., -0., -0.,  0.,  0.,
         0.,  0.,  0.,  0., -0., -0., -0.,  0., -0.,  0., -1.,  0., -0.,
         0., -0.,  0., -0., -0., -0., -0., -0.,  1.,  0., -0., -0., -0.,
        -0.,  0.],
       [-1., -0.,  0.,  0., -0.,  0., -0.,  0.,  1.,  0.,  0., -1.,  0.,
        -0.,  0., -0.,  0.,  1., -0., -0.,  1.,  0.,  0.,  1., -0., -0.,
        -0., -1.,  0.,  0.,  0.,  0., -1., -0.,  0.,  0., -0.,  0., -0.,
         1., -0., -1., -0.,  0.,  0., -0., -1., -0., -0., -0., -0., -0.,
         1.,  0.,  0., -0., -0.,  0., -1.,  0., -0.,  0., -1.,  0., -0.,
         0., -0., -0.,  0., -0.,  0., -0., -0.,  1.,  0., -0.,  0.,  0.,
        -0.,  0.],
       [ 0., -0.,  1., -0., -0., -0., -0., -0., -0.,  0., -0.,  0.,  0.,
        -0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,  1., -0., -0.,  0.,
         0., -0.,  0., -0., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,
        -1.,  0.,  0.,  0., -0., -0.,  0.,  1.,  1., -0., -0., -0., -0.,
         0., -0., -0., -0., -0., -0.,  0., -0., -1.,  0., -0.,  0., -0.,
        -0.,  0., -0., -0., -0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,
        -0., -0.],
       [ 1.,  0., -0.,  0., -0.,  0., -0., -0., -1.,  0.,  0.,  1., -0.,
         0.,  0., -0.,  0., -1., -0., -0., -1., -0.,  0., -1.,  0.,  0.,
         0.,  1.,  0.,  0., -0.,  1.,  1., -0.,  0.,  0., -0.,  0., -0.,
        -0., -0.,  1.,  0., -0., -0.,  0., -0., -0.,  0.,  0., -0., -0.,
        -1.,  0.,  1.,  0., -0.,  0.,  1.,  0.,  0., -0.,  1., -0., -1.,
         0.,  0., -0., -0., -0., -0.,  0., -0., -1., -0., -0.,  1.,  0.,
        -0.,  0.],
       [-0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0.,  0., -1., -0.,
        -0., -0., -0., -0.,  1., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,
         0., -1., -0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0., -0.,
         1., -0., -0., -0.,  0.,  0., -0., -1., -0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0., -0.,  0.,  0., -1.,  0., -0., -0., -0., -0., -0.,
        -0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,
        -0., -0.]], dtype=np.float32), np.array([[ 0., -0., -0.,  0., -0.,  0.,  0., -0., -1., -0., -0., -0., -0.,
         0.,  0.,  0., -0.,  1., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,
         0., -0., -0.,  0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,
        -1.,  0., -1.,  0., -0.,  0.,  0.,  0.,  1., -0., -0., -0., -0.,
         1.,  0.,  1., -0.,  0., -0.,  1.,  0., -1.,  0.,  0., -0., -1.,
        -0.,  0., -0.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,  0.,
        -0., -0.],
       [-0.,  0.,  0.,  0., -0., -0., -0., -0.,  0., -0., -1.,  0., -0.,
        -0., -0., -0., -0., -0.,  0., -0.,  1.,  0.,  0., -0., -0., -0.,
        -1., -0., -0., -0., -0., -0., -1., -0.,  0.,  1., -0.,  0.,  0.,
        -0., -1., -0., -0., -0.,  1.,  0.,  0.,  0., -0.,  0.,  0., -0.,
         0.,  0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,  1., -0.,  0.,
        -1., -0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0., -0.,
         0.,  0.],
       [ 0.,  0., -0.,  0., -0., -0.,  0., -0., -1.,  0., -0., -0., -0.,
         0.,  0.,  0.,  1., -0., -0., -0.,  0.,  0., -0., -0.,  0., -0.,
        -0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,  0.,  0., -0., -0.,
         0.,  1., -1.,  0., -0.,  0.,  0., -0.,  0.,  1., -0., -0., -0.,
         1., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,
         0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -1.,  0.,  0., -0.,
         0., -0.],
       [ 0.,  1., -0., -0., -0.,  0., -0., -0., -0., -0.,  1., -0.,  1.,
        -1.,  0.,  0.,  0., -0.,  0., -1.,  0.,  0., -0.,  0.,  0., -0.,
         0., -0., -0., -0., -1.,  0.,  0.,  0.,  0.,  0.,  0., -1., -1.,
        -0.,  0.,  0.,  1., -0.,  0., -1., -0., -0.,  0.,  0., -0.,  1.,
        -0.,  0., -0.,  0.,  0.,  1., -0.,  1.,  0.,  0., -0., -0., -0.,
         0., -1.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,  1.,  0.,  0.,
         1.,  0.],
       [-0., -0.,  0.,  0.,  0., -1., -1., -1., -0., -0.,  1., -0., -0.,
         0., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0.,
        -0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0., -1.,  0.,  0.,  0.,
         0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,
        -0.,  1., -0., -1.,  0., -0.,  0.,  0.,  0., -0., -0., -0., -0.,
         0.,  0., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,  1.,
         0.,  0.],
       [-0., -0.,  0.,  0.,  0., -0.,  0.,  1., -0.,  0., -0.,  0., -0.,
         0.,  0., -0.,  0.,  0.,  0., -0., -0., -1., -0.,  0.,  0.,  0.,
        -0.,  0., -0., -0.,  0., -0.,  0.,  1., -0., -0., -1., -0., -0.,
         0., -0., -0., -0.,  0., -0., -0.,  0., -0., -1.,  0., -0., -0.,
        -0., -1.,  0., -0., -1., -0., -0.,  0.,  0.,  0., -0., -0., -0.,
         1.,  0.,  0.,  1.,  0.,  0.,  0.,  0., -0., -0., -0., -0., -1.,
         0., -0.],
       [-1., -0.,  1.,  1.,  0.,  0.,  0., -0., -0.,  0.,  0., -1.,  1.,
         0., -0.,  0., -0.,  0., -0.,  0.,  1.,  0.,  1., -0., -0.,  0.,
        -0.,  1., -0., -0.,  0., -1., -0., -0., -0.,  0.,  0.,  0., -0.,
         0.,  0.,  1., -0.,  0.,  0.,  0.,  1., -0., -0., -0.,  0., -0.,
        -0.,  0.,  0., -0., -1., -0., -0.,  0.,  0., -0., -0., -0., -0.,
        -0., -0.,  1.,  0.,  0., -0., -0.,  0., -0., -0., -0., -1.,  0.,
         1., -1.],
       [-1.,  0.,  0.,  0.,  1., -0., -1., -0., -0., -0., -0.,  0., -0.,
         1.,  0., -0., -0.,  0.,  0., -0.,  1.,  0.,  0.,  0.,  1.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0.,
        -0.,  0.,  1.,  0., -0.,  0., -0.,  0.,  0., -0., -1., -0., -0.,
        -0.,  1., -0., -1.,  0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,
         0., -0.,  0., -0., -0., -0., -0., -0., -0., -0.,  0.,  0.,  1.,
        -0.,  0.],
       [-0.,  0.,  0.,  0., -1.,  0.,  1.,  0., -0.,  0., -0., -0., -0.,
        -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  1.,
        -0., -0.,  0., -1., -0., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,
        -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,
         1., -1.,  0., -0., -0.,  1.,  0., -0., -0., -0.,  1.,  0.,  0.,
        -0., -0.,  0., -0., -0.,  0., -0.,  1., -1.,  0.,  0.,  0., -0.,
        -0., -0.],
       [ 0., -1.,  0.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,
         0.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,  1.,  0., -0.,
         0.,  0.,  0.,  0.,  1., -1., -0.,  0., -0., -0.,  0., -0., -0.,
         0.,  0., -1., -1., -1., -0.,  0., -0., -0.,  0.,  0., -0., -0.,
         0.,  0.,  0.,  0., -0., -1., -0., -0.,  0., -0.,  0.,  0.,  1.,
         0.,  0., -0., -0.,  0., -0., -0.,  0.,  0., -0., -1., -0., -0.,
        -0.,  0.],
       [ 1.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  1., -0., -0., -0.,
        -1.,  1.,  0.,  0., -0., -0., -0.,  0., -1., -0.,  0.,  0., -0.,
        -1.,  0., -1.,  1., -0.,  0., -0., -0., -0.,  0.,  0., -1., -0.,
         0.,  0., -1.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,
         0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0., -1.,  1.,
        -0., -0.,  0.,  0., -0., -0., -0., -0.,  0., -0.,  0.,  1.,  0.,
        -0.,  0.],
       [-0.,  0., -0.,  0., -0.,  0., -0.,  0., -1., -1.,  0., -0., -0.,
         0.,  0., -0., -1., -0.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,
         0., -0., -0.,  0., -0., -1.,  0.,  1., -0.,  0., -0.,  0., -0.,
        -0.,  0., -1.,  0., -0.,  0., -0., -0.,  0.,  0., -0.,  1.,  0.,
         0.,  0.,  1., -0.,  0., -1., -0.,  0., -0., -0.,  0., -0., -0.,
        -0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0., -1., -1., -0., -0.,
         0., -0.],
       [-0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,  1.,
         0.,  0., -0.,  0., -0., -0., -1., -0.,  0., -1.,  0.,  0., -0.,
         0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0., -0., -0., -0., -1.,
         0., -0., -0.,  0., -1., -0., -0.,  0., -0., -0., -0.,  0.,  0.,
        -0., -0.,  0.,  0., -0., -0.,  0.,  1., -1., -0., -0., -0.,  0.,
         0., -0., -0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0., -0.,  0.,
         0.,  0.],
       [-0., -0.,  1.,  0., -1., -1., -1., -1., -0.,  1., -0., -0., -0.,
        -0.,  0., -1., -0., -0., -0.,  0., -0., -1., -0., -0., -1., -0.,
        -1., -0., -1.,  1., -0., -0.,  0., -0., -0., -1., -0.,  0.,  0.,
         0., -0., -0., -0.,  0., -0.,  0.,  0.,  0., -0., -0., -0., -1.,
        -0.,  1.,  0., -1.,  0., -0., -0., -0., -1.,  0., -0., -1.,  0.,
        -0.,  0., -0., -0.,  1.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  1.,
        -0.,  0.],
       [ 0., -0.,  0., -1.,  0., -0., -0., -0., -0., -1.,  0.,  0., -0.,
        -0.,  0., -0., -0.,  0., -0.,  0., -0., -0., -1.,  0.,  0., -0.,
         0., -0.,  0.,  0., -1., -0.,  0.,  0., -1.,  0.,  0., -0., -1.,
        -0.,  1., -0.,  0., -0.,  0., -0., -0., -1., -0.,  0., -0., -0.,
        -0., -1.,  0.,  0., -0.,  0.,  0.,  1., -0., -0., -0., -0.,  0.,
         0., -0.,  0.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,
         0.,  0.],
       [-0.,  0.,  0.,  1.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  1.,
        -0., -0.,  0., -0.,  0.,  0.,  1.,  0., -0., -0.,  0., -0.,  0.,
        -0.,  0.,  0., -0., -0., -1.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,
        -0., -0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,
        -0.,  0., -0.,  0., -1.,  0.,  0., -1., -0.,  1., -0., -0.,  1.,
         0., -0.,  1.,  0., -0., -0., -0.,  0., -0.,  0., -0., -0., -0.,
        -0., -1.],
       [ 0., -0.,  0.,  0., -0., -1., -1., -0., -0.,  0.,  0.,  0.,  0.,
         0.,  1., -0.,  0.,  0., -1., -0., -0., -0.,  0.,  0.,  1.,  1.,
        -0., -0., -1.,  0., -0., -0.,  0., -0.,  0.,  0., -0., -1., -0.,
        -0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0.,  0., -1., -0., -0.,
         0.,  1.,  0., -1., -0.,  0., -0.,  1., -0.,  0., -0., -0.,  1.,
        -0.,  0.,  0.,  1., -0., -0.,  0.,  0.,  0., -0.,  0.,  1.,  1.,
        -0., -0.],
       [-0., -0.,  0., -0., -0., -0., -0., -0.,  0., -0.,  0., -0., -0.,
        -0., -0., -0., -1., -0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,
         0., -0.,  0., -0.,  0., -1., -0., -0.,  0., -0., -0.,  0.,  0.,
         0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  1.,  0.,
         0., -1.,  1.,  0., -1.,  0., -0., -1.,  0.,  0.,  0.,  0.,  0.,
         0., -0.,  0.,  0., -0.,  0.,  0.,  1., -0.,  0., -1., -0., -0.,
        -0.,  0.],
       [-0., -0.,  0.,  1.,  0., -0.,  0.,  0., -0.,  0.,  0., -1.,  1.,
        -0.,  0.,  0.,  0., -1., -0.,  0., -0.,  0.,  0., -1., -0., -0.,
         0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,
        -0., -0.,  1.,  0., -0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0.,
         0.,  0., -0.,  0., -1.,  0., -0.,  0., -0.,  0.,  0., -0.,  0.,
         0.,  0.,  1., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0.,
         0., -1.],
       [-1.,  0., -0., -0.,  0.,  0., -1.,  0.,  0.,  0., -0., -0., -0.,
         1.,  0.,  0., -0., -1., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        -0., -1.,  0., -0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0., -0.,
         0.,  0.,  1.,  0., -0.,  0., -1.,  0., -0.,  0., -1., -0.,  0.,
         0.,  1., -0., -1., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,
        -0.,  0., -0.,  0.,  1.,  1., -0.,  0.,  0.,  0., -0., -0.,  1.,
         0.,  0.],
       [-0., -1., -0.,  0., -0.,  0., -0.,  0.,  1.,  0., -0., -1.,  0.,
         0., -0., -0., -0., -0.,  0.,  0.,  0.,  0., -0., -0., -0., -0.,
         0.,  0., -0.,  0.,  0., -0.,  0., -1.,  0.,  0., -0.,  0., -0.,
        -0.,  0.,  1., -0., -0.,  0.,  0.,  0., -0., -1., -0.,  0.,  0.,
        -0., -1., -0.,  0., -1., -0.,  1.,  0., -0., -0., -0.,  0.,  0.,
         0.,  0.,  1., -0.,  0.,  0., -1., -0.,  0.,  1., -0., -0., -0.,
        -0., -0.]], dtype=np.float32))

In [ ]:
#@title Verification
n = 3
m = 5
p = 7
rank = 80

verify_tensor_decomposition(decomposition_357, n, m, p, rank)

## Rank-48 decomposition of <4,4,4> over 0.5*Z[i]


In [ ]:
#@title Data
decomposition_444 = (np.array([[ 0.5+0.5j,  0.5+0.5j,  0. +0.j ,  0. -0.5j,  0.5+0.5j,  0. +0.j ,
         0. +0.5j,  0.5+0.5j,  0. -0.5j, -0.5+0.5j, -0.5+0.5j,  0.5+0.j ,
         0. +0.j , -0. +0.j ,  0. +0.5j,  0. +0.j , -0.5+0.j ,  0.5+0.5j,
         0. +0.5j,  0. -0.j ,  0. +0.j ,  0. +0.5j, -0.5-0.5j, -0. +0.j ,
        -0.5+0.j ,  0. +0.j ,  0. +0.j ,  0. -0.5j, -0.5+0.5j,  0.5+0.5j,
        -0. +0.j ,  0.5+0.j , -0. +0.j ,  0.5+0.j ,  0. -0.5j,  0. +0.j ,
        -0. +0.j ,  0.5+0.j ,  0. +0.j , -0.5+0.j , -0.5-0.5j,  0.5-0.5j,
         0.5+0.5j,  0. +0.5j,  0. -0.j , -0.5+0.5j,  0.5-0.5j,  0.5+0.j ],
       [ 0.5+0.5j,  0. +0.j , -0.5+0.j , -0.5+0.j , -0.5-0.5j,  0. +0.j ,
         0. +0.j , -0.5-0.5j,  0. -0.5j,  0. -0.j ,  0.5-0.5j,  0.5+0.j ,
         0.5+0.5j,  0.5-0.5j, -0.5+0.j ,  0. +0.j , -0. +0.5j,  0.5+0.5j,
        -0. +0.5j,  0. +0.j ,  0.5+0.5j,  0. -0.5j,  0. +0.j ,  0. +0.j ,
         0.5+0.j , -0. +0.j ,  0.5+0.5j,  0. -0.5j, -0.5+0.5j, -0. +0.j ,
         0.5+0.5j, -0.5+0.j ,  0. +0.j , -0. +0.5j,  0. +0.5j, -0. +0.j ,
        -0.5-0.5j,  0. -0.5j,  0.5-0.5j,  0. -0.5j, -0.5-0.5j,  0. -0.j ,
         0. -0.j ,  0.5+0.j ,  0. +0.j ,  0.5-0.5j,  0. +0.j , -0. +0.5j],
       [ 0. +0.j ,  0. +0.j ,  0.5+0.j ,  0.5+0.j ,  0. +0.j ,  0.5-0.5j,
         0. +0.j ,  0. +0.j , -0.5-0.j ,  0. +0.j ,  0. -0.j ,  0. -0.5j,
        -0.5-0.5j, -0.5+0.5j,  0.5+0.j , -0.5+0.5j,  0. +0.5j,  0. +0.j ,
        -0.5+0.j ,  0.5-0.5j, -0.5-0.5j, -0.5-0.j ,  0. -0.j , -0.5-0.5j,
         0. -0.5j,  0.5-0.5j,  0.5+0.5j,  0.5+0.j ,  0. -0.j ,  0. +0.j ,
         0.5+0.5j,  0. -0.5j,  0.5+0.5j,  0. -0.5j, -0.5+0.j ,  0.5-0.5j,
        -0.5-0.5j,  0. -0.5j,  0.5-0.5j,  0. -0.5j,  0. +0.j ,  0. +0.j ,
        -0. +0.j , -0.5-0.j ,  0.5-0.5j,  0. -0.j ,  0. +0.j , -0. +0.5j],
       [ 0. +0.j , -0.5+0.5j,  0. +0.j , -0.5-0.j ,  0. -0.j , -0.5-0.5j,
         0.5+0.j ,  0. +0.j ,  0. -0.5j, -0.5-0.5j,  0. +0.j , -0.5+0.j ,
         0. +0.j , -0. +0.j , -0.5+0.j ,  0.5+0.5j,  0. -0.5j,  0. +0.j ,
        -0. +0.5j,  0.5+0.5j,  0. +0.j ,  0. -0.5j, -0.5+0.5j,  0.5-0.5j,
        -0.5-0.j ,  0.5+0.5j,  0. +0.j , -0. +0.5j,  0. +0.j ,  0.5-0.5j,
         0. -0.j ,  0.5+0.j ,  0.5-0.5j,  0. -0.5j,  0. +0.5j,  0.5+0.5j,
         0. +0.j ,  0. -0.5j,  0. +0.j ,  0. -0.5j,  0. +0.j , -0.5-0.5j,
        -0.5+0.5j, -0.5-0.j , -0.5-0.5j,  0. +0.j ,  0.5+0.5j,  0. -0.5j],
       [ 0.5-0.5j,  0.5+0.5j, -0. +0.j , -0. +0.5j, -0.5+0.5j,  0. +0.j ,
        -0.5+0.j , -0.5-0.5j,  0.5+0.j ,  0.5+0.5j, -0.5+0.5j, -0.5-0.j ,
        -0. +0.j ,  0. -0.j ,  0.5+0.j ,  0. -0.j , -0.5-0.j ,  0.5+0.5j,
        -0. +0.5j,  0. -0.j ,  0. +0.j ,  0. -0.5j,  0.5-0.5j,  0. -0.j ,
         0. -0.5j,  0. +0.j ,  0. +0.j , -0.5+0.j , -0.5-0.5j, -0.5-0.5j,
        -0. +0.j ,  0.5+0.j ,  0. -0.j , -0.5+0.j , -0.5-0.j ,  0. -0.j ,
        -0. +0.j , -0. +0.5j, -0. +0.j , -0.5+0.j ,  0.5+0.5j, -0.5+0.5j,
         0.5-0.5j,  0.5+0.j , -0. +0.j ,  0.5+0.5j,  0.5-0.5j, -0. +0.5j],
       [ 0.5-0.5j, -0. +0.j ,  0. -0.5j, -0.5+0.j ,  0.5-0.5j,  0. +0.j ,
         0. -0.j ,  0.5+0.5j,  0.5+0.j ,  0. +0.j ,  0.5-0.5j, -0.5-0.j ,
        -0.5+0.5j,  0.5-0.5j,  0. -0.5j,  0. -0.j ,  0. -0.5j,  0.5+0.5j,
        -0. +0.5j,  0. +0.j ,  0.5+0.5j,  0. +0.5j,  0. +0.j ,  0. -0.j ,
        -0. +0.5j,  0. +0.j , -0.5-0.5j, -0.5+0.j , -0.5-0.5j, -0. +0.j ,
        -0.5-0.5j, -0.5+0.j ,  0. +0.j , -0. +0.5j,  0.5+0.j ,  0. +0.j ,
        -0.5+0.5j, -0.5+0.j , -0.5-0.5j,  0. +0.5j,  0.5+0.5j,  0. -0.j ,
         0. +0.j ,  0. +0.5j,  0. +0.j , -0.5-0.5j,  0. -0.j ,  0.5+0.j ],
       [ 0. -0.j , -0. +0.j , -0. +0.5j,  0.5+0.j ,  0. +0.j ,  0.5-0.5j,
         0. -0.j , -0. +0.j ,  0. -0.5j,  0. -0.j ,  0. -0.j ,  0. +0.5j,
         0.5-0.5j, -0.5+0.5j, -0. +0.5j,  0.5-0.5j,  0. -0.5j,  0. +0.j ,
        -0.5+0.j ,  0.5-0.5j, -0.5-0.5j,  0.5+0.j ,  0. +0.j ,  0.5-0.5j,
         0.5+0.j , -0.5-0.5j, -0.5-0.5j,  0. -0.5j,  0. -0.j ,  0. +0.j ,
        -0.5-0.5j,  0. -0.5j, -0.5+0.5j,  0. -0.5j, -0. +0.5j, -0.5+0.5j,
        -0.5+0.5j, -0.5+0.j , -0.5-0.5j, -0. +0.5j,  0. -0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.5j, -0.5-0.5j, -0. +0.j , -0. +0.j ,  0.5+0.j ],
       [ 0. +0.j , -0.5+0.5j,  0. +0.j ,  0.5+0.j ,  0. +0.j , -0.5-0.5j,
         0. +0.5j,  0. -0.j ,  0.5+0.j , -0.5+0.5j,  0. +0.j ,  0.5+0.j ,
         0. +0.j ,  0. -0.j , -0. +0.5j, -0.5-0.5j,  0. -0.5j,  0. -0.j ,
         0. +0.5j,  0.5+0.5j,  0. -0.j ,  0. +0.5j, -0.5-0.5j,  0.5+0.5j,
         0. -0.5j,  0.5-0.5j,  0. +0.j ,  0.5+0.j ,  0. +0.j , -0.5+0.5j,
         0. +0.j ,  0.5+0.j ,  0.5+0.5j,  0. +0.5j,  0.5+0.j , -0.5-0.5j,
        -0. +0.j ,  0.5+0.j ,  0. +0.j ,  0. -0.5j,  0. +0.j ,  0.5+0.5j,
         0.5+0.5j, -0. +0.5j, -0.5+0.5j,  0. +0.j ,  0.5+0.5j,  0.5+0.j ],
       [ 0.5-0.5j, -0.5-0.5j,  0. -0.j ,  0. -0.5j, -0.5+0.5j,  0. +0.j ,
         0.5+0.j , -0.5-0.5j, -0.5-0.j , -0.5-0.5j,  0.5-0.5j,  0.5+0.j ,
        -0. +0.j , -0. +0.j ,  0.5+0.j , -0. +0.j , -0.5-0.j ,  0.5+0.5j,
        -0. +0.5j,  0. +0.j ,  0. -0.j ,  0. -0.5j,  0.5-0.5j,  0. +0.j ,
         0. -0.5j,  0. +0.j ,  0. +0.j , -0.5-0.j ,  0.5+0.5j,  0.5+0.5j,
         0. +0.j , -0.5+0.j ,  0. -0.j , -0.5-0.j ,  0.5+0.j ,  0. -0.j ,
         0. +0.j , -0. +0.5j,  0. +0.j ,  0.5+0.j , -0.5-0.5j, -0.5+0.5j,
         0.5-0.5j, -0.5-0.j ,  0. -0.j , -0.5-0.5j,  0.5-0.5j,  0. -0.5j],
       [ 0.5-0.5j, -0. +0.j ,  0. +0.5j, -0.5+0.j ,  0.5-0.5j,  0. +0.j ,
         0. +0.j ,  0.5+0.5j, -0.5-0.j ,  0. +0.j , -0.5+0.5j,  0.5+0.j ,
        -0.5+0.5j,  0.5-0.5j, -0. +0.5j,  0. +0.j , -0. +0.5j,  0.5+0.5j,
        -0. +0.5j,  0. -0.j , -0.5-0.5j,  0. +0.5j,  0. +0.j ,  0. +0.j ,
         0. +0.5j,  0. +0.j ,  0.5+0.5j, -0.5-0.j ,  0.5+0.5j, -0. +0.j ,
        -0.5-0.5j,  0.5+0.j ,  0. +0.j ,  0. -0.5j, -0.5-0.j ,  0. +0.j ,
         0.5-0.5j,  0.5+0.j , -0.5-0.5j, -0. +0.5j, -0.5-0.5j,  0. +0.j ,
         0. -0.j , -0. +0.5j, -0. +0.j ,  0.5+0.5j,  0. +0.j ,  0.5+0.j ],
       [ 0. +0.j , -0. +0.j ,  0. -0.5j,  0.5+0.j ,  0. -0.j , -0.5+0.5j,
        -0. +0.j ,  0. +0.j ,  0. -0.5j,  0. +0.j ,  0. -0.j ,  0. +0.5j,
         0.5-0.5j, -0.5+0.5j,  0. -0.5j,  0.5-0.5j, -0. +0.5j, -0. +0.j ,
         0.5+0.j ,  0.5-0.5j,  0.5+0.5j, -0.5+0.j ,  0. +0.j ,  0.5-0.5j,
        -0.5+0.j ,  0.5+0.5j,  0.5+0.5j, -0. +0.5j,  0. +0.j ,  0. +0.j ,
        -0.5-0.5j,  0. -0.5j,  0.5-0.5j,  0. +0.5j,  0. +0.5j,  0.5-0.5j,
         0.5-0.5j,  0.5+0.j , -0.5-0.5j, -0. +0.5j,  0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. -0.5j, -0.5-0.5j, -0. +0.j , -0. +0.j ,  0.5+0.j ],
       [-0. +0.j ,  0.5-0.5j,  0. +0.j , -0.5-0.j , -0. +0.j ,  0.5+0.5j,
         0. -0.5j,  0. +0.j ,  0.5+0.j ,  0.5-0.5j,  0. +0.j ,  0.5+0.j ,
        -0. +0.j , -0. +0.j , -0. +0.5j, -0.5-0.5j,  0. -0.5j,  0. +0.j ,
         0. -0.5j,  0.5+0.5j,  0. -0.j ,  0. -0.5j, -0.5-0.5j,  0.5+0.5j,
         0. +0.5j, -0.5+0.5j,  0. +0.j , -0.5-0.j ,  0. +0.j ,  0.5-0.5j,
         0. +0.j ,  0.5+0.j , -0.5-0.5j,  0. +0.5j,  0.5+0.j ,  0.5+0.5j,
        -0. +0.j ,  0.5+0.j ,  0. -0.j ,  0. +0.5j,  0. +0.j ,  0.5+0.5j,
         0.5+0.5j,  0. -0.5j, -0.5+0.5j,  0. +0.j ,  0.5+0.5j, -0.5+0.j ],
       [ 0.5-0.5j,  0.5-0.5j, -0. +0.j , -0.5-0.j ,  0.5-0.5j, -0. +0.j ,
        -0.5+0.j , -0.5+0.5j,  0.5+0.j , -0.5-0.5j,  0.5+0.5j,  0. -0.5j,
         0. +0.j , -0. +0.j ,  0.5+0.j ,  0. +0.j ,  0. -0.5j, -0.5+0.5j,
        -0.5+0.j , -0. +0.j ,  0. -0.j , -0.5-0.j , -0.5+0.5j,  0. -0.j ,
        -0. +0.5j,  0. +0.j ,  0. +0.j , -0.5+0.j , -0.5-0.5j,  0.5-0.5j,
         0. +0.j ,  0. -0.5j,  0. +0.j ,  0. +0.5j,  0.5+0.j , -0. +0.j ,
         0. +0.j ,  0. -0.5j, -0. +0.j , -0. +0.5j, -0.5+0.5j,  0.5+0.5j,
         0.5-0.5j, -0.5+0.j ,  0. +0.j , -0.5-0.5j,  0.5+0.5j,  0. +0.5j],
       [ 0.5-0.5j, -0. +0.j ,  0. -0.5j,  0. -0.5j, -0.5+0.5j,  0. -0.j ,
         0. -0.j ,  0.5-0.5j,  0.5+0.j ,  0. -0.j , -0.5-0.5j,  0. -0.5j,
         0.5-0.5j,  0.5+0.5j,  0. -0.5j,  0. +0.j ,  0.5+0.j , -0.5+0.5j,
        -0.5+0.j ,  0. +0.j ,  0.5-0.5j,  0.5+0.j , -0. +0.j , -0. +0.j ,
         0. -0.5j,  0. +0.j ,  0.5-0.5j, -0.5+0.j , -0.5-0.5j, -0. +0.j ,
        -0.5+0.5j, -0. +0.5j,  0. -0.j ,  0.5+0.j , -0.5+0.j ,  0. +0.j ,
         0.5-0.5j,  0.5+0.j , -0.5-0.5j,  0.5+0.j , -0.5+0.5j,  0. +0.j ,
         0. +0.j ,  0. -0.5j, -0. +0.j ,  0.5+0.5j,  0. +0.j ,  0.5+0.j ],
       [ 0. +0.j ,  0. +0.j , -0. +0.5j, -0. +0.5j,  0. +0.j , -0.5-0.5j,
         0. -0.j ,  0. -0.j ,  0. +0.5j,  0. +0.j ,  0. +0.j ,  0.5+0.j ,
        -0.5+0.5j, -0.5-0.5j, -0. +0.5j, -0.5-0.5j,  0.5+0.j ,  0. -0.j ,
        -0. +0.5j,  0.5+0.5j, -0.5+0.5j, -0. +0.5j,  0. +0.j , -0.5+0.5j,
         0.5+0.j ,  0.5+0.5j,  0.5-0.5j,  0. +0.5j, -0. +0.j ,  0. +0.j ,
        -0.5+0.5j,  0.5+0.j , -0.5+0.5j, -0.5+0.j ,  0. +0.5j, -0.5-0.5j,
         0.5-0.5j,  0.5+0.j , -0.5-0.5j,  0.5+0.j ,  0. +0.j ,  0. +0.j ,
         0. -0.j , -0. +0.5j, -0.5-0.5j, -0. +0.j ,  0. +0.j ,  0.5+0.j ],
       [-0. +0.j ,  0.5+0.5j,  0. -0.j , -0. +0.5j,  0. +0.j , -0.5+0.5j,
         0. +0.5j,  0. +0.j , -0.5+0.j ,  0.5-0.5j,  0. +0.j ,  0. -0.5j,
        -0. +0.j ,  0. +0.j , -0. +0.5j, -0.5+0.5j,  0.5+0.j ,  0. +0.j ,
         0.5+0.j , -0.5+0.5j, -0. +0.j , -0.5+0.j ,  0.5+0.5j, -0.5-0.5j,
         0. -0.5j, -0.5+0.5j,  0. -0.j , -0.5-0.j ,  0. +0.j , -0.5-0.5j,
        -0. +0.j , -0. +0.5j,  0.5+0.5j,  0.5+0.j ,  0.5+0.j ,  0.5-0.5j,
         0. -0.j , -0.5-0.j ,  0. +0.j , -0.5-0.j ,  0. -0.j ,  0.5-0.5j,
         0.5+0.5j,  0. -0.5j, -0.5+0.5j,  0. +0.j , -0.5+0.5j,  0.5+0.j ]],
      dtype=np.complex64), np.array([[-0.5-0.j , -0. +0.j ,  0. -0.j ,  0. -0.5j, -0.5-0.j ,  0. +0.j ,
         0. +0.j , -0.5-0.j ,  0.5+0.j , -0. +0.j ,  0. -0.j ,  0. -0.5j,
        -0.5+0.j , -0. +0.5j,  0. +0.j , -0. +0.5j,  0. +0.j ,  0. -0.5j,
         0. -0.j ,  0. -0.5j,  0. +0.j , -0. +0.j ,  0. -0.5j, -0.5-0.j ,
         0. -0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,
        -0.5+0.j ,  0.5+0.j ,  0. +0.j , -0. +0.j ,  0. +0.5j,  0. -0.j ,
         0. -0.j , -0. +0.j , -0. +0.5j,  0. -0.5j,  0. +0.j ,  0.5+0.j ,
        -0. +0.5j,  0.5+0.j ,  0. -0.5j,  0. +0.j , -0.5+0.j ,  0.5+0.j ],
       [ 0. +0.j , -0. +0.5j,  0.5+0.5j,  0. -0.j , -0. +0.j ,  0.5+0.j ,
        -0.5-0.5j,  0. -0.j ,  0. +0.j ,  0. +0.5j, -0. +0.5j,  0. -0.j ,
         0. -0.j ,  0. +0.j , -0.5+0.j ,  0. -0.j ,  0.5+0.j ,  0. -0.j ,
         0. -0.5j,  0. +0.j ,  0. -0.5j, -0.5+0.j ,  0. -0.j , -0. +0.j ,
         0. +0.5j,  0. +0.5j,  0.5+0.j , -0. +0.5j,  0.5+0.j ,  0. +0.5j,
         0. -0.j ,  0. -0.j , -0. +0.5j, -0.5-0.j ,  0. +0.j ,  0. -0.5j,
        -0.5-0.j ,  0. +0.5j,  0. +0.j ,  0. +0.j ,  0. +0.5j,  0. +0.j ,
         0. -0.j ,  0. +0.j ,  0. +0.j ,  0. -0.5j,  0. -0.j , -0. +0.j ],
       [ 0. -0.j ,  0. -0.j ,  0. -0.j ,  0. +0.5j,  0.5+0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j , -0.5+0.j ,  0. +0.5j,  0. +0.j ,  0. -0.j ,
         0.5+0.j ,  0. -0.5j,  0. +0.j , -0. +0.j ,  0.5+0.j , -0. +0.5j,
        -0. +0.j ,  0. +0.5j,  0. +0.j , -0.5+0.j , -0. +0.5j,  0.5+0.j ,
         0. +0.j ,  0. +0.5j,  0.5+0.j , -0. +0.5j,  0. -0.j ,  0. +0.5j,
         0. -0.j , -0.5-0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,  0. -0.5j,
        -0.5+0.j , -0. +0.5j, -0. +0.j ,  0. -0.j ,  0. +0.5j,  0. +0.j ,
        -0. +0.j , -0.5-0.j ,  0. -0.j ,  0. -0.5j,  0.5+0.j ,  0. +0.j ],
       [ 0. -0.j , -0. +0.5j,  0. -0.j ,  0. +0.j ,  0.5+0.j ,  0.5+0.j ,
         0. +0.j ,  0.5+0.j , -0.5+0.j ,  0. +0.5j,  0. +0.5j, -0. +0.5j,
         0.5+0.j , -0. +0.j ,  0. +0.j ,  0. -0.5j,  0. -0.j ,  0. -0.j ,
         0. -0.5j,  0. +0.j ,  0. -0.5j,  0. +0.j ,  0. +0.5j,  0.5+0.j ,
         0. +0.j ,  0. +0.5j,  0. -0.j ,  0. +0.5j,  0. -0.j ,  0. -0.j ,
         0.5+0.j ,  0. +0.j ,  0. +0.j , -0.5+0.j ,  0. +0.j ,  0. +0.j ,
        -0.5+0.j ,  0. +0.5j,  0. +0.j ,  0. +0.5j,  0. -0.j , -0.5-0.j ,
         0. +0.j , -0.5-0.j ,  0. -0.j ,  0. -0.5j, -0. +0.j ,  0. +0.j ],
       [-0.5-0.j ,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0.5+0.j ,  0. +0.j ,
         0. -0.j ,  0.5+0.j ,  0.5+0.j ,  0. +0.j ,  0. +0.j ,  0. -0.5j,
        -0.5+0.j ,  0. -0.5j, -0.5+0.j , -0. +0.5j,  0.5+0.j ,  0. -0.5j,
         0. +0.j , -0. +0.5j,  0. +0.j ,  0. +0.j ,  0. -0.5j, -0.5-0.j ,
         0. +0.j , -0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j ,
        -0.5+0.j , -0.5-0.j , -0. +0.j ,  0.5+0.j ,  0. -0.5j, -0. +0.j ,
        -0. +0.j ,  0. -0.5j,  0. -0.5j,  0. +0.j ,  0. -0.j ,  0.5+0.j ,
         0. -0.5j, -0. +0.j , -0. +0.5j,  0. +0.j ,  0.5+0.j ,  0. -0.j ],
       [ 0. +0.j , -0. +0.5j, -0.5-0.5j,  0. -0.5j,  0. +0.j ,  0.5+0.j ,
         0.5+0.5j,  0. +0.j ,  0. +0.j ,  0. +0.5j,  0. -0.5j, -0. +0.j ,
        -0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,  0. +0.j , -0. +0.j ,
         0. -0.5j,  0. +0.j ,  0. -0.5j,  0.5+0.j ,  0. +0.j , -0. +0.j ,
         0. -0.5j, -0. +0.5j, -0.5-0.j ,  0. +0.5j,  0.5+0.j ,  0. -0.5j,
         0. -0.j ,  0. -0.j ,  0. -0.5j,  0. +0.j ,  0. +0.j ,  0. +0.5j,
        -0.5+0.j ,  0. +0.j ,  0. +0.j , -0. +0.5j,  0. +0.5j,  0. +0.j ,
         0. +0.j , -0.5-0.j ,  0. -0.j , -0. +0.5j, -0. +0.j ,  0.5+0.j ],
       [-0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.5j, -0.5+0.j , -0. +0.j ,
         0. -0.j ,  0. +0.j , -0.5+0.j , -0. +0.5j,  0. +0.j ,  0. -0.j ,
         0.5+0.j , -0. +0.5j,  0. -0.j , -0. +0.j , -0.5-0.j , -0. +0.5j,
         0. +0.j ,  0. -0.5j,  0. +0.j ,  0.5+0.j ,  0. +0.5j,  0.5+0.j ,
         0. +0.j ,  0. +0.5j, -0.5-0.j , -0. +0.5j,  0. -0.j ,  0. -0.5j,
        -0. +0.j ,  0.5+0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.5j,
        -0.5+0.j , -0. +0.5j,  0. +0.j ,  0. +0.j ,  0. +0.5j,  0. +0.j ,
         0. -0.j , -0.5-0.j , -0. +0.j ,  0. +0.5j, -0.5+0.j , -0. +0.j ],
       [ 0. -0.j ,  0. +0.5j,  0. +0.j , -0. +0.j , -0.5+0.j ,  0.5+0.j ,
         0. -0.j , -0.5-0.j , -0.5+0.j ,  0. +0.5j,  0. -0.5j, -0. +0.5j,
         0.5+0.j ,  0. -0.j , -0. +0.j ,  0. -0.5j,  0. +0.j ,  0. -0.j ,
         0. -0.5j, -0. +0.j ,  0. -0.5j,  0. -0.j ,  0. +0.5j,  0.5+0.j ,
         0. +0.j , -0. +0.5j,  0. +0.j ,  0. +0.5j,  0. -0.j , -0. +0.j ,
         0.5+0.j ,  0. -0.j , -0. +0.j , -0.5+0.j ,  0. +0.j ,  0. -0.j ,
        -0.5+0.j ,  0. +0.5j,  0. +0.j ,  0. +0.5j,  0. -0.j , -0.5+0.j ,
         0. -0.j , -0.5-0.j ,  0. +0.j ,  0. +0.5j,  0. +0.j , -0. +0.j ],
       [ 0.5+0.j ,  0. +0.j ,  0. -0.j , -0. +0.j ,  0.5+0.j ,  0. +0.j ,
         0. -0.j , -0.5-0.j , -0. +0.j ,  0. -0.j ,  0. -0.j , -0. +0.j ,
         0.5+0.j , -0. +0.5j,  0.5+0.j ,  0. -0.5j,  0.5+0.j ,  0. -0.5j,
         0. -0.5j,  0. +0.5j, -0. +0.j , -0.5-0.j ,  0. -0.5j, -0.5+0.j ,
         0. -0.5j,  0. +0.j ,  0. -0.j ,  0. -0.5j,  0. -0.j , -0. +0.j ,
        -0.5+0.j , -0. +0.j , -0. +0.j , -0.5-0.j ,  0. +0.j ,  0. +0.j ,
        -0. +0.j ,  0. -0.5j,  0. -0.5j,  0. +0.j ,  0. -0.j , -0.5-0.j ,
        -0. +0.5j,  0. +0.j ,  0. -0.5j,  0. +0.j ,  0.5+0.j ,  0. -0.j ],
       [ 0. -0.j , -0. +0.5j,  0.5+0.5j,  0. +0.5j,  0. +0.j ,  0.5+0.j ,
         0.5+0.5j, -0. +0.j ,  0.5+0.j ,  0. -0.5j,  0. -0.5j, -0. +0.5j,
         0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j ,  0. +0.j , -0. +0.5j,  0. +0.j ,  0. +0.j , -0. +0.j ,
         0. +0.j ,  0. -0.5j, -0.5+0.j ,  0. +0.j ,  0.5+0.j ,  0. +0.5j,
        -0. +0.j , -0.5-0.j ,  0. -0.5j,  0. -0.j , -0. +0.5j,  0. -0.5j,
        -0.5-0.j ,  0. -0.j ,  0. +0.j ,  0. +0.5j,  0. -0.5j,  0. -0.j ,
         0. +0.j ,  0.5+0.j , -0. +0.j ,  0. -0.5j,  0. +0.j ,  0.5+0.j ],
       [ 0. +0.j ,  0. +0.j ,  0. -0.j , -0. +0.5j, -0.5+0.j , -0. +0.j ,
         0. -0.j ,  0. -0.j ,  0. -0.j ,  0. -0.5j, -0. +0.j ,  0. +0.5j,
        -0.5-0.j ,  0. -0.5j,  0. +0.j ,  0. +0.j , -0.5+0.j , -0. +0.5j,
         0. +0.5j,  0. -0.5j,  0. -0.j ,  0. -0.j ,  0. +0.5j,  0.5+0.j ,
        -0. +0.5j,  0. -0.5j, -0.5-0.j ,  0. -0.j , -0. +0.j ,  0. +0.5j,
        -0. +0.j ,  0. +0.j , -0. +0.j ,  0. -0.j , -0. +0.5j,  0. -0.5j,
        -0.5-0.j , -0. +0.5j,  0. +0.j ,  0. -0.j ,  0. -0.5j, -0. +0.j ,
        -0. +0.j ,  0.5+0.j ,  0. -0.j ,  0. -0.5j, -0.5-0.j , -0. +0.j ],
       [ 0. +0.j , -0. +0.5j,  0. -0.j ,  0. +0.j , -0.5-0.j ,  0.5+0.j ,
         0. -0.j ,  0.5+0.j ,  0. -0.j ,  0. -0.5j,  0. -0.5j,  0. +0.j ,
        -0.5+0.j ,  0. +0.j ,  0. +0.j , -0. +0.5j,  0. +0.j ,  0. -0.j ,
         0. -0.j ,  0. +0.j , -0. +0.5j,  0.5+0.j , -0. +0.5j,  0.5+0.j ,
         0. +0.5j,  0. -0.5j,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0.5+0.j , -0.5-0.j , -0. +0.j ,  0.5+0.j ,  0. +0.5j,  0. +0.j ,
        -0.5-0.j , -0. +0.5j,  0. +0.j ,  0. +0.5j,  0. +0.j ,  0.5+0.j ,
        -0. +0.j ,  0.5+0.j ,  0. +0.j ,  0. -0.5j,  0. +0.j ,  0. -0.j ],
       [ 0. -0.5j,  0. +0.j , -0. +0.j ,  0.5+0.j , -0. +0.5j,  0. -0.j ,
        -0. +0.j , -0. +0.5j, -0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,
         0. +0.5j, -0.5-0.j ,  0. +0.j ,  0.5+0.j ,  0. +0.j ,  0.5+0.j ,
         0.5+0.j ,  0.5+0.j ,  0. +0.j ,  0. +0.5j,  0.5+0.j ,  0. +0.5j,
         0.5+0.j , -0. +0.j ,  0. +0.j , -0.5+0.j , -0. +0.j , -0. +0.j ,
        -0. +0.5j, -0. +0.j , -0. +0.j ,  0. -0.j ,  0. -0.j , -0. +0.j ,
         0. +0.j ,  0. -0.j , -0.5-0.j , -0.5-0.j ,  0. +0.j , -0. +0.5j,
         0.5+0.j ,  0. -0.5j,  0.5+0.j ,  0. +0.j , -0. +0.5j,  0. +0.5j],
       [ 0. +0.j ,  0.5+0.j ,  0.5-0.5j,  0. +0.j ,  0. +0.j , -0. +0.5j,
         0.5-0.5j, -0. +0.j ,  0. -0.5j,  0.5+0.j , -0.5+0.j , -0.5+0.j ,
        -0. +0.j , -0. +0.j , -0. +0.5j,  0. -0.j ,  0. -0.5j,  0. +0.j ,
         0. -0.j ,  0. -0.j ,  0.5+0.j ,  0. -0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j , -0.5+0.j ,  0. -0.5j,  0. +0.j ,  0. -0.5j, -0.5-0.j ,
         0. +0.j ,  0. +0.5j,  0.5+0.j ,  0. -0.5j, -0.5-0.j , -0.5+0.j ,
         0. -0.5j, -0.5+0.j ,  0. +0.j ,  0. -0.j ,  0.5+0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.j , -0. +0.j ,  0.5+0.j ,  0. +0.j ,  0. -0.j ],
       [ 0. -0.j ,  0. -0.j , -0. +0.j , -0.5-0.j ,  0. -0.5j,  0. +0.j ,
        -0. +0.j , -0. +0.j ,  0. +0.j ,  0.5+0.j ,  0. +0.j , -0.5-0.j ,
         0. -0.5j,  0.5+0.j ,  0. +0.j ,  0. +0.j ,  0. -0.5j, -0.5+0.j ,
        -0.5-0.j , -0.5-0.j ,  0. -0.j , -0. +0.j , -0.5-0.j ,  0. -0.5j,
        -0.5-0.j , -0.5-0.j ,  0. -0.5j,  0. -0.j , -0. +0.j , -0.5+0.j ,
         0. +0.j ,  0. +0.j , -0. +0.j ,  0. +0.j , -0.5+0.j , -0.5+0.j ,
         0. -0.5j, -0.5-0.j ,  0. -0.j , -0. +0.j ,  0.5+0.j , -0. +0.j ,
         0. +0.j , -0. +0.5j,  0. +0.j ,  0.5+0.j ,  0. -0.5j, -0. +0.j ],
       [ 0. +0.j ,  0.5+0.j , -0. +0.j ,  0. +0.j ,  0. -0.5j, -0. +0.5j,
        -0. +0.j ,  0. -0.5j, -0. +0.j ,  0.5+0.j , -0.5+0.j ,  0. +0.j ,
         0. -0.5j, -0. +0.j ,  0. +0.j , -0.5-0.j ,  0. +0.j ,  0. +0.j ,
         0. +0.j ,  0. +0.j ,  0.5+0.j ,  0. -0.5j, -0.5+0.j ,  0. -0.5j,
        -0.5+0.j , -0.5+0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  0. +0.j ,
         0. -0.5j,  0. +0.5j,  0. -0.j ,  0. -0.5j, -0.5-0.j ,  0. +0.j ,
         0. -0.5j, -0.5-0.j , -0. +0.j ,  0.5+0.j , -0. +0.j ,  0. -0.5j,
         0. +0.j ,  0. +0.5j,  0. +0.j ,  0.5+0.j ,  0. +0.j ,  0. +0.j ]],
      dtype=np.complex64), np.array([[-0. +0.5j,  0. -0.5j,  0. +0.j ,  0. -0.j , -0. +0.j , -0.5+0.j ,
         0. +0.j , -0. +0.j ,  0.5+0.j , -0. +0.5j,  0. +0.j , -0.5+0.5j,
        -0. +0.j , -0. +0.j ,  0.5+0.j ,  0. -0.5j, -0.5-0.5j,  0. +0.5j,
        -0.5-0.5j,  0. +0.j ,  0. +0.j ,  0. +0.j ,  0. -0.j ,  0. +0.j ,
         0. -0.5j,  0. +0.j , -0. +0.5j, -0. +0.5j,  0.5+0.j ,  0. +0.j ,
        -0. +0.5j,  0. +0.j ,  0. -0.5j,  0. -0.j ,  0.5+0.j ,  0. -0.j ,
         0.5+0.j ,  0. -0.5j, -0.5-0.j ,  0.5-0.5j,  0. -0.5j,  0. +0.j ,
        -0.5-0.j , -0.5-0.j , -0.5+0.j ,  0. +0.j ,  0. -0.5j,  0.5+0.j ],
       [-0.5+0.j , -0.5-0.j , -0. +0.j ,  0. +0.j ,  0. +0.j , -0.5-0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.5j,  0. -0.5j,  0. +0.j ,  0.5-0.5j,
         0. +0.j , -0. +0.j ,  0. -0.5j, -0. +0.5j, -0.5+0.5j, -0. +0.5j,
        -0.5-0.5j,  0. +0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,  0. -0.j ,
        -0.5+0.j , -0. +0.j ,  0.5+0.j , -0.5-0.j ,  0. -0.5j,  0. +0.j ,
         0.5+0.j ,  0. +0.j , -0.5+0.j ,  0. +0.j ,  0. +0.5j,  0. +0.j ,
         0.5+0.j , -0.5-0.j , -0.5-0.j , -0.5-0.5j,  0. +0.5j, -0. +0.j ,
         0.5+0.j ,  0. +0.5j,  0. -0.5j,  0. -0.j , -0.5-0.j ,  0. -0.5j],
       [ 0. -0.5j,  0. +0.5j,  0. +0.j ,  0. -0.j ,  0. +0.j ,  0. +0.5j,
        -0. +0.j ,  0. +0.j ,  0. -0.5j,  0.5+0.j , -0. +0.j ,  0.5+0.5j,
        -0. +0.j ,  0. +0.j , -0. +0.5j, -0.5-0.j , -0.5-0.5j,  0.5+0.j ,
        -0.5+0.5j,  0. +0.j ,  0. -0.j ,  0. +0.j , -0. +0.j , -0. +0.j ,
        -0.5-0.j ,  0. -0.j , -0. +0.5j,  0.5+0.j , -0.5+0.j ,  0. +0.j ,
         0. -0.5j, -0. +0.j ,  0. -0.5j,  0. +0.j ,  0. -0.5j,  0. +0.j ,
         0. -0.5j, -0.5+0.j ,  0. -0.5j, -0.5+0.5j, -0.5-0.j ,  0. -0.j ,
         0. -0.5j, -0. +0.5j, -0.5-0.j ,  0. +0.j ,  0. -0.5j,  0. +0.5j],
       [ 0. -0.5j,  0. -0.5j, -0. +0.j ,  0. -0.j ,  0. +0.j ,  0.5+0.j ,
         0. +0.j ,  0. +0.j ,  0. +0.5j, -0. +0.5j,  0. +0.j , -0.5+0.5j,
         0. -0.j , -0. +0.j ,  0. -0.5j,  0. -0.5j,  0.5+0.5j,  0. -0.5j,
         0.5+0.5j,  0. -0.j ,  0. -0.j , -0. +0.j ,  0. +0.j ,  0. +0.j ,
         0.5+0.j , -0. +0.j ,  0. -0.5j,  0.5+0.j ,  0.5+0.j ,  0. -0.j ,
         0. +0.5j,  0. +0.j ,  0. +0.5j,  0. +0.j ,  0. -0.5j, -0. +0.j ,
        -0.5+0.j ,  0.5+0.j , -0.5-0.j ,  0.5-0.5j,  0. -0.5j,  0. +0.j ,
         0.5+0.j ,  0. -0.5j, -0.5-0.j ,  0. +0.j , -0. +0.5j,  0. -0.5j],
       [ 0. -0.5j,  0. -0.j ,  0.5+0.j , -0.5-0.5j,  0. -0.j ,  0.5+0.j ,
         0.5+0.j ,  0. +0.j , -0.5-0.j , -0. +0.j , -0. +0.j ,  0.5-0.5j,
        -0.5+0.j , -0. +0.5j,  0. +0.5j, -0. +0.5j,  0. -0.j ,  0. -0.5j,
         0.5+0.5j, -0. +0.j ,  0.5+0.j , -0. +0.j , -0.5+0.j ,  0. -0.j ,
         0. +0.5j, -0. +0.j , -0. +0.j ,  0. -0.5j, -0.5-0.j ,  0. -0.5j,
        -0. +0.j ,  0. -0.j ,  0. +0.5j, -0.5-0.5j, -0.5+0.j ,  0. +0.j ,
         0. +0.j , -0.5-0.j ,  0. -0.j ,  0. +0.j ,  0. +0.5j, -0. +0.5j,
        -0. +0.j ,  0. -0.5j,  0.5+0.j ,  0. +0.j ,  0. -0.j ,  0. -0.5j],
       [ 0.5+0.j ,  0. -0.j , -0.5-0.j ,  0.5-0.5j,  0. -0.j ,  0.5+0.j ,
         0.5+0.j ,  0. -0.j , -0. +0.5j,  0. +0.j ,  0. -0.j , -0.5+0.5j,
         0.5+0.j , -0.5+0.j , -0.5-0.j ,  0. -0.5j, -0. +0.j ,  0. -0.5j,
         0.5+0.5j,  0. +0.j ,  0. +0.5j,  0. +0.j , -0.5+0.j ,  0. +0.j ,
         0.5+0.j ,  0. +0.j ,  0. +0.j ,  0.5+0.j , -0. +0.5j,  0.5+0.j ,
         0. +0.j ,  0. -0.j ,  0.5+0.j ,  0.5-0.5j,  0. -0.5j,  0. +0.j ,
         0. +0.j ,  0. -0.5j,  0. -0.j ,  0. -0.j ,  0. -0.5j, -0.5-0.j ,
         0. +0.j ,  0.5+0.j ,  0. +0.5j,  0. +0.j ,  0. -0.j ,  0.5+0.j ],
       [ 0. +0.5j,  0. +0.j ,  0. +0.5j, -0.5-0.5j,  0. +0.j ,  0. -0.5j,
        -0. +0.5j,  0. +0.j ,  0. +0.5j,  0. +0.j ,  0. +0.j , -0.5-0.5j,
         0. +0.5j, -0. +0.5j, -0.5+0.j ,  0.5+0.j ,  0. -0.j , -0.5-0.j ,
         0.5-0.5j, -0. +0.j , -0.5+0.j ,  0. -0.j , -0. +0.5j,  0. +0.j ,
         0.5+0.j ,  0. +0.j ,  0. +0.j , -0.5+0.j ,  0.5+0.j ,  0. -0.5j,
         0. +0.j ,  0. +0.j , -0. +0.5j,  0.5+0.5j, -0. +0.5j,  0. -0.j ,
         0. -0.j , -0. +0.5j,  0. +0.j , -0. +0.j ,  0.5+0.j ,  0. -0.5j,
         0. +0.j , -0.5+0.j ,  0.5+0.j ,  0. -0.j ,  0. -0.j ,  0.5+0.j ],
       [-0. +0.5j,  0. +0.j , -0.5-0.j , -0.5-0.5j, -0. +0.j , -0.5+0.j ,
         0.5+0.j ,  0. -0.j ,  0. -0.5j,  0. +0.j ,  0. -0.j ,  0.5-0.5j,
        -0.5+0.j , -0. +0.5j, -0.5-0.j , -0. +0.5j,  0. +0.j , -0. +0.5j,
        -0.5-0.5j,  0. +0.j , -0.5+0.j ,  0. +0.j ,  0.5+0.j , -0. +0.j ,
        -0.5-0.j ,  0. +0.j ,  0. -0.j , -0.5+0.j , -0.5+0.j ,  0. -0.5j,
        -0. +0.j ,  0. -0.j ,  0. -0.5j,  0.5+0.5j, -0. +0.5j,  0. -0.j ,
         0. -0.j ,  0. +0.5j, -0. +0.j ,  0. +0.j , -0. +0.5j,  0. -0.5j,
         0. +0.j , -0.5-0.j ,  0.5+0.j ,  0. -0.j , -0. +0.j ,  0.5+0.j ],
       [ 0. -0.j ,  0. +0.j , -0.5-0.j ,  0.5+0.j ,  0. +0.j , -0.5-0.j ,
         0. +0.j ,  0. +0.j ,  0. -0.5j,  0. -0.j ,  0. -0.j ,  0. +0.5j,
         0.5+0.j ,  0. -0.5j,  0. -0.5j,  0. -0.5j, -0.5+0.j ,  0. -0.j ,
        -0.5+0.j , -0. +0.5j, -0.5-0.j , -0. +0.5j,  0. +0.j , -0.5+0.j ,
         0. -0.5j, -0.5-0.j , -0. +0.5j,  0.5+0.j , -0. +0.j ,  0. -0.j ,
        -0. +0.5j, -0.5-0.j ,  0. -0.5j,  0.5+0.j ,  0.5+0.j ,  0. +0.5j,
         0.5+0.j ,  0. -0.5j, -0.5-0.j ,  0. -0.5j,  0. +0.j ,  0. +0.j ,
         0. +0.j , -0. +0.5j, -0.5-0.j ,  0. +0.j ,  0. +0.j ,  0.5+0.j ],
       [ 0. +0.j ,  0. +0.j ,  0.5+0.j , -0.5-0.j ,  0. -0.j , -0.5-0.j ,
         0. +0.j ,  0. -0.j , -0.5+0.j ,  0. +0.j ,  0. -0.j ,  0. -0.5j,
        -0.5+0.j ,  0.5+0.j ,  0.5+0.j , -0. +0.5j, -0.5-0.j ,  0. -0.j ,
        -0.5+0.j , -0. +0.5j,  0. -0.5j,  0. -0.5j, -0. +0.j ,  0. +0.5j,
        -0.5-0.j ,  0. -0.5j,  0.5+0.j ,  0. +0.5j,  0. +0.j ,  0. +0.j ,
         0.5+0.j , -0.5-0.j , -0.5-0.j , -0.5-0.j , -0. +0.5j,  0. -0.5j,
         0.5+0.j , -0.5+0.j , -0.5-0.j ,  0. -0.5j,  0. -0.j ,  0. -0.j ,
         0. -0.j , -0.5-0.j ,  0. -0.5j,  0. +0.j ,  0. +0.j ,  0. -0.5j],
       [-0. +0.j ,  0. -0.j ,  0. -0.5j,  0.5+0.j ,  0. -0.j ,  0. +0.5j,
         0. +0.j ,  0. +0.j ,  0.5+0.j ,  0. -0.j ,  0. +0.j , -0. +0.5j,
         0. -0.5j,  0. -0.5j,  0.5+0.j , -0.5-0.j , -0.5+0.j ,  0. -0.j ,
        -0.5-0.j , -0.5+0.j ,  0.5+0.j ,  0. -0.5j,  0. +0.j ,  0.5+0.j ,
        -0.5+0.j ,  0.5+0.j , -0. +0.5j, -0. +0.5j,  0. +0.j ,  0. -0.j ,
         0. -0.5j,  0.5+0.j ,  0. -0.5j, -0.5+0.j ,  0. -0.5j, -0.5+0.j ,
         0. -0.5j, -0.5-0.j ,  0. -0.5j,  0. +0.5j,  0. +0.j ,  0. -0.j ,
         0. +0.j ,  0.5+0.j , -0.5+0.j ,  0. -0.j ,  0. +0.j , -0. +0.5j],
       [-0. +0.j , -0. +0.j ,  0.5+0.j , -0. +0.5j,  0. +0.j ,  0.5+0.j ,
         0. +0.j ,  0. -0.j , -0.5-0.j ,  0. +0.j , -0. +0.j , -0.5-0.j ,
         0.5+0.j ,  0. -0.5j,  0.5+0.j ,  0. -0.5j, -0. +0.5j, -0. +0.j ,
        -0. +0.5j,  0. +0.5j,  0.5+0.j ,  0.5+0.j , -0. +0.j , -0.5-0.j ,
         0.5+0.j ,  0.5+0.j ,  0. -0.5j, -0. +0.5j,  0. -0.j ,  0. +0.j ,
        -0. +0.5j,  0. -0.5j,  0. +0.5j,  0. -0.5j,  0. -0.5j,  0. -0.5j,
        -0.5-0.j ,  0.5+0.j , -0.5-0.j ,  0.5+0.j , -0. +0.j ,  0. +0.j ,
         0. -0.j ,  0.5+0.j , -0.5+0.j ,  0. -0.j ,  0. +0.j ,  0. -0.5j],
       [-0. +0.5j,  0. -0.5j,  0. +0.j , -0. +0.5j,  0. -0.5j,  0. +0.j ,
        -0.5-0.j ,  0.5+0.j ,  0.5+0.j , -0. +0.5j, -0.5+0.j , -0.5-0.j ,
         0. +0.j , -0. +0.j ,  0.5+0.j , -0. +0.j ,  0. -0.5j,  0. +0.5j,
         0. -0.5j,  0. +0.j ,  0. +0.j , -0.5+0.j ,  0.5+0.j , -0. +0.j ,
         0.5+0.j , -0. +0.j ,  0. +0.j ,  0. +0.5j,  0.5+0.j ,  0. +0.5j,
         0. +0.j ,  0. -0.5j, -0. +0.j , -0. +0.5j, -0. +0.5j,  0. +0.j ,
         0. +0.j ,  0.5+0.j ,  0. +0.j ,  0.5+0.j ,  0. -0.5j,  0. -0.5j,
        -0.5-0.j , -0.5+0.j ,  0. +0.j ,  0. -0.5j,  0. -0.5j,  0. +0.5j],
       [-0.5+0.j , -0.5+0.j , -0. +0.j ,  0. +0.5j, -0.5-0.j ,  0. +0.j ,
        -0.5+0.j , -0.5-0.j ,  0. -0.5j,  0. -0.5j, -0.5+0.j ,  0.5+0.j ,
         0. +0.j , -0. +0.j ,  0. -0.5j,  0. +0.j ,  0. +0.5j,  0. +0.5j,
         0. -0.5j,  0. +0.j ,  0. +0.j ,  0.5+0.j ,  0.5+0.j ,  0. +0.j ,
         0. -0.5j,  0. -0.j ,  0. -0.j , -0.5+0.j ,  0. -0.5j, -0.5-0.j ,
        -0. +0.j ,  0. -0.5j,  0. +0.j ,  0. +0.5j, -0.5-0.j ,  0. +0.j ,
         0. +0.j , -0. +0.5j,  0. +0.j , -0.5+0.j ,  0. +0.5j,  0.5+0.j ,
         0.5+0.j , -0. +0.5j, -0. +0.j ,  0.5+0.j , -0.5-0.j , -0.5-0.j ],
       [ 0. -0.5j,  0. +0.5j, -0. +0.j , -0. +0.5j,  0. -0.5j,  0. -0.j ,
         0. -0.5j, -0. +0.5j,  0. -0.5j,  0.5+0.j ,  0. -0.5j,  0.5+0.j ,
        -0. +0.j , -0. +0.j , -0. +0.5j,  0. +0.j ,  0. -0.5j,  0.5+0.j ,
         0. +0.5j,  0. -0.j , -0. +0.j , -0.5-0.j ,  0. -0.5j,  0. +0.j ,
         0. +0.5j,  0. +0.j ,  0. +0.j ,  0.5+0.j , -0.5-0.j ,  0. +0.5j,
        -0. +0.j ,  0. -0.5j,  0. +0.j ,  0. -0.5j, -0.5+0.j , -0. +0.j ,
         0. -0.j ,  0. -0.5j, -0. +0.j , -0.5-0.j , -0.5-0.j ,  0. +0.5j,
         0. -0.5j,  0. +0.5j,  0. +0.j ,  0. -0.5j,  0. -0.5j, -0.5-0.j ],
       [ 0. -0.5j,  0. -0.5j,  0. -0.j ,  0.5+0.j ,  0. +0.5j, -0. +0.j ,
        -0.5-0.j , -0.5-0.j , -0. +0.5j,  0. +0.5j, -0.5-0.j ,  0. +0.5j,
         0. +0.j ,  0. +0.j ,  0. -0.5j,  0. +0.j ,  0.5+0.j ,  0. -0.5j,
         0.5+0.j , -0. +0.j , -0. +0.j ,  0. -0.5j, -0.5+0.j ,  0. -0.j ,
         0. -0.5j,  0. +0.j ,  0. +0.j ,  0.5+0.j ,  0.5+0.j , -0. +0.5j,
         0. -0.j , -0.5-0.j ,  0. -0.j , -0.5+0.j , -0.5+0.j ,  0. +0.j ,
         0. -0.j ,  0. -0.5j,  0. -0.j ,  0. -0.5j,  0. -0.5j, -0. +0.5j,
         0.5+0.j ,  0. -0.5j,  0. +0.j ,  0. -0.5j, -0. +0.5j, -0.5-0.j ]],
      dtype=np.complex64))

In [ ]:
#@title Verification
n = 4
m = 4
p = 4
rank = 48

verify_tensor_decomposition(decomposition_444, m, n, p, rank)

## Rank-96 decomposition of <4,4,8> over 0.5*Z[i]


This decomposition can be obtained by doubling the rank-48 decomposition of <4,4,4> provided above.

## Rank-61 decomposition of <4,4,5> over Z

In [ ]:
#@title Data

decomposition_445 = (np.array([[-0.,  0., -0.,  0.,  0., -1.,  0.,  1., -0.,  0.,  0.,  0., -0.,
        -0., -0., -0., -1., -0., -0.,  0., -0., -0., -0.,  0.,  0., -0.,
        -0., -0., -0., -0., -0., -0., -0.,  0.,  0., -0.,  1., -0., -1.,
         1., -0., -1., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,
         0., -0.,  1.,  0., -1., -1., -0.,  0., -1.],
       [ 0.,  1.,  0.,  0., -0.,  1., -1., -0.,  0.,  0., -1.,  0., -0.,
        -0.,  1., -0.,  0., -0., -1., -1., -0., -1., -1.,  0.,  0., -0.,
         0.,  0.,  1., -0., -0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,
        -0.,  1., -1.,  0., -0.,  0.,  0., -1., -0.,  0., -0.,  0., -0.,
         0.,  0., -0., -0.,  0.,  0., -1.,  0., -0.],
       [-0., -0., -0.,  0., -0.,  0., -0.,  1.,  0.,  0.,  0., -1.,  1.,
         0., -0., -0., -1.,  1.,  0.,  0., -0.,  0.,  0., -1.,  0.,  1.,
        -0.,  0., -0., -0., -0., -0., -0.,  0.,  0., -0.,  1., -0., -1.,
         1., -0.,  0., -0.,  0., -1.,  1.,  0.,  0.,  0.,  0., -1.,  1.,
        -0., -0.,  1., -0., -1., -1.,  0.,  0., -1.],
       [-0., -0.,  0.,  0., -0., -0., -1., -0., -0.,  0., -1., -0.,  0.,
         0., -0., -0., -1., -0.,  0., -1.,  0.,  0.,  0., -0.,  0., -0.,
         0., -0.,  1.,  0., -0.,  0., -0., -0., -0., -0., -0., -0.,  0.,
        -0.,  1., -1.,  0.,  0., -0., -1., -1.,  0.,  0., -0., -0.,  0.,
         0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.],
       [ 0., -0., -0.,  0., -1.,  0., -0.,  0., -0., -0.,  0.,  0.,  0.,
         0., -0., -0.,  0., -0., -0., -0.,  0., -0., -0., -0., -0.,  0.,
         0., -0.,  1., -0., -0.,  0., -0., -0., -0., -0., -0., -0.,  1.,
        -0.,  0., -1., -0.,  0.,  0.,  0.,  0.,  0., -1.,  0.,  1.,  0.,
        -1., -0.,  0., -0., -1.,  0., -0.,  0.,  0.],
       [-0., -0., -0.,  0.,  1., -0.,  1., -0., -1., -1.,  0., -1., -0.,
         1.,  0.,  0.,  0.,  1., -0., -0.,  1., -0., -0.,  0., -1., -0.,
        -0.,  0., -0.,  1., -0.,  1., -0., -0.,  0.,  0.,  1.,  0.,  0.,
        -0.,  0.,  0., -0.,  1., -1., -0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0., -1.,  0., -0., -0.,  0.],
       [-0., -0.,  0., -0., -0., -0.,  1., -0.,  0.,  0., -0., -1., -0.,
        -0.,  0., -0.,  0.,  1.,  0., -0., -0.,  0., -0.,  0.,  0., -0.,
         1., -0., -0.,  1.,  0., -0.,  0., -0.,  0.,  0.,  1.,  0.,  0.,
        -0., -0.,  0.,  0., -0., -1., -0., -0.,  0., -1., -0.,  0., -0.,
         0., -0., -0.,  0., -1.,  0.,  0.,  0.,  0.],
       [-0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -1.,  0.,  0.,
        -0., -0., -0.,  0., -0., -0.,  0., -0.,  0., -0., -0.,  0.,  0.,
        -1., -0.,  1., -0., -0., -0., -0., -0.,  1., -0., -0., -0.,  1.,
         0., -0., -1.,  0., -0.,  0.,  0., -1.,  0., -1., -0.,  1., -1.,
        -1.,  0., -1.,  0.,  0.,  0.,  0.,  0., -0.],
       [ 0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0., -0., -1.,
         0.,  1.,  1.,  0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,
         0.,  1., -0.,  0., -0.,  0.,  0., -1., -0.,  1.,  0., -0., -0.,
         0., -0., -1.,  0., -1.,  1.,  0.,  0.,  0., -1., -0.,  1.,  0.,
         0.,  0.,  0., -0.,  0., -0.,  0., -1., -0.],
       [-0.,  1., -0., -1., -0., -0.,  0., -0.,  0., -1.,  0.,  0., -0.,
        -0.,  0., -0.,  0.,  1., -0.,  0., -0.,  0., -0., -0., -1., -0.,
        -0., -1., -0., -0.,  1., -0.,  1.,  0.,  0., -0., -0., -0.,  0.,
        -0., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,
        -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  1.,  0.,  0., -0., -0.,  0., -0., -0.,  0., -0.,  0.,  0.,
        -0., -0.,  1., -0.,  1.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,
         0., -0.,  0.,  0., -0.,  0.,  0., -1., -0.,  1., -0.,  0.,  0.,
        -0., -0.,  0.,  0., -1., -0., -0.,  0., -0., -1.,  0., -0., -0.,
        -0., -0., -0.,  1.,  0.,  0.,  0., -1.,  0.],
       [-0., -0., -0., -1., -0.,  0.,  0., -0., -0., -1.,  0.,  0.,  0.,
        -0., -0., -0., -1.,  1.,  1.,  0., -0., -0.,  1.,  0., -1.,  1.,
        -0.,  0., -0., -0.,  1., -0.,  1.,  0.,  0., -0., -0., -0., -0.,
        -0., -0.,  0., -0.,  0., -0.,  0.,  0., -1.,  0.,  0., -0., -0.,
         0., -0.,  0., -1.,  0.,  0., -0.,  0.,  0.],
       [-0., -0.,  0., -1.,  0.,  0., -0., -0., -0., -1.,  1., -0.,  0.,
        -0., -0., -0., -1., -0.,  1.,  0.,  0.,  1.,  0.,  0., -0., -0.,
        -0., -0.,  0., -0., -0.,  1.,  0.,  1., -0., -0., -0.,  0.,  0.,
         1., -1.,  0.,  0.,  0., -0., -0.,  0., -0.,  0., -1., -0., -0.,
         0.,  1.,  1.,  0., -1., -1.,  0.,  0.,  0.],
       [ 0.,  1., -1.,  0., -0., -0.,  0.,  0.,  0., -1.,  0.,  0.,  0.,
        -0., -0.,  1.,  0., -0., -0., -1.,  0.,  0., -1., -0., -0.,  0.,
         0.,  0., -0., -0., -0., -0., -1., -0., -0., -0., -0., -0., -0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0., -1., -0.,  0., -0., -0.,  0.,
         0., -1.,  0., -0., -0.,  0., -1.,  0.,  1.],
       [-1.,  1., -1.,  0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,  0.,
         0., -0.,  1.,  0., -0., -0., -1., -1., -0., -1., -0.,  1.,  0.,
         0., -0., -0.,  0.,  0., -0., -1.,  0.,  1., -0., -0.,  0., -0.,
        -0.,  0., -0., -1., -1.,  0.,  0., -1.,  1., -1.,  0., -0.,  0.,
         0.,  0., -0., -0.,  0.,  0., -1., -1.,  1.],
       [ 1., -0.,  0., -1., -0.,  0.,  0., -0.,  0., -1., -0.,  0., -0.,
         0.,  0., -0., -1., -0.,  1.,  0., -0., -0., -0.,  0.,  0., -0.,
         0.,  0., -0., -0., -0., -0.,  0.,  1., -0.,  0.,  0.,  1., -0.,
        -0., -1.,  0., -0., -0.,  0.,  0., -1.,  0.,  0., -0., -0.,  0.,
         0., -0., -0.,  0.,  0., -1.,  0., -0., -0.]], dtype=np.float32),
 np.array([[ 0., -0.,  0., -0., -0.,  1., -0., -0., -0.,  0., -1., -0.,  0.,
         0.,  1.,  0.,  0.,  0., -0.,  0.,  0.,  1., -0., -0., -0., -0.,
        -0.,  0.,  1.,  0., -0., -0., -0.,  0.,  0., -0., -0., -0.,  0.,
        -0., -0.,  1., -0., -0., -0.,  0., -0., -0.,  0.,  1.,  0.,  0.,
        -0.,  0.,  0., -0.,  0., -0., -0.,  0., -0.],
       [-0.,  0., -0., -0.,  1., -0., -0.,  0.,  1.,  0., -1., -0.,  0.,
        -1.,  0., -0.,  0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,
        -0., -0.,  1., -0.,  0., -1., -0., -0., -0., -0.,  0.,  0., -0.,
        -0.,  0.,  1.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,
         0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0.],
       [ 1.,  0., -0.,  0., -0., -0.,  0., -0., -0., -0.,  0.,  0., -1.,
         0., -0., -0., -0., -0.,  0., -0., -0., -0., -0.,  0., -0., -1.,
        -0.,  0., -0.,  0., -0.,  0., -0.,  1.,  1.,  1., -0., -1., -0.,
         1.,  0., -0., -1.,  0., -0., -0.,  0., -1.,  0.,  0., -1., -0.,
         1., -0.,  1., -1., -0., -1.,  0., -0.,  0.],
       [ 0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  1., -1.,  0.,
        -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,  0., -0.,
        -0.,  0., -1., -0.,  0., -0.,  0.,  0.,  0.,  0.,  1., -0., -1.,
        -0., -0., -1., -0., -0.,  1., -0., -0., -0.,  0., -0., -1., -1.,
         0.,  0.,  1., -0.,  1., -0., -0.,  0., -0.],
       [ 1.,  0.,  0.,  0., -0.,  0., -0.,  1., -0., -0.,  0., -0.,  0.,
         0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0., -1., -0., -1.,
         0.,  0., -0.,  0., -0., -0., -0.,  1.,  1.,  1.,  0., -1.,  1.,
        -0., -0., -0., -1., -0.,  0.,  0., -0., -1.,  0., -1.,  0.,  1.,
         1.,  0., -0., -1.,  0., -1., -0.,  0., -0.],
       [-0.,  0.,  0.,  1.,  0., -0.,  0., -0., -0., -1.,  0.,  0., -0.,
         0., -0.,  0.,  0.,  0.,  1.,  0., -0.,  1., -0., -0., -0., -0.,
        -0., -0., -0.,  0., -0.,  1.,  0., -0.,  0., -0., -0., -0., -0.,
         0., -0., -0., -0., -0., -0., -0., -0., -0.,  0.,  1.,  0.,  0.,
        -0., -1.,  0.,  0.,  0., -0.,  0., -0., -0.],
       [ 0.,  0., -0.,  1., -0.,  0., -0.,  0.,  1., -1.,  0.,  0., -0.,
        -1., -1., -0.,  0.,  0.,  1., -0., -0., -0., -0.,  0., -0.,  0.,
         0.,  1., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,
        -0.,  0., -0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,
         0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.],
       [-0., -1.,  0.,  1., -0.,  0., -0.,  0., -0., -1.,  0., -0.,  0.,
         0.,  0.,  1., -0., -0.,  1., -0.,  0., -0.,  1.,  0., -1.,  0.,
         0., -0., -0.,  0., -0., -0.,  1., -0., -0., -0.,  0.,  0., -0.,
        -0.,  0.,  0.,  0., -1.,  0.,  0., -0., -1.,  0., -0.,  0., -0.,
        -0.,  0., -0.,  0.,  0.,  0., -0.,  1.,  0.],
       [-0.,  0., -0., -0., -0.,  0., -1.,  0.,  0., -0.,  0.,  0., -0.,
        -0., -0., -0.,  0., -0.,  0., -1., -1.,  0.,  1.,  1., -1.,  1.,
         1.,  0., -0.,  1., -1.,  0.,  0.,  0., -1.,  0., -0.,  0.,  0.,
         0., -0.,  0., -0.,  0., -0.,  1., -0., -0., -0.,  0.,  0., -1.,
         0., -0.,  0.,  0., -0., -0.,  1., -0., -0.],
       [-0.,  0., -1., -0.,  0.,  0.,  1., -0., -0.,  0., -0.,  0., -0.,
         0.,  0., -0.,  0.,  0., -0.,  1., -0.,  0., -0., -1., -0., -1.,
        -1., -0.,  0., -1.,  1.,  0.,  1., -0.,  1., -0., -0., -0., -0.,
         0.,  0.,  0., -1.,  0., -0., -1., -0., -1.,  0.,  0.,  0.,  1.,
        -0., -0.,  0.,  0., -0., -0.,  0., -0., -0.],
       [ 0.,  0.,  1.,  0., -0., -1.,  0., -1.,  0., -0., -0.,  0., -0.,
        -0., -1.,  1., -0.,  1.,  0.,  0.,  0., -1., -0.,  0., -1., -0.,
        -0., -0.,  0.,  1., -1., -1., -0.,  0.,  0., -1., -1.,  0., -0.,
         0.,  0.,  0.,  0., -1., -1., -0., -0.,  0.,  1., -1.,  1., -0.,
        -1.,  1.,  0., -0.,  0., -0., -0.,  0., -1.],
       [ 0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -1., -0., -0.,  0., -0.,
         1., -0., -0.,  0.,  1.,  0.,  0.,  0.,  0., -0.,  0., -1., -0.,
        -0.,  0.,  0., -0., -1., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,
        -0., -0.,  0.,  0., -1., -1.,  0.,  0., -0.,  1.,  0.,  1.,  0.,
        -1., -0.,  0., -0.,  0.,  0., -0.,  0.,  0.],
       [ 0., -0., -0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  1.,
        -0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0., -0., -0.,  1.,
        -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -1.,  0., -0., -0.,
         0., -0., -0., -0.,  0.,  0.,  0., -0.,  1.,  1.,  0.,  1.,  0.,
        -1., -0.,  0.,  1., -0., -0., -0., -1.,  0.],
       [ 0.,  0.,  0.,  0.,  0., -0., -0., -0., -0., -0.,  0.,  1., -0.,
         0., -0.,  0., -0., -1.,  0., -0.,  1., -0.,  0.,  0.,  1.,  0.,
        -1.,  0., -0., -1.,  1., -0., -0.,  0.,  1.,  0.,  0.,  0., -0.,
        -0., -0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0., -0.,  1.,
        -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.],
       [ 0.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,
        -0., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,  1.,  0.,  1.,
         1., -0., -0., -0.,  0., -0.,  0.,  0., -1.,  0., -0., -0.,  0.,
        -0.,  0., -0.,  1., -0., -0., -0.,  0.,  1., -0., -0., -0., -1.,
         0.,  0.,  0.,  1.,  0., -0.,  0.,  0., -0.],
       [-0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  1., -0.,  0.,
        -0.,  0.,  0.,  0.,  0., -1.,  0., -0., -1., -0.,  0.,  0.,  0.,
        -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0., -0.,  0.,  1.,  0.,
         0., -1.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,  0., -0., -0.,
        -0., -0.,  0., -0.,  0., -0., -0., -0.,  0.],
       [ 0., -0.,  0., -1., -1., -0., -0., -0., -1., -0.,  1.,  0.,  0.,
         1.,  1.,  0.,  0.,  0., -1.,  0., -0.,  0., -0., -0., -0., -0.,
         0., -1., -1.,  0.,  1.,  1., -0.,  0.,  0.,  0.,  0., -0., -0.,
         0.,  0.,  0., -0., -0., -0., -0., -0., -0.,  0., -0.,  0.,  0.,
         1.,  0.,  0.,  0., -0., -0., -0.,  0., -0.],
       [-1., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0., -0.,
        -0., -0.,  0., -1., -0.,  0., -0., -0.,  0.,  0., -0., -0., -0.,
         0.,  0.,  0., -0., -0., -0.,  0.,  0., -1.,  0., -0.,  1., -0.,
        -1.,  0., -0.,  1., -0., -0., -0.,  0.,  1.,  0., -0., -0.,  0.,
         0.,  0., -1., -0., -0.,  1.,  0.,  0.,  0.],
       [-0.,  0., -0., -0., -0.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,
         0.,  0., -0.,  0., -0., -0.,  1., -0., -0., -1., -1., -0., -1.,
         0., -0., -0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,
         0., -0., -0.,  0.,  0.,  0., -1., -1.,  0., -0.,  0., -0.,  1.,
         0., -0.,  0., -0.,  0., -0., -1., -0.,  0.],
       [-1.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,
        -0., -0., -0.,  0.,  0., -0.,  0., -0., -0., -0.,  1., -0.,  1.,
        -0.,  0.,  0., -0.,  0., -0.,  0.,  0., -1.,  0.,  0., -0.,  0.,
         0., -0.,  0.,  1.,  0.,  0.,  1., -0.,  1.,  0., -0.,  0., -1.,
         0., -0.,  0., -0.,  0., -0., -0., -0., -0.]], dtype=np.float32),
 np.array([[ 0.,  0.,  0., -0.,  0., -1., -0., -1., -0.,  0., -0.,  0., -0.,
         0., -0.,  0.,  0.,  0., -0.,  0.,  0., -1., -0.,  0., -0.,  0.,
         0.,  0., -0., -0., -0.,  0., -0., -0., -0., -0., -0.,  1.,  0.,
        -1., -1., -0.,  0., -0.,  0.,  0., -0.,  0.,  0., -1., -0., -0.,
         0.,  0., -0., -0., -0.,  1.,  0., -0., -0.],
       [ 0.,  0., -0., -0.,  1., -0.,  0.,  1., -0.,  0., -1.,  0.,  0.,
        -0.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,
        -0.,  0.,  1.,  0., -0.,  1., -0., -0.,  0., -0., -1., -1., -0.,
         1.,  1., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,  1.,  0.,  0.,
        -0.,  0.,  0., -0., -1., -1.,  0., -0., -0.],
       [ 0.,  1., -0., -1.,  0., -0., -0.,  1.,  0.,  0., -0., -0.,  0.,
        -0.,  1.,  1.,  0.,  0., -1.,  0.,  0.,  1.,  0., -0.,  0., -0.,
        -0.,  1.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0., -1., -0.,
         1., -0., -0., -0.,  0., -0.,  0., -0., -0., -0.,  1., -0., -0.,
         0.,  0.,  0., -0.,  0., -1., -0.,  0.,  1.],
       [ 0.,  0., -0.,  0.,  0., -0.,  0., -1.,  0.,  0., -0., -0.,  0.,
         0., -0.,  0., -0.,  0., -0., -0., -0., -0.,  0.,  0., -0., -0.,
        -0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0., -0.,  1.,  0.,
        -1.,  0.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -1., -0., -0.,
        -0.,  1., -0., -0., -0.,  1., -0.,  0., -1.],
       [ 0.,  0.,  0.,  0., -0.,  1., -0.,  1., -1.,  0., -0.,  0., -1.,
         1., -1., -0.,  0.,  0., -0.,  0., -0., -0., -0.,  0., -0.,  0.,
        -0., -0., -1., -0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,  1.,
        -0., -0., -1.,  0.,  0.,  0., -0., -0.,  0.,  0., -0., -1., -0.,
         1., -0.,  0.,  0., -0.,  0.,  0., -0., -0.],
       [ 0., -0.,  0.,  0., -1.,  0.,  0., -1., -0.,  0., -0.,  0.,  1.,
        -1., -0., -0., -0.,  0.,  0., -0., -0., -0.,  0., -0., -0., -0.,
         0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0.,  1., -0., -1.,
         0.,  0., -0., -0., -0.,  1., -0., -0.,  0., -0.,  0.,  1.,  0.,
        -1., -0.,  0.,  0., -0., -0.,  0., -0.,  0.],
       [-0.,  0., -1., -0., -0.,  0.,  0.,  0.,  1.,  0.,  0., -0., -0.,
        -1.,  0., -1.,  0.,  0., -0.,  0.,  1.,  0.,  0., -0.,  1., -0.,
        -0., -1., -0.,  0.,  1., -0., -1., -0., -0.,  0.,  0., -0.,  0.,
         0., -0., -0., -0.,  1., -0.,  0.,  0., -0., -0.,  0., -0., -0.,
         0., -0., -0.,  0.,  0., -0., -0.,  0.,  0.],
       [-0.,  0.,  1.,  1., -0.,  0.,  0., -0.,  0.,  1., -0., -0.,  0.,
         1., -0., -0.,  0., -0., -0., -0., -1.,  0.,  0.,  0., -1., -0.,
         0.,  0., -0.,  0., -1., -1.,  1.,  0., -0.,  0.,  0.,  0.,  0.,
        -0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0., -0., -0., -0., -0.,
         0., -1., -0., -0.,  0.,  0., -0.,  0., -0.],
       [-0., -0., -0.,  0., -0., -0.,  0., -1., -0.,  0., -0., -0.,  0.,
         0., -0., -1.,  1., -0., -0., -0., -0., -0., -1.,  1.,  0.,  1.,
         0.,  0., -0.,  0., -0., -0.,  0., -1.,  0., -1.,  0.,  0.,  0.,
         0., -0., -0.,  1., -0., -0., -0.,  0.,  1., -0., -0., -0.,  0.,
         0.,  0.,  0.,  0.,  0.,  1.,  1., -1., -1.],
       [-0., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0., -1.,
        -0.,  0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,
         0., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0.,
        -1.,  0., -0.,  0., -1., -1., -0., -0.,  0., -1.,  0., -1., -0.,
        -0., -0.,  1., -0.,  1., -0.,  0., -1.,  0.],
       [-0., -1.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  1.,
        -0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,  1., -1.,  0., -1.,
         0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,
         0., -0., -0., -1.,  0.,  0., -0.,  0., -1., -0.,  0., -0., -0.,
        -0., -0.,  0.,  0.,  0.,  0., -1.,  1.,  0.],
       [-0., -0.,  0.,  0., -0., -0., -0.,  1.,  0.,  0.,  0.,  0., -0.,
         0., -0.,  1.,  0., -0., -0.,  0., -0., -0., -0., -0., -0.,  0.,
         0., -0., -0., -0., -0., -0.,  0.,  1.,  0.,  1., -0.,  0.,  0.,
         1.,  0., -0., -0., -0., -0., -0.,  0.,  0., -0., -0., -0.,  0.,
        -0.,  0.,  0.,  0.,  0., -1.,  0.,  1.,  1.],
       [ 0.,  0., -1., -0., -0.,  0.,  1., -0.,  0., -0., -0., -1.,  0.,
        -0.,  0., -0.,  0., -0.,  0., -1.,  0.,  0., -0., -0.,  0.,  0.,
        -0.,  0.,  0.,  1., -0.,  0.,  0., -0.,  0., -0.,  1., -0., -0.,
        -0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
        -0., -0.,  0., -0., -0.,  0., -1., -0.,  1.],
       [ 0.,  0.,  0., -0., -0., -0., -0., -0.,  0.,  0.,  0., -1., -0.,
         0., -0.,  0.,  0., -0.,  0.,  0., -1.,  0., -0.,  1.,  0., -0.,
        -0., -0., -0.,  0.,  0., -0.,  0., -0., -1.,  0.,  0.,  0., -0.,
         1., -0.,  0.,  1., -0., -0.,  0.,  0., -0., -0., -0., -0., -1.,
         0., -0., -1., -0., -1.,  0., -0., -0., -0.],
       [-0.,  1., -0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0.,  1.,  0.,
         0., -0., -0.,  0., -1.,  0., -0.,  1., -0., -1., -0.,  1., -0.,
        -0., -0., -0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0.,
        -0.,  0.,  0., -0.,  1.,  1.,  0.,  0., -0., -0., -0., -0.,  0.,
         0., -0.,  0., -0., -0.,  0.,  1.,  0., -0.],
       [ 0., -0.,  1., -0., -0.,  0., -1.,  0., -0., -0.,  0.,  1.,  0.,
        -0., -0.,  0.,  0., -0.,  0.,  1., -0.,  0., -0., -1.,  0.,  0.,
        -0., -0., -0., -1.,  0.,  0., -0.,  0.,  1.,  0., -1., -0., -0.,
        -1., -0., -0., -1.,  0.,  0.,  0.,  1.,  0., -0.,  0., -0.,  1.,
        -0.,  0.,  1., -0., -0., -0., -0.,  0., -1.],
       [-0.,  0., -1.,  0., -0.,  0., -0.,  1.,  0.,  0.,  0.,  0., -0.,
        -0.,  0.,  0.,  0.,  0.,  0., -1., -0.,  0., -0., -1.,  0.,  0.,
         0., -0.,  0., -0., -0., -0.,  0., -0., -0.,  0., -0., -0.,  0.,
        -0.,  0.,  0., -0., -0., -0., -1., -0.,  0., -0.,  0.,  0., -0.,
        -0.,  0., -0.,  0., -0.,  0., -1., -0.,  1.],
       [-0.,  0.,  0.,  0., -0., -0.,  0.,  1.,  0., -0.,  0.,  0., -0.,
        -0.,  0.,  0.,  0., -0.,  0., -0., -1.,  0., -0., -0.,  0.,  0.,
         1.,  0., -0., -1.,  0., -0.,  0.,  0., -1.,  0., -1.,  0.,  1.,
         1., -0.,  0.,  1., -0., -0., -0., -0., -0., -0., -0., -0.,  0.,
        -0., -0., -1.,  0., -1.,  0., -0., -0., -0.],
       [-0.,  1.,  1.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0., -1.,
        -0., -0.,  1.,  0.,  0.,  0., -0., -0., -0., -1.,  1., -0.,  1.,
         0., -0.,  0., -0.,  0.,  0.,  1., -0., -0.,  1., -0.,  0.,  0.,
        -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0.,  0.,  0.,
         0., -0., -0.,  1., -0.,  0.,  1.,  0., -0.],
       [-1.,  0.,  1., -0.,  0., -0.,  0., -1.,  0., -0.,  0., -0., -0.,
        -0., -0.,  0., -0., -0., -0., -0., -0.,  0., -0.,  0.,  0.,  0.,
         0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,  0.,
        -1.,  0., -0., -1., -0., -0., -0., -0., -0., -0.,  0., -0., -0.,
         0., -0., -0.,  0., -0.,  1., -0.,  0., -1.]], dtype=np.float32))

In [ ]:
#@title Verification
n = 4
m = 4
p = 5
rank = 61

verify_tensor_decomposition(decomposition_445, n, m, p, rank)

## Rank-85 decomposition of <4,4,7> over Z


In [ ]:
#@title Data

decomposition_447 = (np.array([[ 0., -0.,  0., -0., -0.,  0., -0., -0.,  0., -0.,  1., -0.,  0.,
         0., -0.,  0., -0., -0.,  0.,  0.,  0., -1., -0.,  0.,  0., -0.,
        -0.,  1.,  0., -0.,  0.,  0., -1., -0., -1., -0.,  0.,  0., -0.,
        -0.,  0., -0.,  0., -0., -0.,  0., -0.,  1.,  0.,  0., -0.,  1.,
         0.,  1.,  1., -0., -0., -1., -1.,  0., -0.,  0.,  0., -0., -0.,
         0.,  0.,  1., -0.,  0., -0.,  0., -1., -0., -0.,  0., -1., -0.,
         0.,  0.,  0., -0., -0., -0.,  1.],
       [ 0.,  0.,  0., -0.,  0., -1.,  0.,  0.,  0.,  0.,  1.,  0., -1.,
        -0.,  1.,  1., -0.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  0.,
        -0., -0., -0., -0., -0., -0.,  0., -0., -0.,  0., -0., -0., -0.,
         0., -1.,  1., -1.,  0.,  1., -1.,  0., -0., -0., -0., -0., -0.,
         1., -0.,  0.,  0.,  0., -0., -1., -1., -0.,  0., -0.,  1.,  0.,
         0., -0.,  1.,  0.,  0.,  0., -1., -0., -0., -0.,  0., -0.,  1.,
         0., -0., -1.,  1.,  0.,  1.,  0.],
       [ 0., -0., -0., -0.,  0.,  1.,  0.,  0., -0., -0.,  0., -0.,  0.,
         0.,  0.,  0.,  0., -0., -0., -0., -0., -0.,  0., -0., -0.,  0.,
         0., -0.,  0., -1.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,
        -0.,  1.,  0., -0., -0., -1.,  1.,  0.,  0.,  0.,  0., -0., -0.,
        -1., -0.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -1.,  0.,
         1., -1.,  0.,  0.,  1.,  0.,  0., -0.,  0., -0., -0., -0., -1.,
         1., -0., -0.,  0.,  1., -1., -0.],
       [-0., -0., -0., -0., -0.,  0.,  0., -0., -0., -0., -0.,  0., -0.,
        -0., -1.,  0., -0., -0.,  0., -0.,  0., -1., -0.,  0., -0., -0.,
         0., -0., -0.,  1., -0., -1.,  0.,  0., -1.,  0.,  0.,  0.,  0.,
        -0.,  0., -1., -0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0., -0.,
        -0., -0.,  1., -0.,  0., -1., -0.,  1.,  0., -0.,  0.,  0.,  0.,
        -1., -0., -0., -0., -1.,  0.,  1., -1.,  0., -0., -0., -1., -0.,
        -1.,  0.,  1., -1.,  0., -0., -0.],
       [ 0.,  0., -0., -0., -1., -0., -0., -0.,  1.,  1.,  0.,  0.,  0.,
         0., -0.,  0., -0.,  0.,  1.,  0.,  0., -0., -0., -0.,  0., -0.,
         0.,  0.,  0.,  0.,  0., -0., -0., -0.,  1., -0., -0.,  0., -0.,
        -0.,  0., -0., -0.,  0.,  0., -0., -0.,  0., -0., -0.,  1.,  0.,
        -0.,  0.,  0., -1., -1., -0., -0., -0.,  0.,  0., -0., -0.,  0.,
        -0., -0.,  0., -1.,  0., -0., -0., -0., -1., -1.,  0., -0., -0.,
        -0.,  0., -0.,  0.,  0., -0.,  0.],
       [-0.,  0., -1., -0.,  0., -0.,  0.,  0.,  1.,  0.,  0., -1.,  0.,
        -0.,  0., -0., -1., -0.,  0., -0., -0., -0., -0.,  0., -0.,  0.,
        -0.,  0., -0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,
         0.,  0.,  0., -0., -0., -1., -0., -0.,  0., -1., -1., -0.,  0.,
        -1.,  0., -0.,  0., -0., -1.,  1.,  1.,  1.,  1.,  0.,  0., -1.,
         0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,
        -1., -0., -0., -1., -0.,  0.,  0.],
       [-0., -0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  1.,  0.,
        -0.,  0., -0.,  0.,  1.,  0., -0., -0.,  0., -1., -1., -0., -0.,
        -0.,  0.,  1.,  1.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0.,
        -1., -0.,  0., -0.,  1., -0.,  0.,  0.,  0.,  1.,  0., -0., -0.,
         1., -0., -0.,  0., -1.,  0.,  0., -1., -0., -0.,  0., -0., -0.,
         0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -1., -0.,
        -0., -0.,  0., -0.,  0., -0., -0.],
       [-0.,  0.,  1., -0.,  0., -0., -0., -0., -0.,  1.,  0.,  0., -0.,
        -0., -0.,  0.,  1., -1.,  1.,  0.,  0., -0.,  1.,  1., -0., -0.,
        -1.,  0., -1., -1.,  0., -0., -0.,  0.,  1., -0.,  1., -1., -0.,
         1.,  0., -0., -0., -1., -0.,  0.,  0.,  0., -0.,  1., -0.,  0.,
         0.,  0.,  0.,  0.,  0.,  1.,  0., -0., -1., -1., -0.,  0., -0.,
         1.,  0.,  0.,  0.,  0.,  1., -0.,  1.,  0.,  0., -0.,  1.,  0.,
         1.,  0., -0., -0., -0.,  0.,  0.],
       [ 0.,  0.,  0., -1., -0.,  0., -0., -0.,  0., -1., -1.,  0., -0.,
         0.,  0.,  0., -1., -0.,  0.,  1.,  1.,  1.,  0.,  0.,  0., -1.,
        -0.,  0.,  1., -0.,  0., -0., -0., -0., -0., -1.,  0.,  1., -0.,
        -0.,  0., -0., -0., -0., -0., -0., -0.,  0.,  0.,  0., -0.,  0.,
        -0.,  0.,  0., -0., -0., -0., -0.,  0., -0.,  0., -0.,  0., -0.,
         0., -0.,  0., -0., -0., -0.,  0., -0., -0., -0., -1., -0., -0.,
         0.,  0.,  0., -0.,  0., -0.,  0.],
       [-0.,  1.,  1., -0.,  0.,  1.,  0., -0.,  0., -0., -1.,  1.,  1.,
        -0., -1.,  0., -0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,
         0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,
         0.,  1.,  0.,  1.,  0.,  0., -0., -1., -0.,  0., -0.,  0., -0.,
         0., -0., -0.,  0.,  0., -0., -0.,  0., -1.,  0., -1., -1., -0.,
        -0.,  0., -1., -0., -0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0.,
         1., -0.,  1.,  0., -0.,  0.,  0.],
       [-0., -1., -1.,  0.,  0., -1.,  0.,  0., -0., -0., -0., -1.,  0.,
        -0., -0., -0., -0., -1., -0., -0.,  0.,  0.,  1., -0., -0.,  0.,
         0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0., -1.,  0.,  0.,
        -0., -1.,  0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,
         0.,  0., -0., -0., -0.,  0., -0.,  1., -0., -0.,  1., -0.,  0.,
        -1.,  1., -0.,  0., -1.,  0.,  1.,  0.,  0.,  0.,  0., -0., -0.,
        -1., -0., -0.,  0.,  0., -0., -0.],
       [-0., -0., -0.,  0., -0., -0.,  0., -0., -0., -1.,  0., -0.,  0.,
        -0., -0., -0., -1.,  1.,  0., -0.,  1.,  1., -1.,  0., -0.,  0.,
         1.,  0.,  1., -0.,  0., -0., -0.,  0., -0.,  0., -0.,  1.,  0.,
         0., -0., -0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0., -0.,
        -0.,  0.,  0.,  0.,  0.,  0.,  0., -1.,  1., -0.,  0., -0.,  0.,
        -0.,  0., -0., -0.,  1.,  0., -1.,  0.,  0.,  0.,  0.,  0., -0.,
        -0.,  0., -1., -0., -0., -0.,  0.],
       [-0., -0., -0.,  0.,  0.,  0.,  1.,  1., -0.,  0., -0., -0., -0.,
        -0.,  0., -0.,  0., -0., -0., -0., -0., -0., -0., -0., -0.,  0.,
        -0.,  1., -0.,  0., -1.,  0.,  0.,  0., -0.,  0., -0., -0.,  1.,
         1.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  1., -0.,
        -0.,  1., -0.,  0.,  0.,  0., -1.,  0., -0., -1., -0.,  0.,  0.,
         0., -0.,  1.,  0.,  0.,  1.,  0.,  0.,  0.,  0., -1.,  0.,  0.,
         0., -1.,  0.,  0., -0., -0., -0.],
       [ 0.,  1., -0.,  1., -0.,  0., -0.,  0., -0., -0., -0.,  0., -0.,
         1., -0.,  1.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  1.,  0.,
        -0., -0.,  0., -0., -0., -0.,  0., -0., -0.,  0., -0.,  0.,  1.,
         0., -0., -0.,  0.,  0., -0.,  0., -1., -0.,  0.,  0., -0.,  0.,
         0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0., -1.,
        -0.,  0., -0.,  0., -0.,  0.,  0., -0., -0.,  1.,  0., -0., -0.,
        -0., -0.,  0., -0.,  0.,  1., -1.],
       [ 0., -1., -1.,  0., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0.,
         0., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,
        -0., -0., -0., -0., -1., -0.,  0., -1., -0.,  1., -1.,  0., -0.,
         0.,  0., -0., -0.,  1., -1., -0., -0., -1., -0.,  0., -0.,  0.,
         0., -0.,  0.,  1.,  0., -0.,  0.,  0., -0., -0., -0., -1., -0.,
         0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0., -1.,
         0., -0., -0., -0.,  1., -1., -0.],
       [ 1., -0.,  1.,  0., -0., -0., -0.,  1.,  0., -0.,  0., -0., -0.,
         0., -1.,  0., -0.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,  1.,
         0., -0.,  0.,  0.,  0., -1.,  1., -0., -0.,  0.,  1.,  0., -0.,
         1.,  0., -1., -0., -1., -0., -0., -0., -0., -0., -0., -0.,  0.,
         0., -0., -0.,  0., -0., -0.,  0., -0.,  0., -1., -0., -0., -0.,
        -0.,  0., -0.,  1.,  0.,  1., -0., -0., -0.,  0.,  0., -0., -0.,
        -0., -1., -0., -1.,  0., -0., -0.]], dtype=np.float32), np.array([[ 0., -0., -0., -1., -0.,  0.,  0., -0., -0.,  1., -0.,  0., -0.,
         0.,  0., -0., -0., -0.,  0.,  1., -0.,  0.,  0.,  0., -1., -0.,
        -1.,  0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0., -1.,  1.,
         0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,
         0., -1.,  0.,  0., -0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,
        -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  1.,  1.,  0.,  0., -0.,
         0.,  0., -0.,  0.,  0., -0., -1.],
       [-0.,  0., -0., -0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,  0.,
         0., -0.,  0., -0.,  0., -0.,  0., -0.,  1., -0., -0., -0.,  0.,
        -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  1.,  0., -0., -0.,  0.,
         0., -0., -0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0., -0.,  1.,
         0.,  1., -0., -0.,  0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,
        -1.,  1., -0.,  0.,  1., -0., -0., -1.,  0.,  0.,  0., -0.,  0.,
         0.,  0.,  0., -0., -0.,  0., -0.],
       [-0., -0., -0., -0., -1.,  0.,  1.,  0., -0., -1.,  0., -0., -0.,
        -0., -0.,  0., -0.,  0., -0.,  0.,  0.,  1., -0., -0., -0.,  0.,
         1., -0.,  0.,  0.,  0., -0.,  0., -0.,  1., -0.,  0.,  1., -0.,
        -0., -0., -0., -0., -0.,  0.,  0., -0., -0., -0.,  0.,  0., -0.,
        -0., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,
        -1.,  1.,  0.,  0.,  1.,  0., -0., -1.,  0.,  0., -0., -0., -0.,
         0., -0., -0., -0., -0., -0., -0.],
       [ 0.,  0.,  0., -0., -0., -0., -0., -0.,  0., -0.,  1., -0.,  0.,
         0., -0.,  0.,  0., -0., -0., -0., -0., -0., -0.,  0., -0., -0.,
         0.,  1.,  0., -0., -0.,  0., -0., -0., -0., -0., -0.,  0.,  0.,
        -0.,  0.,  0., -1., -0.,  0.,  0.,  0., -0.,  0.,  0., -1., -0.,
        -0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0.,
         0., -0.,  1., -0., -0., -0.,  0.,  0., -1.,  0., -0.,  0., -0.,
        -0.,  0., -0., -0.,  0.,  0., -0.],
       [ 0., -0., -0., -0.,  0., -0.,  0., -0., -0., -1., -1.,  0., -0.,
        -0., -0.,  0.,  0., -0.,  0., -1., -0., -0.,  0.,  0., -0.,  0.,
         1.,  0., -0., -0., -0., -0., -0., -0., -0., -0., -0.,  1.,  0.,
        -0., -0.,  0.,  1.,  0.,  0., -0., -0., -0., -0.,  0., -0., -0.,
         0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,  0., -0.,
         0., -0., -1., -0.,  0., -0.,  0.,  0.,  0.,  0.,  1.,  0., -0.,
         0.,  0.,  0.,  0., -0., -0., -0.],
       [-0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  1.,  0.,  0.,
        -0.,  0., -0., -0., -0., -0., -0.,  0., -0., -0.,  0.,  0., -0.,
         0.,  0., -0., -0., -1., -0., -0., -1., -0., -1.,  0.,  0., -0.,
        -0.,  0., -0., -1., -0.,  0.,  0.,  0., -1.,  0.,  0., -0., -0.,
         0.,  1., -0.,  1., -0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,
         0., -0.,  1.,  0., -0.,  0., -0., -0.,  1., -0., -1.,  0., -0.,
         0.,  0., -0., -0., -0., -0.,  0.],
       [-1., -0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0., -0., -0.,  0.,
        -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  1.,
        -0.,  1., -0., -0.,  0.,  0.,  1.,  0.,  0.,  0., -0., -0., -0.,
        -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0., -0., -0., -1.,  0.,
         0., -1., -0., -0., -0., -0.,  0., -0., -0.,  0., -0., -0., -0.,
         0., -0., -0.,  1., -0.,  0., -0.,  0.,  0., -0.,  1.,  0.,  0.,
         0., -0.,  0., -0.,  0., -0.,  0.],
       [-0., -0., -0.,  1., -0., -0., -0., -0., -1.,  0.,  0., -0., -0.,
        -0.,  0., -1., -1., -0., -0., -1.,  0., -0., -0., -0.,  1., -0.,
         1., -0., -0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,  1.,  0.,
        -0., -0.,  0., -0., -0.,  0.,  0., -0., -0.,  0., -0.,  0., -0.,
        -0., -0., -0.,  0.,  0., -0.,  0., -0.,  1.,  0.,  0., -0.,  0.,
         0., -0., -0.,  0., -0.,  0., -0.,  0., -1., -1., -0., -0., -0.,
        -0., -0., -0.,  0., -0., -0., -0.],
       [-0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0.,
         0., -0.,  1., -0., -0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,
        -0., -1., -0.,  0., -0., -0., -0.,  0.,  0.,  0., -0., -0., -1.,
        -0.,  0.,  0., -0., -0.,  1., -0.,  0.,  0.,  0.,  0.,  0.,  0.,
         1., -1.,  0.,  0.,  0., -0., -0.,  1., -0., -0., -0.,  0.,  0.,
         1., -0.,  0., -0.,  0.,  0.,  1.],
       [-0., -1., -1.,  0., -0., -0., -1.,  0., -0., -0.,  0.,  0., -1.,
        -0., -0., -0., -0., -0., -0., -0., -0.,  0., -0.,  0.,  0.,  0.,
         0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0., -0.,  1.,  0., -1.,
         0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,
         0.,  0., -0., -0., -0., -0., -0., -0.,  0., -1.,  0., -0., -0.,
        -0.,  0.,  0.,  0.,  0.,  1.,  0., -0., -0., -0., -0., -0., -0.,
        -0.,  0.,  0., -0., -0., -0., -0.],
       [-0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  1., -0.,  0.,  0.,  0.,
         0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,
        -0., -1.,  0., -0., -0., -0.,  0., -0., -0., -0., -0., -0., -0.,
         0.,  0.,  0.,  1.,  0., -0.,  0., -0., -0., -0.,  0., -0.,  0.,
         0.,  0.,  0., -0., -0.,  0.,  1.,  0., -0.,  0., -0., -0.,  1.,
        -0.,  0., -1., -0.,  0., -0., -0., -0.,  1.,  1., -0., -0., -0.,
         0.,  0.,  0.,  0., -0., -0.,  0.],
       [-0., -0., -0., -1., -0.,  0., -0.,  0., -0., -0., -0.,  0., -0.,
        -0.,  0., -0.,  1.,  0., -0.,  1.,  0., -0., -0.,  0.,  0., -0.,
        -1., -0.,  0., -0., -0., -0.,  0.,  0., -0., -0.,  0., -1., -0.,
         0.,  0.,  0., -1.,  0.,  0.,  0., -1.,  0., -0.,  0., -0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -1., -0., -0., -0., -0.,
        -0.,  0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,  0., -0.,
         0.,  0., -0.,  0., -0., -0.,  0.],
       [ 0., -0., -0.,  0., -0., -0., -0., -0., -1.,  0.,  0.,  0., -0.,
        -1., -0., -0., -0., -0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,
        -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,
         0.,  0., -0.,  1.,  0.,  1.,  0., -0., -0., -0., -0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0., -1., -1.,
        -0.,  0.,  0.,  0., -0.,  0.,  0., -0., -1., -1., -0., -0., -1.,
        -0., -0.,  0., -0., -1., -1., -0.],
       [-0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,
        -1., -1.,  0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,
         0., -1., -0., -0.,  0.,  1.,  0., -0., -0.,  0., -0.,  0.,  0.,
        -0., -0., -1., -0., -0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,
         0., -0., -0., -0., -0.,  0.,  1.,  0.,  0.,  0.,  0.,  0., -0.,
         0., -0., -1.,  0., -0., -0.,  0.,  0., -0., -0.,  0.,  0., -0.,
         0.,  0., -0., -1.,  0.,  0.,  0.],
       [-0.,  0., -1.,  1.,  0., -0., -0.,  0., -1.,  0.,  0., -0.,  0.,
        -0.,  0.,  0., -1., -0., -0., -1., -0., -0., -0., -0.,  1.,  0.,
         1., -0., -0., -1., -1., -0., -0.,  0., -0.,  0.,  0.,  1., -1.,
         1.,  0.,  0., -0.,  1.,  0., -0., -0.,  1., -1., -0.,  0., -0.,
        -0.,  0., -0., -0.,  0.,  1.,  0.,  0.,  1., -1., -1.,  0.,  0.,
         0., -0., -0.,  0., -0.,  0.,  0.,  0., -1., -1., -0.,  1., -0.,
         1., -0., -0.,  0.,  1.,  0.,  1.],
       [ 0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0., -0.,
         0., -0., -0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,
         0., -0.,  0.,  1., -0.,  0.,  0., -0.,  0., -0., -0.,  0., -0.,
         0.,  0., -0.,  0.,  0., -0.,  0., -0., -1., -0., -0.,  0., -1.,
         0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,  0.,
         1., -1., -0., -0.,  0., -0., -0.,  1., -0., -0.,  0., -1.,  0.,
        -0.,  0.,  0., -0., -1.,  0., -0.],
       [ 0.,  0.,  0.,  0., -0.,  0., -1., -0., -0., -0., -0.,  0.,  0.,
         0., -0.,  0., -0.,  0., -0., -0., -0.,  0., -0., -0.,  0.,  0.,
         0.,  0.,  0., -0.,  1.,  0., -0.,  0.,  0., -0.,  1.,  0.,  0.,
        -1., -0., -0.,  0., -1., -0.,  0., -0.,  0., -0., -0.,  0., -0.,
         0., -0., -0., -0., -0.,  0., -0., -0.,  0.,  0., -0., -0.,  0.,
        -0., -1.,  0., -0.,  0.,  1., -0., -0., -0., -0.,  0.,  0.,  0.,
         0.,  0.,  0., -0.,  0.,  0.,  0.],
       [-0., -0., -0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0., -1.,  0.,
         0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,
        -0., -1., -0., -0., -0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,
         0., -0.,  0.,  1.,  0.,  1., -0., -0.,  0., -0., -0., -0.,  0.,
        -1., -0., -0.,  1.,  1.,  0.,  1.,  0.,  0., -0., -0.,  0., -0.,
        -0.,  0., -1., -0.,  0., -0.,  0., -0.,  1., -0.,  0.,  0., -0.,
         0.,  0., -0.,  0., -0.,  0.,  0.],
       [-0.,  0., -0.,  0.,  0., -1., -0.,  0., -0., -0., -0., -0.,  0.,
        -0.,  0.,  0., -0., -1., -0.,  1.,  0.,  0.,  0., -0.,  0., -0.,
        -1., -0., -1., -0., -0., -0.,  0.,  0., -0.,  1.,  0., -1., -0.,
         0., -0.,  0., -1.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0.,
        -0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0., -0.,  1.,  0.,
        -0.,  0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,  0., -0.,
         0., -0., -0.,  0., -0.,  0.,  0.],
       [ 0.,  0., -0.,  0., -0.,  1., -0., -0.,  0., -0.,  0., -0.,  0.,
         0., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,
        -0., -0.,  0., -0., -0.,  0.,  0.,  1., -0., -0.,  0., -0., -0.,
         0.,  0., -0.,  1., -0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,
        -0.,  0.,  0., -1., -1.,  0.,  0., -0.,  0., -0., -0., -1.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0., -0., -0., -1.,  0., -0., -0.,  0.,
         0., -0.,  0.,  0., -1.,  0.,  0.],
       [-0.,  0.,  0., -0., -0., -0., -0.,  0., -0., -0.,  0., -1.,  0.,
         0., -1.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,
         0., -1.,  0.,  0.,  0.,  1.,  0.,  1., -0., -0., -0.,  0.,  0.,
        -0.,  0., -1., -0., -0., -0.,  1.,  0., -0.,  0., -0., -0., -0.,
         0., -0., -0., -0., -0., -0.,  1., -1., -0.,  0.,  0.,  0., -0.,
         0., -0., -1.,  0.,  0.,  0., -1.,  0., -0., -0.,  0.,  0.,  1.,
        -0.,  0.,  1., -1.,  0., -0., -0.],
       [-0.,  0.,  0.,  1.,  0.,  0., -0.,  0., -1.,  0.,  0., -0.,  0.,
        -0., -0.,  0., -1., -0., -0., -1., -0., -0., -0.,  0.,  1.,  0.,
         1., -0., -0.,  0.,  0.,  1.,  1.,  0., -0.,  0., -0.,  1., -1.,
        -0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,  0.,  1.,  0., -0.,
        -0.,  0.,  1., -0., -0.,  1.,  0., -0., -0., -1.,  0.,  0.,  0.,
         0., -0., -0.,  0., -0.,  0.,  0.,  0., -1., -1., -0., -0.,  0.,
        -0.,  1.,  0., -0., -0.,  0.,  1.],
       [ 0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0., -0., -0., -0.,
        -0., -0., -0., -0., -0.,  0., -0.,  0.,  0., -0., -0.,  0., -0.,
         0., -0.,  0., -0., -0., -1., -1., -0.,  0., -0.,  0.,  0., -0.,
         0.,  0., -0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0., -0., -1.,
        -0.,  0., -1.,  0.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,
         1., -1., -0., -0., -1., -0., -0.,  1., -0., -0.,  0.,  0., -0.,
        -0.,  0.,  0., -0.,  0., -0., -0.],
       [ 0.,  0.,  0.,  0., -0., -0., -1., -0., -0., -0.,  0.,  0.,  0.,
         0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,
        -1.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,
         0., -0., -0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,
        -0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,
         1., -1.,  0., -0., -1.,  1., -0., -0., -0., -0., -0.,  0.,  0.,
        -0., -1., -0.,  0., -0.,  0.,  0.],
       [-0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0., -1.,  0.,
        -0.,  0., -0., -0.,  0., -1.,  0., -0.,  0., -1.,  0.,  0.,  0.,
        -0., -1., -0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0., -0.,
        -0., -0.,  0.,  1.,  0., -0., -0., -0.,  0.,  0.,  0., -0.,  0.,
         0.,  0., -0., -0., -0.,  0.,  1., -1.,  0., -0., -0., -0., -0.,
        -0.,  0., -1.,  1.,  0., -0.,  0., -0.,  1., -0., -0.,  0., -0.,
         0.,  0., -0., -1., -0., -0.,  0.],
       [-0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0., -0.,  0., -0.,  0.,
         0., -1.,  0., -0.,  0., -0.,  1., -1., -0., -0.,  0.,  0.,  1.,
        -1.,  0., -0.,  0.,  0., -0., -0.,  0., -0.,  0., -0., -1., -0.,
        -0.,  0.,  0., -1., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,
        -0.,  0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,
         0., -0., -0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,  0.,
        -0.,  0.,  1.,  0.,  0.,  0.,  0.],
       [ 0.,  0., -0.,  0., -0., -0., -0.,  0.,  0., -0.,  0., -0., -0.,
         0.,  1.,  0., -0.,  0., -0.,  0.,  1.,  0., -0.,  1.,  0., -1.,
        -0., -0., -1.,  0.,  1., -1., -1.,  1., -0.,  1., -0., -0.,  0.,
        -1., -0., -0.,  1.,  0., -0., -0.,  0.,  1., -0.,  0.,  0.,  0.,
        -0.,  0., -1., -1., -1.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,
         0., -0., -0.,  0.,  0.,  0., -0., -0., -1.,  0.,  0.,  1.,  0.,
        -0.,  1., -1., -0.,  0.,  0.,  0.],
       [ 1.,  0.,  0., -0.,  0., -0., -0.,  0.,  0., -0.,  0., -1.,  0.,
         0., -1.,  0.,  0., -0.,  0., -0.,  0.,  0., -1.,  0., -0.,  0.,
         0., -1., -0., -0.,  0.,  1.,  0., -0., -0.,  0., -0.,  0.,  0.,
        -0., -0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0., -0.,
        -0., -0., -0.,  0.,  0., -0.,  1., -1., -0.,  0., -0.,  0.,  0.,
         0., -0., -1., -0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,
         0., -0.,  1., -1., -0., -0., -0.]], dtype=np.float32), np.array([[-0., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0., -0.,
        -0., -0., -1., -0., -0., -0.,  0.,  0., -0.,  0.,  0., -1.,  0.,
        -0., -0.,  0.,  0.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,  0.,
        -0.,  1., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  1., -0., -0.,
         0., -0.,  0.,  0.,  0., -1.,  0., -0.,  0.,  0., -1.,  0.,  0.,
         0., -0.,  0.,  0.,  0., -0., -0., -0., -0.,  0., -0., -0.,  0.,
         1.,  0., -0., -0.,  0.,  0., -1.],
       [-0., -0.,  0., -0.,  0.,  0.,  0., -0., -1.,  0., -0.,  0.,  0.,
        -0., -0.,  0., -0., -0.,  0., -0.,  0.,  0., -0., -0., -1., -0.,
         0., -0.,  0., -0., -0., -0., -0., -0., -0., -0., -0., -0.,  0.,
        -0., -0., -0.,  0.,  0.,  0.,  0., -0., -0., -1.,  1., -0., -0.,
         0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -1.,
        -0.,  0.,  0., -0., -0., -0., -0., -0., -0., -1.,  0., -0.,  0.,
        -0.,  0.,  0.,  0., -0., -0.,  0.],
       [-0., -0.,  0.,  1., -0.,  0., -0., -0.,  0., -0., -0.,  0., -0.,
        -0., -0.,  0.,  1., -0.,  0.,  0.,  0., -0., -0., -0., -1.,  0.,
        -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,
        -0., -0., -0., -0., -0., -0., -0.,  1.,  0.,  0.,  1.,  0., -0.,
        -0.,  0.,  0., -0., -0.,  0., -0.,  0., -1.,  0., -1.,  0., -0.,
         0.,  0.,  0., -0.,  0., -0., -0., -0.,  0.,  0., -0., -0., -0.,
        -0.,  0., -0.,  0.,  0.,  0.,  0.],
       [ 0., -1.,  1.,  0.,  0.,  0., -0., -0., -0.,  0., -0.,  0., -0.,
        -0., -0., -0.,  0.,  0., -0., -0., -0., -0., -0.,  0.,  1., -0.,
        -0., -0., -0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,  1.,
         0.,  0., -0., -0.,  0., -0., -0.,  0., -0.,  0., -1.,  0.,  0.,
        -0., -0.,  0., -0., -0., -0.,  0.,  0., -0.,  1.,  1.,  0., -0.,
        -0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,
         0., -0.,  0.,  0.,  0.,  0.,  0.],
       [ 0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0., -0.,  0.,
        -0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0., -1.,  0., -0.,
        -0.,  0.,  0., -1., -0.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,
         0.,  1.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,  1.,  0.,  1.,
         0.,  0., -1., -0., -0., -1., -0., -0., -0.,  0., -1., -0., -0.,
         0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  1., -0.,
         1., -0., -0.,  0., -0., -0.,  0.],
       [ 0., -0.,  0.,  0., -1.,  0.,  0.,  0.,  0., -0.,  0., -0., -0.,
        -0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,  0., -1.,  0., -0.,
        -0.,  0., -0.,  0., -0.,  0., -0.,  0.,  1., -0.,  0.,  0.,  0.,
         0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0., -1.,  1.,  0., -0.,
        -0.,  0., -1., -0.,  0., -1., -0., -0., -0.,  0.,  0., -0.,  0.,
         0., -0., -0.,  0.,  0., -0.,  0.,  1.,  0.,  0.,  0.,  1., -0.,
        -0.,  0., -0., -0., -0., -0., -0.],
       [ 0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0., -0., -0.,
        -0.,  0.,  0., -0.,  0., -0., -1.,  0.,  1.,  0., -1., -0.,  0.,
         1.,  0., -0., -1., -0.,  0., -0.,  0., -0.,  0.,  0., -1.,  0.,
         0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  1., -0., -0.,
         0.,  0., -1.,  0., -0., -1.,  0., -0.,  0.,  0., -1.,  0.,  0.,
        -1., -0., -0.,  0., -1., -0.,  0.,  1., -0.,  0.,  0.,  1., -0.,
         1.,  0.,  0., -0., -0., -0., -0.],
       [-0., -1.,  1., -0., -0.,  0.,  0.,  1.,  0., -0.,  0., -0., -0.,
         0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,  1., -0.,  0.,
         0.,  0., -0., -0., -1.,  0., -1., -0.,  0.,  0.,  0., -0.,  1.,
         1.,  0.,  0.,  0.,  1., -0.,  0.,  0.,  1.,  0., -1., -0., -1.,
        -0.,  1., -0., -0.,  0., -0., -0.,  0., -0.,  1.,  1., -0., -0.,
        -0., -0., -0., -0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,
         0.,  0., -0.,  0., -0., -0., -1.],
       [-0., -0., -1., -0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  1.,
         0., -0.,  0., -0., -0., -0., -0., -0., -0., -0.,  1.,  0.,  0.,
         0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0., -1., -0., -0.,
         1., -0., -0., -0.,  1., -0.,  0., -0.,  0., -0.,  1.,  0., -1.,
         0., -0., -0., -0., -0.,  0., -0.,  0.,  0., -1., -1.,  0., -0.,
        -1.,  1.,  0., -0.,  0.,  1., -0.,  1.,  0., -0., -0.,  0.,  0.,
        -0., -1., -0.,  0.,  0.,  0.,  0.],
       [-0., -0.,  0.,  0.,  1.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,
         0., -0., -0.,  0., -0., -0., -0.,  0., -0., -0.,  1.,  0.,  0.,
         0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,
         1.,  0., -0., -0., -0.,  0.,  0., -0., -0., -1.,  1.,  0.,  0.,
         0.,  0., -0., -0., -0., -0., -0.,  0.,  0., -1.,  0., -0.,  0.,
         0., -0.,  0., -0.,  0.,  1., -0., -0.,  0.,  0., -0.,  0.,  0.,
         0., -1., -0.,  0.,  0., -0., -0.],
       [-0., -0., -1., -0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0.,
         0., -0., -0., -0., -0.,  0.,  1., -0.,  0., -0.,  1.,  0.,  0.,
        -1.,  0.,  0., -0.,  0.,  0.,  0., -0., -0., -0., -1.,  1.,  0.,
         1.,  0.,  0., -0.,  1., -0.,  0., -0., -0.,  0.,  1.,  0.,  0.,
        -0., -0., -0., -0., -0., -0., -0.,  0.,  0., -1., -1., -0., -0.,
         0., -0.,  0., -0., -0.,  1.,  0., -0.,  0., -0., -0.,  0.,  0.,
         0., -1.,  0.,  0., -0.,  0., -0.],
       [-0., -1.,  1.,  0.,  0.,  0.,  1.,  0.,  0.,  0., -0.,  0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0., -1., -0., -0.,
        -0.,  0., -0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,  0., -0.,
        -1.,  0.,  0.,  0., -1., -0., -0., -0.,  0.,  0., -1., -0., -0.,
        -0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  1.,  1., -0.,  0.,
        -0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,
        -0.,  1., -0.,  0., -0.,  0., -0.],
       [-0.,  0., -0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0., -1., -0.,
         0.,  1., -0., -0., -0.,  1.,  1.,  1., -0.,  0., -0.,  0.,  0.,
         0., -0., -0., -0.,  0., -1., -1., -0.,  0., -0., -0.,  1.,  0.,
         0., -0.,  1., -1., -0.,  0., -1., -0., -0., -0.,  0.,  0., -1.,
        -0.,  0., -0., -1., -0.,  0.,  0.,  0.,  1., -1.,  0.,  0., -1.,
        -0., -0., -0.,  1.,  0., -0.,  0.],
       [-0.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0., -0., -0.,
        -0.,  0., -0.,  0.,  0., -1., -0.,  0., -0., -0., -0., -1., -0.,
        -0., -0.,  0.,  0., -0., -0., -0.,  1.,  0., -0.,  0., -0., -0.,
         0.,  0., -0., -0.,  0., -1., -1., -0., -0.,  0., -0., -0., -0.,
        -1.,  0., -0., -1., -0., -0.,  0., -0., -0., -0., -0.,  0., -1.,
         0., -0., -0.,  0.,  0.,  0., -0.,  0.,  1., -1.,  0.,  0., -1.,
        -0., -0., -0., -0., -0., -0., -0.],
       [-0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -1.,  1.,  0.,
        -0., -1., -0., -0., -0.,  0.,  0., -1.,  0.,  0., -0., -1.,  1.,
         0., -0.,  0.,  0.,  0., -0., -0.,  1.,  0., -0., -0.,  0., -0.,
         0.,  0., -0.,  0.,  0., -1., -1., -0., -0.,  0., -0.,  1., -0.,
        -1.,  0., -0., -1., -0., -0., -1.,  1.,  0., -0.,  0., -0., -1.,
        -0., -0.,  1., -1., -0.,  0., -0.,  0.,  1., -1., -1., -0., -1.,
         0., -0.,  1.,  1., -0., -0., -0.],
       [ 0.,  0., -0.,  0.,  0.,  0.,  0., -1., -0.,  0., -0.,  0., -0.,
        -0., -0.,  0., -0., -0., -0., -0., -0.,  0., -0.,  0.,  1.,  0.,
         0.,  0.,  0., -0., -0.,  0.,  0., -1.,  0., -0.,  0., -0., -0.,
         0., -0., -0., -0.,  0., -0.,  1., -0.,  0., -0.,  0., -1., -0.,
        -0.,  0.,  0.,  1.,  0., -0., -0.,  0.,  0., -0., -0., -0., -0.,
         0., -0.,  0.,  1., -0., -0., -0.,  0., -1.,  1., -0.,  0.,  1.,
        -0.,  0.,  0.,  0.,  0., -1., -0.],
       [ 0., -0., -0.,  1.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,
        -0.,  1.,  0.,  0., -0., -0., -1.,  0., -0., -0.,  1., -1., -1.,
        -0.,  1.,  0.,  1.,  0.,  1.,  1., -0.,  0., -1., -0., -0.,  0.,
         0., -0., -0.,  1.,  0.,  0.,  0.,  1., -1., -0., -0., -0.,  0.,
        -0., -0.,  1.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -1.,  0.,
        -0.,  0., -1.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  1., -1.,  0.,
        -0., -0.,  0., -0., -1.,  0.,  0.],
       [-0.,  0.,  0., -0., -1.,  0., -0., -0., -1., -1.,  0., -0., -0.,
        -0., -0., -0., -1., -0.,  0.,  0., -1.,  0., -0.,  1., -1.,  0.,
         0., -0., -1., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  1., -0.,
         0.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,
         0., -0., -0.,  0., -0., -0.,  0.,  0., -0., -0., -0.,  0., -1.,
        -0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0., -1., -0.,  0.,  0.,
         0., -0., -0., -0.,  0., -0., -0.],
       [-0.,  0., -0.,  1.,  0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,
        -0.,  0., -0., -0.,  1.,  0., -1., -1.,  0., -0.,  1., -1.,  0.,
        -0., -0., -1., -0.,  0., -0., -0., -0.,  0.,  0., -0., -0., -0.,
         0.,  0., -0., -0.,  0.,  0.,  0.,  1.,  0., -0., -0., -0.,  0.,
         0., -0., -0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,
        -0., -0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,
         0., -0., -0., -0.,  0.,  0., -0.],
       [ 0.,  0., -0., -1., -0.,  0.,  0., -1., -0., -0.,  0., -0., -0.,
        -0.,  0.,  0., -0.,  0., -0.,  1., -0.,  0.,  0., -1.,  1.,  1.,
         0.,  0., -0.,  0.,  1.,  0., -0.,  0., -0.,  1.,  0.,  0., -0.,
        -1., -0., -0.,  0., -1.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,
         0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,
        -0., -0., -0., -0., -0., -0.,  0.,  0.,  0., -0., -1., -0., -0.,
        -0.,  0., -0., -0., -0.,  0., -0.],
       [ 0., -0.,  0., -0., -0.,  0., -0.,  0., -0., -0.,  0., -0., -0.,
         0., -0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,  1.,  0.,  0.,
         0., -0.,  0.,  1.,  0., -0., -0.,  1.,  0., -0.,  0.,  0.,  0.,
        -0.,  0., -0., -0., -0.,  0., -1., -0., -1., -0.,  0., -0.,  0.,
         0.,  0., -0.,  0., -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,
         0., -0.,  0.,  0., -0., -0.,  0., -0., -0.,  0., -0., -1., -1.,
         0.,  0., -0., -0., -1., -0.,  0.],
       [-0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,
         0.,  0., -0.,  0., -0.,  0., -0.,  0., -0., -0.,  1.,  0., -0.,
        -0., -0.,  0., -0.,  0., -0., -0.,  1.,  0., -0.,  0., -0., -0.,
        -0., -0., -0.,  0., -0., -1., -1.,  0.,  0.,  0., -0., -0.,  0.,
        -1., -0., -0., -1.,  1., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,
         0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0., -1.,
        -0.,  0., -0., -0.,  0., -0., -0.],
       [-0.,  0., -0., -0.,  0., -1.,  0.,  0., -0.,  0.,  0., -0.,  0.,
         0.,  0., -0., -0.,  1.,  0.,  0.,  0.,  0.,  0.,  1.,  0., -0.,
         0.,  0., -1., -0.,  0.,  0.,  0.,  1., -0.,  1.,  0., -0., -0.,
        -0.,  0., -0.,  0., -0.,  0., -1., -0.,  0., -0., -0., -0., -0.,
         0., -0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,  1.,  0.,
        -0., -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0., -1.,
         0.,  0., -0., -0.,  0., -0., -0.],
       [-0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0.,
        -0., -0., -0., -0.,  0., -0., -0.,  0.,  0.,  0., -1., -0., -0.,
        -0., -0., -0.,  0.,  1.,  0., -0., -1.,  0., -0., -0., -0.,  0.,
        -1.,  0., -0., -0., -1., -0.,  1., -0., -0., -0.,  0.,  0., -0.,
        -0.,  0., -0., -0.,  0., -0., -0.,  0.,  0., -0., -0.,  0.,  0.,
        -0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  1.,
         0., -0.,  0.,  0., -0., -1., -0.],
       [-1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,
        -0., -0., -0., -0., -0., -0., -0., -0., -0.,  0.,  0., -0.,  0.,
        -0.,  0.,  0.,  0.,  0., -1., -1.,  0., -0., -0.,  0., -0., -0.,
        -0.,  0., -1.,  0., -0., -0.,  1.,  0., -0.,  0.,  0.,  0., -0.,
         0., -0., -1., -0., -0., -0., -0., -0.,  0., -0., -0., -0., -0.,
         0., -0., -0., -0.,  0., -0.,  0., -0.,  0., -0.,  0., -0., -0.,
         0.,  0.,  0.,  0., -0.,  0., -0.],
       [-1., -0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,
        -0., -0.,  0., -0., -0.,  1.,  0., -0.,  0., -1.,  0., -0.,  0.,
         0., -0., -0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,
        -0.,  0., -1.,  0., -0., -0., -0., -0.,  0.,  0.,  0., -0.,  0.,
        -0.,  0.,  0., -0., -0.,  0.,  0.,  1.,  0.,  0.,  0., -0., -0.,
        -0.,  0., -0., -1., -0., -0., -1., -0.,  0., -0.,  0., -0., -0.,
         0.,  0.,  0.,  1.,  0.,  0.,  0.],
       [-1., -0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,
        -0.,  1.,  0.,  0.,  0., -0., -0.,  1., -0.,  0.,  0., -0., -1.,
        -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
        -0.,  0., -1., -0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,
        -0.,  0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,  0., -0., -0.,
         0.,  0.,  0., -0.,  0., -0., -1., -0.,  0., -0., -0., -0., -0.,
        -0.,  0., -1.,  0.,  0.,  0., -0.],
       [ 1.,  0., -0., -0., -0.,  0., -0.,  1., -0., -0., -0.,  0.,  0.,
        -1.,  0., -0.,  0., -0.,  0.,  0.,  0., -0., -0., -0.,  0., -0.,
        -0.,  0.,  0.,  0.,  0., -0., -0., -0., -0.,  0., -0.,  0.,  0.,
        -0., -0., -0.,  0., -0., -0., -1.,  0., -0., -0.,  0., -0.,  0.,
         0., -0., -0.,  0., -0.,  0., -0.,  0., -0., -0., -0., -0., -0.,
         0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -0., -1.,
        -0.,  0., -0.,  0., -0.,  1.,  0.]], dtype=np.float32))

In [ ]:
#@title Verification
n = 4
m = 4
p = 7
rank = 85

verify_tensor_decomposition(decomposition_447, n, m, p, rank)

## Rank-90 decomposition of <4,5,6> over Z

In [ ]:
#@title Data

decomposition_456 = (np.array([[ 0., -0.,  0.,  0.,  0.,  0., -1., -0.,  1.,  0.,  0., -0.,  0.,
        -0.,  0., -0., -0.,  0., -0., -1., -1.,  0., -0.,  0., -0., -1.,
        -0.,  0.,  1., -1., -0.,  0., -0.,  1., -0.,  1., -0.,  0.,  0.,
        -0.,  0.,  0., -1.,  1.,  0., -1., -0., -0.,  0.,  0., -0., -0.,
        -1.,  0., -0.,  1.,  0., -0.,  0., -0.,  0.,  1., -0., -0.,  1.,
        -0., -0.,  0., -0.,  0., -1.,  1., -0.,  0.,  1., -0., -0., -0.,
         0., -0.,  0.,  0.,  1.,  0.,  1., -0.,  0.,  0., -0.,  0.],
       [-0.,  0., -0., -0.,  0.,  1., -0., -0.,  0., -1.,  1.,  0., -1.,
         0.,  0.,  0.,  1., -1.,  1., -0.,  0., -1.,  0., -0., -0.,  0.,
         0., -0., -0., -0., -0., -0.,  0., -0., -0., -0., -1.,  0.,  0.,
         1.,  0.,  0., -1., -0.,  0.,  0.,  0., -0., -0.,  1.,  1.,  0.,
         0.,  0., -0., -0., -1., -0.,  0.,  0., -0.,  0.,  1.,  1.,  0.,
         0.,  1.,  0., -0., -1., -0., -0.,  0.,  0., -0.,  1.,  0., -1.,
         1., -0., -0.,  1., -0., -1., -0., -0.,  0., -0.,  1.,  1.],
       [ 0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0., -0.,  0.,  1.,
         0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0., -1., -0.,  0.,
        -0.,  1., -0., -0., -0., -0., -0., -0., -0.,  0.,  0.,  0.,  1.,
         0., -0., -1.,  0., -0., -0.,  0., -0.,  1.,  0., -0., -1., -0.,
        -0., -0., -0., -0., -0., -1.,  0.,  0., -1., -0.,  0., -0.,  0.,
        -0., -0.,  0., -0., -0., -0., -0., -0.,  0., -0., -0., -0., -0.,
         0., -0., -0.,  0., -0., -0., -0.,  0., -0., -0., -1., -0.],
       [ 0.,  0., -0.,  0., -0.,  0., -0., -0.,  1.,  0., -1., -0.,  0.,
        -1., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,
         0., -0.,  0., -1.,  0.,  0., -0.,  1.,  0.,  0.,  0., -1.,  0.,
         0.,  0.,  1., -1.,  1., -0., -1., -0.,  0.,  0., -0.,  0., -0.,
        -1., -0.,  0., -0.,  0.,  0., -0.,  0.,  1., -0., -0., -0.,  1.,
         0., -1., -0.,  0.,  0., -1., -0., -1.,  0., -0., -0., -0.,  0.,
         0., -0.,  0., -0.,  0., -0.,  1., -0., -0., -1., -0.,  0.],
       [-0.,  0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,
         1.,  0.,  0., -1.,  0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,
        -0., -1., -0.,  1.,  1.,  0.,  0.,  0., -0., -0., -0.,  1., -0.,
        -0.,  1., -0., -0., -1., -0.,  1., -0., -1.,  0.,  0.,  0., -0.,
         1., -1., -0.,  0.,  0., -0., -0., -0., -0.,  0., -0., -0., -1.,
         0.,  0., -0., -0., -0.,  1.,  0., -0.,  0.,  0.,  0., -0., -0.,
        -1.,  0.,  0.,  0., -0.,  1., -1.,  0., -0., -0.,  0., -1.],
       [ 0., -0.,  1.,  0.,  0., -0.,  0., -0., -0.,  0.,  0., -1.,  0.,
         0.,  0., -0.,  0., -0., -0., -0.,  0., -0., -1.,  0., -0., -0.,
         0., -0., -0.,  0., -0., -1.,  1., -0., -0., -0., -0.,  0., -0.,
        -0.,  0., -0.,  1.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  1.,
        -0., -0., -1., -0., -0., -0., -1., -0.,  0.,  0., -0.,  0.,  0.,
        -0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,
         0., -1.,  0.,  0.,  0., -0., -0.,  0., -1., -0.,  0., -0.],
       [-0.,  0.,  0., -0., -0., -0., -0., -0., -0.,  1., -0.,  0., -0.,
         0.,  1.,  0., -0.,  1.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,
         0., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,
        -0., -0., -0.,  1., -0.,  0.,  0., -0.,  0., -0., -1.,  0.,  0.,
         1.,  0.,  0.,  0.,  0., -0.,  1., -0.,  0.,  0., -1.,  0.,  0.,
         0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,
        -1., -0., -0., -1., -0., -0., -1.,  0., -0.,  0., -1., -1.],
       [-0.,  0., -0., -0., -0.,  0., -0., -0., -0., -0.,  0.,  0., -0.,
         0., -1., -0., -1., -0., -0.,  0., -0., -0., -0., -0.,  0., -0.,
        -1., -1.,  0., -0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,
         0., -0., -0.,  0., -1.,  0.,  0.,  0., -1., -0.,  0., -0.,  0.,
        -0.,  0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,
        -1., -0., -0., -0., -0.,  1., -0.,  0.,  0.,  0.,  0.,  0.,  0.,
        -0.,  0., -0.,  0., -0.,  1.,  0., -0.,  1.,  0.,  1.,  0.],
       [ 0., -0., -0.,  1., -0.,  0., -0.,  0., -1.,  0.,  0., -0., -0.,
        -0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,
        -0., -0., -0., -0.,  0.,  0.,  1.,  0.,  0., -0.,  1., -0.,  0.,
        -0., -0., -0.,  1.,  0.,  0.,  0.,  0., -0., -0., -0.,  0.,  1.,
        -0.,  1.,  0.,  0.,  0.,  1., -1.,  0.,  0.,  0., -0.,  0., -0.,
        -0., -0.,  0., -0.,  0.,  0., -0.,  1.,  0., -0.,  0.,  0.,  0.,
         0., -0.,  0., -0.,  0., -0.,  0., -0., -1.,  0.,  0.,  0.],
       [-0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0., -0.,
        -0., -0., -0.,  1.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,
         1.,  1.,  0., -1.,  0.,  0., -0., -0., -0.,  0., -0.,  0.,  0.,
        -0., -0.,  0., -0.,  1., -0., -0.,  0.,  1.,  0.,  0.,  0., -1.,
        -1.,  0.,  0., -0.,  0.,  0., -0., -0., -0., -0., -0., -0., -0.,
         1., -0., -0., -0., -0., -1., -0.,  0.,  0., -0.,  0., -0.,  0.,
         1.,  0.,  0.,  0.,  0., -1.,  1., -0., -0., -0., -0.,  1.],
       [-0.,  0., -1., -0.,  1., -0.,  0.,  1., -0.,  0.,  0.,  1., -0.,
         0., -0., -0.,  0.,  0., -0.,  1., -0.,  0.,  1.,  0., -0.,  1.,
        -0., -0.,  0.,  1., -0.,  1., -1., -0.,  0., -1., -0., -0.,  0.,
         0., -0.,  0., -0., -1., -0.,  1.,  0., -0., -0., -0.,  0., -1.,
         1.,  0., -0., -1.,  0.,  0.,  1.,  0., -0., -0.,  0., -0., -1.,
         0.,  0., -0., -0., -0.,  1., -1.,  0.,  1., -1., -0., -0.,  0.,
         0.,  1., -0.,  0., -1.,  0., -1.,  0.,  1., -0.,  0., -0.],
       [-0.,  0.,  1.,  0., -0.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,
        -0., -1.,  0., -1., -0., -1., -1., -0.,  1., -0., -0.,  0., -0.,
        -0.,  0., -0.,  0., -0.,  0., -0.,  0., -0., -0.,  0.,  0., -0.,
        -0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,  0., -1., -0.,
        -1.,  0.,  0., -0., -0., -0., -1., -0.,  0.,  0., -0.,  0., -0.,
         0., -0., -0., -0.,  1., -0.,  0.,  0., -0.,  1., -1., -0.,  0.,
         0., -0.,  1., -0.,  0.,  0.,  1., -0.,  0., -0.,  0.,  0.],
       [ 0.,  0.,  0.,  0., -0., -0., -0., -1.,  0.,  0., -0., -0., -0.,
        -0.,  1., -0.,  1., -0.,  0.,  0., -0., -0., -1.,  0.,  0., -1.,
         1.,  0., -0., -0.,  0.,  0., -0.,  0., -0.,  1., -0.,  0.,  0.,
        -0., -0.,  0., -0.,  1., -0., -0., -1., -0., -0.,  0.,  1.,  0.,
         0., -0.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,
        -0., -0.,  0.,  0.,  0., -1., -0.,  0., -0., -0.,  0.,  0.,  0.,
         0., -0.,  0., -0.,  0., -0., -0., -1., -1., -0.,  0., -0.],
       [-0., -0., -1., -0.,  0.,  0., -0.,  1.,  0., -0., -0.,  1.,  0.,
         0.,  0., -1., -0., -0., -0.,  1.,  0.,  0.,  1.,  1., -1.,  1.,
         0.,  0., -0.,  1., -1.,  1., -1., -1.,  0., -1., -0., -0.,  0.,
        -1.,  0., -0.,  0., -1.,  0.,  1., -0.,  0.,  0.,  0.,  0., -1.,
         1., -0., -0., -1., -0., -0.,  1., -0.,  0.,  0.,  0., -0., -1.,
        -0.,  0., -1., -0., -0.,  1., -1.,  0.,  1., -1.,  0., -0.,  0.,
         0., -0., -1., -0., -1.,  0., -1.,  1.,  1.,  0.,  0.,  0.],
       [ 0., -0., -0., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,
        -0.,  0.,  1., -0.,  0., -0., -0.,  0., -0.,  0., -0., -0.,  0.,
        -1.,  0.,  0.,  0., -0., -1.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
        -0., -0.,  0., -0.,  0.,  0., -1.,  1.,  0.,  0.,  0.,  0.,  1.,
         0., -0.,  0.,  1., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  1.,
         0., -0.,  0.,  0.,  0.,  0.,  0., -0., -1., -0.,  0.,  0.,  0.,
         0., -0.,  0., -0.,  1., -0.,  0.,  0., -0., -0., -0.,  0.],
       [-1.,  0., -0.,  0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0.,
        -0., -0., -0.,  0.,  0., -0., -1., -0.,  0., -0., -0.,  0.,  0.,
         0.,  0.,  1., -1.,  0.,  0., -0.,  0., -0.,  1., -0.,  0., -0.,
         0., -0., -0., -1.,  1., -0., -1.,  0., -0.,  0., -0.,  0.,  0.,
        -1.,  0., -0.,  1., -0., -0., -0., -0.,  0.,  0., -0., -0., -0.,
         0., -0., -0.,  0., -1., -0.,  1., -0.,  0., -0.,  0.,  0.,  0.,
         0., -0.,  0.,  1.,  0., -0.,  0.,  0., -0., -0., -0.,  0.],
       [-0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0., -1.,  0.,  0., -1.,
         0.,  0., -0.,  1., -1.,  1., -0.,  0.,  0.,  0., -0., -0., -0.,
        -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  1., -0.,  0.,  0.,  0.,
         0.,  0., -0., -1., -0.,  0., -0.,  0.,  0., -0., -0.,  1., -0.,
        -0.,  0., -0.,  0., -1.,  0., -0., -1.,  0.,  0.,  1.,  0., -0.,
         0., -0., -0.,  0., -1.,  0., -0.,  0., -0., -0.,  1., -0., -1.,
         1.,  0.,  0.,  1., -0., -1., -0.,  0.,  0., -0.,  1.,  1.],
       [ 0.,  0., -0., -0., -0., -0., -0., -0.,  0., -0.,  0.,  0.,  1.,
        -0., -0.,  0.,  0.,  1., -1., -0.,  0.,  0.,  0., -0.,  0., -0.,
         0., -0.,  0., -0., -0.,  0.,  0.,  0., -1., -0.,  0., -0.,  0.,
        -0.,  0.,  0., -0.,  0., -1.,  0.,  0.,  1.,  0., -0., -1.,  0.,
         0.,  0.,  0.,  0.,  1.,  0., -0.,  0.,  0.,  0., -0., -0.,  0.,
         0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0., -0., -0.,  1.,  0.,
        -0., -0., -0.,  0., -0.,  0.,  0., -0.,  0.,  0., -1.,  0.],
       [-1.,  1.,  0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,
        -0.,  0., -0.,  0., -0., -0., -0.,  1.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0., -1.,  0., -0., -0.,  0.,  0.,  0.,  0., -1.,  0.,
        -0.,  0.,  1., -1.,  1.,  0., -1.,  0.,  0.,  0.,  0., -0.,  0.,
        -1., -0.,  0., -0.,  0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,
         0., -1., -0.,  1., -1., -0., -0., -0.,  0., -0., -0.,  0.,  0.,
        -0., -0., -0.,  1.,  0.,  0.,  0.,  0.,  0., -1., -0., -0.],
       [-0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  1.,  0.,  0.,  0.,
         0., -0.,  0., -1., -0.,  0.,  0., -0.,  0., -0.,  0.,  0., -0.,
         0., -0., -0.,  1.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0.,
         0., -0., -0., -0., -1.,  1.,  1.,  0., -1.,  1.,  0., -0.,  0.,
         1.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0., -1.,  0.,  1.,
        -1.,  0., -0., -0., -0.,  1., -0., -0.,  0.,  0.,  0., -1.]],
      dtype=np.float32), np.array([[-1., -1.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,
        -0., -0.,  0., -0.,  0., -0., -0.,  1., -0.,  1.,  0., -0., -1.,
        -0., -0., -1.,  0., -0., -0., -0., -0., -0.,  1.,  0., -0., -0.,
        -0., -0., -0.,  0., -0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,
        -0., -0.,  0., -0.,  0., -0., -0.,  0.,  0., -1., -0., -0.,  0.,
         0., -0., -0., -0., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0.,
         0., -1.,  0.,  0.,  0., -0., -0.,  1., -0.,  0., -0., -0.],
       [-1., -1.,  0.,  0.,  0.,  0.,  0., -0., -1., -0., -0., -0., -0.,
         0., -0.,  1., -0.,  0., -0.,  0.,  1., -0.,  0.,  0., -0., -0.,
        -0., -0., -1., -0., -0.,  1., -0.,  0.,  0., -0., -0., -0., -0.,
         0., -0.,  0.,  1.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,
         0.,  0.,  1., -1.,  0., -0., -0.,  0., -0., -1., -1., -0., -0.,
        -0., -0.,  0.,  1.,  0.,  0.,  0.,  1.,  1., -0.,  0.,  0.,  0.,
         0., -0.,  0., -1., -1., -0.,  0.,  0.,  0.,  0.,  0., -0.],
       [-0., -0., -0.,  0., -0.,  0., -0.,  0., -1.,  0.,  0., -0.,  0.,
        -0.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,
        -0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,
         0., -0., -0.,  1., -0.,  0., -0.,  0., -0., -0., -0.,  0., -0.,
         0., -0.,  1., -0., -0.,  0., -0.,  0.,  0.,  0., -1.,  0.,  0.,
         0.,  0.,  0.,  1.,  0., -0.,  0.,  1.,  0., -0., -0.,  0.,  0.,
         0.,  1., -0., -1.,  0., -0., -0.,  0.,  0., -0.,  0., -0.],
       [-1., -1.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0., -1., -0.,
        -0., -0., -0.,  0.,  0., -0.,  0.,  1., -0.,  0.,  0., -1., -0.,
        -0., -0., -1., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,
         0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,
         0., -0., -0.,  0., -0.,  0., -0., -0.,  0.,  0., -0., -0., -0.,
         0., -0.,  0., -0., -0.,  0., -1., -0., -0., -0., -0.,  0., -0.,
        -0., -1.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0., -0.],
       [-1., -1., -1., -0., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,
         0., -0.,  0.,  0., -0., -0., -1.,  1., -0.,  0., -0.,  0.,  0.,
        -0.,  0., -1., -0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0.,
        -0.,  0., -0.,  0., -0., -0., -0.,  0., -0., -0.,  0., -0., -0.,
        -0.,  0., -0., -0.,  0., -0., -0., -0.,  0., -1., -0., -0.,  0.,
         0.,  0., -0., -0., -0., -0., -0., -0.,  0., -1., -0., -0.,  0.,
         0., -1.,  1.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.],
       [ 1.,  1., -0., -0., -1., -0., -1.,  0., -1.,  0.,  0.,  0.,  0.,
        -0.,  0.,  0., -0.,  0., -0., -0., -1., -0.,  0., -0.,  0.,  0.,
         0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,
        -0.,  0., -0.,  1., -0.,  0., -0., -0., -0., -0., -0., -0.,  0.,
        -0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0., -0., -1.,  0.,  0.,
        -0.,  0.,  0.,  1.,  0., -0., -0.,  1.,  0.,  0., -0., -0., -0.,
         0.,  0., -0., -1.,  0.,  0., -0., -0., -0.,  0., -0., -0.],
       [-0., -0., -1., -1., -0.,  1.,  0., -0., -0.,  0.,  0.,  1., -0.,
        -0., -0., -0., -0., -1.,  1., -0.,  0.,  1.,  0., -0., -0.,  0.,
        -0.,  0.,  0.,  0., -0., -0.,  1., -0., -1.,  0., -1., -0.,  0.,
        -1., -0.,  0., -0., -0., -0., -0.,  0., -0.,  0., -1., -0., -0.,
        -0., -0.,  0., -0.,  1.,  0., -1., -0., -0., -0., -0., -0.,  0.,
         0.,  0., -1.,  0.,  0., -0., -0., -0.,  0.,  0.,  0., -1., -0.,
         0., -0.,  1., -0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.],
       [-0., -0.,  0., -0.,  0.,  1.,  0., -0., -0.,  1.,  0., -0.,  0.,
         0., -0.,  0.,  0.,  0., -0.,  0., -0., -1., -0., -0.,  0., -0.,
         0.,  0., -0., -0., -0., -0.,  0.,  0.,  0., -0., -0.,  0., -0.,
         1.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  1., -0.,  0., -0.,
         0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  1.,  0., -0.,
         0.,  0.,  1.,  0., -0.,  0.,  0., -0., -0., -0., -1.,  0.,  1.,
        -0.,  0., -1., -0., -0., -0.,  0.,  0.,  0.,  0., -0., -0.],
       [-0., -0.,  1.,  1.,  0., -0., -0., -0.,  0., -0., -0., -1., -0.,
        -0., -0., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,
        -0., -0.,  0., -0., -0.,  0., -1., -0., -0., -0.,  1., -0., -0.,
         0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  1., -0.,  0.,
         0., -0.,  0.,  0., -0.,  0.,  1., -0., -0., -0.,  1., -0., -0.,
        -0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0., -0.,
         0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0., -0.],
       [ 0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,
        -0.,  0.,  0.,  0.,  0.,  0.,  1., -0.,  0., -0., -0.,  0., -0.,
        -0., -0., -0., -1.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
        -0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,
        -1., -0.,  0.,  0.,  0.,  0.,  0., -1.,  0., -0., -0., -0., -0.,
        -0., -1.,  0.,  0.,  0.,  0., -1.,  0., -0., -0.,  0.,  0.,  0.,
        -0.,  0., -0., -0., -0., -0.,  0., -0.,  0.,  1.,  0., -1.],
       [ 0.,  0., -1., -1., -0., -0., -0., -0.,  0.,  0., -0.,  1.,  0.,
        -0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0.,
         0., -0., -0.,  0., -0.,  0.,  1., -0., -0.,  0., -1., -0.,  0.,
        -1.,  0., -0., -0., -0.,  0., -0., -0.,  0., -0., -0., -0.,  0.,
        -0.,  0., -0., -0.,  0.,  0., -1., -1., -0., -0., -0., -1.,  0.,
        -0., -0., -1.,  0.,  0., -0., -0., -0., -0.,  0., -0., -0.,  0.,
         0., -0.,  1., -0., -0.,  0., -0.,  0.,  0., -0., -0., -0.],
       [-1.,  0., -0., -0., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,
         0., -0.,  0.,  0.,  0., -0., -0.,  0.,  1., -0., -0., -0.,  0.,
         0., -0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,
        -1., -0., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,
        -0., -0.,  0., -0., -0., -0., -0., -0.,  0., -0.,  1.,  0.,  0.,
        -0.,  0., -1.,  0., -1., -0., -0.,  0.,  0.,  0., -0., -0., -0.,
        -0., -0.,  1.,  1.,  0., -0., -0., -0., -0.,  0.,  0.,  0.],
       [ 0.,  0., -0., -1.,  0., -0.,  0.,  0., -0.,  0.,  0.,  1.,  0.,
         0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  1., -1., -0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0., -0.,  0., -1.,
         0.,  0., -0., -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0., -0.,
        -0., -0.,  0., -0.,  0., -1.,  0., -0.,  0., -0.,  0., -0.,  0.,
         0.,  0., -1., -0., -0., -0., -0., -0.,  0.,  0., -0., -1.,  0.,
         0., -0.,  0.,  0.,  0.,  0., -0.,  1., -1.,  0.,  0., -0.],
       [ 0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0.,  1.,  0., -0., -1.,
         0.,  0., -0., -1., -0.,  1.,  0.,  0.,  0., -0.,  1.,  0.,  0.,
        -0., -0., -0., -0., -0., -0.,  0.,  0., -0., -0., -0., -0., -1.,
         0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  1., -0.,  1., -0.,
         0.,  0.,  0.,  0., -1.,  0.,  0.,  0.,  0.,  0.,  1.,  0., -0.,
        -0.,  0.,  1.,  0.,  0.,  0., -0.,  0., -0., -0., -1.,  0.,  1.,
         1., -0., -0., -0., -0., -1.,  0., -1., -0.,  0., -0., -0.],
       [ 0.,  0.,  0.,  1.,  0., -0.,  0., -0.,  0.,  0.,  0., -1.,  0.,
         0.,  0., -0., -0., -1., -0.,  0.,  0., -0., -1., -0., -0., -0.,
        -0., -0., -0.,  0., -0., -0., -1.,  0.,  0., -0.,  0., -0.,  0.,
         0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,
         0., -0.,  0.,  0., -0.,  1., -0.,  0.,  0.,  0.,  1.,  0., -0.,
        -0., -0., -0., -0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0., -0.,
         0.,  0., -0., -0., -0.,  0.,  0., -0.,  1., -0.,  1.,  0.],
       [ 0., -0., -0., -0.,  0.,  0., -0., -0., -0., -0., -0.,  0., -0.,
        -0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,
        -0.,  0., -0., -1., -0.,  0., -0.,  0.,  0., -1.,  0.,  0., -0.,
        -0.,  0.,  1.,  0., -1.,  0., -0.,  0.,  1., -0., -0.,  0., -0.,
         0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0., -0.,
        -0.,  0., -0.,  0.,  0.,  0., -1.,  0., -0., -0., -0., -1., -0.,
        -0., -0., -0., -0., -0., -0.,  0., -0.,  0.,  1.,  0.,  0.],
       [ 0., -0., -1., -1.,  0., -0.,  0.,  0.,  0.,  0., -1.,  1., -0.,
         0., -0., -0.,  0., -0., -0., -1., -0.,  0., -0., -0., -0.,  1.,
        -0.,  1., -0.,  0., -0., -0.,  1.,  0.,  0., -1., -1.,  0.,  0.,
        -1.,  0.,  1.,  0., -1., -1., -0., -0.,  0.,  0.,  0., -0., -0.,
         1.,  0.,  0., -0.,  0.,  0., -1.,  0.,  1., -0., -0., -1.,  0.,
        -0.,  1., -1., -0., -0.,  1., -0.,  0.,  0., -1., -0., -1., -0.,
         0., -0.,  1.,  0.,  0., -0., -1., -0., -0., -0., -0., -0.],
       [-1., -0., -0.,  0., -0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,
         0., -1.,  0.,  0., -0., -1., -0.,  0.,  0., -0., -1., -0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0.,
         0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -1.,  0.,
        -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0., -0.,  1.,  0.,  0.,
         0.,  0., -1.,  0., -1., -0.,  0., -0.,  0.,  0., -0., -0.,  0.,
         0., -0.,  0.,  1.,  0.,  0., -0.,  1.,  0.,  0., -0.,  0.],
       [-0.,  1., -0., -1., -0.,  0., -0., -0.,  0.,  0.,  0.,  1., -0.,
         0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0., -0., -1., -0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0.,  1.,  0., -0.,  0.,  0.,  0., -1.,
        -0., -0., -1., -0., -0.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,
        -0.,  0.,  0., -0.,  0., -1., -0.,  0., -1., -0.,  0.,  0.,  0.,
        -0.,  0., -1.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,
         0.,  1.,  0.,  0.,  0., -0., -0., -0., -0.,  0., -0.,  0.],
       [-0.,  1.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,
        -1., -0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,
        -0., -0., -0., -0., -1.,  0., -0.,  0., -0., -0.,  0.,  1., -0.,
        -0., -1.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,
         0.,  1.,  0.,  0.,  0.,  0., -0., -0., -0., -0., -0., -0., -0.,
        -0., -0.,  1., -1.,  0.,  0.,  0., -1., -0., -0., -0.,  0., -0.,
        -0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.],
       [ 0., -0.,  0.,  1.,  0., -0., -0., -0., -0.,  0.,  0., -1., -0.,
        -0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0., -0., -0.,
         0., -0., -0., -0., -0.,  0., -1., -0., -0., -0., -0., -0., -0.,
        -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,
         0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,
        -0.,  0.,  0., -1.,  0.,  0., -0., -1., -0., -0.,  0., -0.,  0.,
         0., -1., -0., -0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.],
       [-0.,  1., -0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,  1., -0.,
        -0.,  0.,  0., -0.,  0., -0., -0.,  0., -0., -0., -0.,  1.,  0.,
         0.,  0.,  0.,  0., -0., -0.,  1., -0., -0.,  0.,  0.,  0., -0.,
        -0., -0., -0., -0., -0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,
        -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,
         0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,
        -0.,  1.,  0., -0.,  0., -0., -0.,  0.,  0.,  1., -0.,  0.],
       [ 0.,  1.,  0., -1.,  0.,  0.,  0.,  0., -0.,  0., -1.,  1.,  0.,
         0.,  0.,  0.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,
        -0., -0., -0.,  0., -0., -0.,  1., -0.,  0., -0., -1., -0., -0.,
        -1.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,
        -0., -0.,  0., -0.,  0., -0., -0., -0., -0.,  0., -0., -1.,  0.,
         0.,  1., -1.,  0.,  0., -0., -0., -0.,  0.,  0., -0., -0.,  0.,
         0.,  1.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.],
       [-0., -1., -0.,  0.,  0.,  0.,  1.,  0.,  1.,  0., -0.,  0., -0.,
         0., -0., -0., -0., -0.,  0., -0.,  1., -0., -0.,  0.,  0.,  0.,
        -0.,  0., -0.,  0., -0.,  0., -0.,  1., -0., -0., -0.,  0., -0.,
         0., -0.,  0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,
        -0.,  0., -0., -0.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,  0.,
         0., -0., -1., -1., -0., -0., -0., -1.,  0.,  0.,  0.,  0., -0.,
         0.,  0.,  0., -0.,  0., -0., -0., -0.,  0., -0., -0.,  0.],
       [ 0.,  0.,  0., -1.,  0., -0.,  0.,  0., -0.,  0.,  0.,  1., -0.,
         1., -0.,  0., -0., -0., -0., -0., -0.,  0.,  1., -1.,  0., -1.,
         0.,  0.,  0.,  0.,  0., -0.,  1., -0., -0.,  1.,  0., -1., -1.,
         0., -0., -1.,  0.,  1.,  0., -1.,  1., -0.,  1., -0.,  0., -0.,
        -0.,  0., -0.,  1., -0., -1., -0., -0., -1., -0.,  0.,  0., -1.,
        -1., -0., -1., -0., -0., -1., -0.,  0., -0.,  0.,  0.,  0.,  0.,
         0., -0.,  0.,  0.,  1., -0., -0.,  1., -1.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,  1.,  0., -0., -0.,
        -0., -0.,  1.,  0., -0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,
        -0.,  0., -0.,  0., -1.,  0.,  0.,  0., -0., -0., -0., -0., -0.,
        -0., -1., -0., -0.,  0., -0.,  0.,  0., -0.,  1., -0.,  0., -0.,
         0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,  0.,  1.,  0., -0.,
        -0., -0.,  1., -0.,  0.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,
         1.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0.],
       [ 0., -0., -0.,  1., -0., -0.,  0., -0.,  0.,  1., -0., -1.,  0.,
        -0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,
         0., -0., -0., -0.,  0., -1., -1., -0., -0., -0., -0., -0., -0.,
         0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0.,  1.,
         0., -1., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  1., -0., -0.,
        -0.,  0., -0., -0., -0.,  0.,  0.,  0., -0., -0., -0., -0.,  0.,
         1., -0.,  0., -0., -0., -0.,  0.,  0.,  0., -0.,  0., -0.],
       [ 0.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,
         0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0., -0., -0.,  0.,
        -0., -0.,  0., -1., -0., -0., -0., -0.,  0.,  0., -0., -1., -0.,
         0.,  0., -0., -0., -0., -0., -1., -0., -0.,  1., -0., -0., -0.,
        -0., -0., -0.,  1.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,
         0.,  0., -0.,  0.,  0., -0., -1., -0.,  0.,  0., -0., -0., -0.,
         0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,  1., -0.,  0.],
       [ 0.,  0., -1., -1., -0.,  0.,  0., -0.,  0., -0., -1.,  1., -0.,
         1., -0.,  0., -0., -0.,  0., -1.,  0., -0., -0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0., -0., -0.,  1.,  0.,  0.,  0., -1., -1., -0.,
        -1., -0., -0., -0., -0., -0., -1.,  0., -0.,  1., -0.,  0.,  0.,
         1.,  0., -0.,  1.,  0.,  0., -1., -0.,  0.,  0.,  0., -1., -1.,
         0.,  1., -1.,  0.,  0., -0., -0., -0.,  0., -1., -0., -0.,  0.,
         0.,  0.,  1.,  0.,  1.,  0., -1.,  0., -0., -0., -0.,  0.],
       [-1., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,
        -0., -1., -1., -1.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,
        -1.,  0.,  0.,  0.,  1.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,
         0., -0.,  0.,  0., -0.,  0., -0., -0.,  0., -0., -0.,  0.,  0.,
        -0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,  1.,  0.,  0.,
        -0., -0., -1., -0., -1., -0., -0., -0.,  0.,  0., -1., -0., -0.,
         0.,  0.,  0.,  1.,  0.,  0., -0., -0., -0., -0.,  0.,  0.]],
      dtype=np.float32), np.array([[ 0.,  0., -0.,  0.,  0., -0., -0.,  1.,  0.,  0., -0.,  0.,  1.,
        -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0., -0., -0.,  1.,
         0., -1., -0., -0.,  0.,  0., -0., -0.,  1., -0., -0., -0., -1.,
        -0.,  0.,  0.,  0., -0., -0., -0., -0.,  0.,  0., -0., -0.,  0.,
        -0., -0., -0., -0., -1., -0.,  0.,  0., -1., -0., -0., -0.,  0.,
        -1., -0., -0.,  0.,  0., -1.,  0., -0., -0.,  0., -0.,  0.,  0.,
        -0., -0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,  0., -0.],
       [-0., -0.,  0.,  0.,  0., -0., -0.,  1.,  0.,  0., -0.,  0.,  1.,
        -0., -0., -0., -0., -1.,  0., -0., -0., -0., -1.,  0., -0., -0.,
        -0., -0.,  0.,  0., -0., -0., -0., -0., -0., -0., -0.,  0., -1.,
         0.,  0., -0., -0., -0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,
         0.,  0., -0., -0., -1., -1., -0., -0.,  0., -0., -0.,  0., -0.,
        -1.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,
         0.,  0., -0., -0.,  0.,  0., -0., -0., -1., -0.,  1.,  0.],
       [ 0.,  0.,  0.,  0., -0., -0., -0.,  1., -0.,  0., -0.,  0.,  1.,
         0., -0., -0.,  0., -0., -1., -0.,  0.,  0., -0., -1.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0., -1.,
         0., -0.,  0., -0., -0., -0., -0.,  1., -0.,  0.,  0.,  1.,  0.,
         0., -0.,  0., -0., -1., -0.,  0., -0., -0., -0., -0., -0.,  0.,
         0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,
        -0., -0., -0., -0.,  0., -0.,  0., -1., -0., -0.,  0.,  0.],
       [ 0., -0., -0., -0.,  0., -0., -0., -0., -0.,  0., -0., -0.,  0.,
        -0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0., -1.,
         0., -0., -0., -0.,  0., -0.,  0.,  0., -1.,  1., -0.,  0.,  0.,
         0., -0., -1., -0., -1., -1.,  0., -0., -1.,  0., -0., -0.,  0.,
         0.,  0., -0.,  0., -0.,  0.,  0., -0.,  1.,  0.,  0., -0.,  0.,
        -0.,  0., -0., -0.,  0.,  1.,  0.,  0., -0.,  0.,  0., -1., -0.,
        -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0., -0.],
       [ 0.,  0.,  0.,  0., -0.,  1., -0., -0., -0.,  0.,  0.,  0., -1.,
         1.,  0., -0.,  0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,
        -0.,  0., -0.,  0.,  0.,  0., -0.,  0., -1., -0.,  0., -0.,  0.,
         0., -1.,  0., -0., -0., -0., -0.,  0.,  0., -0., -0.,  0.,  0.,
         0.,  0., -0., -0.,  1.,  0., -0.,  0.,  0.,  0.,  0., -0., -1.,
         0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  1.,  0.,  0., -0.,  0.,
        -0., -0., -0., -0., -1., -0.,  0.,  0., -0., -0., -0., -0.],
       [ 0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  1.,  0.,  0., -0.,
         0., -0., -0.,  0., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,
        -0.,  0.,  0.,  0., -0., -1., -0.,  0.,  0., -0.,  0.,  0., -0.,
         0., -1., -0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  1.,
         0.,  1.,  0., -0., -0., -0., -0.,  0.,  0., -0., -0.,  0., -0.,
        -0., -0.,  0., -0.,  0., -0.,  0.,  0.,  1., -0., -0., -0., -1.,
         1., -0.,  0.,  0.,  0., -1., -0., -0., -0.,  0.,  0., -0.],
       [ 0., -0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,
         0., -0.,  1., -1.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,
         1., -0., -0., -0.,  1., -0.,  0.,  0., -0., -0.,  0., -0.,  0.,
        -0., -1.,  0., -0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0.,
        -0., -0., -0., -0.,  0., -0.,  0.,  0., -0., -0.,  0., -0., -0.,
        -0.,  0., -0., -0., -0., -0.,  0., -0.,  1., -0.,  1., -0., -1.,
         0., -0., -0., -0.,  0., -1., -0.,  0.,  0.,  0., -0., -0.],
       [-0.,  0., -0., -0., -0., -1.,  0.,  0.,  0., -0., -0., -0.,  0.,
        -1.,  0., -0., -0., -0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0.,
         0.,  0., -0.,  0.,  0., -0.,  0., -0.,  1.,  0., -0., -1.,  0.,
         0.,  0., -0.,  0.,  0.,  0.,  1.,  0.,  0.,  1.,  0.,  0., -0.,
        -0.,  0., -0., -1., -1., -0.,  0., -0.,  0., -0.,  0., -0.,  1.,
        -0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -1.,
        -0., -0., -0., -0.,  1.,  0., -0., -0.,  0., -0., -0., -0.],
       [-0., -0., -0.,  1.,  0., -0., -1., -0., -1., -0.,  0., -0.,  1.,
        -1.,  0., -0., -0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,
        -0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  1.,  0., -1.,  0., -1.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,
        -0.,  1.,  0.,  0., -1., -1.,  0., -0., -0.,  0.,  0., -1.,  1.,
        -0.,  0., -0.,  0.,  0.,  0., -0.,  1., -1., -0., -0., -0.,  0.,
        -0.,  0., -0.,  0.,  1., -0.,  0., -0., -0.,  0.,  0.,  0.],
       [-0.,  0.,  0.,  1.,  0., -0.,  0.,  0., -0., -0.,  0.,  0.,  1.,
        -0., -0., -0., -0., -1.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,
         0.,  0., -0.,  0.,  0.,  1., -0., -0., -0.,  0., -1.,  0., -1.,
        -0., -0.,  0., -0., -0., -0.,  0.,  0., -0., -0., -1.,  0., -1.,
         0.,  0., -1.,  0., -1., -1., -0., -0.,  0.,  0.,  0., -1., -0.,
         0., -0.,  0., -0., -0., -0., -0.,  0., -1.,  0., -0., -0.,  0.,
         0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  1.,  0.],
       [ 0., -0.,  1., -0.,  0.,  0., -0.,  0., -0., -0., -0., -1.,  1.,
        -0., -0.,  0.,  0., -0., -1.,  0., -0., -1.,  1., -1., -1., -0.,
        -0.,  0., -0.,  0., -0.,  1.,  0., -0.,  0.,  0., -0., -0., -1.,
         1., -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  1.,  0.,
        -0., -0., -1.,  0., -1., -0.,  0.,  0.,  0.,  0.,  0., -1., -0.,
        -0.,  0., -0., -0.,  0.,  0.,  0.,  0., -1., -0., -0.,  0.,  0.,
         0.,  1.,  1.,  0.,  0.,  0., -0., -1.,  0., -0., -0.,  0.],
       [ 1.,  0.,  0.,  0.,  0.,  0.,  1.,  0., -0.,  1., -0.,  0.,  0.,
         1.,  0.,  0., -0., -1., -0., -0., -1., -0.,  0., -0., -0.,  0.,
         0.,  0.,  1., -0., -0.,  0., -0., -0., -1.,  0.,  0.,  1., -0.,
        -0.,  0., -0.,  0.,  0., -0., -1.,  0., -0., -1., -1.,  0.,  0.,
         0., -0., -0.,  1.,  0., -0., -0.,  0.,  0.,  0.,  1.,  0., -1.,
         0.,  0.,  0., -1., -0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,
        -0., -0., -0., -1., -1., -0.,  0.,  0.,  0.,  0., -0.,  0.],
       [ 0., -1.,  0.,  0., -0., -0.,  1., -0.,  0., -0., -1., -0., -0.,
        -1.,  0.,  0.,  0.,  0.,  0., -0., -1., -0., -0.,  0., -0.,  0.,
        -0., -0.,  0., -0., -0., -0., -0.,  0., -0.,  0.,  0., -1.,  0.,
         0., -0., -1.,  0.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,
         0.,  0., -0.,  0., -0., -0., -0., -0.,  1.,  1., -0., -0., -0.,
        -0., -1.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,
         0.,  0., -0.,  0.,  0., -0., -0., -0.,  0., -1.,  0.,  0.],
       [-0.,  0.,  0.,  1., -0., -0.,  0.,  0., -0., -0., -0.,  1.,  0.,
        -0.,  0.,  0.,  0., -0., -0., -1.,  0., -0.,  0., -0.,  0.,  1.,
         0.,  0., -1.,  1.,  0., -0.,  1.,  0., -0., -1.,  0., -0., -0.,
         0.,  0.,  0.,  0.,  1.,  0., -1.,  0.,  0.,  0.,  0.,  0., -0.,
        -1., -0., -0.,  1., -0.,  0., -0.,  0., -0.,  1.,  0., -0., -1.,
         0.,  0.,  0., -0., -0., -1.,  1., -0.,  0., -1., -0.,  0., -0.,
         0.,  0., -0., -0., -1.,  0.,  1.,  0., -0., -0.,  0., -0.],
       [-0., -0.,  0., -0., -0.,  0., -0.,  0.,  0., -0., -0.,  0.,  0.,
        -0., -0.,  0.,  0., -0.,  0., -1.,  0., -0.,  0., -0., -1.,  1.,
        -0.,  0., -1.,  0., -0.,  0., -0.,  0., -0., -1., -0.,  0.,  0.,
        -0., -0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,
         0., -0.,  0.,  1.,  0., -0.,  0., -0.,  0.,  1., -0.,  0.,  0.,
         0., -0., -0.,  0., -0., -0.,  1.,  0., -0., -1.,  0.,  0., -0.,
        -0.,  0.,  0., -0., -1., -0.,  0., -0., -0.,  0.,  0., -0.],
       [-0.,  1.,  0.,  0.,  0.,  0., -1.,  0., -0., -0.,  1.,  0.,  0.,
         1.,  0., -0., -0., -0.,  0., -1.,  1., -0., -0., -0., -0.,  1.,
         0., -0., -1., -0.,  0.,  0., -0.,  0., -0., -1.,  0.,  1.,  0.,
         0.,  0.,  1., -0.,  1., -0., -1.,  0.,  1.,  0.,  0.,  0.,  0.,
        -1., -0.,  0.,  1.,  0.,  0., -0., -0., -1., -0., -0.,  0., -1.,
         0.,  1.,  0., -0., -0., -1., -0., -0., -0., -1., -0., -0., -0.,
         0.,  0., -0., -0., -1., -0.,  1.,  0., -0.,  0.,  0., -1.],
       [ 0., -0., -0.,  0., -0., -0., -0., -1., -0., -0.,  1., -0., -0.,
        -0., -0., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0., -0., -1.,
         0.,  1., -0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0.,
        -0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0., -0.,
        -0., -0.,  0.,  0., -0., -0.,  0.,  0.,  0., -1.,  0., -1.,  1.,
         1., -0., -0., -0., -0.,  1., -0., -0., -1.,  0.,  0., -0., -0.,
        -0., -0.,  0.,  0.,  1., -0., -0., -0.,  0.,  0.,  0.,  0.],
       [-0., -0., -1., -0.,  0.,  0., -0., -1.,  0.,  0., -0., -0., -0.,
         0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0., -1.,
        -0., -0., -0.,  0., -0.,  0.,  0.,  0., -0., -0., -1., -0., -0.,
         0., -0., -0.,  0.,  0., -0., -0., -0.,  0., -0., -1., -0., -0.,
        -0.,  0., -0.,  0., -0.,  0., -1.,  0.,  0., -1.,  0., -1.,  1.,
         1.,  0., -0.,  0., -0.,  1.,  0.,  0., -1.,  1., -0., -0., -0.,
        -0., -0.,  0.,  0.,  1., -0., -1., -0.,  0.,  0.,  0.,  0.],
       [-0.,  0., -0., -0., -0.,  0., -0., -1.,  0.,  0.,  0., -0.,  0.,
        -0.,  0.,  0.,  0., -0., -0.,  0.,  0., -1., -0.,  0., -0., -1.,
         0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,
         1.,  0., -0.,  0.,  0., -0.,  0., -1., -0., -0., -0.,  0., -0.,
        -0.,  0., -0.,  0., -0., -0.,  0., -0., -0., -1.,  0., -1., -0.,
        -0., -0., -0., -0., -0.,  0.,  0.,  0., -1.,  1., -0., -0.,  0.,
        -0.,  0.,  1.,  0.,  1.,  0., -0., -0.,  0.,  0.,  0.,  0.],
       [ 0.,  0., -0., -0.,  0., -0., -0., -0.,  0.,  0., -1., -0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0.,  1., -0.,  0., -0.,  0.,  0.,  0.,
        -0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0.,
         0.,  0., -0.,  0.,  0.,  1.,  0., -0., -0.,  0., -0.,  0., -0.,
         1., -0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0., -0., -0.,
        -0., -1., -0., -0.,  0.,  0.,  0., -0., -0.,  1., -0.,  0.,  0.,
        -0., -0.,  0.,  0.,  0., -0., -1., -0.,  0.,  0.,  0.,  1.],
       [ 0., -0.,  0.,  0.,  0.,  1.,  1.,  0.,  0., -0., -0., -0., -2.,
         0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0.,  1.,  0., -0.,
        -0., -0.,  0.,  0.,  1.,  0., -0.,  1., -2., -0., -0., -0.,  1.,
        -1., -1., -0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0., -0., -0.,  2., -0., -0.,  0., -0.,  0.,  0.,  1.,  0.,
        -0.,  0.,  1.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,  0., -0.,
        -0., -0.,  0., -0.,  0., -0., -0., -0., -0., -0.,  0., -0.],
       [ 0., -0., -0.,  0.,  0.,  0., -0., -0., -1., -0., -0.,  0., -1.,
         0.,  1., -0., -1.,  0.,  1.,  0., -0.,  1., -0.,  1.,  0., -0.,
        -0., -0.,  0.,  0.,  1., -0., -0.,  1., -0., -0.,  0., -0.,  1.,
        -1., -1.,  0.,  1.,  0.,  0., -0.,  0.,  0., -0., -0., -1., -0.,
         0.,  0.,  1., -0.,  1.,  0.,  0.,  0., -0.,  0., -0.,  1.,  0.,
        -0.,  0.,  1., -0.,  1., -0., -0., -0., -0., -0.,  1., -0., -1.,
         0., -0.,  0., -1., -0., -1., -0., -0., -0., -0.,  0.,  0.],
       [-0., -0., -0., -0., -1.,  0.,  0., -0., -0.,  0.,  0.,  0., -1.,
         0., -0., -0., -1., -0.,  1.,  0.,  0.,  1.,  0.,  1.,  0., -0.,
         1.,  0., -0.,  0.,  1.,  0.,  0., -0., -0.,  0.,  0., -0.,  1.,
        -1., -1.,  0., -0., -0., -0.,  0.,  0.,  0., -0., -0., -1.,  0.,
        -0.,  0., -0., -0.,  1., -0., -0., -0.,  0., -0., -0.,  1., -0.,
        -0.,  0.,  1.,  0., -0., -0.,  0., -0.,  0.,  0.,  1., -0., -1.,
        -0., -0., -0.,  0.,  0., -1.,  0., -0.,  0.,  0.,  0.,  0.],
       [-1., -0., -0., -0., -0., -1., -1.,  0., -0.,  0., -0., -0., -0.,
         0.,  0.,  0., -0., -0.,  1., -0.,  1.,  1., -0., -0.,  0.,  0.,
        -0.,  0., -1., -0., -0.,  0.,  0., -0.,  2.,  0., -0., -0., -0.,
         0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0., -0.,
        -0.,  0.,  0.,  0., -1., -0., -0., -0., -0., -0.,  0.,  0., -0.,
        -0.,  0., -0., -0.,  1.,  0., -0., -0., -0., -0.,  1., -0., -1.,
        -0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.]],
      dtype=np.float32))

In [ ]:
#@title Verification
n = 4
m = 5
p = 6
rank = 90

verify_tensor_decomposition(decomposition_456, n, m, p, rank)

## Rank-93 decomposition of <5,5,5> over Z



In [ ]:
#@title Data

decomposition_555 = (np.array([[ 0.,  0.,  0., -1., -0., -0.,  0.,  0., -0., -0., -0.,  1.,  0.,
         0.,  0.,  0., -0., -0.,  1.,  0., -0., -1., -0.,  0.,  0., -0.,
         0.,  0.,  0., -1., -1.,  1., -1.,  0., -0., -0.,  0., -0., -0.,
        -0.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,
        -1., -0., -0.,  0.,  0., -0.,  0., -0., -0.,  0., -0., -0.,  1.,
        -0., -0.,  0.,  0., -0.,  0.,  0., -1.,  0., -0., -0.,  0., -0.,
        -0., -1.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,
         0., -1.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0.,
         0., -0.,  0.,  0.,  0., -1., -0., -1.,  1., -0., -0., -1., -1.,
        -1.,  0.,  1.,  1.,  1., -1.,  1., -0.,  0.,  0., -0., -0., -0.,
        -0.,  0., -0.,  0.,  0.,  0.,  0.,  1., -0.,  0., -1.,  1., -0.,
         1.,  0., -0.,  0., -0.,  1.,  0., -0., -0.,  0., -0., -0., -1.,
         1.,  0.,  0.,  0.,  1.,  0.,  0.,  1., -0.,  1., -0., -0.,  1.,
        -1.,  0., -1.,  0.,  0., -0., -0.,  0., -0., -0., -1., -0., -1.,
         0.,  0.],
       [-0.,  0., -0.,  0.,  1.,  0.,  0., -0.,  0.,  1., -1., -0.,  0.,
         1.,  0., -1.,  0., -0.,  0., -0., -0.,  0., -0., -0., -0., -1.,
        -0., -0.,  0., -0., -0., -1.,  1., -1.,  0., -1., -0., -1.,  0.,
        -0., -1., -0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0.,
         1.,  0., -0.,  0., -1., -0.,  0.,  0., -0.,  0., -0., -1.,  0.,
        -0., -0., -0., -0.,  0.,  0.,  0.,  1.,  0.,  0., -0., -0.,  0.,
        -0.,  0.,  0., -0.,  1., -0.,  0.,  0., -0., -1., -0., -0.,  0.,
         1.,  0.],
       [-0., -0.,  1., -1.,  0., -0.,  0., -0.,  0.,  0., -0.,  1., -0.,
        -0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0., -0., -0., -0., -1.,
        -1.,  1.,  0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0., -0.,
         0., -0.,  0.,  0., -0.,  0., -1., -0.,  1., -0., -0.,  1., -0.,
        -0., -0.,  0.,  0.,  0.,  1., -0.,  0., -0.,  0.,  0.,  0., -0.,
         0., -1.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,
        -0., -1.,  0.,  0.,  0.,  1.,  1., -0.,  0.,  0., -0., -0., -1.,
        -0., -1.],
       [ 0.,  0.,  0., -0., -1., -0.,  0., -0.,  0., -1.,  0., -0.,  0.,
        -1.,  0.,  1.,  0.,  0.,  0.,  0., -0.,  1., -0., -0.,  0.,  0.,
        -0., -0., -0.,  1.,  1., -0.,  0.,  1., -0.,  1., -0.,  1.,  0.,
        -0.,  1., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -1.,  1., -0.,
         0.,  0., -0., -0.,  1.,  1.,  0.,  0.,  0.,  0., -0.,  1.,  0.,
        -0.,  0., -0., -0.,  0.,  1., -0.,  0., -0., -0.,  0.,  0.,  0.,
         0., -0., -0.,  0.,  0., -0., -0.,  0.,  0.,  1., -0., -0.,  0.,
        -1., -0.],
       [ 0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,
        -0.,  1., -0.,  0.,  0., -0., -0., -0.,  0., -0., -1., -0.,  0.,
        -0.,  0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,  0., -0.,  0.,
        -0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0., -0.,
         0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  1., -0., -0., -0.,  0.,
        -1.,  0., -1., -0., -1.,  0., -0.,  1., -0.,  0.,  1., -0., -0.,
         0.,  1., -0., -1.,  0.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,
         0.,  1.],
       [-0.,  0., -0.,  0., -0.,  1., -0.,  0.,  0.,  0., -0., -1.,  0.,
        -0., -1.,  0.,  0., -0., -0.,  0.,  0.,  0.,  1.,  1.,  1., -0.,
         0.,  0., -1.,  0.,  0.,  0., -0., -0., -1., -0., -0.,  0.,  1.,
        -0.,  0., -0.,  0., -0., -0.,  0.,  0., -0., -1.,  0., -0., -0.,
        -0.,  0.,  1., -0., -0.,  0., -0.,  0., -1.,  0., -0.,  0.,  0.,
         0.,  0.,  0., -0.,  0.,  0.,  0., -1., -0., -0., -0.,  0., -1.,
        -0., -0.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0.,
         0., -0.],
       [-0.,  0., -0.,  0., -1.,  0., -0.,  0., -0.,  0.,  0., -1., -0.,
         0., -0., -0., -0.,  0., -0.,  0., -0., -0.,  0.,  1.,  1., -0.,
         0.,  0., -0., -0.,  0., -0.,  0.,  0., -0., -0.,  0.,  1.,  0.,
        -0.,  1., -0., -0., -0.,  0., -0.,  0.,  0., -1.,  0., -0.,  1.,
         0., -0.,  1., -0.,  1.,  0.,  0., -0.,  0.,  0., -0., -0.,  0.,
        -0.,  0.,  0., -0., -0.,  0.,  0., -1.,  0.,  0.,  0.,  0., -1.,
        -0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0., -0., -1.,  0.,
        -0.,  0.],
       [ 0.,  1., -1., -0., -0., -0., -0., -0.,  1., -0.,  0., -1., -1.,
        -0., -0., -0., -0.,  0., -0., -1., -0.,  0.,  0.,  0., -0.,  0.,
        -0., -1., -0.,  0., -0.,  0.,  0., -0., -1., -0.,  0.,  0.,  1.,
        -0.,  0.,  1.,  0., -0.,  0.,  1., -0., -1.,  0.,  0., -0., -0.,
         0., -0.,  1.,  0., -0.,  0., -0.,  0.,  0., -0.,  1., -0.,  0.,
         0.,  1., -1., -0.,  0., -0., -1., -0., -0.,  0.,  0.,  0.,  0.,
        -1.,  1.,  0., -0., -0., -1., -1., -1., -0., -0., -1.,  0., -0.,
         0.,  1.],
       [-0.,  0.,  0., -0.,  1.,  1.,  0.,  1., -0.,  1., -0., -0.,  0.,
        -0.,  0., -1.,  1.,  0.,  0., -0.,  0., -1.,  0., -0., -0., -0.,
        -0., -0., -1., -0., -1.,  0., -0., -0., -1.,  0., -0., -1.,  0.,
         0., -1., -0.,  0., -0., -0.,  0.,  0., -0., -0.,  0., -1.,  0.,
         0., -0.,  0., -1., -1., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,
         0., -0.,  0.,  0.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  0.,
        -0.,  0., -0.,  0.,  0.,  0., -0., -0., -1.,  0., -0.,  0., -0.,
         0.,  0.],
       [ 0.,  0., -0.,  1., -0., -0., -0., -0.,  0., -0., -0., -1., -0.,
         1.,  0.,  0.,  0., -0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,
        -0.,  0., -0., -0.,  0., -1.,  1., -0.,  0.,  0., -0., -1.,  0.,
        -1., -0., -0., -0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0., -0.,
         1., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,
         0.,  0.,  0., -0., -0.,  0., -0.,  1., -0., -0.,  0., -1.,  0.,
         0.,  1., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,
         0.,  0.],
       [ 0.,  0., -0., -0., -0., -0.,  0.,  0.,  0., -0., -0., -0., -0.,
        -1.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0.,  1.,  1.,
        -0., -0., -1., -0., -0.,  1., -1., -0., -0.,  0.,  1.,  1.,  0.,
         1., -0.,  0., -1., -0., -0.,  0., -1.,  0.,  0.,  0.,  0., -0.,
        -1., -0., -0.,  0.,  1.,  0., -0., -0.,  0.,  0.,  0.,  1., -0.,
        -1., -0., -0.,  0., -0.,  0.,  0., -1., -0.,  0., -0.,  1.,  0.,
         1.,  0., -0., -0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,
         0.,  0.],
       [ 0., -0., -0., -0.,  0., -0.,  0., -0., -0., -0., -0., -0.,  0.,
        -1., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  1.,
        -0., -0.,  0., -0.,  0.,  1., -1., -0., -0.,  1., -0.,  1.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,
        -1., -0.,  0., -0.,  1.,  0.,  0.,  0.,  0., -0.,  0.,  1.,  0.,
         0.,  0., -0.,  1., -0.,  0.,  0., -1.,  0.,  0.,  0.,  1.,  0.,
         0., -0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,
         0.,  0.],
       [ 0.,  0.,  0.,  1.,  0.,  0., -0.,  0.,  0., -0.,  0., -1.,  0.,
         0., -0., -0., -0., -0., -0., -0., -0.,  0.,  0., -0., -0.,  1.,
         0., -0., -0.,  0.,  0.,  0., -0.,  0., -0., -0.,  1.,  0., -0.,
         0.,  0.,  0.,  0.,  0.,  0.,  1.,  0., -0., -0., -0.,  0., -0.,
        -0., -0., -0., -0.,  1.,  0.,  0.,  0.,  0., -0., -0.,  1., -0.,
         0.,  1., -0.,  0., -0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0.,
        -0.,  1., -0.,  0.,  0., -1., -1.,  0., -0., -0.,  0., -0., -0.,
         0., -0.],
       [-1., -0.,  0., -0., -0.,  0., -0., -0., -0.,  0., -0., -0., -0.,
        -0.,  0., -0., -0.,  0., -1., -0., -0.,  0., -0.,  0.,  0.,  0.,
        -1., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -1.,  1.,  0.,  0.,
         1., -0., -0., -1.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,
        -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,
        -0., -0.,  0., -1., -0., -1., -0., -0., -0., -0.,  0.,  0.,  0.,
        -0., -0., -1.,  0., -1., -0.,  0.,  0.,  0., -0.,  0.,  0., -0.,
         0.,  0.],
       [-0.,  0., -0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,
         0.,  0., -0.,  0., -1., -0., -0., -0., -0., -0.,  1.,  0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,
         0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  1., -0., -0., -0.,  0.,
         0., -1.,  0.,  0., -0., -0., -0.,  0., -0., -0., -0., -0., -0.,
         0.,  0.,  1.,  0.,  0.,  0.,  0.,  1.,  0.,  0., -0.,  0.,  0.,
         0., -1., -0., -0.,  0.,  1., -0.,  1.,  0., -0.,  0., -0., -0.,
         0., -1.],
       [-0.,  0.,  1.,  0.,  0.,  0., -0.,  0., -1., -0., -0.,  1.,  0.,
        -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0., -1.,  0., -0.,
         0., -0.,  0., -0.,  0., -0., -0., -0.,  1., -0.,  0.,  0.,  1.,
        -0., -0., -0.,  0.,  0.,  1., -1.,  0., -0.,  0., -0.,  0., -1.,
         0.,  0., -1.,  0., -0.,  0.,  1., -1., -1.,  1.,  0., -0., -0.,
         0., -0.,  0., -0., -0.,  0.,  1., -1.,  0.,  0.,  0., -0., -0.,
        -0.,  0., -0.,  0.,  0.,  0.,  1.,  0.,  1., -0., -0., -0., -0.,
         0., -0.],
       [ 0.,  0.,  1., -0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  1.,  0.,
         0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0., -1., -0., -0.,
         0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  1.,  0.,
         0.,  1.,  0., -0.,  0.,  1., -1., -0., -0., -0., -0.,  0.,  0.,
        -0., -0., -1.,  0.,  1.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,
         0.,  0.,  0.,  0.,  0., -0.,  1., -1.,  0., -0., -0.,  0.,  0.,
         0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,
         0.,  0.],
       [ 0., -0.,  1., -0., -0.,  0., -0., -0., -1., -0.,  0.,  1., -0.,
         0., -0.,  0., -0., -1., -0., -0.,  0., -0.,  0., -0., -0.,  0.,
         0., -0.,  0., -0., -0.,  0.,  0.,  0.,  1.,  0.,  0., -0.,  0.,
        -0.,  0.,  0., -0.,  1., -0., -1., -0.,  1., -0.,  0.,  0.,  0.,
        -0.,  0., -1., -0., -0.,  0., -0.,  0., -0.,  0., -1., -0., -0.,
         0.,  0.,  1.,  0.,  0., -0.,  1., -0., -0.,  0.,  0., -0.,  0.,
         0., -1., -0., -0.,  0.,  1.,  1.,  1.,  0., -0.,  0.,  0., -0.,
         0., -1.],
       [-0.,  0.,  0., -0.,  0., -0.,  0.,  1., -1.,  1., -0., -0., -0.,
         0., -0., -0., -0.,  0.,  0., -0., -0., -1.,  0., -0., -0., -0.,
         0., -0.,  0.,  0.,  0.,  0., -0., -0.,  1., -0., -0., -1.,  0.,
        -0., -1., -0., -0., -0.,  0., -0., -0.,  0.,  0.,  0., -1.,  0.,
        -0., -0., -0.,  0., -1., -0.,  1.,  0., -0., -0.,  0.,  0., -0.,
         0., -0., -0., -0., -0.,  0., -0., -0., -0.,  0., -0., -0., -0.,
         0.,  0., -0., -0., -0.,  0.,  1.,  0.,  0., -0.,  0., -0., -0.,
        -0., -0.],
       [-0., -0., -0., -1., -0.,  0., -0.,  0.,  0., -0., -0.,  1., -0.,
         0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0., -1., -0., -0.,
        -0.,  0., -0., -0., -0.,  1., -1.,  0., -0.,  0.,  0.,  0.,  0.,
         0.,  0., -0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,
        -1., -0., -0.,  0., -0.,  0., -0., -0., -0., -0., -0.,  0., -0.,
         0.,  0., -1.,  0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0., -0., -0., -0., -0., -0.,  0., -0., -0.,  0.,
        -0.,  0.],
       [ 0., -1.,  0.,  0.,  0.,  1.,  0.,  0.,  0., -0., -0., -1., -0.,
        -0., -0.,  0., -0.,  0., -0., -0., -0., -0.,  1.,  1., -0., -1.,
        -0., -0., -0., -0., -0., -1.,  1.,  0., -1., -0., -0.,  0.,  0.,
         0., -0.,  0., -0., -0., -0.,  0.,  0., -0., -1.,  0.,  0.,  0.,
         1.,  0.,  1.,  0.,  0., -0., -0., -0., -0.,  0., -0.,  0., -0.,
        -0.,  0.,  0.,  0.,  0., -0., -0., -0.,  1.,  0.,  1.,  0.,  0.,
         0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,  0., -0.,  0., -0.,
         0.,  0.],
       [-0.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0., -1.,  0.,
         1., -0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,  1.,  0., -1.,
        -0.,  0., -0., -0.,  0., -1.,  1.,  0., -0., -1., -0., -0., -0.,
        -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,
         1., -0.,  1., -0., -0.,  0., -0., -0., -0., -0.,  0., -1., -0.,
        -0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0., -0., -0.,
         0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,
         1., -0.],
       [-0., -0.,  0., -1., -0.,  0.,  0., -0.,  1.,  0., -0., -0., -0.,
        -0., -0., -0., -0., -0., -0., -0.,  0., -0., -0., -0.,  0., -1.,
         0., -0., -0., -0., -0., -0.,  0.,  0., -1., -0.,  0.,  0., -0.,
        -0.,  0.,  1.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0., -0.,
        -0.,  0.,  1.,  0., -0., -0.,  0.,  0.,  0., -0.,  1., -0.,  0.,
         0., -0., -1.,  0.,  0., -0., -1., -0.,  0., -0.,  0.,  0., -0.,
        -0.,  0.,  0., -0., -0.,  0., -0., -1., -0., -0., -0., -0.,  0.,
        -0., -0.],
       [ 0.,  0., -0., -0., -0.,  0., -1., -0., -0., -0.,  0.,  0.,  0.,
        -1.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,
        -0., -0., -0.,  1., -0., -0., -0.,  1., -1.,  1.,  0., -0., -0.,
         0., -0.,  0.,  0., -0.,  0., -0., -0., -0., -0., -1., -0.,  0.,
         0., -0.,  0., -0., -0.,  1.,  0., -0., -0.,  0.,  0.,  1., -0.,
        -0.,  0., -0.,  0., -0.,  0., -0., -0., -0., -0., -0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,
        -1.,  0.]], dtype=np.float32), np.array([[ 0.,  1.,  0.,  0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,  1.,
        -0.,  0.,  0., -0.,  0.,  0.,  0.,  1., -1.,  0.,  0.,  0.,  0.,
         1., -0.,  0.,  0.,  0., -0.,  0., -0.,  0., -1., -1.,  1., -1.,
         0., -0., -0., -0.,  0., -0.,  0.,  1.,  0., -0.,  1., -1., -0.,
        -1., -1.,  0.,  0.,  1., -1.,  0., -0.,  1., -0., -0., -1., -0.,
        -0.,  0.,  0.,  0., -0.,  0., -0., -1., -1., -0., -0.,  0., -0.,
         1., -0., -1.,  0.,  0., -0., -0., -0.,  0., -0.,  1.,  0., -1.,
         0.,  0.],
       [-0., -0.,  1.,  1.,  0., -0.,  0.,  0., -1., -0.,  0.,  1., -0.,
         0.,  0., -0.,  0.,  0.,  0.,  0., -1., -0.,  0.,  1., -0.,  0.,
         0.,  0., -0.,  0., -0., -0., -0., -0., -0., -0.,  0.,  0., -0.,
        -0., -0.,  0., -0., -0.,  1., -1.,  0.,  1., -0., -0.,  0., -0.,
        -0.,  0., -1.,  0., -0., -0.,  1.,  0.,  0.,  0.,  0.,  0.,  1.,
         0.,  0.,  0.,  0., -1., -0., -1.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0., -0.,  0., -0.,  0.,  1., -0.,  0.,  0., -0., -0., -0.,
        -0., -1.],
       [ 0., -0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,
         1., -0., -0., -0.,  0., -0., -0., -0., -0., -0.,  0.,  0.,  0.,
        -0.,  0., -0.,  1.,  0., -1.,  0.,  0.,  0., -1., -0., -0.,  0.,
        -0., -0., -0., -1., -0.,  0.,  0., -0., -0., -0.,  1.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  1.,  0., -0.,
         0.,  0., -1., -0., -0.,  0., -0.,  0., -1.,  0.,  1.,  0., -0.,
        -0., -0.,  0.,  0.,  0., -0.,  0.,  1.,  0., -0., -0., -0., -0.,
        -0.,  0.],
       [-0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,
        -0.,  0., -0., -0., -0., -1., -0., -1., -0., -0.,  0., -0., -0.,
         0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -0., -0.,
         1., -0.,  0., -1.,  0., -0., -0., -1., -0.,  0., -0.,  0.,  0.,
        -0.,  1.,  0., -0., -0., -0., -0.,  0., -0.,  0.,  1., -0.,  1.,
         1., -0., -1.,  0., -1.,  0., -0., -0.,  0., -0.,  0.,  0., -0.,
        -0., -0.,  1., -1., -0., -0.,  0.,  1., -0., -0.,  0.,  0.,  0.,
         0.,  0.],
       [-0.,  0., -0., -1.,  0., -0.,  0.,  0.,  1.,  0.,  0., -0.,  0.,
        -0., -0., -0.,  0.,  0., -1., -0.,  0., -0., -0.,  0.,  0., -0.,
        -0., -0., -0., -0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,  0.,
         1.,  0., -0., -1., -0.,  0., -0., -1.,  0., -0.,  0., -0.,  0.,
         0.,  0., -0.,  0., -0.,  0., -1., -0.,  0.,  0.,  1.,  0.,  0.,
         1., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0., -0.,
        -0., -1.,  1., -0., -0.,  1., -1.,  0.,  0., -0.,  0., -0.,  0.,
        -0.,  0.],
       [ 0.,  1., -0., -0., -0., -0.,  0.,  0., -0., -0.,  0.,  0.,  1.,
         0., -0., -0., -0., -0.,  0.,  0.,  1., -0., -0., -0.,  0.,  0.,
         0.,  0., -0.,  0., -0.,  0., -0.,  0., -0., -0.,  0.,  0., -1.,
        -0., -0., -0.,  0., -0.,  0., -0.,  1.,  0.,  0.,  0., -0., -0.,
        -0., -0.,  0.,  0.,  0., -0.,  0.,  1.,  0.,  1.,  0.,  0.,  0.,
        -0., -0., -0., -0.,  0., -0.,  0., -0., -1., -1., -0.,  0., -0.,
         1., -0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  1.,  0.,  0.,
        -0.,  0.],
       [-0.,  0., -0.,  0., -0.,  1.,  0., -0., -0., -0.,  0., -0.,  0.,
        -0., -0.,  0., -1.,  0., -0., -0., -1.,  0., -0.,  0., -1.,  0.,
         0.,  0., -1.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,
        -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  1.,  0., -0., -1.,
         0., -0., -0.,  0.,  0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,
         0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0., -0., -1.,
         0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  1., -0., -0.,  1., -0.,
         0., -0.],
       [-1., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
        -0., -0., -0.,  0., -0.,  0., -0., -0.,  0., -1., -0., -0., -0.,
         0., -0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,
         0., -0., -0., -1.,  0., -0.,  0.,  0., -0., -0., -0., -0., -0.,
        -0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,  1.,  0., -0., -0.,
         0.,  0., -0.,  0.,  0., -0., -0.,  0., -1., -1., -0., -0.,  0.,
         0.,  0.,  1., -0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,
        -0., -0.],
       [-1., -0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0., -0., -0., -0.,
         0.,  1., -0.,  0., -0.,  0.,  0., -1., -0., -1.,  0., -0.,  0.,
         0., -0.,  0., -0., -0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,
        -0., -0.,  0., -1., -0., -0., -0., -1.,  0., -0., -0.,  0., -0.,
        -0.,  0., -0., -0., -0., -0.,  0., -1.,  1.,  0.,  0.,  0.,  0.,
         1.,  0., -0., -0., -1., -0.,  0.,  0., -0.,  0., -1.,  0.,  0.,
         0., -0.,  1., -1.,  0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,
         0.,  0.],
       [-1.,  0.,  0., -0.,  0., -1., -0., -0., -0.,  0.,  0.,  0., -0.,
         0.,  0.,  0.,  1., -0., -0., -0., -0., -0., -1., -0.,  0., -0.,
        -0.,  0.,  1., -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0.,
        -0.,  0.,  0., -1., -0., -0.,  0., -1., -0.,  0.,  0.,  0., -0.,
         0., -0., -0.,  0., -0.,  0., -0., -1.,  0., -0., -0.,  0.,  0.,
        -0., -0.,  0., -0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,  0.,
         0.,  0.,  1.,  0.,  0.,  0., -0., -0., -1., -0., -0., -0., -0.,
         0., -0.],
       [-0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,  1.,  0.,  0.,  0.,
        -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,
         1., -0., -0.,  0.,  0.,  0.,  0., -0.,  0., -1., -1.,  0.,  0.,
        -0.,  1., -0.,  0., -0., -1., -0., -0., -0., -0.,  1., -1., -1.,
        -0., -0.,  0., -0.,  1., -1.,  0., -1.,  0., -1., -0., -1.,  0.,
         0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,  1., -0., -0., -0.,
        -0.,  0., -1.,  0.,  0., -0., -0.,  0.,  0., -0.,  0., -0., -1.,
         0., -0.],
       [-0., -0., -0.,  1., -0., -1.,  0., -0., -1.,  0., -1.,  1., -0.,
         0., -0., -0.,  1.,  0., -0., -0.,  0., -0.,  0., -0.,  1.,  0.,
         0.,  0.,  1., -0.,  0., -0.,  1., -0.,  0., -0., -0., -0.,  0.,
         0., -0., -0.,  0.,  0.,  1., -1.,  0.,  0., -1.,  0.,  0.,  1.,
        -0.,  0., -1.,  0., -0.,  0.,  1.,  0.,  0., -0.,  0.,  0., -0.,
        -0.,  0.,  0., -0.,  0., -0., -1., -0., -0.,  0., -0.,  0., -0.,
         0.,  0., -0.,  0.,  0.,  0.,  1., -0., -1.,  0.,  0., -1., -0.,
        -0.,  0.],
       [ 1.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0., -0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  1.,  0., -0., -0.,
         0.,  0., -0.,  0.,  0.,  0.,  0.,  1., -0., -1.,  0., -0., -0.,
         0., -0., -0., -0., -0., -0.,  0.,  0.,  0., -1.,  1.,  0., -0.,
         0.,  0., -1.,  0., -0.,  0.,  0., -0.,  0., -1.,  1.,  0.,  0.,
         0., -0., -0., -0., -0.,  0., -1.,  0.,  0.,  1.,  0.,  0.,  0.,
         0., -0., -1., -0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,
        -1.,  0.],
       [ 1., -0.,  0.,  0.,  1.,  0.,  0., -0., -0., -1.,  0., -0., -0.,
        -1.,  0., -1.,  0., -0., -1., -0.,  0., -1.,  1., -0.,  0.,  0.,
        -0., -0., -0., -1., -1., -0.,  0.,  1.,  0.,  0., -0.,  1.,  0.,
         1., -1., -0., -0., -0.,  1., -0., -0.,  0., -1.,  0.,  0.,  1.,
        -0., -0., -1., -0.,  0., -0.,  0.,  1.,  0., -0.,  1.,  0.,  1.,
         0., -0., -0.,  0.,  0., -0., -1.,  0.,  0.,  0., -0.,  1.,  0.,
         0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0., -1.,  0.,
        -1., -0.],
       [ 1., -0., -0., -1., -0.,  1.,  0., -0.,  1., -0.,  0., -1., -0.,
        -0., -0.,  0., -1.,  0.,  0., -0., -0.,  0.,  1., -0., -1., -0.,
        -0.,  0., -1.,  0., -0., -0., -1.,  0.,  0.,  0.,  0.,  0., -0.,
         0.,  0., -0., -0., -0., -0.,  1., -0.,  0., -0., -0., -0.,  0.,
        -0., -0.,  0.,  0.,  0.,  0., -1.,  1.,  0.,  0.,  1., -0., -0.,
         0., -0.,  0., -1., -0., -0., -0., -0.,  0., -0., -0.,  0., -0.,
         0., -0.,  0., -0., -1., -0., -1.,  0.,  1.,  0.,  0., -0., -0.,
         0.,  0.],
       [-0., -1., -0.,  0.,  0., -0.,  0., -0., -0., -0.,  0., -0., -1.,
         0.,  0., -0.,  0.,  0.,  0., -0., -1., -0., -0.,  0., -0., -1.,
        -1.,  0., -0., -0.,  0.,  0.,  0., -0., -0.,  1.,  1.,  0.,  0.,
         0., -0., -0., -0.,  1.,  0.,  0., -1., -0., -0., -1., -0.,  0.,
        -0.,  0.,  0., -0., -0.,  1.,  0., -0.,  0.,  0.,  0.,  1., -0.,
         0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  1.,  0., -0., -0.,  0.,
        -1.,  0.,  1.,  0., -0.,  0., -0., -0.,  0.,  0., -1.,  0.,  1.,
        -0.,  0.],
       [ 0.,  0., -1., -0.,  0., -0.,  0.,  0.,  1., -0., -0., -0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0., -0.,  0., -0.,  0.,
         0.,  1., -0.,  0.,  0., -0.,  0.,  0., -0., -0., -0., -0., -0.,
        -0., -0., -0., -0., -0., -1.,  1., -0.,  0., -0., -0., -0., -0.,
        -0., -0.,  0., -0.,  0., -0., -1., -0., -0.,  0., -0.,  0., -0.,
         0., -0.,  0., -0., -0.,  0.,  1., -0.,  0., -0.,  0.,  0., -0.,
         0., -0.,  0.,  0.,  0., -0., -1., -0.,  0.,  0.,  1., -0., -1.,
        -0., -0.],
       [-0., -1., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0., -0., -0.,
         0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0., -0., -0.,  0., -1.,
         0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,  1.,  0., -0.,  0.,
        -0., -0.,  1.,  1.,  0.,  0., -0.,  0., -0.,  0., -1., -0.,  0.,
        -0.,  0., -0.,  0.,  0.,  1.,  0., -0., -0., -0., -1.,  1., -0.,
         0., -0.,  0., -0.,  0., -0., -0., -0.,  1., -0., -0., -0., -0.,
        -0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,
        -0.,  0.],
       [-0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0., -0., -0.,  0.,
        -0.,  0.,  0., -0., -1., -0., -1.,  1., -0.,  0., -0., -0.,  0.,
         1.,  1.,  0., -0.,  0.,  0.,  0., -0.,  0., -0., -1., -0.,  0.,
         0., -0., -0.,  1.,  0., -0.,  0.,  1.,  1., -0., -0., -0., -0.,
        -0., -1.,  0.,  0., -0., -0.,  0., -0., -0.,  0., -1.,  0.,  0.,
         0., -1., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,  0., -0.,
         1.,  0., -1., -0., -0.,  1., -0., -1.,  0.,  0.,  1., -0., -1.,
        -0., -0.],
       [-0.,  0.,  0., -0.,  0., -0., -0.,  0., -1.,  0.,  0.,  0., -0.,
        -0., -0.,  0., -0.,  0.,  0.,  0., -0., -0., -0., -0., -0., -0.,
         1.,  0.,  0., -0.,  0., -0., -0., -0.,  0., -0., -1.,  0.,  0.,
         0.,  0., -0.,  1.,  0., -0.,  0.,  1.,  0., -0., -0.,  0.,  0.,
        -0.,  0.,  0.,  0.,  0., -0.,  1.,  0.,  0.,  0., -1., -0., -0.,
        -0., -1., -0.,  0., -0., -0., -0., -0., -0.,  0.,  0., -0.,  0.,
         1.,  0., -1.,  0.,  0.,  0.,  1., -0.,  0., -0.,  0.,  0., -0.,
         0.,  0.],
       [-0.,  0.,  0., -0.,  0., -0., -0., -1., -0.,  0.,  0.,  0., -0.,
        -0., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,
         1., -0.,  0.,  0., -0., -0.,  0.,  0., -0., -0., -0.,  0., -0.,
         0., -0., -0., -0., -0., -0., -0.,  0., -0., -0.,  1., -1., -0.,
         0., -0., -0., -0.,  0., -1.,  1., -1., -0., -1., -0., -0., -0.,
         0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,  1., -0., -0.,  0.,
        -0.,  0., -1., -0.,  0., -0.,  0., -0., -1.,  0.,  0.,  0., -1.,
        -0.,  0.],
       [ 0., -0., -0.,  0., -0.,  0.,  1.,  1.,  0.,  1., -1., -0.,  0.,
         0.,  0.,  1.,  1., -0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0.,
         0., -0.,  0.,  0.,  0., -0., -0., -1., -0., -0.,  0.,  0., -0.,
        -0., -0., -0., -0.,  0.,  0., -0., -0., -0.,  0., -0., -0., -0.,
        -0., -0.,  0.,  1.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0.,
        -0.,  0., -0.,  0., -0., -1.,  0.,  0.,  0., -0., -0.,  0., -0.,
        -0., -0.,  0.,  0., -1., -0.,  0., -0.,  0., -1., -0., -0., -0.,
        -0.,  0.],
       [ 1.,  0.,  0., -0., -0.,  1.,  1.,  0.,  1.,  0.,  0.,  0.,  0.,
         0., -0.,  0.,  0., -0., -0., -0., -0., -0.,  1., -0., -0., -0.,
        -0.,  0.,  0., -0., -0., -0., -0.,  0., -1.,  0., -0.,  0., -0.,
        -0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,  1., -0., -0.,
         0., -0., -0., -0., -0.,  0.,  0.,  0.,  0., -1.,  1.,  0., -0.,
         0., -0., -0., -0., -0., -0., -0., -0.,  0.,  1., -0.,  0.,  0.,
        -0.,  0., -1.,  0., -0.,  0.,  0., -0., -0., -0.,  0., -0.,  0.,
         0., -0.],
       [ 1.,  0.,  0., -0.,  0.,  1.,  1.,  1.,  1.,  0., -0., -0.,  0.,
        -0.,  0.,  0., -0., -0., -1.,  0.,  0., -1.,  1., -0.,  0.,  0.,
         0., -0., -0., -1., -1., -0.,  0., -0., -1., -0.,  0.,  0., -0.,
         0.,  0., -0.,  0., -0., -0., -0., -0., -0., -0., -0.,  0.,  0.,
        -0., -0., -0.,  1., -0., -0., -1.,  1., -0.,  0.,  1., -0.,  1.,
        -0., -0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,
         0., -0., -0.,  0., -0., -0.,  0.,  0.,  1.,  0., -0., -0., -0.,
         0.,  0.],
       [ 1., -0., -0.,  0., -0.,  1.,  0., -0.,  1.,  0., -0., -0., -0.,
         0.,  0.,  0., -1., -0.,  0., -0., -0., -0.,  1.,  0., -0.,  0.,
         0., -0.,  0.,  0., -0.,  0.,  0., -0., -1., -0., -0., -0., -0.,
         0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  0.,
        -0., -0., -0.,  0.,  0., -0., -1.,  1., -0., -0.,  1.,  0., -0.,
        -0., -0.,  0.,  0., -0.,  1., -0., -0.,  0., -0., -0.,  0.,  0.,
        -0., -0.,  0.,  0.,  0., -0.,  0., -0.,  1., -0., -0., -0., -0.,
         0., -0.]], dtype=np.float32), np.array([[-0., -0., -0.,  0.,  0., -0.,  0., -1.,  0.,  1., -0.,  0.,  1.,
        -0., -0.,  1., -0., -0., -0.,  0.,  0.,  1., -0., -0., -0., -0.,
        -0.,  1., -0.,  0., -1., -0., -0., -0., -0., -0.,  0., -0.,  0.,
        -0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -1., -0.,
         0., -0.,  0., -1., -0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0.,
        -0.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,
        -0.,  0.,  0., -0.,  0., -0.,  0., -0., -0.,  0., -1.,  0., -1.,
         0., -0.],
       [ 0., -0.,  0., -0.,  0., -0., -0., -0., -0., -0.,  0., -0.,  1.,
        -0., -1.,  0., -1., -0., -0., -0., -0., -0., -0., -0.,  0., -0.,
         0., -0., -0.,  0., -0., -0.,  0., -0., -0., -0.,  0.,  0., -1.,
         0.,  0., -0.,  0., -0., -0., -0., -0., -0.,  0., -0.,  0., -1.,
         0., -0., -0., -1., -0.,  0., -0.,  1.,  1.,  0., -0.,  0., -0.,
        -0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0., -0.,
         0., -0.,  0., -0.,  0.,  0., -0., -0.,  1.,  0., -0.,  1.,  0.,
        -0.,  0.],
       [-0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,
        -0., -0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,
        -1.,  1.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,  1., -1.,  0.,
        -0., -1.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,
         0.,  0.,  0., -0.,  1., -0., -0., -0., -0., -0., -0., -0.,  0.,
         0., -1.,  0., -0.,  0., -0., -0., -0.,  0., -0., -0.,  1.,  0.,
         1., -0.,  0., -0.,  0.,  0., -0., -0., -0.,  0., -1., -0., -1.,
        -0.,  0.],
       [-0.,  0.,  0.,  0.,  0., -0.,  0., -1.,  0.,  1., -0.,  0.,  0.,
         0.,  1.,  1.,  1., -0.,  0.,  0., -0.,  1., -0., -0., -0.,  0.,
        -0.,  0.,  0.,  0., -1.,  0.,  0., -0., -0.,  0.,  0.,  1.,  0.,
        -0.,  1.,  0.,  0.,  1.,  0.,  0.,  0.,  0., -0., -0., -0.,  1.,
         0., -0., -0., -0.,  0.,  0.,  0., -1., -1.,  0.,  0.,  0.,  0.,
         0., -0.,  0.,  0.,  0., -0., -0., -1.,  0.,  0.,  0., -1.,  0.,
        -0., -0.,  0., -0., -0., -0.,  0., -0., -1., -0.,  0., -1.,  0.,
         0.,  0.],
       [ 0., -1., -0.,  0.,  0., -0., -0.,  1., -0., -1., -0., -0., -1.,
         0.,  0., -1., -0.,  0., -0., -0.,  0., -1.,  0., -0., -0.,  1.,
         0.,  0.,  0.,  0.,  1., -0., -0., -0., -0.,  0., -0., -1., -0.,
         0., -1., -1.,  0., -0., -0.,  0., -0.,  0., -0., -0.,  1.,  0.,
         1., -0.,  0.,  1.,  1., -1.,  0.,  0., -0., -0.,  0.,  1.,  0.,
        -0., -0., -0.,  0., -0., -0., -0.,  0., -0., -0., -0.,  1.,  0.,
         0., -0., -0., -0., -0.,  0., -0., -0.,  0., -0.,  0., -0., -0.,
        -0.,  0.],
       [ 0., -0., -0., -0., -0., -0., -0., -0., -0.,  0.,  1.,  0.,  1.,
        -0.,  0.,  0.,  0.,  1.,  0., -1.,  1., -0., -0.,  0.,  0., -0.,
        -0.,  1., -0., -0., -0., -0., -0.,  0., -0., -0., -0.,  0.,  0.,
        -0., -0.,  0.,  0.,  0.,  0., -0., -0., -1., -0., -0., -0.,  0.,
        -0., -0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0., -0.,
         0.,  0.,  0.,  0.,  1.,  0.,  0., -0.,  0., -0., -0.,  0.,  0.,
        -0.,  0.,  0., -1., -0., -0.,  0.,  0.,  0., -1., -1., -0.,  0.,
         0.,  1.],
       [ 0.,  0.,  0.,  0.,  1.,  0.,  0.,  0., -0.,  0., -0., -0.,  1.,
         0., -0., -1.,  0.,  0., -0., -1.,  1., -0., -0.,  0., -0.,  0.,
        -0.,  0., -0., -0., -0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,
         0.,  0., -0., -0.,  0.,  0., -0., -0., -0.,  0., -0., -0.,  0.,
        -0.,  0.,  0., -0., -0.,  0., -0.,  0., -0., -0., -0., -0., -0.,
         0., -0., -0., -0.,  1.,  0., -0., -0.,  0.,  0., -0.,  0.,  1.,
         0., -0., -0., -1., -0.,  0., -0., -0.,  0., -1., -1.,  1.,  0.,
         0.,  0.],
       [ 0., -0., -1.,  0., -0., -0., -0., -0.,  0.,  0.,  1., -1.,  0.,
        -0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0., -1., -1.,  0.,
         0., -0., -0.,  0., -0.,  0., -1.,  0., -0., -0., -0., -0., -0.,
         0., -0., -0.,  0.,  0., -0.,  1.,  0., -0.,  0., -0.,  0.,  0.,
        -0., -0., -0., -0., -0., -0., -0., -0., -0.,  0.,  0.,  0., -0.,
        -0., -0.,  0., -1., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -1.,
        -0., -0., -0.,  0.,  1.,  0., -0., -0.,  0.,  0.,  0.,  0., -0.,
         0.,  1.],
       [ 0., -0., -1., -0., -1.,  0.,  0., -0., -0.,  1., -0., -0.,  0.,
        -0.,  0.,  1.,  0., -1., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,
        -0., -1.,  0., -0., -0., -0.,  0., -0., -0., -0.,  0., -0.,  0.,
         0.,  1., -0.,  0.,  0.,  1., -0.,  0.,  1., -0., -0., -0.,  1.,
        -0.,  0., -0.,  0.,  0., -0.,  0., -0., -0., -0.,  0., -0.,  0.,
         0., -0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0.,
         0., -0., -0., -0., -0., -0., -0.,  0., -0.,  0.,  0., -1., -0.,
        -0., -0.],
       [ 0.,  0., -1., -0.,  0.,  0.,  0.,  0., -0.,  0., -0., -0., -1.,
        -0., -0., -0., -0., -1., -0.,  1., -1.,  0., -0., -1., -0., -0.,
         0., -1., -0.,  0.,  0., -0.,  0., -1.,  0.,  0.,  0.,  0., -0.,
        -0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  1., -1., -0.,  0., -0.,
         0., -0., -1.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,  0.,
        -0., -0., -0.,  0., -1.,  0., -1., -0.,  0., -0.,  0., -0., -1.,
        -0., -0., -0.,  1., -0., -0., -0.,  0.,  0.,  1.,  1.,  0., -0.,
         1., -0.],
       [ 0.,  0., -0.,  0.,  0.,  0.,  1.,  0., -0., -0.,  0.,  0., -1.,
         0., -0., -1., -0., -0., -0., -0.,  0., -0., -0.,  0., -0., -0.,
        -0., -1., -0., -1.,  1., -0.,  0., -1., -0., -0., -0.,  0.,  0.,
        -0.,  0., -0., -0., -0., -0., -0., -0.,  0., -0., -1., -0.,  0.,
        -0., -0., -0.,  1.,  0.,  1.,  0., -0., -0.,  0.,  0., -0., -0.,
         0., -0.,  0., -0., -0.,  0., -0., -0., -0., -1.,  0., -0., -0.,
         0.,  0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0.,  1., -0.,  1.,
         0.,  0.],
       [ 0., -1.,  0., -0.,  0.,  1.,  0., -0., -0.,  0.,  0.,  0., -1.,
        -0.,  1.,  0.,  1.,  0., -0.,  0.,  0.,  0., -1., -0., -0.,  0.,
        -0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0., -0.,
         0., -0.,  0.,  0., -0.,  0., -0., -0., -0.,  1.,  0.,  0., -0.,
        -0.,  0.,  0.,  1.,  0.,  0., -0., -0., -0.,  0.,  0., -0., -0.,
         0.,  0.,  0., -0.,  0., -0., -0.,  0.,  1., -0.,  1.,  0.,  0.,
         0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0., -1.,  0.,
         0., -0.],
       [ 0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,
         1.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0.,
         1., -1.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -1., -1., -0.,  0.,
        -0.,  0.,  0.,  1., -0., -0., -0., -1.,  0., -0., -0., -0., -0.,
        -0.,  0.,  0., -0., -0.,  0., -0.,  0., -0., -0.,  0.,  1.,  0.,
        -0.,  1.,  0.,  0.,  0.,  0.,  0., -0.,  0., -1.,  0., -1.,  0.,
        -1.,  0.,  1.,  0., -0., -0., -0.,  0.,  0.,  0.,  1.,  0.,  1.,
        -1.,  0.],
       [-0.,  0., -0.,  0., -0., -0.,  0.,  1., -1., -1., -0., -0.,  0.,
        -0., -1., -1., -1.,  0., -0.,  0.,  0., -1., -0., -0.,  0., -0.,
         0., -0., -0.,  0.,  1.,  0.,  0., -0., -0., -0.,  0., -1.,  0.,
        -0., -1.,  1., -0., -0., -1., -0., -0.,  0.,  0., -0.,  0., -1.,
        -0.,  1., -0.,  0., -0.,  0.,  1.,  1.,  1.,  1.,  1., -0., -0.,
         0.,  0.,  0., -0., -0., -0., -1.,  1.,  0., -0., -0.,  1., -0.,
         0.,  0.,  0., -0.,  0., -0., -0.,  1.,  1.,  0.,  0.,  1., -0.,
        -0., -0.],
       [ 0.,  1., -0.,  0.,  0.,  0., -1.,  0., -0., -0., -0., -0.,  1.,
         1., -0.,  1.,  0., -0., -0.,  0., -0.,  0., -0., -0.,  0., -0.,
         0., -0., -0.,  1., -1., -1., -0.,  1., -0., -0., -0.,  0.,  0.,
         0., -0.,  1., -0., -0., -0., -0.,  0.,  0., -0., -0.,  0.,  0.,
         0., -0., -0., -1.,  0., -0.,  0., -0.,  0., -0., -0.,  0.,  0.,
         0., -0.,  0.,  0., -0.,  0.,  0.,  0., -1.,  0.,  0., -1.,  0.,
        -0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,
        -1.,  0.],
       [ 0.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,
         0., -0.,  1., -0., -1., -0.,  1.,  0., -0.,  0.,  0., -0., -0.,
        -0.,  0., -0., -0., -1.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,
        -0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  1.,  0., -0., -0.,  0.,
        -0.,  0.,  0., -1., -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  1.,
        -0., -0.,  0.,  0., -1., -0., -0., -0., -0., -0., -0., -0.,  0.,
        -0.,  0., -0.,  1.,  0.,  0., -0.,  0., -0.,  1., -0.,  0.,  0.,
         0., -1.],
       [-0., -0., -0., -0., -1.,  0.,  0., -0., -0., -0.,  0., -0., -0.,
         0., -1.,  1.,  0., -0.,  0.,  1., -0.,  0.,  0.,  0.,  0., -0.,
        -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0., -0., -0., -0.,
         0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,
        -0.,  0.,  0., -1.,  0., -0., -0., -0., -0., -0.,  0.,  0.,  0.,
         0., -0.,  0.,  0., -0., -0., -0., -0.,  0., -0., -0.,  0.,  0.,
         0., -0., -0.,  1.,  0., -0., -0.,  0.,  0.,  1., -0., -0.,  0.,
        -0., -0.],
       [-0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,
         0., -0.,  0., -0.,  0.,  1.,  0.,  0., -0., -0., -0., -0., -0.,
         0.,  0., -0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,
        -1., -0., -0.,  0.,  0.,  0., -0., -0.,  1., -0., -0.,  0., -0.,
         0., -0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,  1.,
        -1., -0.,  0., -0., -1., -0., -0., -0., -0.,  0.,  0.,  1.,  0.,
        -0.,  1.,  0.,  0.,  0., -1., -0., -0., -0., -0., -0., -0.,  0.,
         0., -1.],
       [-0.,  0., -0., -0.,  1., -0., -0., -0.,  0.,  0., -0.,  0., -0.,
         0.,  1., -0.,  0.,  1., -0.,  0.,  0.,  1., -0., -0.,  0.,  0.,
         0., -0.,  0.,  0., -1., -0., -0.,  0., -0., -0.,  0.,  1., -0.,
         0.,  0.,  0., -0.,  0., -0., -0.,  0., -0., -0., -0., -0., -0.,
        -0., -1.,  0.,  0.,  0.,  0., -0.,  0., -1., -0.,  0.,  0.,  0.,
        -0., -0.,  0.,  0.,  0., -0., -0., -1., -0., -0.,  0., -1., -0.,
         0., -0., -0., -0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0., -0.,
         0., -0.],
       [-0., -0.,  0., -0., -0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,
        -1.,  0., -1.,  0.,  1.,  0., -1.,  0.,  0., -0.,  0.,  0., -0.,
        -0.,  0.,  0., -1.,  1.,  1., -0.,  0.,  0., -0., -0.,  0.,  0.,
         0., -0.,  0.,  0., -0.,  0., -0., -0., -0., -0., -0., -0.,  0.,
        -0., -0.,  0.,  1.,  0.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,
        -0., -0.,  1.,  0., -0., -0., -0.,  0., -0., -0., -1.,  1.,  0.,
         0., -0., -0., -1.,  0., -0.,  0.,  1., -0., -1., -0., -0., -0.,
         0., -0.],
       [-1., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,
        -0.,  0.,  0.,  0.,  1., -1., -1., -0., -0., -0., -0.,  0., -0.,
        -1.,  1.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,
        -0.,  0.,  0., -0., -0., -0., -0.,  0., -1., -0., -0., -0., -0.,
        -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,  0., -1.,
         0.,  0., -0., -0.,  1.,  1., -0., -0.,  0.,  1.,  0., -0., -0.,
        -0.,  0., -1., -1., -1.,  0., -0.,  0.,  0., -1., -1.,  0., -1.,
        -0.,  1.],
       [ 0., -0., -0.,  0.,  1.,  0.,  0.,  0.,  0., -0., -0., -0.,  1.,
        -0.,  0., -1., -1.,  0.,  0., -1., -0., -0.,  0., -0., -1.,  0.,
         0., -0., -1., -0.,  0., -0.,  0., -0.,  0., -0., -0.,  0., -0.,
         0.,  0.,  0.,  0., -0.,  0., -0., -1., -0.,  0.,  0.,  0.,  0.,
         0.,  0., -0.,  0.,  0., -0., -0., -0.,  0., -0., -0., -0., -0.,
        -1.,  0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,  0.,  0., -0.,
        -1., -0., -0., -1., -0.,  0., -0.,  0.,  0., -1.,  0.,  1., -0.,
         0., -0.],
       [-1., -0.,  0.,  0., -0., -0., -0.,  0., -0.,  0., -0., -0.,  0.,
        -0.,  0.,  0.,  0., -0., -1.,  0.,  0., -0., -0., -0.,  0.,  0.,
        -1.,  1., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,
        -0.,  0.,  0.,  0.,  0.,  0., -0.,  1., -1.,  0.,  0., -0., -0.,
         0., -0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0., -0., -0., -1.,
         1., -1.,  0., -1.,  1.,  0., -0.,  0.,  0.,  1., -0., -0.,  0.,
         1., -1., -1.,  0., -0.,  1., -0., -0., -0.,  0., -1.,  0., -1.,
         0.,  1.],
       [-0.,  0., -0., -0., -1., -0.,  0., -1.,  0.,  1., -0.,  0.,  0.,
        -0.,  0.,  1.,  1., -1., -0., -0.,  0., -0., -0.,  0., -0.,  0.,
        -0.,  0., -0., -0., -0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,
        -0.,  1.,  0.,  0.,  0.,  1., -1., -0., -0., -0., -0.,  0.,  1.,
        -0., -0.,  0.,  0.,  0., -0., -1.,  0., -0., -0.,  0.,  0.,  0.,
         0., -1.,  0., -0.,  0.,  0., -0.,  0., -0., -0., -0.,  0., -0.,
        -0., -0.,  0., -0.,  0.,  1.,  1.,  0., -1., -0.,  0., -1.,  0.,
        -0.,  0.],
       [-0.,  0.,  0.,  1., -0., -1.,  1.,  0., -1.,  0.,  0.,  1., -1.,
        -0., -0.,  0., -0., -1., -0.,  1., -0., -0.,  0.,  0.,  1.,  0.,
        -0., -0.,  1.,  0.,  0.,  0., -0., -1.,  1.,  0., -0., -0.,  0.,
         0.,  0.,  0.,  0., -0., -0., -1.,  1., -0., -1., -0., -0., -0.,
         0., -0., -1.,  0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,
         1., -1., -0., -0.,  0., -0., -1., -0., -0., -0., -0.,  0., -0.,
         1., -1.,  0.,  1.,  0.,  1.,  1.,  0.,  0.,  1.,  0., -0., -0.,
         1., -0.]], dtype=np.float32))

In [ ]:
#@title Verification
n = 5
m = 5
p = 5
rank = 93

verify_tensor_decomposition(decomposition_555, n, m, p, rank)